# 🛡️ Microsoft Sentinel Security Analysis — 12-Month Threat Detection

## Overview
Comprehensive security analysis across Microsoft Sentinel Data Lake tables covering a **fixed 12-month window: 2025-02-20 → 2026-02-20**.  
Designed for SOC analysts, threat hunters, and CISO-level reporting.

---

## Analysis Sections

| # | Section | Key Detections | Primary Tables |
|---|---------|---------------|----------------|
| **1** | **Setup & Authentication** | Workspace config, `device_code` / browser auth, `run_kql()` helper | — |
| **2** | **Data Source Availability** | Table presence check across 19 data sources | All |
| **3** | **Brute Force Detection** | High-frequency failed sign-ins per user and IP | SigninLogs, AADNonInteractiveUserSignInLogs |
| **3.1** | **Success After Failure** | Successful login following brute force pattern | SigninLogs |
| **4** | **Password Spray** | Single IP → many accounts; botnet distributed spray | SigninLogs, AADNonInteractiveUserSignInLogs |
| **5** | **Risky Users & Sign-ins** | AAD Identity Protection risk correlation, risky sign-in enrichment | AADRiskyUsers, AADUserRiskEvents, RiskySignIns |
| **5.1** | **Geolocation Analysis** | Sign-in map, country breakdown, multi-country sessions | SigninLogs |
| **6** | **Baseline Anomaly Detection** | Z-score time-series, spike detection (3σ threshold), hourly baseline | SigninLogs |
| **7** | **IOC Matching — Full-Spectrum (11 tables)** | TI indicator correlation across every data source in the workbook | ThreatIntelligenceIndicator + all below |
| **7.1** | → IP Address IOC | DeviceNetworkEvents, DeviceLogonEvents, SigninLogs, AADNonInteractiveUserSignInLogs, AADServicePrincipalSignInLogs, AuditLogs, AzureActivity, IdentityLogonEvents, IdentityDirectoryEvents, CloudAppEvents, SecurityAlert | All IP-bearing tables |
| **7.2** | → File Hash IOC | SHA256/SHA1/MD5 match across DeviceFileEvents, DeviceProcessEvents, EmailAttachmentInfo | DeviceFileEvents, DeviceProcessEvents, EmailAttachmentInfo |
| **7.3** | → Domain / Hostname IOC | DNS contacts, LDAP queries, email sender domains, UPN domains, AuditLogs resource targets | DeviceNetworkEvents, IdentityQueryEvents, EmailEvents, SigninLogs, AuditLogs |
| **7.4** | → URL IOC | Device web requests, MDO Safe Links clicks, email embedded URLs, CloudAppEvents link shares | DeviceNetworkEvents, UrlClickEvents, EmailUrlInfo, CloudAppEvents |
| **7.5** | → Process / Command-line IOC | Image hash vs TI + LOLBin/abuse patterns + SecurityAlert process entities | DeviceProcessEvents, SecurityAlert |
| **7.6** | → Email / Collaboration IOC | Sender IP + address, attachment hashes, Teams/SharePoint sender matching | EmailEvents, EmailAttachmentInfo, CloudAppEvents |
| **7.7** | → IOC Summary | Combined hit count pivot, by-source heatmap, stacked bar by entity type | All TI |
| **7.8** | → Threat Actor & Analytics | TI actor grouping, SecurityAlert enrichment, UEBA BehaviorAnalytics | ThreatIntelligenceIndicator, SecurityAlert, BehaviorAnalytics |
| **8** | **Lateral Movement Detection** | Accounts on many devices, pass-the-hash / ticket indicators | DeviceLogonEvents |
| **8.1** | **Lateral Device Propagation** | Device-to-device connection analysis | DeviceNetworkEvents |
| **8.2** | **Recon Tool Detection** | LOLBin discovery commands (nltest, dsquery, nmap…) | DeviceProcessEvents |
| **9** | **Defender Alerts** | High/medium fidelity SecurityAlert enrichment by provider | SecurityAlert |
| **9.1** | **Alert Enrichment** | Cross-correlate alerts with IOC hits, risky users, geo | SecurityAlert + multiple |
| **9.2** | **Cloud App Correlation** | CloudAppEvents anomaly and OAuth abuse detection | CloudAppEvents |
| **10** | **Spike Detection** | Statistical spike vs 180-day historical baseline | SigninLogs |
| **10.1** | **Identity Alerts** | IdentityInfo + IdentityLogonEvents fusion | IdentityLogonEvents, IdentityInfo |
| **10.2** | **Malicious OAuth Apps & Service Principals** | Consent grant abuse, SP geo anomalies, credential abuse, geo heatmap | AuditLogs, AADServicePrincipalSignInLogs |
| **11** | **Defender for Office 365** | Full MDO threat hunting suite | EmailEvents, EmailAttachmentInfo, CloudAppEvents |
| **11.1** | → Spam & Phishing | Blocked/quarantined email, sender geo, category breakdown | EmailEvents |
| **11.2** | → Email Malware | Attachment detonation, delivered malware, malware families | EmailAttachmentInfo, EmailEvents |
| **11.3** | → Teams / SharePoint / OneDrive Malware | FileMalwareDetected, AntiVirus events | CloudAppEvents |
| **11.4** | → Business Email Compromise (BEC) | Suspicious inbox/transport rules, high-volume external forwarding | AuditLogs, EmailEvents |
| **11.5** | → Adversary-in-the-Middle (AiTM) | MFA-bypass sign-ins, phishing link click correlation | SigninLogs, UrlClickEvents |
| **16** | **🏛️ Threat Hunting — On-Prem Identity & Domain Controllers** | Kerberoasting, DCSync, LDAP recon, NTLM/PTH, privileged group changes, MDI alerts | SecurityEvent, IdentityLogonEvents, IdentityQueryEvents, IdentityDirectoryEvents |
| **16.1** | → Kerberoasting / AS-REP Roasting | RC4 TGS/TGT requests, T1558.003/004 | SecurityEvent (4768/4769) |
| **16.2** | → DCSync / Directory Replication Abuse | Replication GUID 1131f6aa/1131f6ab via DC, T1003.006 | IdentityDirectoryEvents, SecurityEvent (4662) |
| **16.3** | → LDAP Recon / BloodHound-style Queries | Mass LDAP/SAMR queries per account, T1087.002 | IdentityQueryEvents |
| **16.4** | → NTLM Relay & Pass-the-Hash | Multi-device NTLM from single identity, T1550.002 | IdentityLogonEvents, SecurityEvent (4776) |
| **16.5** | → Privileged Group Membership Changes | Additions to Domain Admins / Enterprise Admins, T1098 | SecurityEvent (4728/4732/4756) |
| **16.6** | → Suspicious Processes on Domain Controllers | MDI attack category alerts (DCSync, Golden Ticket, Skeleton Key…) | IdentityDirectoryEvents |
| **17** | **☁️ Threat Hunting — Azure Environment** | Cryptojacking, Azure privesc, mass deletion, Key Vault exfil, storage abuse, impossible travel | AzureActivity, AzureDiagnostics |
| **17.1** | → Cryptojacking | GPU/HPC VM deployments; large compute bursts, T1496 | AzureActivity |
| **17.2** | → Azure Privilege Escalation | Owner/UAA role assignment at subscription scope, T1098.003 | AzureActivity, AuditLogs |
| **17.3** | → Mass Resource Deletion | Bulk DELETE operations; ransomware / sabotage pattern, T1485 | AzureActivity |
| **17.4** | → Key Vault Enumeration | High-volume SecretGet/SecretList; secret exfiltration, T1552.001 | AzureDiagnostics |
| **17.5** | → Storage Exfiltration | Anonymous / SAS blob reads, large egress volumes, T1530 | AzureDiagnostics |
| **17.6** | → Impossible Travel (Azure Management) | CallerIpAddress geo velocity in control-plane, T1078.004 | AzureActivity |
| **17.7** | → Suspicious Automation Deployments | New Automation Accounts, Runbooks, Logic Apps, Function Apps, T1059.009 | AzureActivity |
| **12** | **🔬 Forensic Investigation Hub** | Deep-dive entity pivot — set target and run | All |
| **12.1** | → IP Forensics | Full pivot on a single IP across all tables + activity timeline | SigninLogs, DeviceNetworkEvents, EmailEvents, TI |
| **12.2** | → Device Forensics | Logons, processes, file changes, network, alerts timeline | DeviceLogonEvents, DeviceProcessEvents, DeviceFileEvents, SecurityAlert |
| **12.3** | → Identity Forensics | Sign-ins, audit, risk events, device logons, email timeline | SigninLogs, AuditLogs, AADRiskyUsers, EmailEvents |
| **13** | **📊 SOC KPI Dashboard** | MTTD, alert volume/severity trend, TP rate, provider breakdown | SecurityAlert |
| **14** | **🏢 C-Level / CISO Dashboard** | Risk gauge (RAG), business risk areas, compliance gaps, top-5 recommendations | Aggregated findings |
| **15** | **🧩 MITRE ATT&CK Heatmap** | Detection coverage across 22 techniques / 8 tactics | Aggregated findings |
| **18** | **Summary Dashboard** | All-up chart grid across every section | Aggregated findings |
| **19** | **Data Volume** | Ingestion volume per table over analysis period | All |
| **20** | **Executive Summary** | Plain-text narrative ready for briefings and reports | Aggregated findings |

---

## Data Sources

| Category | Tables |
|----------|--------|
| **Azure AD** | SigninLogs, AADNonInteractiveUserSignInLogs, AADServicePrincipalSignInLogs, AuditLogs |
| **Identity Protection** | AADRiskyUsers, AADUserRiskEvents, RiskySignIns |
| **Behavior Analytics** | BehaviorAnalytics |
| **Threat Intelligence** | ThreatIntelligenceIndicator |
| **Defender for Cloud Apps** | CloudAppEvents |
| **Defender for Identity (MDI)** | IdentityLogonEvents, IdentityQueryEvents, IdentityDirectoryEvents, IdentityInfo |
| **Defender for Endpoint** | DeviceLogonEvents, DeviceNetworkEvents, DeviceProcessEvents, DeviceFileEvents |
| **Defender for Office 365** | EmailEvents, EmailAttachmentInfo, EmailUrlInfo, UrlClickEvents |
| **On-Prem / Windows Security** | SecurityEvent (4624/4648/4662/4768/4769/4771/4776/4728/4732/4756) |
| **Azure Control Plane** | AzureActivity, AzureDiagnostics |
| **Sentinel Core** | SecurityAlert, SecurityIncident |

---

## SOC KPI Targets (Section 13)

| KPI | Target | Description |
|-----|--------|-------------|
| **MTTD** | < 4 h | Mean Time to Detect |
| **High+Medium Signal Rate** | > 30% | Alert quality ratio |
| **Credential Attack Trend** | ↓ 0 | Brute force + spray volume |
| **IOC Hit Rate** | 0 | Confirmed TI matches |
| **Risky Users** | ↓ | Active high-risk identities |

---

## MITRE ATT&CK Coverage

| Tactic | Techniques Mapped |
|--------|-------------------|
| **TA0001 Initial Access** | T1566 Phishing, T1566.001 Spearphishing Attachment, T1566.002 Spearphishing Link, T1078 Valid Accounts, T1078.004 Cloud Accounts |
| **TA0003 Persistence** | T1098 Account Manipulation, T1098.003 Additional Cloud Roles, T1059.009 Cloud Admin Scripts |
| **TA0004 Privilege Escalation** | T1098.003 Azure Role Assignment Abuse, T1548 Abuse Elevation Mechanisms |
| **TA0005 Defense Evasion** | T1078.004 Cloud Account Abuse, T1550.002 Pass-the-Hash |
| **TA0006 Credential Access** | T1110 Brute Force, T1110.003 Password Spraying, T1539 Steal Web Session Cookie, T1557 AiTM, T1558.003 Kerberoasting, T1558.004 AS-REP Roasting, T1003.006 DCSync, T1552.001 Credentials in Files (Key Vault) |
| **TA0007 Discovery** | T1087 Account Discovery, T1087.002 Domain Account Discovery, T1083 File Discovery, T1046 Network Service Discovery |
| **TA0008 Lateral Movement** | T1021 Remote Services, T1550 Alternate Auth Material, T1550.002 Pass-the-Hash |
| **TA0009 Collection** | T1114 Email Collection, T1213 Data from Repositories, T1530 Data from Cloud Storage |
| **TA0010 Exfiltration** | T1048 Exfil over Alternative Protocol, T1567 Exfil to Cloud Storage, T1530 Cloud Storage Object |
| **TA0011 C2** | T1071 Application Layer Protocol, T1102 Web Service |
| **TA0040 Impact** | T1486 Data Encrypted for Impact, T1485 Data Destruction (Mass Delete), T1496 Resource Hijacking (Cryptojacking), T1565 Data Manipulation |

---

## 🔧 Bug Fixes Applied

| Cell | Fix |
|------|-----|
| **Process IOC (7.5)** | `SEM0100: MaxConfidence` — `ConfidenceScore` is dynamic in TI; added schema-anchor `datatable` + `column_ifexists()` |
| **Spam/Phishing (11.1)** | `SEM0100: PhishFilterVerdict` / `SpamFilterVerdict` — optional columns; wrapped with `column_ifexists()` |
| **Email Malware (11.2)** | `SEM0100: MalwareFamily`, `DetectionMethods`, `FileType` — optional columns; wrapped with `column_ifexists()` |
| **Threat Actor (7.8)** | `SEM0100: IndicatorType` dynamic + `DeviceName` missing from UEBA; used `tostring(column_ifexists())`, removed `dcount(DeviceName)`, fixed `array_length(ActivityInsights) > 0` |
| **MITRE ATT&CK (15)** | Missing `import plotly.graph_objects as go`; VS Code cell-ID desync → rebuilt via delete + insert |
| **OAuth / SP (10.2)** | `SEM0100: IsHighRisk` out-of-scope after `project` — moved `order by` before `project` |
| **IOC Matching (7.1–7.6)** | Expanded from 6 tables to 11+ tables per entity type; all use `union isfuzzy=true` to handle unavailable tables |

---


## 1. Setup and Configuration

Import required libraries and configure the Microsoft Sentinel Data Lake connection using MSTICPy.

In [2]:

# ── Install packages into THIS kernel's Python ────────────────────────────────
import subprocess, sys

# Show exactly which Python the kernel is using
print(f"🐍 Kernel Python: {sys.executable}")
print(f"   Version: {sys.version.split()[0]}\n")

required = [
    "azure-monitor-query",
    "azure-identity",
    "pandas",
    "numpy",
    "plotly",
    "matplotlib",
    "scipy",
]

errors = []
for pkg in required:
    # Always install (--upgrade ensures correct version even if partially present)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", "--quiet",
         "--trusted-host", "pypi.org",
         "--trusted-host", "files.pythonhosted.org",
         pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {pkg}")
    else:
        print(f"  ❌ {pkg}")
        for line in (result.stderr or result.stdout).strip().splitlines():
            if line.strip():
                print(f"     {line}")
        errors.append(pkg)

if errors:
    print(f"\n❌ {len(errors)} package(s) failed: {errors}")
    print(f"\nManual fix — run this in the VS Code terminal:")
    print(f'  "{sys.executable}" -m pip install {" ".join(errors)}')
    print(f"\nIf that fails, select the correct kernel in VS Code:")
    print(f"  Kernel → Select Kernel → Python Environments")
    print(f"  Choose: C:\\Users\\dalonso\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe")
else:
    print("\n✅ All packages installed — proceed to the next cell.")


🐍 Kernel Python: c:\Users\dalonso\AppData\Local\Python\pythoncore-3.14-64\python.exe
   Version: 3.14.2

  ✅ azure-monitor-query
  ✅ azure-identity
  ✅ pandas
  ✅ numpy
  ✅ plotly
  ✅ matplotlib
  ✅ scipy

✅ All packages installed — proceed to the next cell.


In [3]:

# ── Core libraries ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy import stats
import warnings, json, urllib.request, urllib.error
warnings.filterwarnings('ignore')

# ── Azure SDK ─────────────────────────────────────────────────────────────────
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from azure.identity import (
    DefaultAzureCredential,
    InteractiveBrowserCredential,
    DeviceCodeCredential,
    ClientSecretCredential,
)

print("✅ All imports successful")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Analysis Period: Last 12 months "
      f"({(datetime.now()-timedelta(days=365)).strftime('%Y-%m-%d')} to "
      f"{datetime.now().strftime('%Y-%m-%d')})")


✅ All imports successful
📅 Analysis Date: 2026-02-24 11:21:26
📊 Analysis Period: Last 12 months (2025-02-24 to 2026-02-24)


In [4]:

# nbformat was installed after the kernel started, so plotly cached nbformat=None.
# Fix: inject the now-importable nbformat directly into plotly's renderer module.
import importlib, sys
importlib.invalidate_caches()

import nbformat                          # loads it into sys.modules

import plotly.io._renderers as _pr
_pr.nbformat = nbformat                  # patch the module-level None → real module

import plotly.io as pio
print(f"✅ nbformat {nbformat.__version__} injected into plotly renderer")
print(f"   Plotly default renderer: {pio.renderers.default}")


✅ nbformat 5.10.4 injected into plotly renderer
   Plotly default renderer: vscode


In [5]:

# ── Imports (self-contained so this cell can run standalone) ─────────────────
from datetime import timedelta
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from azure.identity import (
    DefaultAzureCredential,
    InteractiveBrowserCredential,
    DeviceCodeCredential,
    ClientSecretCredential,
)

# ── Workspace configuration ───────────────────────────────────────────────────
WORKSPACE_ID  = "xxxxx-xxxx-xxxx-xxx-xxxx"
TENANT_ID     = "xxxxx-xxxx-xxxx-xxx-xxxx"

# ── Authentication method ────────────────────────────────────────────────────
# Choose ONE of the options below by setting AUTH_METHOD:
#
#   "default"       - Tries: az CLI login → VS Code login → environment vars
#                     Prerequisites: run  az login  in the terminal first (recommended)
#
#   "browser"       - Opens a browser popup for interactive login
#
#   "device_code"   - Prints a code + URL; useful when browser popups are blocked
#
#   "service_principal" - Non-interactive; fill in CLIENT_ID and CLIENT_SECRET below
#
AUTH_METHOD   = "device_code"   # ← prints a code+URL to the cell output (no browser popup required)

CLIENT_ID     = ""              # Only needed for AUTH_METHOD = "service_principal"
CLIENT_SECRET = ""              # Only needed for AUTH_METHOD = "service_principal"

# ── Build credential ──────────────────────────────────────────────────────────
print(f"🔐 Authenticating with method: '{AUTH_METHOD}'")

if AUTH_METHOD == "default":
    # Tip: run  `az login --tenant {TENANT_ID}`  in the terminal before this cell
    _credential = DefaultAzureCredential(
        exclude_interactive_browser_credential=False,
        additionally_allowed_tenants=[TENANT_ID],
    )
elif AUTH_METHOD == "browser":
    _credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)
elif AUTH_METHOD == "device_code":
    _credential = DeviceCodeCredential(tenant_id=TENANT_ID)
elif AUTH_METHOD == "service_principal":
    if not CLIENT_ID or not CLIENT_SECRET:
        raise ValueError("Set CLIENT_ID and CLIENT_SECRET above for service_principal auth")
    _credential = ClientSecretCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    )
else:
    raise ValueError(f"Unknown AUTH_METHOD: '{AUTH_METHOD}'")

# ── Connect to Log Analytics ──────────────────────────────────────────────────
_logs_client = LogsQueryClient(_credential)

# Quick connectivity test
try:
    _test = _logs_client.query_workspace(
        workspace_id=WORKSPACE_ID,
        query="print('ok')",
        timespan=timedelta(minutes=1),
    )
    print(f"✅ Connected to Microsoft Sentinel workspace")
    print(f"   Workspace : {WORKSPACE_ID}")
    print(f"   Tenant    : {TENANT_ID}")
    print(f"   Auth      : {AUTH_METHOD}")
except Exception as e:
    print(f"⚠️  Connectivity test failed: {e}")
    print("\n  _logs_client is created — you will be prompted to authenticate")
    print("  when you run the first query cell (browser popup will appear).")
    print("\n  Tip: run  az login --tenant " + TENANT_ID + "  in the terminal")
    print("  then re-run this cell to skip the popup entirely.")


🔐 Authenticating with method: 'device_code'
To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code CQLW5RSK8 to authenticate.
✅ Connected to Microsoft Sentinel workspace
   Workspace : xxxxx-xxxx-xxxx-xxx-xxxx
   Tenant    : xxxx-xxxx-xxxx-xxxx-xxxxx
   Auth      : device_code


In [6]:

# ── Imports (safe fallback if cell 4 was skipped) ─────────────────────────────
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from scipy import stats

try:
    from azure.monitor.query import LogsQueryStatus
except ImportError:
    LogsQueryStatus = None

# ── Time range ────────────────────────────────────────────────────────────────
END_DATE   = datetime.now()
START_DATE = END_DATE - timedelta(days=365)

TIME_RANGE = (
    f"TimeGenerated between "
    f"(datetime({START_DATE.strftime('%Y-%m-%d')}) .. "
    f"datetime({END_DATE.strftime('%Y-%m-%d')}))"
)
print(f"📅 Analysis Time Range: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")

# ── KQL execution helper ──────────────────────────────────────────────────────
def run_kql(query: str, description: str = "") -> pd.DataFrame:
    """Execute a KQL query against the Sentinel workspace and return a DataFrame."""
    if description:
        print(f"🔍 {description}...")

    # Guard: auth cell must have run first
    if "_logs_client" not in dir() and "_logs_client" not in globals():
        print("   ❌ _logs_client not found — run the Authentication cell (cell 5) first.")
        print("      Tip: run  az login --tenant 0527ecb7-06fb-4769-b324-fd4a3bb865eb  in the terminal,")
        print("           then run cell 5, then re-run this cell and the query cell.")
        return pd.DataFrame()

    try:
        response = _logs_client.query_workspace(
            workspace_id=WORKSPACE_ID,
            query=query,
            timespan=timedelta(days=366),
        )
        if LogsQueryStatus and response.status == LogsQueryStatus.PARTIAL:
            print(f"   ⚠️ Partial result: {response.partial_error}")
        table  = response.tables[0]
        # SDK v1.x: columns are objects with .name; v2.x: columns are plain strings
        columns = [col if isinstance(col, str) else col.name for col in table.columns]
        result = pd.DataFrame(
            data=table.rows,
            columns=columns,
        )
        print(f"   ✅ Returned {len(result)} rows")
        return result
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return pd.DataFrame()

# ── Statistical helpers ───────────────────────────────────────────────────────
def detect_anomalies(data: pd.Series, threshold: float = 3.0) -> pd.Series:
    """Return a boolean Series: True where z-score exceeds threshold."""
    return np.abs(stats.zscore(data.fillna(0))) > threshold

def calculate_baseline(data: pd.Series) -> dict:
    """Return basic descriptive stats for a numeric Series."""
    return {
        "mean":   data.mean(),
        "std":    data.std(),
        "median": data.median(),
        "p95":    data.quantile(0.95),
        "p99":    data.quantile(0.99),
    }

print("✅ TIME_RANGE, run_kql(), and statistical helpers defined")

# ── Auth check ────────────────────────────────────────────────────────────────
if "_logs_client" not in dir() and "_logs_client" not in globals():
    print("\n⚠️  Authentication not yet completed.")
    print("   → Run cell 5 to authenticate before running query cells.")
    print("     If az login failed, set  AUTH_METHOD = 'browser'  in cell 5.")
else:
    print("✅ _logs_client is ready — query cells can run.")


📅 Analysis Time Range: 2025-02-24 to 2026-02-24
✅ TIME_RANGE, run_kql(), and statistical helpers defined
✅ _logs_client is ready — query cells can run.


---
## 2. Data Source Connectivity Check

Verify which data sources are available and have data in the workspace.

In [8]:

# ── Section 2: Data Source Connectivity — all tables used in this workbook ────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ── KQL: count rows per table in the analysis window (isfuzzy = silently skip
#         tables not available in this workspace) ─────────────────────────────
data_source_check_query = f"""
union isfuzzy=true
// ── Identity / Entra ID ──────────────────────────────────────────────────────
    (SigninLogs                         | where {TIME_RANGE} | summarize Count=count() | extend TableName="SigninLogs"),
    (AADNonInteractiveUserSignInLogs    | where {TIME_RANGE} | summarize Count=count() | extend TableName="AADNonInteractiveUserSignInLogs"),
    (AADServicePrincipalSignInLogs      | where {TIME_RANGE} | summarize Count=count() | extend TableName="AADServicePrincipalSignInLogs"),
    (AuditLogs                          | where {TIME_RANGE} | summarize Count=count() | extend TableName="AuditLogs"),
    (AADRiskyUsers                      | where {TIME_RANGE} | summarize Count=count() | extend TableName="AADRiskyUsers"),
    (AADUserRiskEvents                  | where {TIME_RANGE} | summarize Count=count() | extend TableName="AADUserRiskEvents"),
// ── Threat Intelligence ──────────────────────────────────────────────────────
    (ThreatIntelligenceIndicator        | where {TIME_RANGE} | summarize Count=count() | extend TableName="ThreatIntelligenceIndicator"),
// ── Defender for Endpoint (MDE) ──────────────────────────────────────────────
    (DeviceLogonEvents                  | where {TIME_RANGE} | summarize Count=count() | extend TableName="DeviceLogonEvents"),
    (DeviceNetworkEvents                | where {TIME_RANGE} | summarize Count=count() | extend TableName="DeviceNetworkEvents"),
    (DeviceProcessEvents                | where {TIME_RANGE} | summarize Count=count() | extend TableName="DeviceProcessEvents"),
    (DeviceFileEvents                   | where {TIME_RANGE} | summarize Count=count() | extend TableName="DeviceFileEvents"),
// ── Defender for Identity (MDI) ──────────────────────────────────────────────
    (IdentityLogonEvents                | where {TIME_RANGE} | summarize Count=count() | extend TableName="IdentityLogonEvents"),
    (IdentityQueryEvents                | where {TIME_RANGE} | summarize Count=count() | extend TableName="IdentityQueryEvents"),
    (IdentityDirectoryEvents            | where {TIME_RANGE} | summarize Count=count() | extend TableName="IdentityDirectoryEvents"),
// ── Defender for Cloud Apps (MCAS) ───────────────────────────────────────────
    (CloudAppEvents                     | where {TIME_RANGE} | summarize Count=count() | extend TableName="CloudAppEvents"),
// ── Defender for Office 365 (MDO) ────────────────────────────────────────────
    (EmailEvents                        | where {TIME_RANGE} | summarize Count=count() | extend TableName="EmailEvents"),
    (EmailAttachmentInfo                | where {TIME_RANGE} | summarize Count=count() | extend TableName="EmailAttachmentInfo"),
    (EmailUrlInfo                       | where {TIME_RANGE} | summarize Count=count() | extend TableName="EmailUrlInfo"),
    (UrlClickEvents                     | where {TIME_RANGE} | summarize Count=count() | extend TableName="UrlClickEvents"),
    (FileMaliciousContentInfo           | where {TIME_RANGE} | summarize Count=count() | extend TableName="FileMaliciousContentInfo"),
// ── Security Alerts & Windows Events ─────────────────────────────────────────
    (SecurityAlert                      | where {TIME_RANGE} | summarize Count=count() | extend TableName="SecurityAlert"),
    (SecurityEvent                      | where {TIME_RANGE} | summarize Count=count() | extend TableName="SecurityEvent"),
// ── UEBA ─────────────────────────────────────────────────────────────────────
    (BehaviorAnalytics                  | where {TIME_RANGE} | summarize Count=count() | extend TableName="BehaviorAnalytics"),
// ── Azure ─────────────────────────────────────────────────────────────────────
    (AzureActivity                      | where {TIME_RANGE} | summarize Count=count() | extend TableName="AzureActivity"),
    (AzureDiagnostics                   | where {TIME_RANGE} | summarize Count=count() | extend TableName="AzureDiagnostics"),
// ── Office 365 ────────────────────────────────────────────────────────────────
    (OfficeActivity                     | where {TIME_RANGE} | summarize Count=count() | extend TableName="OfficeActivity")
| project TableName, Count
| order by Count desc
"""

data_sources = run_kql(data_source_check_query, "Checking data source availability for all tables")

# ── Category mapping ──────────────────────────────────────────────────────────
CATEGORY_MAP = {
    "SigninLogs":                      "Identity / Entra ID",
    "AADNonInteractiveUserSignInLogs": "Identity / Entra ID",
    "AADServicePrincipalSignInLogs":   "Identity / Entra ID",
    "AuditLogs":                       "Identity / Entra ID",
    "AADRiskyUsers":                   "Identity / Entra ID",
    "AADUserRiskEvents":               "Identity / Entra ID",
    "ThreatIntelligenceIndicator":     "Threat Intelligence",
    "DeviceLogonEvents":               "Defender for Endpoint",
    "DeviceNetworkEvents":             "Defender for Endpoint",
    "DeviceProcessEvents":             "Defender for Endpoint",
    "DeviceFileEvents":                "Defender for Endpoint",
    "IdentityLogonEvents":             "Defender for Identity",
    "IdentityQueryEvents":             "Defender for Identity",
    "IdentityDirectoryEvents":         "Defender for Identity",
    "CloudAppEvents":                  "Defender for Cloud Apps",
    "EmailEvents":                     "Defender for Office 365",
    "EmailAttachmentInfo":             "Defender for Office 365",
    "EmailUrlInfo":                    "Defender for Office 365",
    "UrlClickEvents":                  "Defender for Office 365",
    "FileMaliciousContentInfo":        "Defender for Office 365",
    "SecurityAlert":                   "Security / Alerts",
    "SecurityEvent":                   "Security / Alerts",
    "BehaviorAnalytics":               "UEBA",
    "AzureActivity":                   "Azure",
    "AzureDiagnostics":                "Azure",
    "OfficeActivity":                  "Office 365",
}

# ── All 26 expected tables (for tables that returned 0 rows / were skipped) ──
ALL_TABLES = list(CATEGORY_MAP.keys())

if not data_sources.empty:
    data_sources["Category"] = data_sources["TableName"].map(CATEGORY_MAP).fillna("Other")
    data_sources["Status"]   = data_sources["Count"].apply(
        lambda x: "✅ Available" if x > 0 else "⚠️  No Data"
    )
else:
    data_sources = pd.DataFrame({"TableName": [], "Count": [], "Category": [], "Status": []})

# Add any tables that were completely absent (skipped by isfuzzy)
present = set(data_sources["TableName"].tolist()) if not data_sources.empty else set()
missing_rows = []
for t in ALL_TABLES:
    if t not in present:
        missing_rows.append({
            "TableName": t,
            "Count":     0,
            "Category":  CATEGORY_MAP.get(t, "Other"),
            "Status":    "❌ Not Found",
        })
if missing_rows:
    data_sources = pd.concat([data_sources, pd.DataFrame(missing_rows)], ignore_index=True)

data_sources = data_sources.sort_values(["Category", "Count"], ascending=[True, False])

# ── Console summary ────────────────────────────────────────────────────────────
total   = len(data_sources)
avail   = (data_sources["Count"] > 0).sum()
no_data = (data_sources["Count"] == 0).sum()

print(f"\n{'='*70}")
print(f"  DATA SOURCE CONNECTIVITY — {total} TABLES CHECKED")
print(f"{'='*70}")
print(f"  ✅ Available (data in period) : {avail}")
print(f"  ⚠️  No data / not found       : {no_data}")
print(f"{'='*70}")

current_cat = ""
for _, row in data_sources.iterrows():
    cat = row["Category"]
    if cat != current_cat:
        print(f"\n  [{cat}]")
        current_cat = cat
    count_str = f"{int(row['Count']):>12,}" if row["Count"] > 0 else f"{'—':>12}"
    print(f"    {row['Status']}  {row['TableName']:<45} {count_str} rows")

print(f"\n{'='*70}")

# ── Tabular display ────────────────────────────────────────────────────────────
display(data_sources[["Category", "TableName", "Count", "Status"]])

# ── Visualisation 1: bar chart of available tables by row count ────────────────
avail_df = data_sources[data_sources["Count"] > 0].sort_values("Count", ascending=False)
if not avail_df.empty:
    category_colors = {
        "Identity / Entra ID":      "#0984e3",
        "Threat Intelligence":       "#d63031",
        "Defender for Endpoint":     "#e17055",
        "Defender for Identity":     "#6c5ce7",
        "Defender for Cloud Apps":   "#00b894",
        "Defender for Office 365":   "#fdcb6e",
        "Security / Alerts":         "#e84393",
        "UEBA":                      "#fd79a8",
        "Azure":                     "#74b9ff",
        "Office 365":                "#55efc4",
    }
    avail_df = avail_df.copy()
    avail_df["Color"] = avail_df["Category"].map(category_colors).fillna("#b2bec3")

    fig = px.bar(
        avail_df,
        x="TableName",
        y="Count",
        color="Category",
        color_discrete_map=category_colors,
        title="📊 Data Source Event Counts — All Available Tables (Analysis Period)",
        labels={"TableName": "Table", "Count": "Event Count"},
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        legend_title="Data Source Category",
        height=480,
    )
    fig.show()

# ── Visualisation 2: availability status by category ──────────────────────────
cat_summary = (
    data_sources.groupby("Category", observed=True)
    .apply(lambda g: pd.Series({
        "Available":  int((g["Count"] > 0).sum()),
        "No Data":    int((g["Count"] == 0).sum()),
    }))
    .reset_index()
)
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    name="✅ Available", x=cat_summary["Category"],
    y=cat_summary["Available"], marker_color="#00b894",
))
fig2.add_trace(go.Bar(
    name="❌ No Data / Not Found", x=cat_summary["Category"],
    y=cat_summary["No Data"], marker_color="#d63031",
))
fig2.update_layout(
    barmode="stack",
    title="📋 Table Availability by Data Source Category",
    xaxis_tickangle=-30,
    yaxis_title="Table Count",
    legend_title="Status",
    height=400,
)
fig2.show()


🔍 Checking data source availability for all tables...
   ✅ Returned 26 rows

  DATA SOURCE CONNECTIVITY — 26 TABLES CHECKED
  ✅ Available (data in period) : 24
  ⚠️  No data / not found       : 2

  [Azure]
    ✅ Available  AzureDiagnostics                                52,643,889 rows
    ✅ Available  AzureActivity                                      532,726 rows

  [Defender for Cloud Apps]
    ✅ Available  CloudAppEvents                                  23,150,210 rows

  [Defender for Endpoint]
    ✅ Available  DeviceProcessEvents                             21,301,361 rows
    ✅ Available  DeviceFileEvents                                20,131,069 rows
    ✅ Available  DeviceNetworkEvents                             14,054,649 rows
    ✅ Available  DeviceLogonEvents                                  524,009 rows

  [Defender for Identity]
    ✅ Available  IdentityLogonEvents                              1,752,152 rows
    ✅ Available  IdentityQueryEvents                          

,Category,TableName,Count,Status
1,Azure,AzureDiagnostics,52643889,✅ Available
15,Azure,AzureActivity,532726,✅ Available
3,Defender for Cloud Apps,CloudAppEvents,23150210,✅ Available
4,Defender for Endpoint,DeviceProcessEvents,21301361,✅ Available
5,Defender for Endpoint,DeviceFileEvents,20131069,✅ Available
6,Defender for Endpoint,DeviceNetworkEvents,14054649,✅ Available
16,Defender for Endpoint,DeviceLogonEvents,524009,✅ Available
10,Defender for Identity,IdentityLogonEvents,1752152,✅ Available
18,Defender for Identity,IdentityQueryEvents,235153,✅ Available
19,Defender for Identity,IdentityDirectoryEvents,115686,✅ Available


---
## 3. Brute Force Attack Detection

Identify accounts experiencing high volumes of failed sign-in attempts, indicating potential brute force attacks.

**Detection Criteria:**
- More than 10 failed logins per account per hour
- Success-after-failure pattern (potential account compromise)

In [9]:
# Brute Force Detection Query
brute_force_query = f"""
// Brute Force Attack Detection - Failed Sign-ins per Account
let FailureThreshold = 10;
let TimeWindow = 1h;
union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where {TIME_RANGE}
| where ResultType != 0  // Failed sign-ins
| summarize 
    FailedAttempts = count(),
    UniqueIPs = dcount(IPAddress),
    IPs = make_set(IPAddress, 10),
    FailureCodes = make_set(ResultType, 10),
    FirstAttempt = min(TimeGenerated),
    LastAttempt = max(TimeGenerated)
    by UserPrincipalName, bin(TimeGenerated, TimeWindow)
| where FailedAttempts >= FailureThreshold
| extend AttackDuration = datetime_diff('minute', LastAttempt, FirstAttempt)
| extend RiskLevel = case(
    FailedAttempts > 100, "🔴 Critical",
    FailedAttempts > 50, "🟠 High",
    FailedAttempts > 20, "🟡 Medium",
    "🟢 Low"
)
| order by FailedAttempts desc
| take 100
"""

brute_force_results = run_kql(brute_force_query, "Detecting brute force attacks")

if not brute_force_results.empty:
    print(f"\n🚨 Found {len(brute_force_results)} potential brute force incidents")
    display(brute_force_results.head(20))
    
    # Visualization
    fig = px.scatter(brute_force_results, 
                     x='TimeGenerated', 
                     y='FailedAttempts',
                     color='RiskLevel',
                     size='UniqueIPs',
                     hover_data=['UserPrincipalName', 'IPs'],
                     title='Brute Force Attack Timeline')
    fig.show()
else:
    print("✅ No brute force attacks detected in the analysis period")

🔍 Detecting brute force attacks...
   ✅ Returned 100 rows

🚨 Found 100 potential brute force incidents


,UserPrincipalName,TimeGenerated,FailedAttempts,UniqueIPs,IPs,FailureCodes,FirstAttempt,LastAttempt,AttackDuration,RiskLevel
0,u17092@int.zava-corp.com,2026-01-26 09:00:00+00:00,1365,1,"[""20.236.10.66""]","[""70043""]",2026-01-26 09:00:00.689931+00:00,2026-01-26 09:47:56.581829+00:00,47,🔴 Critical
1,u17092@int.zava-corp.com,2026-01-26 08:00:00+00:00,1289,1,"[""20.236.10.66""]","[""70043""]",2026-01-26 08:00:00.712687+00:00,2026-01-26 08:59:59.500733+00:00,59,🔴 Critical
2,u17092@int.zava-corp.com,2026-01-26 07:00:00+00:00,979,1,"[""20.236.10.66""]","[""70043""]",2026-01-26 07:38:41.374184+00:00,2026-01-26 07:59:58.506101+00:00,21,🔴 Critical
3,u1060@int.zava-corp.com,2026-01-19 05:00:00+00:00,510,1,"[""4.255.188.99""]","[""70043""]",2026-01-19 05:00:01.779899+00:00,2026-01-19 05:59:48.075554+00:00,59,🔴 Critical
4,u1060@int.zava-corp.com,2026-01-19 04:00:00+00:00,495,3,"[""20.97.10.99"",""136.37.42.170"",""4.255.188.99""]","[""70043""]",2026-01-19 04:00:01.624357+00:00,2026-01-19 04:59:54.604578+00:00,59,🔴 Critical
5,u3555@int.zava-corp.com,2025-12-18 10:00:00+00:00,473,2,"[""40.69.22.128"",""40.78.239.108""]","[""50158"",""50207"",""130507"",""54006"",""65002"",""700...",2025-12-18 10:00:21.212825+00:00,2025-12-18 10:43:52.855306+00:00,43,🔴 Critical
6,u11652@int.zava-corp.com,2026-02-17 05:00:00+00:00,443,1,"[""155.93.206.134""]","[""70043""]",2026-02-17 05:00:03.847201+00:00,2026-02-17 05:59:57.848759+00:00,59,🔴 Critical
7,u11652@int.zava-corp.com,2026-02-17 04:00:00+00:00,419,1,"[""155.93.206.134""]","[""70043""]",2026-02-17 04:03:53.067720+00:00,2026-02-17 04:59:14.136534+00:00,56,🔴 Critical
8,u787@int.zava-corp.com,2026-01-13 21:00:00+00:00,386,3,"[""40.86.183.173"",""2a01:111:f400:7e19::100"",""2a...","[""700084"",""70043"",""500131""]",2026-01-13 21:00:01.220435+00:00,2026-01-13 21:59:58.521506+00:00,59,🔴 Critical
9,u11420@int.zava-corp.com,2026-01-08 08:00:00+00:00,349,1,"[""4.194.122.170""]","[""53003""]",2026-01-08 08:38:31.983043+00:00,2026-01-08 08:47:46.026664+00:00,9,🔴 Critical


In [10]:
# Detect Success After Failure (Potential Compromise)
success_after_failure_query = f"""
// Success After Multiple Failures - Potential Account Compromise
let FailedLogins = SigninLogs
| where {TIME_RANGE}
| where ResultType != 0
| summarize FailedCount = count(), FailedIPs = make_set(IPAddress, 10) 
    by UserPrincipalName, bin(TimeGenerated, 1h);
let SuccessfulLogins = SigninLogs
| where {TIME_RANGE}
| where ResultType == 0
| project UserPrincipalName, SuccessTime = TimeGenerated, SuccessIP = IPAddress, 
          Location, DeviceDetail, AppDisplayName;
FailedLogins
| where FailedCount >= 5
| join kind=inner (
    SuccessfulLogins
) on UserPrincipalName
| where SuccessTime between (TimeGenerated .. (TimeGenerated + 2h))
| extend CompromiseIndicator = iff(set_has_element(FailedIPs, SuccessIP) == false, "🔴 New IP - Likely Compromise", "🟡 Same IP")
| project UserPrincipalName, FailedCount, FailedIPs, SuccessTime, SuccessIP, 
          Location, AppDisplayName, CompromiseIndicator
| order by SuccessTime desc
| take 50
"""

compromised_accounts = run_kql(success_after_failure_query, "Detecting success-after-failure patterns")

if not compromised_accounts.empty:
    print(f"\n⚠️ Found {len(compromised_accounts)} potential compromised accounts")
    display(compromised_accounts)

🔍 Detecting success-after-failure patterns...
   ✅ Returned 50 rows

⚠️ Found 50 potential compromised accounts


,UserPrincipalName,FailedCount,FailedIPs,SuccessTime,SuccessIP,Location,AppDisplayName,CompromiseIndicator
0,u1596@int.zava-corp.com,6,"[""13.72.241.239""]",2026-02-23 23:40:08.982970+00:00,13.72.241.239,AU,Security Copilot Portal,🟡 Same IP
1,u1596@int.zava-corp.com,6,"[""13.72.241.239""]",2026-02-23 23:39:11.783551+00:00,13.72.241.239,AU,Security Copilot Portal,🟡 Same IP
2,u1596@int.zava-corp.com,6,"[""13.72.241.239""]",2026-02-23 23:38:43.593194+00:00,13.72.241.239,AU,Security Copilot Portal,🟡 Same IP
3,u1596@int.zava-corp.com,6,"[""13.72.241.239""]",2026-02-23 23:38:43.248775+00:00,13.72.241.239,AU,Security Copilot Portal,🟡 Same IP
4,u16518@int.zava-corp.com,8,"[""13.72.241.239""]",2026-02-23 23:29:01.405858+00:00,13.72.241.239,AU,Microsoft Account Controls V2,🟡 Same IP
5,u16518@int.zava-corp.com,8,"[""13.72.241.239""]",2026-02-23 23:27:08.127750+00:00,13.72.241.239,AU,Azure AD Identity Governance - Entitlement Man...,🟡 Same IP
6,u16518@int.zava-corp.com,8,"[""13.72.241.239""]",2026-02-23 23:26:42.867128+00:00,13.72.241.239,AU,Microsoft Edge,🟡 Same IP
7,u3762@int.zava-corp.com,5,"[""40.69.22.128""]",2026-02-23 23:16:53.868982+00:00,40.69.22.128,IE,Office365 Shell WCSS-Client,🟡 Same IP
8,u3762@int.zava-corp.com,5,"[""40.69.22.128""]",2026-02-23 23:16:38.875681+00:00,40.69.22.128,IE,Office365 Shell WCSS-Client,🟡 Same IP
9,u3762@int.zava-corp.com,5,"[""40.69.22.128""]",2026-02-23 23:16:17.564877+00:00,40.69.22.128,IE,Office365 Shell WCSS-Client,🟡 Same IP


---
## 4. Password Spray Attack Detection

Identify password spray attacks where attackers attempt to authenticate against multiple accounts from the same source.

**Detection Criteria:**
- Single IP targeting 5+ unique accounts
- Distributed attacks across multiple IPs targeting same accounts (botnet pattern)

In [11]:
# Password Spray Detection
password_spray_query = f"""
// Password Spray Detection - Single IP targeting multiple accounts
let AccountThreshold = 5;
union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where {TIME_RANGE}
| where ResultType != 0  // Failed sign-ins
| summarize 
    TargetedAccounts = dcount(UserPrincipalName),
    Accounts = make_set(UserPrincipalName, 20),
    TotalAttempts = count(),
    FailureCodes = make_set(ResultType),
    UserAgents = make_set(UserAgent, 5),
    FirstSeen = min(TimeGenerated),
    LastSeen = max(TimeGenerated)
    by IPAddress, bin(TimeGenerated, 1h)
| where TargetedAccounts >= AccountThreshold
| extend AttackType = case(
    TargetedAccounts > 50, "🔴 Mass Password Spray",
    TargetedAccounts > 20, "🟠 Large Password Spray",
    TargetedAccounts > 10, "🟡 Medium Password Spray",
    "🟢 Small Password Spray"
)
| extend GeoInfo = geo_info_from_ip_address(IPAddress)
| extend Country = tostring(GeoInfo.country)
| order by TargetedAccounts desc
| take 100
"""

password_spray_results = run_kql(password_spray_query, "Detecting password spray attacks")

if not password_spray_results.empty:
    print(f"\n🚨 Found {len(password_spray_results)} password spray campaigns")
    display(password_spray_results.head(20))
    
    # Top attacking IPs
    fig = px.bar(password_spray_results.head(20), 
                 x='IPAddress', 
                 y='TargetedAccounts',
                 color='AttackType',
                 title='Top 20 Password Spray Source IPs')
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

🔍 Detecting password spray attacks...
   ✅ Returned 100 rows

🚨 Found 100 password spray campaigns


,IPAddress,TimeGenerated,TargetedAccounts,Accounts,TotalAttempts,FailureCodes,UserAgents,FirstSeen,LastSeen,AttackType,GeoInfo,Country
0,172.200.70.89,2026-02-17 15:00:00+00:00,227,"[""u13758@int.zava-corp.com"",""u13384@int.zava-c...",2336,"[""70044"",""50158"",""50140"",""50089"",""500121"",""500...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-17 15:00:01.089662+00:00,2026-02-17 15:59:59.808382+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
1,172.200.70.89,2026-02-23 15:00:00+00:00,216,"[""u1812@int.zava-corp.com"",""u1522@int.zava-cor...",1984,"[""70044"",""50158"",""50140"",""50074"",""50076"",""7000...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-23 15:00:02.190486+00:00,2026-02-23 15:59:56.137027+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
2,172.200.70.89,2026-02-19 14:00:00+00:00,207,"[""u3548@int.zava-corp.com"",""u3414@int.zava-cor...",2295,"[""50126"",""70044"",""50158"",""50140"",""50133"",""5008...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-19 14:00:00.830990+00:00,2026-02-19 14:59:59.244237+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
3,172.200.70.89,2026-02-02 17:00:00+00:00,206,"[""u1929@int.zava-corp.com"",""u1252@int.zava-cor...",2273,"[""50074"",""16003"",""50158"",""70044"",""50020"",""1305...","[""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15...",2026-02-02 17:00:04.280653+00:00,2026-02-02 17:59:59.268260+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
4,172.200.70.89,2026-02-02 15:00:00+00:00,205,"[""u12258@int.zava-corp.com"",""u6235@int.zava-co...",2119,"[""50158"",""70044"",""50140"",""50074"",""500121"",""500...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-02 15:00:02.724929+00:00,2026-02-02 15:59:58.354782+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
5,172.200.70.89,2026-02-02 16:00:00+00:00,202,"[""u2603@int.zava-corp.com"",""u924@int.zava-corp...",2287,"[""50158"",""50207"",""50074"",""70044"",""50140"",""5001...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-02 16:00:00.778674+00:00,2026-02-02 16:59:56.688823+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
6,172.200.70.89,2026-02-23 14:00:00+00:00,200,"[""u1323@int.zava-corp.com"",""u3498@int.zava-cor...",2109,"[""70044"",""50158"",""16003"",""50140"",""50074"",""7004...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-23 14:00:01.932549+00:00,2026-02-23 14:59:57.332793+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
7,172.200.70.89,2026-02-19 15:00:00+00:00,197,"[""u1476@int.zava-corp.com"",""u13732@int.zava-co...",2420,"[""50158"",""70044"",""50207"",""50140"",""50076"",""5012...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-19 15:00:02.685478+00:00,2026-02-19 15:59:59.405070+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
8,172.200.70.89,2026-01-20 14:00:00+00:00,193,"[""u1450@int.zava-corp.com"",""u2676@int.zava-cor...",1945,"[""70044"",""50158"",""50126"",""500121"",""50074"",""501...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-01-20 14:00:03.113272+00:00,2026-01-20 14:59:59.488031+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States
9,172.200.70.89,2026-02-02 18:00:00+00:00,191,"[""u722@int.zava-corp.com"",""u969@int.zava-corp....",2646,"[""70044"",""50158"",""50020"",""16003"",""50074"",""5001...","[""Mozilla/5.0 (Windows NT 10.0; Win64; x64) Ap...",2026-02-02 18:00:00.378223+00:00,2026-02-02 18:59:58.356341+00:00,🔴 Mass Password Spray,"{""country"":""United States"",""state"":""Virginia"",...",United States


In [12]:
# Botnet Pattern Detection - Distributed Password Spray
botnet_spray_query = f"""
// Botnet Password Spray - Multiple IPs targeting same accounts
let TimeWindow = 1h;
let MinIPs = 3;
union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where {TIME_RANGE}
| where ResultType != 0
| summarize 
    AttackingIPs = dcount(IPAddress),
    IPs = make_set(IPAddress, 20),
    TotalAttempts = count(),
    Countries = make_set(Location, 10)
    by UserPrincipalName, bin(TimeGenerated, TimeWindow)
| where AttackingIPs >= MinIPs
| extend BotnetIndicator = case(
    AttackingIPs > 20, "🔴 High - Likely Botnet",
    AttackingIPs > 10, "🟠 Medium - Distributed Attack",
    "🟡 Low - Multiple Sources"
)
| order by AttackingIPs desc
| take 50
"""

botnet_results = run_kql(botnet_spray_query, "Detecting botnet password spray patterns")

if not botnet_results.empty:
    print(f"\n🤖 Found {len(botnet_results)} potential botnet attack patterns")
    display(botnet_results)

🔍 Detecting botnet password spray patterns...
   ✅ Returned 50 rows

🤖 Found 50 potential botnet attack patterns


,UserPrincipalName,TimeGenerated,AttackingIPs,IPs,TotalAttempts,Countries,BotnetIndicator
0,u1917@int.zava-corp.com,2026-02-05 08:00:00+00:00,38,"[""2404:c0:5c40::2cd3:a7b3"",""2404:c0:5c40::2cd3...",65,"[""ID""]",🔴 High - Likely Botnet
1,u5546@int.zava-corp.com,2025-12-04 15:00:00+00:00,28,"[""2a01:110:b080:0:7f97::15"",""2a01:110:b080:0:7...",46,"[""IE""]",🔴 High - Likely Botnet
2,u1917@int.zava-corp.com,2026-02-05 07:00:00+00:00,22,"[""2404:c0:5c40::2cca:df4e"",""2404:c0:5c40::2cca...",51,"[""ID""]",🔴 High - Likely Botnet
3,u1917@int.zava-corp.com,2026-02-05 09:00:00+00:00,21,"[""2404:c0:5c40::2cd9:1b5d"",""2404:c0:5c40::2cd9...",30,"[""ID""]",🔴 High - Likely Botnet
4,p10876@ctf.alpineskihouse.co,2026-01-28 00:00:00+00:00,18,"[""2001:4898:80e8:20:2ffa:caec:b48c:c532"",""148....",31,"[""US"",""GB""]",🟠 Medium - Distributed Attack
5,u1226@int.zava-corp.com,2026-02-20 08:00:00+00:00,18,"[""240d:18:a9:2100:1888:4f2d:e667:fe88"",""240d:1...",43,"[""JP""]",🟠 Medium - Distributed Attack
6,u13720@int.zava-corp.com,2026-01-13 05:00:00+00:00,17,"[""2404:f801:9000:1a:6d97:2979:4f1c:d06e"",""2404...",24,"[""SG""]",🟠 Medium - Distributed Attack
7,p10876@ctf.alpineskihouse.co,2026-01-27 17:00:00+00:00,16,"[""2001:4898:80e8:12:3008:caec:b48c:c532"",""2001...",106,"[""US"",""GB""]",🟠 Medium - Distributed Attack
8,u1226@int.zava-corp.com,2026-02-20 12:00:00+00:00,15,"[""240d:18:a9:2100:6924:c8cf:397c:e570"",""240d:1...",39,"[""JP""]",🟠 Medium - Distributed Attack
9,u1226@int.zava-corp.com,2025-12-17 10:00:00+00:00,15,"[""240d:18:a9:2100:1591:c98e:d152:506c"",""13.71....",49,"[""JP""]",🟠 Medium - Distributed Attack


---
## 5. Risky User Correlation

Correlate Azure AD Identity Protection risk signals with sign-in activity to identify high-risk users and their behavior patterns.

In [13]:
# Query Risky Users and Risk Events
risky_users_query = f"""
// Risky Users with Sign-in Correlation
let RiskyUsersList = AADRiskyUsers
| where {TIME_RANGE}
| where RiskLevel in ("medium", "high")
| project UserPrincipalName, RiskLevel, RiskState, RiskLastUpdatedDateTime, 
          RiskDetail, UserDisplayName;
let UserRiskEvents = AADUserRiskEvents
| where {TIME_RANGE}
| summarize 
    RiskEventCount = count(),
    RiskTypes = make_set(RiskEventType),
    DetectionTypes = make_set(DetectionTimingType)
    by UserPrincipalName;
let SignInActivity = SigninLogs
| where {TIME_RANGE}
| summarize 
    TotalSignIns = count(),
    FailedSignIns = countif(ResultType != 0),
    SuccessfulSignIns = countif(ResultType == 0),
    UniqueIPs = dcount(IPAddress),
    UniqueLocations = dcount(Location),
    LastSignIn = max(TimeGenerated)
    by UserPrincipalName;
RiskyUsersList
| join kind=leftouter UserRiskEvents on UserPrincipalName
| join kind=leftouter SignInActivity on UserPrincipalName
| extend FailureRate = round(todouble(FailedSignIns) / TotalSignIns * 100, 2)
| extend RiskScore = case(
    RiskLevel == "high" and FailureRate > 50, "🔴 Critical",
    RiskLevel == "high", "🟠 High",
    RiskLevel == "medium" and FailureRate > 30, "🟠 High",
    "🟡 Medium"
)
| project UserPrincipalName, UserDisplayName, RiskLevel, RiskState, RiskScore,
          RiskEventCount, RiskTypes, TotalSignIns, FailedSignIns, FailureRate,
          UniqueIPs, UniqueLocations, LastSignIn
| order by RiskScore asc, FailedSignIns desc
"""

risky_users = run_kql(risky_users_query, "Querying risky users with sign-in correlation")

if not risky_users.empty:
    print(f"\n⚠️ Found {len(risky_users)} risky users")
    display(risky_users)
    
    # Risk Level Distribution
    fig = px.pie(risky_users, names='RiskScore', title='Risky User Distribution by Risk Score')
    fig.show()

🔍 Querying risky users with sign-in correlation...
   ✅ Returned 407 rows

⚠️ Found 407 risky users


,UserPrincipalName,UserDisplayName,RiskLevel,RiskState,RiskScore,RiskEventCount,RiskTypes,TotalSignIns,FailedSignIns,FailureRate,UniqueIPs,UniqueLocations,LastSignIn
0,u6208@int.zava-corp.com,Tyler Szalai,high,confirmedCompromised,🔴 Critical,3.0,"[""adminConfirmedUserCompromised""]",784.0,430.0,54.85,7.0,2.0,2026-02-23 18:09:35.728133+00:00
1,u6208@int.zava-corp.com,Tyler Szalai,high,confirmedCompromised,🔴 Critical,3.0,"[""adminConfirmedUserCompromised""]",784.0,430.0,54.85,7.0,2.0,2026-02-23 18:09:35.728133+00:00
2,u3355@int.zava-corp.com,Mastura Walter,high,confirmedCompromised,🔴 Critical,1.0,"[""adminConfirmedUserCompromised""]",684.0,378.0,55.26,5.0,3.0,2026-02-23 09:22:28.955318+00:00
3,u3968@int.zava-corp.com,Sam Mukeshimana,high,confirmedCompromised,🔴 Critical,NaN,NaN,524.0,309.0,58.97,3.0,1.0,2026-02-19 02:54:11.375749+00:00
4,u5542@int.zava-corp.com,Attila Pinter,high,confirmedCompromised,🔴 Critical,1.0,"[""adminConfirmedUserCompromised""]",466.0,292.0,62.66,5.0,3.0,2026-02-23 22:01:01.809679+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,u2803@int.zava-corp.com,Vlastimil Schneider,high,confirmedCompromised,🟠 High,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
403,daisyp@zava-corp.com,Daisy Phillips,high,confirmedCompromised,🟠 High,3.0,"[""maliciousIPAddress"",""anonymizedIPAddress""]",NaN,NaN,NaN,NaN,NaN,NaT
404,u5674@int.zava-corp.com,Jokubas Jansson,high,confirmedCompromised,🟠 High,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
405,u1236@int.zava-corp.com,Thilak Mukarukundo,medium,atRisk,🟡 Medium,5.0,"[""unlikelyTravel"",""unfamiliarFeatures""]",136.0,39.0,28.68,1.0,1.0,2026-02-13 22:47:24.973933+00:00


In [14]:

# Risky Sign-ins Analysis
# NOTE: The 'RiskySignIns' table is not available in all Sentinel workspaces.
# Using SigninLogs with RiskLevelDuringSignIn column instead (always available).
risky_signins_query = f"""
// Risky Sign-ins — SigninLogs filtered by RiskLevelDuringSignIn
SigninLogs
| where {TIME_RANGE}
| where RiskLevelDuringSignIn in ("medium", "high")
| summarize
    RiskySignInCount   = count(),
    RiskLevels         = make_set(RiskLevelDuringSignIn),
    Locations          = make_set(Location, 10),
    IPs                = make_set(IPAddress, 10),
    Apps               = make_set(AppDisplayName, 10),
    FirstRiskySignIn   = min(TimeGenerated),
    LastRiskySignIn    = max(TimeGenerated),
    FailedCount        = countif(ResultType != 0),
    SuccessCount       = countif(ResultType == 0)
    by UserPrincipalName
| extend DaysAtRisk    = datetime_diff('day', LastRiskySignIn, FirstRiskySignIn)
| extend FailureRate   = round(100.0 * FailedCount / RiskySignInCount, 1)
| order by RiskySignInCount desc
| take 50
"""

risky_signins = run_kql(risky_signins_query, "Analyzing risky sign-ins")

if not risky_signins.empty:
    print(f"\n🔐 Found {len(risky_signins)} users with risky sign-ins")
    display(risky_signins)
    # Top risky users chart
    top_risky = risky_signins.head(15)
    fig = px.bar(top_risky, x="UserPrincipalName", y="RiskySignInCount",
                 color="FailureRate",
                 color_continuous_scale="Reds",
                 title="🔐 Users with Risky Sign-ins (colour = failure rate %)",
                 labels={"UserPrincipalName": "User", "RiskySignInCount": "Risky Sign-ins"})
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("✅ No risky sign-ins (medium/high risk) found in SigninLogs for this period")


🔍 Analyzing risky sign-ins...
   ✅ Returned 50 rows

🔐 Found 50 users with risky sign-ins


,UserPrincipalName,RiskySignInCount,RiskLevels,Locations,IPs,Apps,FirstRiskySignIn,LastRiskySignIn,FailedCount,SuccessCount,DaysAtRisk,FailureRate
0,elviaa@zava-corp.com,2037,"[""medium"",""high""]","[""DE"",""US"",""GB"",""CH"",""NL"",""AT"",""SE"",""PL"",""FR"",...","[""185.220.101.9"",""192.159.99.168"",""176.65.134....","[""Microsoft_AAD_RegisteredApps"",""Azure Portal""...",2025-12-03 09:14:32.478684+00:00,2026-02-19 07:14:37.890161+00:00,144,1893,78,7.1
1,u12952@int.zava-corp.com,33,"[""medium""]","[""US"",""IN""]","[""162.254.52.204"",""219.65.88.20""]","[""Highlights""]",2025-12-11 11:48:48+00:00,2026-01-20 20:15:21.715048+00:00,28,5,40,84.8
2,u7978@int.zava-corp.com,32,"[""medium""]","[""GB""]","[""2a01:110:8012:1012:c9e1:27c4:b6d5:5cc4""]","[""Security Copilot Portal""]",2026-01-29 11:37:18.201978+00:00,2026-01-29 12:32:15.316449+00:00,29,3,0,90.6
3,u1863@int.zava-corp.com,28,"[""medium""]","[""IN""]","[""167.220.238.197"",""223.31.35.58""]","[""Highlights"",""Microsoft 365 Security and Comp...",2025-12-01 06:09:36.362336+00:00,2025-12-03 06:52:32.622289+00:00,24,4,2,85.7
4,u2378@int.zava-corp.com,28,"[""medium""]","[""IN"",""IE""]","[""167.220.238.203"",""40.69.22.128""]","[""Microsoft 365 Security and Compliance Center""]",2025-12-23 08:15:57.394164+00:00,2026-01-20 09:34:36.432870+00:00,22,6,28,78.6
5,u3248@int.zava-corp.com,27,"[""medium"",""high""]","[""NL"",""DE""]","[""108.142.230.59"",""4.184.232.211""]","[""Azure Portal""]",2026-01-21 08:29:37.538018+00:00,2026-01-23 13:21:44.047623+00:00,21,6,2,77.8
6,u2980@int.zava-corp.com,26,"[""medium""]","[""JP"",""AU""]","[""13.71.146.83"",""13.72.241.239""]","[""Microsoft 365 Security and Compliance Center...",2026-01-15 00:46:25.067246+00:00,2026-01-20 08:55:05.044665+00:00,19,7,5,73.1
7,u4179@int.zava-corp.com,24,"[""medium"",""high""]","[""IE"",""NL""]","[""40.69.22.128"",""108.142.230.59"",""20.107.5.167""]","[""Microsoft 365 Security and Compliance Center""]",2025-11-27 08:08:23.252811+00:00,2026-01-21 09:22:37.071352+00:00,19,5,55,79.2
8,u2150@int.zava-corp.com,24,"[""medium""]","[""JP""]","[""2404:f801:8050:1:c068:704c:4e7d:62f0""]","[""Microsoft Office 365 Portal""]",2026-01-20 05:01:39.793618+00:00,2026-01-20 05:33:34.939147+00:00,21,3,0,87.5
9,lydiab@zava-corp.com,22,"[""high"",""medium""]","[""US"",""UA""]","[""192.159.99.162"",""185.156.72.7""]","[""ZavaBot""]",2026-01-08 08:52:53.704500+00:00,2026-01-12 09:49:26.388905+00:00,15,7,4,68.2


In [15]:

# ── 5b.  Sign-in Geolocation Analysis ────────────────────────────────────────
# Maps sign-in events by country/city; flags impossible travel & new locations.

geo_signin_query = f"""
SigninLogs
| where {TIME_RANGE}
| extend GeoInfo = geo_info_from_ip_address(IPAddress)
| extend Country  = tostring(GeoInfo.country),
         City     = tostring(GeoInfo.city),
         Lat      = todouble(GeoInfo.latitude),
         Lon      = todouble(GeoInfo.longitude)
| where isnotempty(Country)
| summarize
    TotalSignIns  = count(),
    FailedSignIns = countif(ResultType != 0),
    UniqueUsers   = dcount(UserPrincipalName),
    UniqueIPs     = dcount(IPAddress),
    FirstSeen     = min(TimeGenerated),
    LastSeen      = max(TimeGenerated)
  by Country, City, Lat, Lon
| order by TotalSignIns desc
| take 100
"""

geo_risky_query = f"""
SigninLogs
| where {TIME_RANGE}
| where RiskLevelDuringSignIn in ("medium", "high")
| extend GeoInfo = geo_info_from_ip_address(IPAddress)
| extend Country = tostring(GeoInfo.country),
         City    = tostring(GeoInfo.city)
| where isnotempty(Country)
| summarize
    RiskySignIns = count(),
    UniqueUsers  = dcount(UserPrincipalName),
    UniqueIPs    = dcount(IPAddress)
  by Country, City
| order by RiskySignIns desc
| take 50
"""

geo_multiuser_query = f"""
// Users signing in from 3+ distinct countries — strong impossible-travel signal
SigninLogs
| where {TIME_RANGE}
| extend GeoInfo = geo_info_from_ip_address(IPAddress)
| extend Country = tostring(GeoInfo.country)
| where isnotempty(Country)
| summarize
    Countries        = make_set(Country),
    CountryCount     = dcount(Country),
    TotalSignIns     = count(),
    FailedSignIns    = countif(ResultType != 0),
    FirstSeen        = min(TimeGenerated),
    LastSeen         = max(TimeGenerated)
  by UserPrincipalName
| where CountryCount >= 3
| extend SpanDays = datetime_diff('day', LastSeen, FirstSeen)
| order by CountryCount desc, TotalSignIns desc
| take 50
"""

print("🌍 Running geolocation analysis...")
geo_by_location = run_kql(geo_signin_query,   "Sign-ins by country/city")
geo_risky       = run_kql(geo_risky_query,    "Risky sign-ins by geography")
geo_multiuser   = run_kql(geo_multiuser_query,"Multi-country users")

# ─ Print summary ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("🌍 SIGN-IN GEOLOCATION SUMMARY")
print(f"{'='*70}")

if not geo_by_location.empty:
    total_countries = geo_by_location["Country"].nunique() if "Country" in geo_by_location.columns else 0
    total_cities    = geo_by_location.shape[0]
    print(f"   • Sign-in activity detected from {total_countries} countries / {total_cities} cities")
    print(f"\n   Top 10 sign-in locations:")
    if "Country" in geo_by_location.columns and "TotalSignIns" in geo_by_location.columns:
        for _, row in geo_by_location.head(10).iterrows():
            location = f"{row.get('City','?')}, {row.get('Country','?')}"
            print(f"     {'🔴' if row.get('FailedSignIns',0) > row.get('TotalSignIns',1)*0.5 else '🟢'} "
                  f"{location:<35} {int(row.get('TotalSignIns',0)):>6} sign-ins  "
                  f"({int(row.get('UniqueUsers',0))} users)")

if not geo_risky.empty:
    print(f"\n   ⚠️  Risky sign-ins span {geo_risky['Country'].nunique()} countries:")
    display(geo_risky.head(10))

if not geo_multiuser.empty:
    print(f"\n   ⚠️  {len(geo_multiuser)} users signed in from 3+ countries (potential account compromise):")
    display(geo_multiuser.head(15))
else:
    print("\n   ✅ No users signed in from 3+ distinct countries")

# ─ Charts ─────────────────────────────────────────────────────────────────────
if not geo_by_location.empty and "Country" in geo_by_location.columns:
    country_totals = (geo_by_location
                      .groupby("Country", as_index=False)["TotalSignIns"].sum()
                      .sort_values("TotalSignIns", ascending=False)
                      .head(20))
    fig = px.bar(country_totals, x="Country", y="TotalSignIns",
                 title="🌍 Sign-in Volume by Country (Top 20)",
                 color="TotalSignIns", color_continuous_scale="Blues",
                 labels={"TotalSignIns": "Sign-ins", "Country": "Country"})
    fig.update_layout(xaxis_tickangle=-45, coloraxis_showscale=False)
    fig.show()

if not geo_by_location.empty and all(c in geo_by_location.columns for c in ["Lat","Lon","TotalSignIns"]):
    geo_map = geo_by_location.dropna(subset=["Lat","Lon"])
    if not geo_map.empty:
        fig2 = px.scatter_geo(geo_map,
                              lat="Lat", lon="Lon",
                              size="TotalSignIns",
                              hover_name="City",
                              hover_data={"Country": True, "TotalSignIns": True,
                                          "UniqueUsers": True, "Lat": False, "Lon": False},
                              color="TotalSignIns",
                              color_continuous_scale="Reds",
                              projection="natural earth",
                              title="🌐 Sign-in Map — 12 Month Period")
        fig2.update_layout(height=500)
        fig2.show()


🌍 Running geolocation analysis...
🔍 Sign-ins by country/city...
   ✅ Returned 100 rows
🔍 Risky sign-ins by geography...
   ✅ Returned 48 rows
🔍 Multi-country users...
   ✅ Returned 50 rows

🌍 SIGN-IN GEOLOCATION SUMMARY
   • Sign-in activity detected from 33 countries / 100 cities

   Top 10 sign-in locations:
     🟢 Boydton, United States               76801 sign-ins  (917 users)
     🟢 Amsterdam, Netherlands               58986 sign-ins  (875 users)
     🟢 San Antonio, United States           51205 sign-ins  (530 users)
     🟢 Dublin, Ireland                      39862 sign-ins  (555 users)
     🟢 Sydney, Australia                    35377 sign-ins  (391 users)
     🟢 London, United Kingdom               35335 sign-ins  (723 users)
     🔴 Tokyo, Japan                         33195 sign-ins  (323 users)
     🟢 , United States                      30859 sign-ins  (587 users)
     🟢 Singapore, Singapore                 22784 sign-ins  (319 users)
     🟢 San Jose, United States          

,Country,City,RiskySignIns,UniqueUsers,UniqueIPs
0,Germany,Brandenburg,550,1,8
1,Germany,,332,1,4
2,Sweden,,322,2,4
3,United States,New York,228,3,4
4,Ireland,Dublin,177,17,3
5,Singapore,Singapore,141,11,11
6,Iceland,,126,1,1
7,Austria,Vienna,123,1,4
8,United States,,119,14,4
9,United Kingdom,London,116,10,8



   ⚠️  50 users signed in from 3+ countries (potential account compromise):


,UserPrincipalName,Countries,CountryCount,TotalSignIns,FailedSignIns,FirstSeen,LastSeen,SpanDays
0,,"[""Greece"",""United States"",""Japan"",""Israel"",""In...",24,730,729,2025-12-11 03:36:10+00:00,2026-02-21 12:29:04.872339+00:00,72
1,elviaa@zava-corp.com,"[""Germany"",""Austria"",""Switzerland"",""France"",""I...",10,2042,149,2025-12-03 09:14:32.478684+00:00,2026-02-19 07:14:37.890161+00:00,78
2,u2485@int.zava-corp.com,"[""United States"",""Australia"",""Brazil"",""South A...",7,702,308,2025-12-02 00:25:55.886955+00:00,2026-02-22 22:46:54.450832+00:00,82
3,u4066@int.zava-corp.com,"[""Austria"",""Australia"",""Germany"",""Netherlands""...",6,474,200,2025-11-26 15:02:42.352750+00:00,2026-02-19 14:53:10.319930+00:00,85
4,u2016@int.zava-corp.com,"[""Australia"",""Türkiye"",""Netherlands"",""Sweden"",...",6,345,189,2025-11-25 22:08:42.239610+00:00,2026-02-15 20:25:04.263366+00:00,82
5,u747@int.zava-corp.com,"[""Ireland"",""United Kingdom"",""Netherlands"",""Uni...",5,698,370,2025-11-26 08:51:59.544887+00:00,2026-02-20 10:50:22.131437+00:00,86
6,u3606@int.zava-corp.com,"[""Netherlands"",""United Kingdom"",""France"",""Unit...",5,505,235,2025-11-26 09:35:21.765400+00:00,2026-02-22 19:21:06.009090+00:00,88
7,u6205@int.zava-corp.com,"[""Saudi Arabia"",""United Kingdom"",""Netherlands""...",5,488,191,2025-11-25 11:37:08.955030+00:00,2026-02-19 11:47:03.635586+00:00,86
8,u383@int.zava-corp.com,"[""Germany"",""Netherlands"",""Poland"",""Australia"",...",5,440,150,2025-11-25 22:33:40.474541+00:00,2026-02-21 21:22:02.888128+00:00,88
9,u399@int.zava-corp.com,"[""Italy"",""Netherlands"",""United Kingdom"",""Germa...",5,417,180,2025-11-26 14:19:56.472174+00:00,2026-02-20 16:15:22.980821+00:00,86


---
## 6. Baseline Anomaly Detection

Establish normal baselines for sign-in activity and detect anomalies using statistical time-series analysis.

**Methods:**
- Z-score analysis for hourly/daily failed login counts (configurable bin size)
- Rolling 7-day average trend overlay
- Spike detection with configurable σ threshold
- Day-of-week × hour-of-day heatmap to spot patterned attacks
- Anomaly drill-down: top users/IPs driving each spike

> **Config block at the top of the cell** — change `BASELINE_DAYS`, `BIN_SIZE`, `ANOMALY_SIGMA`, etc. to tune the analysis without editing any KQL.


In [18]:

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — CONFIGURATION  (edit only this block to tune the analysis)
# ═══════════════════════════════════════════════════════════════════════════════

BASELINE_DAYS   = 365   # ← analysis window in days (365 = 12 months, 180 = 6 months, 90 = 3 months)
BIN_SIZE        = "1h"  # ← time bucket: "1h" (hourly) or "1d" (daily)
ANOMALY_SIGMA   = 3.0   # ← z-score threshold to flag a bucket as anomalous (2.5 / 3.0 / 3.5)
ROLLING_WINDOW  = 168   # ← rolling-average window in buckets (168 h = 7 days for hourly; use 7 for daily)
TOP_N_ANOMALIES = 15    # ← how many anomaly rows to display in the drill-down table
DRILL_DOWN_TOP  = 5     # ← top users / IPs shown per anomalous bucket in the drill-down

# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as _stats
from datetime import datetime, timedelta

# ── Derived time range ─────────────────────────────────────────────────────────
_end   = datetime.now()
_start = _end - timedelta(days=BASELINE_DAYS)
_kql_range = (
    f"TimeGenerated between "
    f"(datetime({_start.strftime('%Y-%m-%d')}) .. "
    f"datetime({_end.strftime('%Y-%m-%d')}))"
)

print(f"⚙️  Config  →  window: {BASELINE_DAYS}d  ({_start.strftime('%Y-%m-%d')} → {_end.strftime('%Y-%m-%d')})")
print(f"             bin: {BIN_SIZE}  |  σ threshold: {ANOMALY_SIGMA}  |  rolling window: {ROLLING_WINDOW} buckets")

# ── 6.1  Main baseline query ───────────────────────────────────────────────────
baseline_query = f"""
// Section 6 — Failed Login Baseline
// Bin size and time range are set in the Python config block above.
union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where {_kql_range}
| where ResultType != 0
| summarize
    FailedCount = count(),
    UniqueUsers = dcount(UserPrincipalName),
    UniqueIPs   = dcount(IPAddress)
    by bin(TimeGenerated, {BIN_SIZE})
| order by TimeGenerated asc
"""

hourly_failures = run_kql(baseline_query, f"6.1 Baseline — {BASELINE_DAYS}-day failed login counts (bin={BIN_SIZE})")

if hourly_failures.empty:
    print("\n⚠️  No data returned — check authentication and time range.")
else:
    hourly_failures["TimeGenerated"] = pd.to_datetime(hourly_failures["TimeGenerated"])
    hourly_failures = hourly_failures.sort_values("TimeGenerated").reset_index(drop=True)

    # ── Statistics ─────────────────────────────────────────────────────────────
    fc = hourly_failures["FailedCount"]
    baseline = {
        "mean":   fc.mean(),
        "std":    fc.std(),
        "median": fc.median(),
        "p95":    fc.quantile(0.95),
        "p99":    fc.quantile(0.99),
    }
    anomaly_threshold = baseline["mean"] + ANOMALY_SIGMA * baseline["std"]

    print(f"\n{'='*56}")
    print(f"  📊 BASELINE STATISTICS  ({BASELINE_DAYS}-day window)")
    print(f"{'='*56}")
    print(f"  Mean          : {baseline['mean']:>10.1f} failed logins / {BIN_SIZE}")
    print(f"  Std deviation : {baseline['std']:>10.1f}")
    print(f"  Median        : {baseline['median']:>10.1f}")
    print(f"  95th pct      : {baseline['p95']:>10.1f}")
    print(f"  99th pct      : {baseline['p99']:>10.1f}")
    print(f"  Anomaly line  : {anomaly_threshold:>10.1f}  ({ANOMALY_SIGMA}σ above mean)")
    print(f"{'='*56}")

    # ── Z-score & anomaly flag ──────────────────────────────────────────────────
    hourly_failures["z_score"]    = _stats.zscore(fc.fillna(0))
    hourly_failures["is_anomaly"] = hourly_failures["z_score"].abs() > ANOMALY_SIGMA
    hourly_failures["rolling_avg"] = (
        fc.rolling(window=min(ROLLING_WINDOW, len(fc)), min_periods=1).mean()
    )

    anomalies = hourly_failures[hourly_failures["is_anomaly"]].copy()
    print(f"\n🚨 Detected {len(anomalies)} anomalous buckets  "
          f"({100*len(anomalies)/max(len(hourly_failures),1):.1f}% of period)")

    # ── Chart 1: Time-series with rolling avg, baseline & anomaly markers ──────
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hourly_failures["TimeGenerated"], y=fc,
        mode="lines", name=f"Failed Logins ({BIN_SIZE})",
        line=dict(color="#74b9ff", width=1), opacity=0.8,
    ))
    fig.add_trace(go.Scatter(
        x=hourly_failures["TimeGenerated"], y=hourly_failures["rolling_avg"],
        mode="lines", name=f"Rolling Avg ({ROLLING_WINDOW} buckets)",
        line=dict(color="#0984e3", width=2, dash="dot"),
    ))
    if not anomalies.empty:
        fig.add_trace(go.Scatter(
            x=anomalies["TimeGenerated"], y=anomalies["FailedCount"],
            mode="markers", name="🚨 Anomaly",
            marker=dict(color="#d63031", size=9, symbol="circle-open",
                        line=dict(width=2)),
        ))
    fig.add_hline(y=baseline["mean"],  line_dash="dash", line_color="#00b894",
                  annotation_text=f"Mean ({baseline['mean']:.0f})",
                  annotation_position="top left")
    fig.add_hline(y=anomaly_threshold, line_dash="dash", line_color="#e17055",
                  annotation_text=f"{ANOMALY_SIGMA}σ threshold ({anomaly_threshold:.0f})",
                  annotation_position="top left")
    fig.add_hline(y=baseline["p95"],   line_dash="dot",  line_color="#fdcb6e",
                  annotation_text=f"P95 ({baseline['p95']:.0f})",
                  annotation_position="top left")
    fig.update_layout(
        title=(f"Failed Login Attempts — {BASELINE_DAYS}-day Baseline "
               f"with Anomaly Detection  (bin={BIN_SIZE}, σ={ANOMALY_SIGMA})"),
        xaxis_title="Time", yaxis_title="Failed Login Count",
        legend=dict(orientation="h", y=-0.18),
        height=460,
    )
    fig.show()

    # ── Anomaly drill-down table ────────────────────────────────────────────────
    if not anomalies.empty:
        print(f"\n📋 Top {TOP_N_ANOMALIES} Anomalous Buckets:")
        display(
            anomalies[["TimeGenerated", "FailedCount", "UniqueUsers", "UniqueIPs", "z_score"]]
            .sort_values("FailedCount", ascending=False)
            .head(TOP_N_ANOMALIES)
            .reset_index(drop=True)
        )

    # ── Chart 2: Z-score over time ──────────────────────────────────────────────
    fig_z = px.line(
        hourly_failures, x="TimeGenerated", y="z_score",
        title=f"Z-Score Over Time  (threshold ±{ANOMALY_SIGMA}σ highlighted)",
        labels={"z_score": "Z-Score", "TimeGenerated": "Time"},
        color_discrete_sequence=["#a29bfe"],
    )
    fig_z.add_hline(y= ANOMALY_SIGMA, line_dash="dash", line_color="#d63031",
                    annotation_text=f"+{ANOMALY_SIGMA}σ")
    fig_z.add_hline(y=-ANOMALY_SIGMA, line_dash="dash", line_color="#d63031",
                    annotation_text=f"-{ANOMALY_SIGMA}σ")
    fig_z.add_hline(y=0, line_dash="dot", line_color="#636e72")
    fig_z.update_layout(height=340)
    fig_z.show()

    # ── Chart 3: Day-of-week × Hour-of-day heatmap (hourly bins only) ──────────
    if BIN_SIZE == "1h":
        hm = hourly_failures.copy()
        hm["DayOfWeek"] = hm["TimeGenerated"].dt.day_name()
        hm["Hour"]      = hm["TimeGenerated"].dt.hour
        day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        pivot = (
            hm.groupby(["DayOfWeek", "Hour"])["FailedCount"]
            .mean()
            .reset_index()
            .pivot(index="DayOfWeek", columns="Hour", values="FailedCount")
            .reindex(day_order)
        )
        fig_hm = px.imshow(
            pivot,
            labels=dict(x="Hour of Day", y="Day of Week", color="Avg Failed Logins"),
            title=f"Avg Failed Logins by Day × Hour  ({BASELINE_DAYS}-day window)",
            color_continuous_scale="Blues",
            aspect="auto",
        )
        fig_hm.update_layout(height=340)
        fig_hm.show()

    # ── 6.2  Per-user drill-down for the top anomaly windows ───────────────────
    if not anomalies.empty:
        top_anomaly_times = anomalies.nlargest(3, "FailedCount")["TimeGenerated"].tolist()
        _bin_delta = pd.Timedelta(BIN_SIZE)

        drilldown_parts = []
        for _ts in top_anomaly_times:
            _ts_end  = _ts + _bin_delta
            _ts_s    = _ts.strftime("%Y-%m-%dT%H:%M:%SZ")
            _ts_e    = _ts_end.strftime("%Y-%m-%dT%H:%M:%SZ")
            drilldown_query = f"""
union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where TimeGenerated between (datetime({_ts_s}) .. datetime({_ts_e}))
| where ResultType != 0
| summarize
    Failures  = count(),
    FailCodes = make_set(ResultType, 5),
    IPs       = make_set(IPAddress, 5)
    by UserPrincipalName
| top {DRILL_DOWN_TOP} by Failures
| extend BucketStart = datetime('{_ts_s}')
"""
            _df = run_kql(drilldown_query, f"  → drill-down {_ts.strftime('%Y-%m-%d %H:%M')}")
            if not _df.empty:
                drilldown_parts.append(_df)

        if drilldown_parts:
            drilldown_df = pd.concat(drilldown_parts, ignore_index=True)
            print(f"\n🔬 Top-user drill-down for the {len(top_anomaly_times)} largest anomaly buckets:")
            display(drilldown_df)


⚙️  Config  →  window: 365d  (2025-02-24 → 2026-02-24)
             bin: 1h  |  σ threshold: 3.0  |  rolling window: 168 buckets
🔍 6.1 Baseline — 365-day failed login counts (bin=1h)...
   ✅ Returned 2149 rows

  📊 BASELINE STATISTICS  (365-day window)
  Mean          :     6218.5 failed logins / 1h
  Std deviation :     2102.0
  Median        :     5831.0
  95th pct      :     9920.0
  99th pct      :    11639.6
  Anomaly line  :    12524.4  (3.0σ above mean)

🚨 Detected 12 anomalous buckets  (0.6% of period)



📋 Top 15 Anomalous Buckets:


,TimeGenerated,FailedCount,UniqueUsers,UniqueIPs,z_score
0,2026-01-13 20:00:00+00:00,14909,792,245,4.135417
1,2026-02-02 18:00:00+00:00,13601,1080,314,3.513002
2,2026-01-12 21:00:00+00:00,13595,802,252,3.510147
3,2026-01-29 15:00:00+00:00,13525,975,246,3.476837
4,2026-01-13 19:00:00+00:00,13363,840,249,3.399749
5,2026-01-13 17:00:00+00:00,12964,944,277,3.209884
6,2026-01-12 22:00:00+00:00,12890,730,258,3.174670
7,2026-01-13 18:00:00+00:00,12819,922,272,3.140885
8,2026-01-28 14:00:00+00:00,12765,1077,268,3.115189
9,2026-01-27 16:00:00+00:00,12667,1070,303,3.068555


🔍   → drill-down 2026-01-13 20:00...
   ✅ Returned 5 rows
🔍   → drill-down 2026-02-02 18:00...
   ✅ Returned 5 rows
🔍   → drill-down 2026-01-12 21:00...
   ✅ Returned 5 rows

🔬 Top-user drill-down for the 3 largest anomaly buckets:


,UserPrincipalName,Failures,FailCodes,IPs,BucketStart
0,u1316@int.zava-corp.com,242,"[""70043""]","[""20.97.10.99""]",2026-01-13 20:00:00+00:00
1,u1305@int.zava-corp.com,125,"[""700082"",""70043""]","[""20.217.218.129""]",2026-01-13 20:00:00+00:00
2,u595@int.zava-corp.com,99,"[""50158"",""70044"",""50140"",""50074"",""70043""]","[""172.200.70.89""]",2026-01-13 20:00:00+00:00
3,u336@int.zava-corp.com,94,"[""70043""]","[""50.47.234.241"",""71.227.196.202"",""2607:fb90:e...",2026-01-13 20:00:00+00:00
4,u6244@int.zava-corp.com,94,"[""70043"",""50020""]","[""5.163.154.82"",""40.69.22.128""]",2026-01-13 20:00:00+00:00
5,u1686@int.zava-corp.com,201,"[""70043""]","[""13.71.146.83""]",2026-02-02 18:00:00+00:00
6,u2738@int.zava-corp.com,161,"[""70043""]","[""40.86.181.13"",""40.86.183.173"",""13.88.17.9"",""...",2026-02-02 18:00:00+00:00
7,u1316@int.zava-corp.com,99,"[""50158"",""70043""]","[""20.97.10.99""]",2026-02-02 18:00:00+00:00
8,u2693@int.zava-corp.com,88,"[""700082"",""70043""]","[""13.72.241.239"",""172.200.70.89"",""4.8.21.86""]",2026-02-02 18:00:00+00:00
9,u4503@int.zava-corp.com,74,"[""70044"",""70043""]","[""172.167.23.70""]",2026-02-02 18:00:00+00:00


In [19]:
# Impossible Travel Detection
impossible_travel_query = f"""
// Impossible Travel Detection
let MaxTravelSpeed = 500; // mph - impossible for physical travel
SigninLogs
| where {TIME_RANGE}
| where ResultType == 0  // Successful logins only
| where isnotempty(Location)
| extend GeoInfo = geo_info_from_ip_address(IPAddress)
| extend Latitude = toreal(GeoInfo.latitude), Longitude = toreal(GeoInfo.longitude)
| where isnotempty(Latitude) and isnotempty(Longitude)
| order by UserPrincipalName asc, TimeGenerated asc
| serialize
| extend 
    PrevLat = prev(Latitude, 1),
    PrevLon = prev(Longitude, 1),
    PrevTime = prev(TimeGenerated, 1),
    PrevUser = prev(UserPrincipalName, 1),
    PrevLocation = prev(Location, 1)
| where UserPrincipalName == PrevUser
| extend TimeDiffHours = datetime_diff('hour', TimeGenerated, PrevTime)
| where TimeDiffHours > 0 and TimeDiffHours < 24
| extend Distance = geo_distance_2points(Longitude, Latitude, PrevLon, PrevLat) / 1609.34 // Convert to miles
| extend TravelSpeed = Distance / TimeDiffHours
| where TravelSpeed > MaxTravelSpeed
| project UserPrincipalName, TimeGenerated, Location, IPAddress,
          PrevTime, PrevLocation, Distance, TimeDiffHours, TravelSpeed,
          AppDisplayName, DeviceDetail
| extend ImpossibleTravelAlert = "🔴 Impossible Travel Detected"
| order by TravelSpeed desc
| take 50
"""

impossible_travel = run_kql(impossible_travel_query, "Detecting impossible travel")

if not impossible_travel.empty:
    print(f"\n✈️ Found {len(impossible_travel)} impossible travel incidents")
    display(impossible_travel)
else:
    print("✅ No impossible travel detected")

🔍 Detecting impossible travel...
   ✅ Returned 50 rows

✈️ Found 50 impossible travel incidents


,UserPrincipalName,TimeGenerated,Location,IPAddress,PrevTime,PrevLocation,Distance,TimeDiffHours,TravelSpeed,AppDisplayName,DeviceDetail,ImpossibleTravelAlert
0,u171@int.zava-corp.com,2026-02-20 14:58:55.318179+00:00,GB,172.167.23.70,2026-02-20 13:25:48.759962+00:00,AU,10559.116119,1,10559.116119,Microsoft 365 Copilot extension,"{""deviceId"":""f6708177-965d-477d-a602-b403c664c...",🔴 Impossible Travel Detected
1,u16377@int.zava-corp.com,2026-02-10 12:36:06.115567+00:00,AU,13.72.241.239,2026-02-10 11:13:25.546396+00:00,GB,10559.116119,1,10559.116119,Microsoft 365 Copilot extension,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",🔴 Impossible Travel Detected
2,u171@int.zava-corp.com,2026-02-04 19:42:54.516096+00:00,AU,13.72.241.239,2026-02-04 18:23:22.995049+00:00,GB,10559.116119,1,10559.116119,Microsoft 365 Copilot extension,"{""deviceId"":""f6708177-965d-477d-a602-b403c664c...",🔴 Impossible Travel Detected
3,u171@int.zava-corp.com,2026-02-04 09:50:52.327735+00:00,GB,172.167.23.70,2026-02-04 08:57:12.105772+00:00,AU,10559.116119,1,10559.116119,Microsoft 365 Copilot extension,"{""deviceId"":""f6708177-965d-477d-a602-b403c664c...",🔴 Impossible Travel Detected
4,u807@int.zava-corp.com,2026-01-20 19:59:19.132379+00:00,NL,40.68.200.63,2026-01-20 18:50:10.043252+00:00,AU,10341.025350,1,10341.025350,Microsoft 365 Copilot extension,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",🔴 Impossible Travel Detected
5,u491@int.zava-corp.com,2026-02-17 13:59:15.627826+00:00,NL,40.68.200.63,2026-02-17 12:09:05.880198+00:00,AU,10341.025350,1,10341.025350,Microsoft 365 Copilot extension,"{""deviceId"":""37feaf06-4c3b-4db8-84e4-c273451bf...",🔴 Impossible Travel Detected
6,u491@int.zava-corp.com,2026-02-10 13:24:48.307853+00:00,AU,13.72.241.239,2026-02-10 12:17:07.395972+00:00,NL,10341.025350,1,10341.025350,Office365 Shell WCSS-Client,"{""deviceId"":""37feaf06-4c3b-4db8-84e4-c273451bf...",🔴 Impossible Travel Detected
7,u6443@int.zava-corp.com,2026-02-18 14:47:43.650246+00:00,NL,40.68.200.63,2026-02-18 13:22:58.749414+00:00,AU,10341.025350,1,10341.025350,Office365 Shell WCSS-Client,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",🔴 Impossible Travel Detected
8,u4066@int.zava-corp.com,2026-01-08 15:09:45.641813+00:00,AU,13.72.241.239,2026-01-08 14:50:50.849009+00:00,AT,9934.805184,1,9934.805184,Office365 Shell WCSS-Client,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",🔴 Impossible Travel Detected
9,u6772@int.zava-corp.com,2026-02-15 21:11:55.318557+00:00,AU,13.72.241.239,2026-02-15 20:59:57.550961+00:00,HU,9807.189016,1,9807.189016,Microsoft 365 Security and Compliance Center,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",🔴 Impossible Travel Detected



## 7. IOC Matching — Full-Spectrum Threat Intelligence Correlation

Scans **every table in this workbook** against `ThreatIntelligenceIndicator` for all IOC entity types.  
Each query uses `union isfuzzy=true` so unavailable tables are silently skipped.

| # | Entity Type | Tables Searched |
|---|-------------|----------------|
| **7.1** | **IP Address** | DeviceNetworkEvents, DeviceLogonEvents, SigninLogs, AADNonInteractiveUserSignInLogs, AADServicePrincipalSignInLogs, AuditLogs, AzureActivity, IdentityLogonEvents, IdentityDirectoryEvents, CloudAppEvents, SecurityAlert, **CommonSecurityLog** (SourceIP + DestinationIP) |
| **7.2** | **File Hash** | DeviceFileEvents, DeviceProcessEvents, EmailAttachmentInfo, SecurityAlert (file entities) |
| **7.3** | **Domain / Hostname** | DeviceNetworkEvents, IdentityQueryEvents (LDAP/DNS), EmailEvents (sender domain), **EmailUrlInfo** (link domains), **UrlClickEvents** (Safe Links click domain), **EmailAttachmentInfo** (sender domain), SigninLogs (UPN domain), AuditLogs, **CommonSecurityLog** (DestinationHostName) |
| **7.4** | **URL** | DeviceNetworkEvents, UrlClickEvents (MDO), EmailUrlInfo, CloudAppEvents, **CommonSecurityLog** (RequestURL — proxy) |
| **7.5** | **Process / Command-line** | DeviceProcessEvents (hash match + LOLBin patterns), SecurityAlert (process entities) |
| **7.6** | **Email Sender / Attachment** | EmailEvents (sender IP + address), EmailAttachmentInfo (hash + sender), CloudAppEvents (Teams / SharePoint) |
| **7.7** | **IOC Summary** | Consolidated pivot + chart across all entity types |
| **7.8** | **Threat Actor & Analytics** | TI actor grouping, SecurityAlert enrichment, BehaviorAnalytics UEBA |

> **Defender for Office 365** tables added to 7.3:
> - **EmailUrlInfo** — extracts the registrable domain from every URL embedded in email bodies; catches malicious link domains even if the IP is clean
> - **UrlClickEvents** — domain extracted from Safe Links-rewritten URLs at click time; detects users who actually opened a malicious link
> - **EmailAttachmentInfo** — sender domain on emails that carried attachments; catches malicious-domain senders that bypassed body-only filters
>
> **CommonSecurityLog** (CEF format) is included in three IOC sections:
> - **7.1 IP** — `SourceIP` (inbound) and `DestinationIP` (outbound C2 / exfil)
> - **7.3 Domain** — `DestinationHostName` (DNS-resolved connections through NGFW / proxy)
> - **7.4 URL** — `RequestURL` (full URLs proxied through web gateways)


In [32]:

# ── 7.1  IOC Matching — IP Address (all tables) ───────────────────────────────
# Matches malicious IPs from ThreatIntelligenceIndicator across every table
# in the workbook that carries an IP address column.

ip_ioc_query = f"""
let ThreatIPs = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(NetworkIP) or isnotempty(NetworkDestinationIP)
       or isnotempty(NetworkSourceIP)
| where ConfidenceScore >= 50
| extend IOC_IP = coalesce(NetworkIP, NetworkDestinationIP, NetworkSourceIP)
| where isnotempty(IOC_IP)
| summarize ThreatTypes    = make_set(ThreatType, 5),
            MaxConfidence  = max(ConfidenceScore),
            ThreatDesc     = any(Description)
  by IOC_IP;
// ── Endpoint network connections ───────────────────────────────────────────
let Net = DeviceNetworkEvents
| where {TIME_RANGE}
| where isnotempty(RemoteIP)
| join kind=inner ThreatIPs on $left.RemoteIP == $right.IOC_IP
| project TimeGenerated, Source="DeviceNetworkEvents",
    Actor=InitiatingProcessAccountName, Host=DeviceName,
    MatchedIP=RemoteIP, AdditionalInfo=RemoteUrl,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Endpoint logons from remote IP ────────────────────────────────────────
let DLE = DeviceLogonEvents
| where {TIME_RANGE}
| where isnotempty(RemoteIP) and LogonType in (3, 10)
| join kind=inner ThreatIPs on $left.RemoteIP == $right.IOC_IP
| project TimeGenerated, Source="DeviceLogonEvents",
    Actor=AccountName, Host=DeviceName,
    MatchedIP=RemoteIP, AdditionalInfo=strcat("LogonType:", tostring(LogonType)),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Interactive sign-ins ───────────────────────────────────────────────────
let SL = SigninLogs
| where {TIME_RANGE}
| where isnotempty(IPAddress)
| join kind=inner ThreatIPs on $left.IPAddress == $right.IOC_IP
| project TimeGenerated, Source="SigninLogs",
    Actor=UserPrincipalName, Host=tostring(DeviceDetail.displayName),
    MatchedIP=IPAddress, AdditionalInfo=ResultDescription,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Non-interactive sign-ins ───────────────────────────────────────────────
let NISL = AADNonInteractiveUserSignInLogs
| where {TIME_RANGE}
| where isnotempty(IPAddress)
| join kind=inner ThreatIPs on $left.IPAddress == $right.IOC_IP
| project TimeGenerated, Source="AADNonInteractiveSignIn",
    Actor=UserPrincipalName, Host=tostring(""),
    MatchedIP=IPAddress, AdditionalInfo=AppDisplayName,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Service principal sign-ins ─────────────────────────────────────────────
let SPSL = AADServicePrincipalSignInLogs
| where {TIME_RANGE}
| where isnotempty(IPAddress)
| join kind=inner ThreatIPs on $left.IPAddress == $right.IOC_IP
| project TimeGenerated, Source="SPSignInLogs",
    Actor=ServicePrincipalName, Host=tostring(""),
    MatchedIP=IPAddress, AdditionalInfo=ResourceDisplayName,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Azure Audit Logs ───────────────────────────────────────────────────────
let AL = AuditLogs
| where {TIME_RANGE}
| extend ActorIP = tostring(InitiatedBy.user.ipAddress)
| where isnotempty(ActorIP) and ActorIP != "<null>"
| join kind=inner ThreatIPs on $left.ActorIP == $right.IOC_IP
| project TimeGenerated, Source="AuditLogs",
    Actor=tostring(InitiatedBy.user.userPrincipalName), Host=tostring(""),
    MatchedIP=ActorIP, AdditionalInfo=OperationName,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Azure Activity (control-plane) ────────────────────────────────────────
let AA = AzureActivity
| where {TIME_RANGE}
| where isnotempty(CallerIpAddress)
| join kind=inner ThreatIPs on $left.CallerIpAddress == $right.IOC_IP
| project TimeGenerated, Source="AzureActivity",
    Actor=Caller, Host=ResourceGroup,
    MatchedIP=CallerIpAddress, AdditionalInfo=OperationNameValue,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Defender for Identity logons ───────────────────────────────────────────
let ILE = IdentityLogonEvents
| where {TIME_RANGE}
| where isnotempty(IPAddress)
| join kind=inner ThreatIPs on $left.IPAddress == $right.IOC_IP
| project TimeGenerated, Source="IdentityLogonEvents",
    Actor=AccountName, Host=DeviceName,
    MatchedIP=IPAddress, AdditionalInfo=strcat(Protocol, " / ", ActionType),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Defender for Identity directory events ─────────────────────────────────
let IDE = IdentityDirectoryEvents
| where {TIME_RANGE}
| where isnotempty(IPAddress)
| join kind=inner ThreatIPs on $left.IPAddress == $right.IOC_IP
| project TimeGenerated, Source="IdentityDirectoryEvents",
    Actor=AccountName, Host=TargetDeviceName,
    MatchedIP=IPAddress, AdditionalInfo=ActionType,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Cloud App Events ────────────────────────────────────────────────────────
let CAE = CloudAppEvents
| where {TIME_RANGE}
| where isnotempty(IPAddress)
| join kind=inner ThreatIPs on $left.IPAddress == $right.IOC_IP
| project TimeGenerated, Source="CloudAppEvents",
    Actor=AccountDisplayName, Host=Application,
    MatchedIP=IPAddress, AdditionalInfo=ActionType,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Security Alerts with embedded IP entities ──────────────────────────────
let SA = SecurityAlert
| where {TIME_RANGE}
| mv-expand Entity = todynamic(Entities)
| where tostring(Entity.Type) == "ip"
| extend AlertIP = tostring(Entity.Address)
| where isnotempty(AlertIP)
| join kind=inner ThreatIPs on $left.AlertIP == $right.IOC_IP
| project TimeGenerated, Source="SecurityAlert",
    Actor=Tactics, Host=CompromisedEntity,
    MatchedIP=AlertIP, AdditionalInfo=AlertName,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── CommonSecurityLog — firewall / proxy / network device traffic ──────────
// Checks both SourceIP (inbound) and DestinationIP (outbound) against TI.
// DeviceVendor + DeviceProduct identify the originating security appliance.
let CSL_Src = CommonSecurityLog
| where {TIME_RANGE}
| where isnotempty(SourceIP)
| join kind=inner ThreatIPs on $left.SourceIP == $right.IOC_IP
| project TimeGenerated, Source="CommonSecurityLog-SourceIP",
    Actor=SourceUserName,
    Host=strcat(DeviceVendor, " / ", DeviceProduct),
    MatchedIP=SourceIP,
    AdditionalInfo=strcat("Dest:", DestinationIP, " Port:", tostring(DestinationPort),
                          " Action:", DeviceAction),
    ThreatTypes, MaxConfidence, ThreatDesc;
let CSL_Dst = CommonSecurityLog
| where {TIME_RANGE}
| where isnotempty(DestinationIP)
| join kind=inner ThreatIPs on $left.DestinationIP == $right.IOC_IP
| project TimeGenerated, Source="CommonSecurityLog-DestIP",
    Actor=SourceUserName,
    Host=strcat(DeviceVendor, " / ", DeviceProduct),
    MatchedIP=DestinationIP,
    AdditionalInfo=strcat("Src:", SourceIP, " Port:", tostring(DestinationPort),
                          " Action:", DeviceAction),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Union all sources ──────────────────────────────────────────────────────
union isfuzzy=true Net, DLE, SL, NISL, SPSL, AL, AA, ILE, IDE, CAE, SA, CSL_Src, CSL_Dst
| extend AlertSeverity = case(
    MaxConfidence >= 80, "High",
    MaxConfidence >= 60, "Medium",
    "Low"
)
| order by MaxConfidence desc, TimeGenerated desc
| take 300
"""

network_ioc_matches = run_kql(ip_ioc_query, "7.1 IP IOC — scanning all 13 data sources (incl. CommonSecurityLog)")

if not network_ioc_matches.empty:
    total = len(network_ioc_matches)
    print(f"\n  Found {total} IP IOC matches across all data sources")
    import plotly.express as px
    if "Source" in network_ioc_matches.columns:
        by_source = network_ioc_matches["Source"].value_counts().reset_index()
        by_source.columns = ["Source", "Hits"]
        fig = px.bar(by_source, x="Source", y="Hits", color="Hits",
                     color_continuous_scale="Reds",
                     title="7.1 — IP IOC Matches by Data Source")
        fig.update_layout(xaxis_tickangle=-30)
        fig.show()
    display(network_ioc_matches.head(30))
else:
    print("  No IP IOC matches found across any data source")


🔍 7.1 IP IOC — scanning all 13 data sources (incl. CommonSecurityLog)...
   ✅ Returned 0 rows
  No IP IOC matches found across any data source


In [33]:

# ── 7.2  IOC Matching — File Hash ─────────────────────────────────────────────
# Matches TI file hashes against:
#   DeviceFileEvents     — file create / modify / delete on endpoints
#   DeviceProcessEvents  — process image execution (SHA256)
#   EmailAttachmentInfo  — malicious attachment delivered via email
#   SecurityAlert        — file entities embedded in alert records

file_ioc_query = f"""
let ThreatHashes = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(FileHashValue)
| where ConfidenceScore >= 50
| summarize ThreatTypes   = make_set(ThreatType, 5),
            MaxConfidence = max(ConfidenceScore),
            ThreatDesc    = any(Description)
  by FileHashValue;
// ── Device file events (create / write / rename) ───────────────────────────
let DFE = DeviceFileEvents
| where {TIME_RANGE}
| where isnotempty(SHA256)
| join kind=inner ThreatHashes on $left.SHA256 == $right.FileHashValue
| project TimeGenerated, Source="DeviceFileEvents",
    Host=DeviceName, Actor=InitiatingProcessAccountName,
    MatchedHash=SHA256, FileName, AdditionalInfo=FolderPath,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Process execution (image hash) ────────────────────────────────────────
let DPE = DeviceProcessEvents
| where {TIME_RANGE}
| where isnotempty(SHA256)
| join kind=inner ThreatHashes on $left.SHA256 == $right.FileHashValue
| project TimeGenerated, Source="DeviceProcessEvents",
    Host=DeviceName, Actor=AccountName,
    MatchedHash=SHA256, FileName, AdditionalInfo=ProcessCommandLine,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Email attachment hash ──────────────────────────────────────────────────
let EAI = EmailAttachmentInfo
| where {TIME_RANGE}
| where isnotempty(SHA256)
| join kind=inner ThreatHashes on $left.SHA256 == $right.FileHashValue
| join kind=leftouter (
    EmailEvents | where {TIME_RANGE}
    | project NetworkMessageId, RecipientEmailAddress,
              SenderFromAddress, Subject, DeliveryAction
  ) on NetworkMessageId
| project TimeGenerated, Source="EmailAttachmentInfo",
    Host=RecipientEmailAddress, Actor=SenderFromAddress,
    MatchedHash=SHA256, FileName, AdditionalInfo=strcat("Subject:", Subject,
                                                         " Delivery:", DeliveryAction),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── SecurityAlert — embedded file entities ────────────────────────────────
let SA = SecurityAlert
| where {TIME_RANGE}
| mv-expand Entity = todynamic(Entities)
| where tostring(Entity.Type) == "file"
| extend FileHash256 = tostring(Entity.Hashes["SHA256"])
| where isnotempty(FileHash256)
| join kind=inner ThreatHashes on $left.FileHash256 == $right.FileHashValue
| project TimeGenerated, Source="SecurityAlert-FileEntity",
    Host=CompromisedEntity, Actor=Tactics,
    MatchedHash=FileHash256, FileName=tostring(Entity.Name),
    AdditionalInfo=AlertName,
    ThreatTypes, MaxConfidence, ThreatDesc;
union isfuzzy=true DFE, DPE, EAI, SA
| extend AlertSeverity = case(
    MaxConfidence >= 80, "High",
    MaxConfidence >= 60, "Medium",
    "Low"
)
| order by MaxConfidence desc, TimeGenerated desc
| take 300
"""

file_ioc_matches = run_kql(file_ioc_query,
    "7.2 File Hash IOC — DeviceFileEvents + DeviceProcessEvents + EmailAttachmentInfo + SecurityAlert")

if not file_ioc_matches.empty:
    print(f"\n  Found {len(file_ioc_matches)} file hash IOC matches")
    import plotly.express as px
    if "Source" in file_ioc_matches.columns:
        src = file_ioc_matches["Source"].value_counts().reset_index()
        src.columns = ["Source", "Hits"]
        px.bar(src, x="Source", y="Hits", color="Hits",
               color_continuous_scale="Reds",
               title="7.2 — File Hash IOC Matches by Data Source"
        ).show()
    if "FileName" in file_ioc_matches.columns:
        top_f = file_ioc_matches["FileName"].value_counts().head(15).reset_index()
        top_f.columns = ["FileName", "Hits"]
        px.bar(top_f, x="FileName", y="Hits", color="Hits",
               color_continuous_scale="Oranges",
               title="7.2 — Top Malicious Files Detected"
        ).update_layout(xaxis_tickangle=-30).show()
    display(file_ioc_matches.head(30))
else:
    print("  No file hash IOC matches found")


🔍 7.2 File Hash IOC — DeviceFileEvents + DeviceProcessEvents + EmailAttachmentInfo + SecurityAlert...
   ✅ Returned 0 rows
  No file hash IOC matches found


In [ ]:

# ── 7.5  IOC Matching — Process / Command-line ────────────────────────────────
# (a) DeviceProcessEvents: image SHA256 vs TI file hashes
# (b) DeviceProcessEvents: known LOLBin / abuse command-line patterns
# (c) SecurityAlert: process-related entities from all alert providers

process_ioc_query = f"""
// Schema anchor
let _Empty = datatable(
    TimeGenerated:datetime, Source:string, Host:string, Actor:string,
    FileName:string, ProcessCommandLine:string, FileHash:string,
    ThreatTypes:dynamic, MaxConfidence:long, ThreatDesc:string, MatchType:string)[];
// ── (a) Process image SHA256 vs TI hashes ──────────────────────────────────
let ThreatHashes = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(FileHashValue)
| where tolong(column_ifexists("ConfidenceScore", long(0))) >= 50
| summarize ThreatTypes   = make_set(ThreatType, 5),
            MaxConfidence = max(tolong(column_ifexists("ConfidenceScore", long(0)))),
            ThreatDesc    = any(Description)
  by FileHashValue;
let HashHits = DeviceProcessEvents
| where {TIME_RANGE}
| where isnotempty(SHA256)
| join kind=inner ThreatHashes on $left.SHA256 == $right.FileHashValue
| project TimeGenerated, Source="DeviceProcessEvents-Hash",
    Host=DeviceName, Actor=AccountName,
    FileName, ProcessCommandLine, FileHash=SHA256,
    ThreatTypes, MaxConfidence, ThreatDesc, MatchType="Process Hash - TI Match";
// ── (b) LOLBin / abuse command-line patterns ──────────────────────────────
let LOLBins = DeviceProcessEvents
| where {TIME_RANGE}
| where ProcessCommandLine has_any (
    "powershell -enc", "powershell -e ", "-ExecutionPolicy Bypass", "-nop -w hidden",
    "certutil -urlcache", "certutil -decode", "bitsadmin /transfer",
    "mshta http", "regsvr32 /s /u /i:", "wmic process call create",
    "rundll32.exe javascript:", "cmstp.exe /ni", "installutil.exe",
    "net user /add", "net localgroup administrators /add",
    "schtasks /create", "odbcconf.exe /s /a", "ieexec.exe http",
    "xwizard.exe", "appsyncpublishingserver.exe", "pcalua.exe -a",
    "syncappvpublishingserver.exe", "ftp.exe -s:", "nltest /domain_trusts",
    "dsquery * -filter", "BloodHound", "SharpHound", "mimikatz", "lsadump"
  )
| project TimeGenerated, Source="DeviceProcessEvents-LOLBin",
    Host=DeviceName, Actor=AccountName,
    FileName, ProcessCommandLine, FileHash=SHA256,
    ThreatTypes=dynamic(["LOLBin/Abuse"]),
    MaxConfidence=long(70),
    ThreatDesc="Suspicious command-line technique matched",
    MatchType="LOLBin / Abuse Pattern";
// ── (c) SecurityAlert process entities ────────────────────────────────────
let SAProc = SecurityAlert
| where {TIME_RANGE}
| where AlertSeverity in ("High","Medium")
| mv-expand Entity = todynamic(Entities)
| where tostring(Entity.Type) == "process"
| extend ProcCmdLine = tostring(Entity.commandLine)
| project TimeGenerated, Source="SecurityAlert-ProcessEntity",
    Host=CompromisedEntity, Actor=Tactics,
    FileName=AlertName, ProcessCommandLine=ProcCmdLine, FileHash="",
    ThreatTypes=dynamic(["Alert"]),
    MaxConfidence=iff(AlertSeverity=="High", long(80), long(65)),
    ThreatDesc=Description,
    MatchType=strcat("Alert: ", AlertName);
union isfuzzy=true HashHits, LOLBins, SAProc, _Empty
| extend MaxConfidence = tolong(column_ifexists("MaxConfidence", long(0)))
| order by MaxConfidence desc, TimeGenerated desc
| take 300
"""

process_ioc_matches = run_kql(process_ioc_query, "7.5 Process IOC — hash + LOLBin + SecurityAlert entities")

if not process_ioc_matches.empty:
    print(f"\n  Found {len(process_ioc_matches)} process IOC / suspicious command-line matches")
    import plotly.express as px
    if "MatchType" in process_ioc_matches.columns:
        mt = process_ioc_matches["MatchType"].apply(lambda x: x.split(" - ")[0] if isinstance(x, str) else x).value_counts().reset_index()
        mt.columns = ["MatchType", "Count"]
        px.bar(mt, x="MatchType", y="Count",
               title="7.5 — Process IOC Matches by Type",
               color="Count", color_continuous_scale="Reds"
        ).update_layout(xaxis_tickangle=-30).show()
    display(process_ioc_matches.head(30))
else:
    print("  No process IOC / LOLBin matches found")


🔍 7.5 Process IOC — hash + LOLBin + SecurityAlert entities...


In [ ]:

# ── 7.3  IOC Matching — Domain / Hostname ─────────────────────────────────────
# Covers: device DNS/network lookups, Defender for Identity LDAP/DNS queries,
#         email sender domains, Azure audit target resources,
#         CommonSecurityLog (proxy / NGFW destination hostname), and
#         Defender for Office 365 (EmailUrlInfo, UrlClickEvents, EmailAttachmentInfo).

domain_ioc_query = f"""
let ThreatDomains = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(DomainName)
| where ConfidenceScore >= 50
| summarize ThreatTypes   = make_set(ThreatType, 5),
            MaxConfidence = max(ConfidenceScore),
            ThreatDesc    = any(Description)
  by DomainName;
// ── Endpoint DNS / network contacts ───────────────────────────────────────
let DNE = DeviceNetworkEvents
| where {TIME_RANGE}
| where isnotempty(RemoteUrl)
| extend ExtractedDomain = tostring(split(trim_start(@"https?://", RemoteUrl), "/")[0])
| join kind=inner ThreatDomains on $left.ExtractedDomain == $right.DomainName
| project TimeGenerated, Source="DeviceNetworkEvents",
    Host=DeviceName, Actor=InitiatingProcessAccountName,
    MatchedDomain=ExtractedDomain, AdditionalInfo=RemoteUrl,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Defender for Identity DNS/LDAP queries ─────────────────────────────────
let IQE = IdentityQueryEvents
| where {TIME_RANGE}
| where QueryType in ("Dns", "Ldap")
| where isnotempty(QueryTarget)
| join kind=inner ThreatDomains on $left.QueryTarget == $right.DomainName
| project TimeGenerated, Source="IdentityQueryEvents",
    Host=DeviceName, Actor=AccountName,
    MatchedDomain=QueryTarget, AdditionalInfo=strcat(QueryType, " query"),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── MDO EmailEvents — sender domain ────────────────────────────────────────
let EMail = EmailEvents
| where {TIME_RANGE}
| where isnotempty(SenderFromAddress)
| extend SenderDomain = tostring(split(SenderFromAddress, "@")[1])
| where isnotempty(SenderDomain)
| join kind=inner ThreatDomains on $left.SenderDomain == $right.DomainName
| project TimeGenerated, Source="EmailEvents-SenderDomain",
    Host=RecipientEmailAddress, Actor=SenderFromAddress,
    MatchedDomain=SenderDomain, AdditionalInfo=Subject,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── MDO EmailUrlInfo — domain extracted from body links ───────────────────
// Matches the registrable domain of every URL embedded in delivered emails.
let EUI = EmailUrlInfo
| where {TIME_RANGE}
| where isnotempty(Url)
| extend UrlDomain = tostring(split(trim_start(@"https?://", Url), "/")[0])
| where isnotempty(UrlDomain)
| join kind=inner ThreatDomains on $left.UrlDomain == $right.DomainName
| join kind=leftouter (
    EmailEvents | where {TIME_RANGE}
    | project NetworkMessageId, RecipientEmailAddress, SenderFromAddress, Subject, DeliveryAction
  ) on NetworkMessageId
| project TimeGenerated, Source="EmailUrlInfo-LinkDomain",
    Host=RecipientEmailAddress, Actor=SenderFromAddress,
    MatchedDomain=UrlDomain, AdditionalInfo=strcat("URL:", Url, " Delivery:", DeliveryAction),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── MDO UrlClickEvents — Safe Links click — domain ────────────────────────
// Triggered when a user clicks a link; captures the real destination domain
// even after link rewriting by Safe Links.
let SLC = UrlClickEvents
| where {TIME_RANGE}
| where isnotempty(Url)
| extend ClickDomain = tostring(split(trim_start(@"https?://", Url), "/")[0])
| where isnotempty(ClickDomain)
| join kind=inner ThreatDomains on $left.ClickDomain == $right.DomainName
| project TimeGenerated, Source="UrlClickEvents-ClickDomain",
    Host=tostring(""), Actor=AccountUpn,
    MatchedDomain=ClickDomain, AdditionalInfo=strcat("URL:", Url, " IsClickedThrough:", tostring(IsClickedThrough)),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── MDO EmailAttachmentInfo — sender domain of attachment emails ───────────
// Catches malicious-domain senders that delivered attachments (phishing kits,
// malware droppers) — sender domain may differ from From: address domain.
let EAI = EmailAttachmentInfo
| where {TIME_RANGE}
| where isnotempty(SenderFromAddress)
| extend AttachSenderDomain = tostring(split(SenderFromAddress, "@")[1])
| where isnotempty(AttachSenderDomain)
| join kind=inner ThreatDomains on $left.AttachSenderDomain == $right.DomainName
| project TimeGenerated, Source="EmailAttachmentInfo-SenderDomain",
    Host=RecipientEmailAddress, Actor=SenderFromAddress,
    MatchedDomain=AttachSenderDomain, AdditionalInfo=strcat("File:", FileName, " SHA256:", SHA256),
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── Azure AD sign-in spoofed domains ──────────────────────────────────────
let SLDom = SigninLogs
| where {TIME_RANGE}
| where isnotempty(UserPrincipalName)
| extend UPNDomain = tostring(split(UserPrincipalName, "@")[1])
| join kind=inner ThreatDomains on $left.UPNDomain == $right.DomainName
| project TimeGenerated, Source="SigninLogs-UPNDomain",
    Host=tostring(""), Actor=UserPrincipalName,
    MatchedDomain=UPNDomain, AdditionalInfo=AppDisplayName,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── AuditLogs resource target domains ─────────────────────────────────────
let ALDom = AuditLogs
| where {TIME_RANGE}
| extend TargetDomain = tostring(parse_json(tostring(TargetResources))[0].displayName)
| where isnotempty(TargetDomain)
| join kind=inner ThreatDomains on $left.TargetDomain == $right.DomainName
| project TimeGenerated, Source="AuditLogs-TargetResource",
    Host=tostring(""), Actor=tostring(InitiatedBy.user.userPrincipalName),
    MatchedDomain=TargetDomain, AdditionalInfo=OperationName,
    ThreatTypes, MaxConfidence, ThreatDesc;
// ── CommonSecurityLog — proxy / NGFW destination hostname ─────────────────
// Firewalls and proxies log DestinationHostName for DNS-resolved connections.
// Covers Palo Alto, Fortinet, Check Point, Cisco ASA, Zscaler, etc.
let CSL_Domain = CommonSecurityLog
| where {TIME_RANGE}
| where isnotempty(DestinationHostName)
| join kind=inner ThreatDomains on $left.DestinationHostName == $right.DomainName
| project TimeGenerated, Source="CommonSecurityLog-DestHostname",
    Host=strcat(DeviceVendor, " / ", DeviceProduct),
    Actor=SourceUserName,
    MatchedDomain=DestinationHostName,
    AdditionalInfo=strcat("DestIP:", DestinationIP,
                          " Port:", tostring(DestinationPort),
                          " Action:", DeviceAction),
    ThreatTypes, MaxConfidence, ThreatDesc;
union isfuzzy=true DNE, IQE, EMail, EUI, SLC, EAI, SLDom, ALDom, CSL_Domain
| extend AlertSeverity = case(
    MaxConfidence >= 80, "High",
    MaxConfidence >= 60, "Medium",
    "Low"
)
| order by MaxConfidence desc, TimeGenerated desc
| take 300
"""

domain_ioc_matches = run_kql(domain_ioc_query,
    "7.3 Domain IOC — 9 sources (DeviceNetworkEvents + IdentityQueryEvents + EmailEvents + EmailUrlInfo + UrlClickEvents + EmailAttachmentInfo + SigninLogs + AuditLogs + CommonSecurityLog)")

if not domain_ioc_matches.empty:
    print(f"\n  Found {len(domain_ioc_matches)} malicious domain matches across all data sources")
    import plotly.express as px
    if "Source" in domain_ioc_matches.columns:
        src = domain_ioc_matches["Source"].value_counts().reset_index()
        src.columns = ["Source", "Hits"]
        px.bar(src, x="Source", y="Hits", color="Hits", color_continuous_scale="Reds",
               title="7.3 — Domain IOC Matches by Data Source"
        ).update_layout(xaxis_tickangle=-30).show()
    if "MatchedDomain" in domain_ioc_matches.columns:
        top_d = domain_ioc_matches["MatchedDomain"].value_counts().head(15).reset_index()
        top_d.columns = ["Domain", "Hits"]
        px.bar(top_d, x="Domain", y="Hits", color="Hits", color_continuous_scale="Oranges",
               title="7.3 — Top Malicious Domains Contacted"
        ).update_layout(xaxis_tickangle=-30).show()
    display(domain_ioc_matches.head(30))
else:
    print("  No malicious domain IOC matches found")


🔍 7.3 Domain IOC — DeviceNetworkEvents + IdentityQueryEvents + EmailEvents + SigninLogs + CommonSecurityLog...
   ✅ Returned 0 rows
  No malicious domain IOC matches found


In [ ]:

# ── 7.4  IOC Matching — URL ────────────────────────────────────────────────────
# Covers: device web requests, MDO Safe Links clicks, email embedded URLs,
#         CloudAppEvents (Teams / SharePoint link shares), and
#         CommonSecurityLog (proxy full URL from firewalls / NGFW / web gateways).

url_ioc_query = f"""
let ThreatUrls = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(Url)
| where ConfidenceScore >= 50
| summarize ThreatTypes   = make_set(ThreatType, 5),
            MaxConfidence = max(ConfidenceScore),
            ThreatDesc    = any(Description)
  by Url;
// ── Endpoint web requests ─────────────────────────────────────────────────
let DNE_URL = DeviceNetworkEvents
| where {TIME_RANGE}
| where isnotempty(RemoteUrl)
| join kind=inner ThreatUrls on $left.RemoteUrl == $right.Url
| project TimeGenerated, Source="DeviceNetworkEvents",
    Host=DeviceName, Actor=InitiatingProcessAccountName,
    MatchedUrl=RemoteUrl, ThreatTypes, MaxConfidence, ThreatDesc;
// ── MDO Safe Links clicks ─────────────────────────────────────────────────
let SLC = UrlClickEvents
| where {TIME_RANGE}
| where isnotempty(Url)
| join kind=inner ThreatUrls on $left.Url == $right.Url
| project TimeGenerated, Source="UrlClickEvents (MDO)",
    Host=tostring(""), Actor=AccountUpn,
    MatchedUrl=Url, ThreatTypes, MaxConfidence, ThreatDesc;
// ── Email embedded URLs ────────────────────────────────────────────────────
let EML_URL = EmailUrlInfo
| where {TIME_RANGE}
| where isnotempty(Url)
| join kind=inner ThreatUrls on $left.Url == $right.Url
| join kind=leftouter (
    EmailEvents | where {TIME_RANGE}
    | project NetworkMessageId, Recipient=RecipientEmailAddress, Subject
  ) on NetworkMessageId
| project TimeGenerated, Source="EmailUrlInfo",
    Host=tostring("Mail"), Actor=Recipient,
    MatchedUrl=Url, ThreatTypes, MaxConfidence, ThreatDesc;
// ── CloudAppEvents (Teams / SharePoint link shares) ───────────────────────
let CAE_URL = CloudAppEvents
| where {TIME_RANGE}
| extend EventUrl = tostring(todynamic(RawEventData).Url)
| where isnotempty(EventUrl)
| join kind=inner ThreatUrls on $left.EventUrl == $right.Url
| project TimeGenerated, Source="CloudAppEvents",
    Host=Application, Actor=AccountDisplayName,
    MatchedUrl=EventUrl, ThreatTypes, MaxConfidence, ThreatDesc;
// ── CommonSecurityLog — proxy / web gateway full URL match ────────────────
// Forward proxies (Zscaler, Squid, BlueCoat, McAfee Web Gateway, etc.) and
// NGFW with App-ID log RequestURL containing the full request URL.
let CSL_URL = CommonSecurityLog
| where {TIME_RANGE}
| where isnotempty(RequestURL)
| join kind=inner ThreatUrls on $left.RequestURL == $right.Url
| project TimeGenerated, Source="CommonSecurityLog-ProxyURL",
    Host=strcat(DeviceVendor, " / ", DeviceProduct),
    Actor=SourceUserName,
    MatchedUrl=RequestURL,
    ThreatTypes, MaxConfidence, ThreatDesc;
union isfuzzy=true DNE_URL, SLC, EML_URL, CAE_URL, CSL_URL
| extend AlertSeverity = case(
    MaxConfidence >= 80, "High",
    MaxConfidence >= 60, "Medium",
    "Low"
)
| order by MaxConfidence desc, TimeGenerated desc
| take 200
"""

url_ioc_matches = run_kql(url_ioc_query,
    "7.4 URL IOC — DeviceNetworkEvents + UrlClickEvents + EmailUrlInfo + CloudAppEvents + CommonSecurityLog")

if not url_ioc_matches.empty:
    print(f"\n  Found {len(url_ioc_matches)} malicious URL matches")
    import plotly.express as px
    if "Source" in url_ioc_matches.columns:
        src = url_ioc_matches["Source"].value_counts().reset_index()
        src.columns = ["Source", "Hits"]
        px.pie(src, values="Hits", names="Source",
               title="7.4 — URL IOC Hits by Source").show()
    display(url_ioc_matches.head(30))
else:
    print("  No malicious URL IOC matches found")


🔍 7.4 URL IOC — DeviceNetworkEvents + UrlClickEvents + EmailUrlInfo + CloudAppEvents + CommonSecurityLog...
   ✅ Returned 0 rows
  No malicious URL IOC matches found


In [ ]:

# ── 7.5  IOC Matching — Process / Command-line ────────────────────────────────
# (a) DeviceProcessEvents: image SHA256 vs TI file hashes
# (b) DeviceProcessEvents: known LOLBin / abuse command-line patterns
# (c) SecurityAlert: process-related entities from all alert providers

process_ioc_query = f"""
// Schema anchor
let _Empty = datatable(
    TimeGenerated:datetime, Source:string, Host:string, Actor:string,
    FileName:string, ProcessCommandLine:string, FileHash:string,
    ThreatTypes:dynamic, MaxConfidence:long, ThreatDesc:string, MatchType:string)[];
// ── (a) Process image SHA256 vs TI hashes ──────────────────────────────────
let ThreatHashes = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(FileHashValue)
| where tolong(column_ifexists("ConfidenceScore", long(0))) >= 50
| summarize ThreatTypes   = make_set(ThreatType, 5),
            MaxConfidence = max(tolong(column_ifexists("ConfidenceScore", long(0)))),
            ThreatDesc    = any(Description)
  by FileHashValue;
let HashHits = DeviceProcessEvents
| where {TIME_RANGE}
| where isnotempty(SHA256)
| join kind=inner ThreatHashes on $left.SHA256 == $right.FileHashValue
| project TimeGenerated, Source="DeviceProcessEvents-Hash",
    Host=DeviceName, Actor=AccountName,
    FileName, ProcessCommandLine, FileHash=SHA256,
    ThreatTypes, MaxConfidence, ThreatDesc, MatchType="Process Hash - TI Match";
// ── (b) LOLBin / abuse command-line patterns ──────────────────────────────
let LOLBins = DeviceProcessEvents
| where {TIME_RANGE}
| where ProcessCommandLine has_any (
    "powershell -enc", "powershell -e ", "-ExecutionPolicy Bypass", "-nop -w hidden",
    "certutil -urlcache", "certutil -decode", "bitsadmin /transfer",
    "mshta http", "regsvr32 /s /u /i:", "wmic process call create",
    "rundll32.exe javascript:", "cmstp.exe /ni", "installutil.exe",
    "net user /add", "net localgroup administrators /add",
    "schtasks /create", "odbcconf.exe /s /a", "ieexec.exe http",
    "xwizard.exe", "appsyncpublishingserver.exe", "pcalua.exe -a",
    "syncappvpublishingserver.exe", "ftp.exe -s:", "nltest /domain_trusts",
    "dsquery * -filter", "BloodHound", "SharpHound", "mimikatz", "lsadump"
  )
| project TimeGenerated, Source="DeviceProcessEvents-LOLBin",
    Host=DeviceName, Actor=AccountName,
    FileName, ProcessCommandLine, FileHash=SHA256,
    ThreatTypes=dynamic(["LOLBin/Abuse"]),
    MaxConfidence=long(70),
    ThreatDesc="Suspicious command-line technique matched",
    MatchType="LOLBin / Abuse Pattern";
// ── (c) SecurityAlert process entities ────────────────────────────────────
let SAProc = SecurityAlert
| where {TIME_RANGE}
| where AlertSeverity in ("High","Medium")
| mv-expand Entity = todynamic(Entities)
| where tostring(Entity.Type) == "process"
| extend ProcCmdLine = tostring(Entity.commandLine)
| project TimeGenerated, Source="SecurityAlert-ProcessEntity",
    Host=CompromisedEntity, Actor=Tactics,
    FileName=AlertName, ProcessCommandLine=ProcCmdLine, FileHash="",
    ThreatTypes=dynamic(["Alert"]),
    MaxConfidence=iff(AlertSeverity=="High", long(80), long(65)),
    ThreatDesc=Description,
    MatchType=strcat("Alert: ", AlertName);
union isfuzzy=true HashHits, LOLBins, SAProc, _Empty
| extend MaxConfidence = tolong(column_ifexists("MaxConfidence", long(0)))
| order by MaxConfidence desc, TimeGenerated desc
| take 300
"""

process_ioc_matches = run_kql(process_ioc_query, "7.5 Process IOC — hash + LOLBin + SecurityAlert entities")

if not process_ioc_matches.empty:
    print(f"\n  Found {len(process_ioc_matches)} process IOC / suspicious command-line matches")
    import plotly.express as px
    if "MatchType" in process_ioc_matches.columns:
        mt = process_ioc_matches["MatchType"].apply(lambda x: x.split(" - ")[0] if isinstance(x, str) else x).value_counts().reset_index()
        mt.columns = ["MatchType", "Count"]
        px.bar(mt, x="MatchType", y="Count",
               title="7.5 — Process IOC Matches by Type",
               color="Count", color_continuous_scale="Reds"
        ).update_layout(xaxis_tickangle=-30).show()
    display(process_ioc_matches.head(30))
else:
    print("  No process IOC / LOLBin matches found")


🔍 7.5 Process IOC — hash + LOLBin + SecurityAlert entities...
   ✅ Returned 300 rows

  Found 300 process IOC / suspicious command-line matches


,TimeGenerated,Source,Host,Actor,FileName,ProcessCommandLine,FileHash,ThreatTypes,MaxConfidence,ThreatDesc,MatchType
0,2026-02-23 12:49:56.016377+00:00,SecurityAlert-ProcessEntity,aaronb-pc.zava-corp.com,LateralMovement,Hands-on-keyboard attack involving multiple de...,,,"[""Alert""]",80,A suspicious process was created on a remote d...,Alert: Hands-on-keyboard attack involving mult...
1,2026-02-23 12:49:56.016377+00:00,SecurityAlert-ProcessEntity,aaronb-pc.zava-corp.com,LateralMovement,Hands-on-keyboard attack involving multiple de...,,,"[""Alert""]",80,A suspicious process was created on a remote d...,Alert: Hands-on-keyboard attack involving mult...
2,2026-02-23 12:49:54.975544+00:00,SecurityAlert-ProcessEntity,rayt-pc.zava-corp.com,CredentialAccess,Malicious credential theft tool execution dete...,,,"[""Alert""]",80,A known credential theft tool execution comman...,Alert: Malicious credential theft tool executi...
3,2026-02-23 12:49:54.975544+00:00,SecurityAlert-ProcessEntity,rayt-pc.zava-corp.com,CredentialAccess,Malicious credential theft tool execution dete...,,,"[""Alert""]",80,A known credential theft tool execution comman...,Alert: Malicious credential theft tool executi...
4,2026-02-23 12:49:54.975544+00:00,SecurityAlert-ProcessEntity,rayt-pc.zava-corp.com,CredentialAccess,Malicious credential theft tool execution dete...,,,"[""Alert""]",80,A known credential theft tool execution comman...,Alert: Malicious credential theft tool executi...
5,2026-02-23 12:49:53.947226+00:00,SecurityAlert-ProcessEntity,rayt-pc.zava-corp.com,CredentialAccess,Malicious credential theft tool execution dete...,,,"[""Alert""]",80,A known credential theft tool execution comman...,Alert: Malicious credential theft tool executi...
6,2026-02-23 12:49:53.947226+00:00,SecurityAlert-ProcessEntity,rayt-pc.zava-corp.com,CredentialAccess,Malicious credential theft tool execution dete...,,,"[""Alert""]",80,A known credential theft tool execution comman...,Alert: Malicious credential theft tool executi...
7,2026-02-23 12:49:53.947226+00:00,SecurityAlert-ProcessEntity,rayt-pc.zava-corp.com,CredentialAccess,Malicious credential theft tool execution dete...,,,"[""Alert""]",80,A known credential theft tool execution comman...,Alert: Malicious credential theft tool executi...
8,2026-02-23 12:49:53.947226+00:00,SecurityAlert-ProcessEntity,rayt-pc.zava-corp.com,CredentialAccess,Malicious credential theft tool execution dete...,,,"[""Alert""]",80,A known credential theft tool execution comman...,Alert: Malicious credential theft tool executi...
9,2026-02-23 12:49:53.272400+00:00,SecurityAlert-ProcessEntity,aaronb-pc.zava-corp.com,"InitialAccess, Persistence, PrivilegeEscalatio...",Hands-on-keyboard attack involving multiple de...,,,"[""Alert""]",80,A suspicious process was created on a remote d...,Alert: Hands-on-keyboard attack involving mult...


In [ ]:

# ── 7.6  IOC Matching — Email Sender / Attachment / Collaboration ─────────────
# Sources: EmailEvents (sender IP + address), EmailAttachmentInfo (file hash + sender),
#          CloudAppEvents (Teams / SharePoint file-sharing with malicious senders)

email_ioc_query = f"""
// ── TI IP set ──────────────────────────────────────────────────────────────
let ThreatIPs = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(NetworkIP) or isnotempty(NetworkSourceIP)
| where ConfidenceScore >= 50
| extend IOC_IP = coalesce(NetworkIP, NetworkSourceIP)
| summarize ThreatTypes=make_set(ThreatType,5), MaxConfidence=max(ConfidenceScore),
            ThreatDesc=any(Description) by IOC_IP;
// ── TI Email / Domain set ──────────────────────────────────────────────────
let ThreatAddrs = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(EmailSenderAddress) or isnotempty(DomainName)
| where ConfidenceScore >= 50
| extend IOC_Addr = coalesce(EmailSenderAddress, DomainName)
| summarize ThreatTypes=make_set(ThreatType,5), MaxConfidence=max(ConfidenceScore),
            ThreatDesc=any(Description) by IOC_Addr;
// ── TI File Hash set ───────────────────────────────────────────────────────
let ThreatHashes = ThreatIntelligenceIndicator
| where TimeGenerated > ago(30d) and ExpirationDateTime > now()
| where isnotempty(FileHashValue)
| where ConfidenceScore >= 50
| summarize ThreatTypes=make_set(ThreatType,5), MaxConfidence=max(ConfidenceScore),
            ThreatDesc=any(Description) by FileHashValue;
// ── EmailEvents: sender IP match ──────────────────────────────────────────
let SenderIPHits = EmailEvents
| where {TIME_RANGE}
| where isnotempty(SenderIPv4)
| join kind=inner ThreatIPs on $left.SenderIPv4 == $right.IOC_IP
| project TimeGenerated, Source="EmailEvents-SenderIP",
    Actor=SenderFromAddress, Host=RecipientEmailAddress,
    MatchedValue=SenderIPv4, Subject, DeliveryAction,
    ThreatTypes, MaxConfidence, ThreatDesc, MatchType="Sender IP";
// ── EmailEvents: sender address / domain match ─────────────────────────────
let SenderAddrHits = EmailEvents
| where {TIME_RANGE}
| where isnotempty(SenderFromAddress)
| extend SenderDomain = tostring(split(SenderFromAddress, "@")[1])
| join kind=leftouter ThreatAddrs on $left.SenderFromAddress == $right.IOC_Addr
| join kind=leftouter ThreatAddrs on $left.SenderDomain == $right.IOC_Addr
| where isnotempty(MaxConfidence) or isnotempty(MaxConfidence1)
| extend ThreatTypes   = coalesce(ThreatTypes, ThreatTypes1)
| extend MaxConfidence = coalesce(MaxConfidence, MaxConfidence1)
| extend ThreatDesc    = coalesce(ThreatDesc, ThreatDesc1)
| project TimeGenerated, Source="EmailEvents-SenderAddr",
    Actor=SenderFromAddress, Host=RecipientEmailAddress,
    MatchedValue=SenderFromAddress, Subject, DeliveryAction,
    ThreatTypes, MaxConfidence, ThreatDesc, MatchType="Sender Address/Domain";
// ── EmailAttachmentInfo: attachment hash match ─────────────────────────────
let AttachHashHits = EmailAttachmentInfo
| where {TIME_RANGE}
| where isnotempty(SHA256)
| join kind=inner ThreatHashes on $left.SHA256 == $right.FileHashValue
| join kind=leftouter (
    EmailEvents | where {TIME_RANGE}
    | project NetworkMessageId, RecipientEmailAddress,
              SenderFromAddress, Subject, DeliveryAction
  ) on NetworkMessageId
| project TimeGenerated, Source="EmailAttachmentInfo-Hash",
    Actor=SenderFromAddress, Host=RecipientEmailAddress,
    MatchedValue=SHA256, Subject, DeliveryAction,
    ThreatTypes, MaxConfidence, ThreatDesc, MatchType="Attachment Hash";
// ── CloudAppEvents: Teams/SharePoint messages from TI senders ─────────────
let CAE_Email = CloudAppEvents
| where {TIME_RANGE}
| where Application in ("Microsoft Teams","Microsoft SharePoint Online","Microsoft OneDrive for Business")
| extend SenderEmail = tostring(todynamic(RawEventData).UserId)
| where isnotempty(SenderEmail)
| join kind=inner ThreatAddrs on $left.SenderEmail == $right.IOC_Addr
| project TimeGenerated, Source="CloudAppEvents-CollabSender",
    Actor=SenderEmail, Host=Application,
    MatchedValue=SenderEmail, Subject=ActionType, DeliveryAction=tostring(""),
    ThreatTypes, MaxConfidence, ThreatDesc, MatchType="Collab Platform Sender";
union isfuzzy=true SenderIPHits, SenderAddrHits, AttachHashHits, CAE_Email
| extend AlertSeverity = case(
    MaxConfidence >= 80, "High",
    MaxConfidence >= 60, "Medium",
    "Low"
)
| extend DeliveryRisk = iff(DeliveryAction == "Delivered", "REACHED INBOX", "Blocked/Quarantined")
| order by MaxConfidence desc, TimeGenerated desc
| take 200
"""

email_ioc_matches = run_kql(email_ioc_query, "7.6 Email IOC — EmailEvents + EmailAttachmentInfo + CloudAppEvents")

if not email_ioc_matches.empty:
    print(f"\n  Found {len(email_ioc_matches)} malicious email / collaboration IOC matches")
    import plotly.express as px

    if "DeliveryAction" in email_ioc_matches.columns:
        delivered = email_ioc_matches[email_ioc_matches["DeliveryAction"] == "Delivered"]
        if not delivered.empty:
            print(f"\n  !! {len(delivered)} DELIVERED to inboxes — immediate remediation required:")
            for _, r in delivered.head(10).iterrows():
                print(f"     {r.get('Host','?')} <= {r.get('Actor','?')}  ({r.get('MatchedValue','?')})")

    if "Source" in email_ioc_matches.columns:
        src = email_ioc_matches["Source"].value_counts().reset_index()
        src.columns = ["Source", "Hits"]
        px.bar(src, x="Source", y="Hits", color="Hits", color_continuous_scale="Reds",
               title="7.6 — Email IOC Matches by Source").show()

    display(email_ioc_matches.head(30))
else:
    print("  No malicious email / collaboration IOC matches found")


🔍 7.6 Email IOC — EmailEvents + EmailAttachmentInfo + CloudAppEvents...
   ✅ Returned 0 rows
  No malicious email / collaboration IOC matches found


In [ ]:

# ── 7.7  IOC Summary — All Entity Types ──────────────────────────────────────
import plotly.graph_objects as go
import pandas as pd

def _safe_len(var_name):
    """Return row count of a DataFrame variable, 0 if missing or empty."""
    try:
        df = globals().get(var_name)
        return len(df) if df is not None and not df.empty else 0
    except Exception:
        return 0

_ioc_summary = {
    "7.1 IP Address": _safe_len("network_ioc_matches"),
    "7.2 File Hash":  _safe_len("file_ioc_matches"),
    "7.3 Domain":     _safe_len("domain_ioc_matches"),
    "7.4 URL":        _safe_len("url_ioc_matches"),
    "7.5 Process/LOLBin": _safe_len("process_ioc_matches"),
    "7.6 Email/Collab":   _safe_len("email_ioc_matches"),
}

total_ioc_hits = sum(_ioc_summary.values())

print("=" * 64)
print("  IOC MATCHING — CONSOLIDATED SUMMARY (all tables)")
print("=" * 64)
for entity, count in _ioc_summary.items():
    status = "ALERT" if count > 5 else "HIT" if count > 0 else "CLEAN"
    bar = "#" * min(count, 40) if count > 0 else ""
    print(f"  [{status:5s}]  {entity:<28}  {count:>5} matches  {bar}")
print("-" * 64)
print(f"  {'TOTAL THREAT INTELLIGENCE HITS':<28}  {total_ioc_hits:>5}")
print("=" * 64)

if total_ioc_hits > 0:
    print(f"\n  RISK: {total_ioc_hits} confirmed IOC hits across {sum(v>0 for v in _ioc_summary.values())} of 6 entity types")
    print("\n  RECOMMENDED ACTIONS:")
    actions = {
        "7.1 IP Address":        "Block IPs at NSG / firewall; investigate originating hosts",
        "7.2 File Hash":         "Quarantine devices — initiate EDR full response",
        "7.3 Domain":            "Add to DNS sinkhole / proxy block list",
        "7.4 URL":               "Block in proxy; enforce MDO Safe Links for all users",
        "7.5 Process/LOLBin":    "Review LOLBin executions; check for persistence (tasks, services, registry)",
        "7.6 Email/Collab":      "Block senders in MDO; notify recipients of delivered TI-matched mail",
    }
    for entity, count in _ioc_summary.items():
        if count > 0:
            print(f"    • {entity}: {actions[entity]}")

# ── Source breakdown across ALL IOC results ────────────────────────────────
all_ioc_dfs = [
    ("network_ioc_matches",  "7.1 IP"),
    ("file_ioc_matches",     "7.2 Hash"),
    ("domain_ioc_matches",   "7.3 Domain"),
    ("url_ioc_matches",      "7.4 URL"),
    ("process_ioc_matches",  "7.5 Process"),
    ("email_ioc_matches",    "7.6 Email"),
]

source_rows = []
for var_name, label in all_ioc_dfs:
    df = globals().get(var_name)
    if df is not None and not df.empty and "Source" in df.columns:
        for src, cnt in df["Source"].value_counts().items():
            source_rows.append({"EntityType": label, "DataSource": src, "Hits": cnt})

# ── Charts ────────────────────────────────────────────────────────────────
colors_ioc = ["#d63031" if v > 5 else "#e17055" if v > 0 else "#636e72"
              for v in _ioc_summary.values()]

fig_summary = go.Figure(go.Bar(
    x=list(_ioc_summary.keys()),
    y=list(_ioc_summary.values()),
    marker_color=colors_ioc,
    text=list(_ioc_summary.values()),
    textposition="outside"
))
fig_summary.update_layout(
    title="Threat Intelligence Hits by Entity Type (All Tables)",
    xaxis_title="IOC Entity Type",
    yaxis_title="Total Matches",
    height=420
)
fig_summary.show()

if source_rows:
    import plotly.express as px
    src_df = pd.DataFrame(source_rows)
    fig_src = px.bar(src_df, x="DataSource", y="Hits", color="EntityType",
                     title="IOC Hits by Data Source and Entity Type",
                     barmode="stack")
    fig_src.update_layout(xaxis_tickangle=-35, height=450)
    fig_src.show()


  IOC MATCHING — CONSOLIDATED SUMMARY (all tables)
  [CLEAN]  7.1 IP Address                    0 matches  
  [CLEAN]  7.2 File Hash                     0 matches  
  [CLEAN]  7.3 Domain                        0 matches  
  [CLEAN]  7.4 URL                           0 matches  
  [ALERT]  7.5 Process/LOLBin              300 matches  ########################################
  [CLEAN]  7.6 Email/Collab                  0 matches  
----------------------------------------------------------------
  TOTAL THREAT INTELLIGENCE HITS    300

  RISK: 300 confirmed IOC hits across 1 of 6 entity types

  RECOMMENDED ACTIONS:
    • 7.5 Process/LOLBin: Review LOLBin executions; check for persistence (tasks, services, registry)


In [ ]:

# ── 7.8  Threat Actor & Threat Analytics Intelligence ────────────────────────
import pandas as pd
import plotly.express as px

# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIG — change only this block                            ║
# ║  THREAT_ACTOR_DAYS : lookback window for TI feed queries    ║
# ║    365 = 12 months (default)                                ║
# ║    180 = 6 months  |  90 = 3 months  |  730 = 24 months    ║
# ╚══════════════════════════════════════════════════════════════╝
THREAT_ACTOR_DAYS = 365   # ← change here

_ta_window = f"ago({THREAT_ACTOR_DAYS}d)"
print(f"  Threat actor lookback: {THREAT_ACTOR_DAYS} days ({THREAT_ACTOR_DAYS//30} months)")

# ── (a) TI feed summary — active & ingested in the window ────────────────────
ti_summary_query = f"""
ThreatIntelligenceIndicator
| where TimeGenerated > {_ta_window} and ExpirationDateTime > now()
| extend _type = coalesce(
    tostring(column_ifexists("IndicatorType", "")),
    tostring(column_ifexists("type", "")),
    "Unknown"
)
| summarize
    ActiveIndicators = count(),
    ThreatTypes      = make_set(ThreatType, 20),
    AvgConfidence    = round(avg(ConfidenceScore), 1),
    HighConfCount    = countif(ConfidenceScore >= 75)
  by IndicatorType = _type
| order by ActiveIndicators desc
"""

# ── (b) SecurityAlert: actors & tactics seen in the analysis period ───────────
ta_alerts_query = f"""
SecurityAlert
| where {TIME_RANGE}
| where isnotempty(Tactics)
| summarize
    AlertCount       = count(),
    UniqueAlertNames = dcount(AlertName),
    Entities         = make_set(Entities, 5)
  by Tactics, AlertName, AlertSeverity
| order by AlertSeverity asc, AlertCount desc
| take 100
"""

# ── (c) Threat actor references in TI descriptions — 12-month window ─────────
# Scans free-text Description fields for known APT group names, ransomware
# families, and offensive tooling. Expanded to cover more actor aliases.
ta_actor_query = f"""
ThreatIntelligenceIndicator
| where TimeGenerated > {_ta_window}
| where isnotempty(Description)
| extend
    HasLazarus   = Description has_any ("Lazarus","APT38","BlueNoroff","Guardians of Peace","Hidden Cobra"),
    HasCozy      = Description has_any ("APT29","Cozy Bear","SVR","Midnight Blizzard","Nobelium","The Dukes"),
    HasFancy     = Description has_any ("APT28","Fancy Bear","GRU","Forest Blizzard","Strontium","Sofacy","Pawn Storm"),
    HasLockbit   = Description has_any ("LockBit","ALPHV","BlackCat","ransomware","Cl0p","BlackBasta","Royal","Play","Akira"),
    HasChinaAPT  = Description has_any ("APT41","APT10","Volt Typhoon","Double Dragon","Wicked Panda","Stone Panda","Winnti","Salt Typhoon"),
    HasIRAN      = Description has_any ("APT33","APT34","Charming Kitten","Phosphorus","IRGC","MuddyWater","OilRig","Cobalt Gypsy"),
    HasCobalt    = Description has_any ("Cobalt Strike","Brute Ratel","Sliver","Metasploit","Mimikatz","SharpHound","BloodHound"),
    HasSandworm  = Description has_any ("Sandworm","BlackEnergy","NotPetya","Voodoo Bear","Seashell Blizzard","Elektra","Industroyer"),
    HasTA505     = Description has_any ("TA505","Dridex","Evil Corp","Evil Corp","Indrik Spider","BitPaymer"),
    HasScattered = Description has_any ("Scattered Spider","Octo Tempest","0ktapus","Muddled Libra","UNC3944")
| summarize
    TotalIndicators = count(),
    Lazarus_NK      = countif(HasLazarus),
    Cozy_Bear_RU    = countif(HasCozy),
    Fancy_Bear_RU   = countif(HasFancy),
    LockBit_RW      = countif(HasLockbit),
    ChinaAPT        = countif(HasChinaAPT),
    Iran_APT        = countif(HasIRAN),
    CobaltTools     = countif(HasCobalt),
    Sandworm_RU     = countif(HasSandworm),
    TA505_ECorp     = countif(HasTA505),
    ScatteredSpider = countif(HasScattered)
"""

# ── (d) TI trend — indicators ingested per month over the window ──────────────
ti_trend_query = f"""
ThreatIntelligenceIndicator
| where TimeGenerated > {_ta_window}
| summarize
    Count         = count(),
    HighConf      = countif(ConfidenceScore >= 75),
    UniqueActors  = dcount(ThreatType)
  by Month = startofmonth(TimeGenerated)
| order by Month asc
"""

# ── (e) UEBA BehaviorAnalytics — entity anomalies over full window ────────────
ueba_query = f"""
BehaviorAnalytics
| where TimeGenerated > {_ta_window}
| where array_length(ActivityInsights) > 0
| summarize
    UEBA_Events      = count(),
    UniqueUsers      = dcount(UserName),
    HighRiskEntities = countif(InvestigationPriority >= 7)
  by ActivityType
| order by UEBA_Events desc
| take 20
"""

print(f"  Querying TI feed, SecurityAlert tactics, actor references, monthly trend, and UEBA...")
ti_summary = run_kql(ti_summary_query, f"TI feed summary ({THREAT_ACTOR_DAYS}d)")
ta_alerts  = run_kql(ta_alerts_query,  "Tactics in SecurityAlert")
ta_actors  = run_kql(ta_actor_query,   f"Threat actor references ({THREAT_ACTOR_DAYS}d)")
ti_trend   = run_kql(ti_trend_query,   f"TI monthly trend ({THREAT_ACTOR_DAYS}d)")
ueba_df    = run_kql(ueba_query,       f"UEBA BehaviorAnalytics ({THREAT_ACTOR_DAYS}d)")

# ─ Print results ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  THREAT ACTOR & ANALYTICS INTELLIGENCE  (last {THREAT_ACTOR_DAYS} days / {THREAT_ACTOR_DAYS//30} months)")
print(f"{'='*70}")

# TI feed
if not ti_summary.empty:
    total_ti  = int(ti_summary["ActiveIndicators"].sum()) if "ActiveIndicators" in ti_summary.columns else 0
    high_conf = int(ti_summary["HighConfCount"].sum())    if "HighConfCount"    in ti_summary.columns else 0
    print(f"\n  Threat Intelligence Feed (active, not expired — last {THREAT_ACTOR_DAYS}d):")
    print(f"   Total active indicators : {total_ti:,}")
    print(f"   High-confidence (>= 75) : {high_conf:,}")
    display(ti_summary)
else:
    print("\n   ThreatIntelligenceIndicator — no active indicators (check TI connector)")

# Actor references
if not ta_actors.empty and "TotalIndicators" in ta_actors.columns:
    row = ta_actors.iloc[0]
    actor_map = {
        "Lazarus / APT38 (NK)":     int(row.get("Lazarus_NK",      0)),
        "Cozy Bear / APT29 (RU)":   int(row.get("Cozy_Bear_RU",    0)),
        "Fancy Bear / APT28 (RU)":  int(row.get("Fancy_Bear_RU",   0)),
        "LockBit / Ransomware":     int(row.get("LockBit_RW",      0)),
        "China APT41/10/VT":        int(row.get("ChinaAPT",        0)),
        "Iran APT33/34":            int(row.get("Iran_APT",        0)),
        "Cobalt Strike / Tools":    int(row.get("CobaltTools",     0)),
        "Sandworm (RU)":            int(row.get("Sandworm_RU",     0)),
        "TA505 / Evil Corp":        int(row.get("TA505_ECorp",     0)),
        "Scattered Spider":         int(row.get("ScatteredSpider", 0)),
    }
    total_actor_hits = sum(actor_map.values())
    print(f"\n  Threat Actor Intelligence — last {THREAT_ACTOR_DAYS} days  (total TI indicators: {int(row.get('TotalIndicators', 0)):,})")
    print(f"  {'Actor Group':<38} {'Indicators':>10}  {'Bar':}")
    print(f"  {'-'*65}")
    for actor, cnt in sorted(actor_map.items(), key=lambda x: -x[1]):
        status = "ALERT" if cnt > 10 else "HIT" if cnt > 0 else "CLEAN"
        bar    = "#" * min(cnt, 30) if cnt > 0 else ""
        print(f"  [{status:5s}]  {actor:<34} {cnt:>6}  {bar}")
    print(f"  {'-'*65}")
    print(f"  {'TOTAL ACTOR-TAGGED INDICATORS':<38} {total_actor_hits:>10}")

# ATT&CK-mapped alerts
if not ta_alerts.empty:
    print(f"\n  ATT&CK Tactics in SecurityAlert ({len(ta_alerts)} tactic/alert combos):")
    display(ta_alerts.head(20))
else:
    print("\n   No SecurityAlert records with Tactics in this period")

# UEBA
if not ueba_df.empty:
    total_ueba = int(ueba_df["UEBA_Events"].sum())      if "UEBA_Events"      in ueba_df.columns else 0
    high_risk  = int(ueba_df["HighRiskEntities"].sum()) if "HighRiskEntities" in ueba_df.columns else 0
    print(f"\n  UEBA BehaviorAnalytics ({THREAT_ACTOR_DAYS}d): {total_ueba:,} events | {high_risk} high-risk entities (InvestigationPriority >= 7)")
    display(ueba_df)
else:
    print("\n   BehaviorAnalytics — no data (enable UEBA in Sentinel Settings > Entity behavior)")

# ─ Chart 1: Actor indicators bar ──────────────────────────────────────────────
if not ta_actors.empty and "TotalIndicators" in ta_actors.columns:
    actor_fig_data = {k: v for k, v in actor_map.items() if v > 0}
    if actor_fig_data:
        fig = px.bar(
            x=list(actor_fig_data.keys()),
            y=list(actor_fig_data.values()),
            title=f"Threat Actor Indicators in TI Feed (last {THREAT_ACTOR_DAYS} days)",
            labels={"x": "Threat Actor / Group", "y": "Indicator Count"},
            color=list(actor_fig_data.values()),
            color_continuous_scale="Reds",
        )
        fig.update_layout(xaxis_tickangle=-30, coloraxis_showscale=False, height=420)
        fig.show()

# ─ Chart 2: Monthly TI ingestion trend ────────────────────────────────────────
if not ti_trend.empty and "Month" in ti_trend.columns and "Count" in ti_trend.columns:
    ti_trend["Month"] = pd.to_datetime(ti_trend["Month"])
    fig_trend = px.bar(ti_trend, x="Month", y="Count",
                       title=f"TI Indicator Ingestion per Month (last {THREAT_ACTOR_DAYS} days)",
                       labels={"Count": "Indicators", "Month": "Month"},
                       color="HighConf",
                       color_continuous_scale="Reds",
                       hover_data=["HighConf", "UniqueActors"])
    fig_trend.update_layout(xaxis_tickangle=-30, height=400,
                             coloraxis_colorbar_title="High-Conf")
    fig_trend.show()

# ─ Chart 3: ATT&CK tactics bar ────────────────────────────────────────────────
if not ta_alerts.empty and "Tactics" in ta_alerts.columns and "AlertCount" in ta_alerts.columns:
    tactic_totals = (ta_alerts
                     .groupby("Tactics", as_index=False)["AlertCount"].sum()
                     .sort_values("AlertCount", ascending=False))
    fig2 = px.bar(tactic_totals, x="Tactics", y="AlertCount",
                  title="SecurityAlert Count by MITRE ATT&CK Tactic",
                  color="AlertCount", color_continuous_scale="Oranges",
                  labels={"AlertCount": "Alerts", "Tactics": "Tactic"})
    fig2.update_layout(xaxis_tickangle=-30, coloraxis_showscale=False, height=400)
    fig2.show()


🕵️  Loading threat actor & analytics intelligence...
🔍 TI feed summary...
   ✅ Returned 0 rows
🔍 Tactics in SecurityAlert...
   ✅ Returned 100 rows
🔍 Threat actor references in TI...
   ✅ Returned 1 rows
🔍 UEBA BehaviorAnalytics...
   ✅ Returned 0 rows

🕵️  THREAT ACTOR & ANALYTICS INTELLIGENCE

   ℹ️  ThreatIntelligenceIndicator — no active indicators (check TI connector)

👥 Threat Actor Intelligence (last 60 days of TI feed):
   🟢 Lazarus / APT38 (NK)                   0 indicator(s)
   🟢 Cozy Bear / APT29 (RU)                 0 indicator(s)
   🟢 Fancy Bear / APT28 (RU)                0 indicator(s)
   🟢 LockBit / Ransomware                   0 indicator(s)
   🟢 China APT41/10/VT                      0 indicator(s)
   🟢 Iran APT33/34                          0 indicator(s)
   🟢 Cobalt Strike / Tools                  0 indicator(s)

⚔️  ATT&CK Tactics in SecurityAlert (100 tactic/alert combos):


,Tactics,AlertName,AlertSeverity,AlertCount,UniqueAlertNames,Entities
0,CredentialAccess,Malicious credential theft tool execution dete...,High,734,1,"[""[{\""$id\"":\""4\"",\""Name\"":\""rayt\"",\""NTDomain..."
1,Unknown,Phishing alert,High,504,1,"[""[{\""$id\"":\""3\"",\""Name\"":\""u6434\"",\""UPNSuff..."
2,Exfiltration,DLP policy (Block sharing sensitive informatio...,High,487,1,"[""[{\""$id\"":\""3\"",\""Name\"":\""u3014\"",\""UPNSuff..."
3,LateralMovement,Hands-on-keyboard attack involving multiple de...,High,432,1,"[""[{\""$id\"":\""4\"",\""Name\"":\""rayt\"",\""NTDomain..."
4,LateralMovement,Compromised account conducting hands-on-keyboa...,High,247,1,"[""[{\""$id\"":\""4\"",\""Name\"":\""rayt\"",\""NTDomain..."
5,Unknown,Potential human-operated malicious activity,High,238,1,"[""[{\""$id\"":\""4\"",\""Name\"":\""rayt\"",\""NTDomain..."
6,Unknown,A potentially malicious URL click was detected,High,235,1,"[""[{\""$id\"":\""3\"",\""Recipient\"":\""u6765@int.za..."
7,"InitialAccess, Probing",A potentially malicious URL click was detected,High,195,1,"[""[{\""$id\"":\""3\"",\""Recipient\"":\""u3124@int.za..."
8,Unknown,Purview IRM ('80062f0d') test 12/14,High,192,1,"[""[{\""$id\"":\""3\"",\""Name\"":\""agentphishtriage\..."
9,"CredentialAccess, LateralMovement, CommandAndC...",Multiple dual-purpose tools were dropped,High,182,1,"[""[{\""$id\"":\""4\"",\""Directory\"":\""C:\\\\M365DA..."



   ℹ️  BehaviorAnalytics — no data (enable UEBA in Sentinel Settings → Entity behavior)


---
## 8. Lateral Movement Detection

Identify potential lateral movement by analyzing:
- Accounts logging into unusually high number of devices
- Devices initiating connections to many new internal hosts
- Credential abuse patterns

In [26]:
# Lateral Movement - Account Analysis
lateral_account_query = f"""
// Lateral Movement Detection - Accounts Accessing Multiple Devices
let NormalDeviceCount = 3; // Baseline: most users access <= 3 devices
let Multiplier = 3;
// Calculate historical baseline per user
let UserBaseline = DeviceLogonEvents
| where TimeGenerated between (ago(365d) .. ago(7d))
| summarize BaselineDevices = dcount(DeviceName) by AccountName
| extend Threshold = max_of(BaselineDevices * Multiplier, NormalDeviceCount * Multiplier);
// Recent activity
let RecentActivity = DeviceLogonEvents
| where {TIME_RANGE}
| summarize 
    RecentDevices = dcount(DeviceName),
    Devices = make_set(DeviceName, 30),
    LogonTypes = make_set(LogonType),
    FirstLogon = min(TimeGenerated),
    LastLogon = max(TimeGenerated),
    TotalLogons = count()
    by AccountName;
RecentActivity
| join kind=leftouter UserBaseline on AccountName
| extend BaselineDevices = coalesce(BaselineDevices, NormalDeviceCount)
| extend Threshold = coalesce(Threshold, NormalDeviceCount * Multiplier)
| where RecentDevices > Threshold
| extend LateralMovementIndicator = case(
    RecentDevices > BaselineDevices * 5, "🔴 Critical - Significant Anomaly",
    RecentDevices > BaselineDevices * 3, "🟠 High - Unusual Activity",
    "🟡 Medium - Monitor"
)
| extend DeviceIncrease = RecentDevices - BaselineDevices
| project AccountName, BaselineDevices, RecentDevices, DeviceIncrease, Threshold,
          LateralMovementIndicator, LogonTypes, TotalLogons, FirstLogon, LastLogon, Devices
| order by DeviceIncrease desc
| take 50
"""

lateral_accounts = run_kql(lateral_account_query, "Detecting lateral movement by account")

if not lateral_accounts.empty:
    print(f"\n🔄 Found {len(lateral_accounts)} accounts with potential lateral movement")
    display(lateral_accounts)
    
    # Visualization
    fig = px.bar(lateral_accounts.head(20), 
                 x='AccountName', 
                 y=['BaselineDevices', 'RecentDevices'],
                 barmode='group',
                 title='Account Device Access: Baseline vs Recent')
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

🔍 Detecting lateral movement by account...
   ✅ Returned 0 rows


In [27]:
# Lateral Movement - Device Network Analysis
lateral_device_query = f"""
// Lateral Movement - Devices Connecting to Many New Internal Hosts
let InternalIPRanges = dynamic(["10.", "172.16.", "172.17.", "172.18.", "172.19.", 
                                 "172.20.", "172.21.", "172.22.", "172.23.", "172.24.",
                                 "172.25.", "172.26.", "172.27.", "172.28.", "172.29.",
                                 "172.30.", "172.31.", "192.168."]);
// Historical baseline
let DeviceBaseline = DeviceNetworkEvents
| where TimeGenerated between (ago(365d) .. ago(7d))
| where RemoteIP has_any (InternalIPRanges)
| summarize BaselineHosts = dcount(RemoteIP) by DeviceName;
// Recent activity (last 7 days)
let RecentConnections = DeviceNetworkEvents
| where TimeGenerated > ago(7d)
| where RemoteIP has_any (InternalIPRanges)
| summarize 
    RecentHosts = dcount(RemoteIP),
    Hosts = make_set(RemoteIP, 50),
    Ports = make_set(RemotePort, 20),
    Processes = make_set(InitiatingProcessFileName, 10),
    TotalConnections = count()
    by DeviceName;
RecentConnections
| join kind=leftouter DeviceBaseline on DeviceName
| extend BaselineHosts = coalesce(BaselineHosts, 10)
| extend HostIncrease = RecentHosts - BaselineHosts
| extend IncreasePercent = round(todouble(HostIncrease) / BaselineHosts * 100, 2)
| where IncreasePercent > 50 or HostIncrease > 20
| extend LateralIndicator = case(
    IncreasePercent > 200 or HostIncrease > 50, "🔴 Critical - Scanning Behavior",
    IncreasePercent > 100, "🟠 High - Unusual Spread",
    "🟡 Medium - Monitor"
)
| project DeviceName, BaselineHosts, RecentHosts, HostIncrease, IncreasePercent,
          LateralIndicator, Ports, Processes, TotalConnections
| order by HostIncrease desc
| take 50
"""

lateral_devices = run_kql(lateral_device_query, "Detecting lateral movement by device")

if not lateral_devices.empty:
    print(f"\n🖥️ Found {len(lateral_devices)} devices with potential lateral movement")
    display(lateral_devices)

🔍 Detecting lateral movement by device...


   ✅ Returned 0 rows


In [ ]:
# Reconnaissance Tool Detection
recon_tools_query = f"""
// Reconnaissance Tool Usage Detection
let ReconTools = dynamic(["net.exe", "nltest.exe", "dsquery.exe", "nslookup.exe", 
                          "ping.exe", "arp.exe", "route.exe", "netstat.exe", 
                          "tasklist.exe", "systeminfo.exe", "whoami.exe", 
                          "quser.exe", "query.exe", "cmdkey.exe", "ipconfig.exe"]);
DeviceProcessEvents
| where {TIME_RANGE}
| where FileName in~ (ReconTools)
| summarize 
    ToolUsageCount = count(),
    ToolsUsed = make_set(FileName),
    UniqueCommands = dcount(ProcessCommandLine),
    Commands = make_set(ProcessCommandLine, 10),
    FirstUse = min(TimeGenerated),
    LastUse = max(TimeGenerated)
    by DeviceName, AccountName
| where ToolUsageCount > 5 or array_length(ToolsUsed) > 3
| extend ReconIndicator = case(
    array_length(ToolsUsed) > 5, "🔴 High - Multiple Recon Tools",
    ToolUsageCount > 20, "🟠 Medium - Frequent Usage",
    "🟡 Low - Monitor"
)
| order by array_length(ToolsUsed) desc, ToolUsageCount desc
| take 50
"""

recon_activity = run_kql(recon_tools_query, "Detecting reconnaissance tool usage")

if not recon_activity.empty:
    print(f"\n🔍 Found {len(recon_activity)} instances of reconnaissance activity")
    display(recon_activity)

🔍 Detecting reconnaissance tool usage...
   ✅ Returned 50 rows

🔍 Found 50 instances of reconnaissance activity


,DeviceName,AccountName,ToolUsageCount,ToolsUsed,UniqueCommands,Commands,FirstUse,LastUse,ReconIndicator
0,kenvins-pc.zava-corp.com,system,2695,"[""tasklist.exe"",""ipconfig.exe"",""net.exe"",""PING...",39,"[""tasklist /FI \""IMAGENAME eq Secure System\""...",2025-09-02 21:06:00.573850+00:00,2026-02-03 14:14:37.196481+00:00,🔴 High - Multiple Recon Tools
1,babaks-pc.zava-corp.com,system,1744,"[""ROUTE.EXE"",""ARP.EXE"",""NETSTAT.EXE"",""ipconfig...",39,"[""\""route.exe\"" print"",""\""arp.exe\"" /a"",""\""net...",2025-09-25 11:33:34.016420+00:00,2026-02-03 06:58:35.665158+00:00,🔴 High - Multiple Recon Tools
2,cpc-u126-5hkyoq,system,493,"[""tasklist.exe"",""net.exe"",""ROUTE.EXE"",""ipconfi...",40,"[""tasklist /FI \""IMAGENAME eq Secure System\""...",2025-09-18 18:29:21.340431+00:00,2025-10-25 17:11:22.812148+00:00,🔴 High - Multiple Recon Tools
3,rayt-pc.zava-corp.com,system,2231,"[""net.exe"",""ipconfig.exe"",""ARP.EXE"",""NETSTAT.E...",37,"[""\""net.exe\"" accounts"",""\""ipconfig.exe\"" /all...",2025-08-24 13:24:20.906093+00:00,2026-02-03 13:10:44.250495+00:00,🔴 High - Multiple Recon Tools
4,alpineski-u4090,system,1934,"[""ipconfig.exe"",""tasklist.exe"",""net.exe"",""PING...",34,"[""\""ipconfig.exe\"" /flushdns"",""tasklist /FI \...",2025-08-24 15:16:10.552139+00:00,2026-02-19 19:54:22.470057+00:00,🔴 High - Multiple Recon Tools
5,desktop-f4t55h2,system,804,"[""tasklist.exe"",""net.exe"",""systeminfo.exe"",""NE...",33,"[""tasklist /FI \""IMAGENAME eq Secure System\""...",2025-08-25 12:56:10.784116+00:00,2026-02-19 14:30:49.099857+00:00,🔴 High - Multiple Recon Tools
6,desktop-k561uoc,system,707,"[""net.exe"",""tasklist.exe"",""PING.EXE"",""NETSTAT....",33,"[""\""net.exe\"" accounts"",""tasklist /FI \""IMAGE...",2025-09-02 14:34:54.552488+00:00,2026-02-19 07:06:48.494198+00:00,🔴 High - Multiple Recon Tools
7,desktop-raau95q,system,197,"[""net.exe"",""tasklist.exe"",""PING.EXE"",""query.ex...",33,"[""\""net.exe\"" accounts"",""tasklist /FI \""IMAGE...",2025-09-22 16:31:18.982523+00:00,2025-12-10 03:27:14.333377+00:00,🔴 High - Multiple Recon Tools
8,vm01-demo,system,171,"[""tasklist.exe"",""net.exe"",""ipconfig.exe"",""ARP....",36,"[""tasklist /FI \""IMAGENAME eq Secure System\""...",2025-09-05 15:25:53.564913+00:00,2025-09-10 05:04:32.271012+00:00,🔴 High - Multiple Recon Tools
9,eng-ws-motors.vnevado.alpineskihouse.co,system,1204,"[""net.exe"",""PING.EXE"",""systeminfo.exe"",""NETSTA...",31,"[""\""net.exe\"" accounts"",""ping -n 2 127.0.0.1 ...",2025-08-25 08:24:44.451452+00:00,2026-02-19 05:01:22.126013+00:00,🔴 High - Multiple Recon Tools


---
## 9. Alert Enrichment & Cross-Telemetry Correlation

Enrich detected events with contextual data from multiple sources and correlate with existing Defender alerts.

In [28]:
# Correlate with Defender Alerts
defender_alerts_query = f"""
// Defender Alerts - Identity & Credential Related
let IdentityKeywords = dynamic(["brute", "password", "spray", "credential", "lateral", 
                                 "reconnaissance", "suspicious", "anomal", "impossible",
                                 "risky", "compromise", "attack", "malicious"]);
SecurityAlert
| where {TIME_RANGE}
| where AlertName has_any (IdentityKeywords) or Description has_any (IdentityKeywords)
| summarize 
    AlertCount = count(),
    Severities = make_set(AlertSeverity),
    Tactics = make_set(Tactics),
    FirstAlert = min(TimeGenerated),
    LastAlert = max(TimeGenerated)
    by AlertName, ProviderName
| extend SeverityScore = case(
    Severities has "High", "🔴 High",
    Severities has "Medium", "🟠 Medium",
    "🟡 Low"
)
| order by AlertCount desc
"""

defender_alerts = run_kql(defender_alerts_query, "Querying Defender alerts")

if not defender_alerts.empty:
    print(f"\n🚨 Found {len(defender_alerts)} unique alert types")
    display(defender_alerts)
    
    # Alert distribution
    fig = px.bar(defender_alerts, x='AlertName', y='AlertCount', 
                 color='SeverityScore', title='Defender Alert Distribution')
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

🔍 Querying Defender alerts...
   ✅ Returned 4677 rows

🚨 Found 4677 unique alert types


,AlertName,ProviderName,AlertCount,Severities,Tactics,FirstAlert,LastAlert,SeverityScore
0,Malicious credential theft tool execution dete...,MDATP,774,"[""High""]","[""CredentialAccess"",""PrivilegeEscalation, Defe...",2025-11-26 00:09:20.329684+00:00,2026-02-22 00:34:43.859648+00:00,🔴 High
1,Suspicious app authorization in M365 - Test,MicrosoftThreatProtection,769,"[""Medium""]","[""PrivilegeEscalation""]",2025-11-26 09:38:04.658478+00:00,2026-02-05 17:53:02.431920+00:00,🟠 Medium
2,CC_Risky behavior in Copilot for Microsoft 365,OATP,606,"[""Medium""]","[""Probing""]",2025-11-25 17:32:06.375474+00:00,2026-02-22 20:47:04.075769+00:00,🟠 Medium
3,Hands-on-keyboard attack involving multiple de...,MDATP,519,"[""High""]","[""LateralMovement"",""InitialAccess, Persistence...",2025-11-26 14:29:29.413637+00:00,2026-02-22 00:34:52.196391+00:00,🔴 High
4,A potentially malicious URL click was detected,OATP,441,"[""High""]","[""InitialAccess, Probing"",""Unknown"",""InitialAc...",2025-11-26 00:08:03.652480+00:00,2026-02-22 01:08:02.928758+00:00,🔴 High
...,...,...,...,...,...,...,...,...
4672,Purview IRM ('7814e264') Departures,Office 365 Security and Compliance,1,"[""High""]","[""Exfiltration""]",2025-12-16 02:53:06.081400+00:00,2025-12-16 02:53:06.081400+00:00,🔴 High
4673,Purview IRM ('43e42f6d') Risky AI usage quick ...,Office 365 Security and Compliance,1,"[""Medium""]","[""Unknown""]",2025-12-16 02:53:05.518855+00:00,2025-12-16 02:53:05.518855+00:00,🟠 Medium
4674,Purview IRM ('e1fda84c') Data Exfiltration Mon...,Office 365 Security and Compliance,1,"[""Low""]","[""Exfiltration""]",2026-02-03 02:35:04.135423+00:00,2026-02-03 02:35:04.135423+00:00,🟡 Low
4675,Purview IRM ('aff7eee2') DSPM for AI - Detect ...,Office 365 Security and Compliance,1,"[""Low""]","[""Unknown""]",2026-01-08 23:59:03.241794+00:00,2026-01-08 23:59:03.241794+00:00,🟡 Low


In [29]:
# Enrich compromised account activity
enrichment_query = f"""
// Post-Compromise Activity Enrichment
// Find accounts with failed logins followed by success, then track subsequent activity
let CompromisedAccounts = SigninLogs
| where {TIME_RANGE}
| where ResultType != 0
| summarize FailedCount = count() by UserPrincipalName, bin(TimeGenerated, 1h)
| where FailedCount >= 10
| join kind=inner (
    SigninLogs
    | where {TIME_RANGE}
    | where ResultType == 0
    | project UserPrincipalName, SuccessTime = TimeGenerated
) on UserPrincipalName
| where SuccessTime between (TimeGenerated .. (TimeGenerated + 2h))
| distinct UserPrincipalName, CompromiseTime = SuccessTime;
// Track post-compromise audit activity
CompromisedAccounts
| join kind=inner (
    AuditLogs
    | where {TIME_RANGE}
    | extend InitiatedBy = tostring(InitiatedBy.user.userPrincipalName)
    | project TimeGenerated, InitiatedBy, OperationName, Category, Result,
              TargetResources
) on $left.UserPrincipalName == $right.InitiatedBy
| where TimeGenerated between (CompromiseTime .. (CompromiseTime + 24h))
| summarize 
    PostCompromiseActions = count(),
    Operations = make_set(OperationName, 20),
    Categories = make_set(Category)
    by UserPrincipalName, CompromiseTime
| extend RiskLevel = case(
    PostCompromiseActions > 50, "🔴 Critical - High Activity Post-Compromise",
    PostCompromiseActions > 20, "🟠 High",
    "🟡 Medium"
)
| order by PostCompromiseActions desc
"""

post_compromise = run_kql(enrichment_query, "Enriching post-compromise activity")

if not post_compromise.empty:
    print(f"\n🔒 Found {len(post_compromise)} accounts with post-compromise activity")
    display(post_compromise)

🔍 Enriching post-compromise activity...
   ✅ Returned 12459 rows

🔒 Found 12459 accounts with post-compromise activity


,UserPrincipalName,CompromiseTime,PostCompromiseActions,Operations,Categories,RiskLevel
0,,2026-01-13 18:50:28.089440+00:00,15997,"[""Import"",""Update user"",""Add member to group"",...","[""ProvisioningManagement"",""UserManagement"",""Gr...",🔴 Critical - High Activity Post-Compromise
1,u101@a.alpineskihouse.co,2025-12-09 09:54:11.232061+00:00,677,"[""Add member to group"",""GroupsODataV4_Get"",""Va...","[""GroupManagement"",""Authentication"",""Device"",""...",🔴 Critical - High Activity Post-Compromise
2,u101@a.alpineskihouse.co,2025-12-09 09:54:33.681114+00:00,677,"[""Add member to group"",""GroupsODataV4_Get"",""Va...","[""GroupManagement"",""Authentication"",""Device"",""...",🔴 Critical - High Activity Post-Compromise
3,u101@a.alpineskihouse.co,2025-12-09 09:55:08.472733+00:00,677,"[""Add member to group"",""GroupsODataV4_Get"",""Va...","[""GroupManagement"",""Authentication"",""Device"",""...",🔴 Critical - High Activity Post-Compromise
4,u101@a.alpineskihouse.co,2025-12-09 09:53:53.825664+00:00,677,"[""Add member to group"",""GroupsODataV4_Get"",""Va...","[""GroupManagement"",""Authentication"",""Device"",""...",🔴 Critical - High Activity Post-Compromise
...,...,...,...,...,...,...
12454,u1034@int.zava-corp.com,2026-01-14 16:43:02.615898+00:00,1,"[""Validate user authentication""]","[""Authentication""]",🟡 Medium
12455,u12217@int.zava-corp.com,2025-12-09 19:27:07.371908+00:00,1,"[""Validate user authentication""]","[""Authentication""]",🟡 Medium
12456,u402@int.zava-corp.com,2025-12-31 06:05:50.036094+00:00,1,"[""Validate user authentication""]","[""Authentication""]",🟡 Medium
12457,u402@int.zava-corp.com,2025-12-31 06:02:16.206878+00:00,1,"[""Validate user authentication""]","[""Authentication""]",🟡 Medium


In [30]:
# Cloud App Activity Correlation
cloud_app_query = f"""
// Cloud App Events for Compromised Accounts
let SuspiciousAccounts = SigninLogs
| where {TIME_RANGE}
| where RiskLevelDuringSignIn in ("medium", "high") or RiskLevelAggregated in ("medium", "high")
| distinct UserPrincipalName;
CloudAppEvents
| where {TIME_RANGE}
| where AccountId in (SuspiciousAccounts) or AccountDisplayName in (SuspiciousAccounts)
| summarize 
    EventCount = count(),
    ActionTypes = make_set(ActionType, 20),
    Applications = make_set(Application, 10),
    Countries = make_set(CountryCode, 10),
    FirstActivity = min(TimeGenerated),
    LastActivity = max(TimeGenerated)
    by AccountDisplayName
| extend SuspicionLevel = case(
    EventCount > 1000, "🔴 High - Excessive Activity",
    array_length(Countries) > 5, "🟠 Medium - Multi-Country",
    "🟡 Monitor"
)
| order by EventCount desc
| take 50
"""

cloud_app_activity = run_kql(cloud_app_query, "Analyzing cloud app activity for suspicious accounts")

if not cloud_app_activity.empty:
    print(f"\n☁️ Found {len(cloud_app_activity)} suspicious accounts with cloud app activity")
    display(cloud_app_activity)

🔍 Analyzing cloud app activity for suspicious accounts...
   ✅ Returned 20 rows

☁️ Found 20 suspicious accounts with cloud app activity


,AccountDisplayName,EventCount,ActionTypes,Applications,Countries,FirstActivity,LastActivity,SuspicionLevel
0,u13040@int.zava-corp.com,123,"[""Search"",""Validate"",""EnablePlugin"",""Authorize...","[""Microsoft 365"",""Microsoft 365 Copilot Chat""]","["""",""US""]",2025-11-06 21:12:41+00:00,2025-11-06 21:32:18+00:00,🟡 Monitor
1,u13056@int.zava-corp.com,33,"[""Validate"",""Search"",""DEX Reporting API read G...","[""Microsoft 365""]","[""""]",2025-11-07 21:45:10+00:00,2025-11-07 21:46:14+00:00,🟡 Monitor
2,u17529@int.zava-corp.com,21,"[""Validate"",""Search""]","[""Microsoft 365""]","[""""]",2026-02-03 11:23:23+00:00,2026-02-03 11:43:37+00:00,🟡 Monitor
3,u12691@int.zava-corp.com,19,"[""Search"",""Validate""]","[""Microsoft 365""]","[""""]",2025-10-18 02:01:26+00:00,2025-10-18 02:01:36+00:00,🟡 Monitor
4,u13720@int.zava-corp.com,19,"[""Validate"",""Search""]","[""Microsoft 365""]","[""""]",2025-12-09 03:53:17+00:00,2025-12-09 03:56:03+00:00,🟡 Monitor
5,u2831@int.zava-corp.com,6,"[""Search"",""Validate""]","[""Microsoft 365""]","[""""]",2025-10-17 18:47:18+00:00,2025-10-17 18:47:22+00:00,🟡 Monitor
6,u575@int.zava-corp.com,5,"[""CopilotInteraction""]","[""Microsoft 365 Copilot Chat""]","[""AU""]",2025-10-20 07:34:19+00:00,2025-10-20 07:34:45+00:00,🟡 Monitor
7,u8367@int.zava-corp.com,5,"[""Search"",""Validate""]","[""Microsoft 365""]","[""""]",2025-10-17 18:52:12+00:00,2025-10-17 18:52:14+00:00,🟡 Monitor
8,u11684@int.zava-corp.com,4,"[""AutoSensitivityLabelRuleMatch""]","[""Microsoft Exchange Online""]","[""""]",2025-10-18 15:21:32+00:00,2025-10-18 16:21:50+00:00,🟡 Monitor
9,u3313@int.zava-corp.com,3,"[""AutoSensitivityLabelRuleMatch""]","[""Microsoft Exchange Online""]","[""""]",2025-10-18 04:18:09+00:00,2025-10-18 04:18:09+00:00,🟡 Monitor


---
## 10. Defender Built-in Detection Rules

Leverage Defender detection rules and custom analytics to surface medium to high fidelity signals.

In [31]:
# Spike Detection - Hourly Failed Logons vs Baseline
spike_detection_query = f"""
// Spike Detection - Failed Logons Compared to Historical Baseline
let HistoricalBaseline = union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where TimeGenerated between (ago(180d) .. ago(7d))
| where ResultType != 0
| summarize HistoricalFailures = count() by bin(TimeGenerated, 1h)
| summarize 
    BaselineMean = avg(HistoricalFailures),
    BaselineStd = stdev(HistoricalFailures),
    Baseline95thPct = percentile(HistoricalFailures, 95);
let RecentActivity = union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where TimeGenerated > ago(7d)
| where ResultType != 0
| summarize CurrentFailures = count() by bin(TimeGenerated, 1h);
RecentActivity
| extend BaselineMean = toscalar(HistoricalBaseline | project BaselineMean)
| extend BaselineStd = toscalar(HistoricalBaseline | project BaselineStd)
| extend Baseline95thPct = toscalar(HistoricalBaseline | project Baseline95thPct)
| extend ZScore = (CurrentFailures - BaselineMean) / BaselineStd
| extend IsSpike = ZScore > 3
| extend SpikeLevel = case(
    ZScore > 5, "🔴 Critical Spike (>5σ)",
    ZScore > 4, "🟠 Major Spike (>4σ)",
    ZScore > 3, "🟡 Spike (>3σ)",
    "Normal"
)
| where IsSpike == true
| project TimeGenerated, CurrentFailures, BaselineMean, ZScore, SpikeLevel, Baseline95thPct
| order by ZScore desc
"""

spikes = run_kql(spike_detection_query, "Detecting failed login spikes")

if not spikes.empty:
    print(f"\n📈 Detected {len(spikes)} spike events")
    display(spikes)
    
    # Visualization
    fig = px.scatter(spikes, x='TimeGenerated', y='CurrentFailures', 
                     color='SpikeLevel', size='ZScore',
                     title='Failed Login Spikes (Last 7 Days)')
    fig.add_hline(y=spikes['BaselineMean'].mean(), line_dash="dash", 
                  annotation_text="Baseline Mean")
    fig.show()

🔍 Detecting failed login spikes...
   ✅ Returned 0 rows


In [32]:
# Identity Protection Alert Summary
identity_alerts_query = f"""
// Identity Protection Alerts Summary
let MDIAlerts = SecurityAlert
| where {TIME_RANGE}
| where ProviderName has_any ("Microsoft Defender for Identity", "Azure ATP", "AATP")
| summarize 
    AlertCount = count(),
    arg_max(TimeGenerated, *)
    by AlertName
| project AlertName, AlertSeverity, AlertCount, Description = substring(Description, 0, 200),
          Tactics, ProviderName, TimeGenerated
| order by case(AlertSeverity == "High", 1, AlertSeverity == "Medium", 2, 3), AlertCount desc;
let AADAlerts = SecurityAlert
| where {TIME_RANGE}
| where ProviderName has_any ("Azure Active Directory", "Microsoft Entra")
| summarize 
    AlertCount = count(),
    arg_max(TimeGenerated, *)
    by AlertName
| project AlertName, AlertSeverity, AlertCount, Description = substring(Description, 0, 200),
          Tactics, ProviderName, TimeGenerated;
union MDIAlerts, AADAlerts
| extend SeverityIcon = case(
    AlertSeverity == "High", "🔴",
    AlertSeverity == "Medium", "🟠",
    "🟡"
)
| project SeverityIcon, AlertName, AlertSeverity, AlertCount, Description, Tactics, ProviderName
| order by case(AlertSeverity == "High", 1, AlertSeverity == "Medium", 2, 3), AlertCount desc
"""

identity_alerts = run_kql(identity_alerts_query, "Summarizing identity protection alerts")

if not identity_alerts.empty:
    print(f"\n🛡️ Found {len(identity_alerts)} unique identity alert types")
    display(identity_alerts)

🔍 Summarizing identity protection alerts...
   ✅ Returned 0 rows


---
## 🔑 Section 10.2 — Malicious OAuth Apps & Service Principal Threat Hunting

Detects OAuth app abuse and rogue Service Principals — a common post-compromise persistence mechanism.  
Attackers register or compromise apps with over-permissive API scopes to silently harvest email, files, and directory data while bypassing MFA.

| Hunt | Signal | Tables |
|------|--------|--------|
| **10.2.1 Suspicious Consent Grants** | Users consenting to apps with Mail/Files/Directory write scopes | AuditLogs |
| **10.2.2 Service Principal Geo Anomalies** | SP sign-ins from new or atypical countries | AADServicePrincipalSignInLogs |
| **10.2.3 High-Risk OAuth Permissions** | Apps granted dangerous Graph API delegated/app permissions | AuditLogs, CloudAppEvents |
| **10.2.4 SP Credential Abuse** | Rapid SP auth from many IPs / countries (likely credential stuffing or stolen secret) | AADServicePrincipalSignInLogs |
| **10.2.5 Geolocation Heat-map** | Combined OAuth/SP activity by country with risk scoring | AuditLogs, AADServicePrincipalSignInLogs |

**Key threat patterns:**
- **Illicit consent grant** — attacker tricks user into consenting to a malicious app → persistent delegated access with no MFA
- **App-only token abuse** — compromised service principal secret used from attacker infrastructure
- **Over-privileged legacy apps** — apps with `Mail.ReadWrite`, `Files.ReadWrite.All`, `Directory.ReadWrite.All` are high-value targets


In [33]:

# ── 10.2.1  Suspicious OAuth Consent Grants ───────────────────────────────────
# Detects users who consented to apps requesting high-risk API permissions.
# Illicit consent grant = attacker-controlled app tricks user → persistent access bypassing MFA.

oauth_consent_query = f"""
AuditLogs
| where {TIME_RANGE}
| where OperationName in ("Consent to application", "Add app role assignment to service principal",
                          "Add delegated permission grant", "Add oauth2PermissionGrant")
| extend Actor       = tostring(InitiatedBy.user.userPrincipalName)
| extend ActorIP     = tostring(InitiatedBy.user.ipAddress)
| extend GeoInfo     = geo_info_from_ip_address(ActorIP)
| extend Country     = tostring(GeoInfo.country)
| extend City        = tostring(GeoInfo.city)
| extend AppName     = tostring(parse_json(tostring(TargetResources))[0].displayName)
| extend Permissions = tostring(parse_json(tostring(AdditionalDetails)))
| extend IsHighRisk  = Permissions has_any (
    "Mail.ReadWrite", "Mail.Read", "Mail.Send",
    "Files.ReadWrite.All", "Files.Read.All",
    "Directory.ReadWrite.All", "Directory.Read.All",
    "User.ReadWrite.All", "Group.ReadWrite.All",
    "RoleManagement.ReadWrite.Directory",
    "offline_access", "full_access_as_user"
  )
| extend RiskLevel = case(IsHighRisk, "High Risk — Sensitive Permissions", "Standard")
| order by IsHighRisk desc, TimeGenerated desc
| project TimeGenerated, OperationName, RiskLevel, Actor, ActorIP, Country, City,
          AppName, Permissions
| take 200
"""

oauth_consent_results = run_kql(oauth_consent_query, "Hunting suspicious OAuth consent grants")

# ── 10.2.2  Service Principal Sign-ins — Geo Anomalies ────────────────────────
# SP sign-ins from unusual / new countries signal credential compromise or supply-chain attack.

sp_signin_query = f"""
AADServicePrincipalSignInLogs
| where {TIME_RANGE}
| where ResultType == 0   // successful
| extend GeoInfo       = geo_info_from_ip_address(IPAddress)
| extend Country       = tostring(GeoInfo.country)
| extend City          = tostring(GeoInfo.city)
| summarize
    TotalSignIns    = count(),
    UniqueIPs       = dcount(IPAddress),
    Countries       = make_set(Country, 20),
    Cities          = make_set(City, 10),
    SampleIPs       = make_set(IPAddress, 5),
    FirstSeen       = min(TimeGenerated),
    LastSeen        = max(TimeGenerated)
  by ServicePrincipalName, ServicePrincipalId, AppId
| extend CountryCount = array_length(Countries)
| extend GeoRisk = case(
    CountryCount >= 5,  "Critical — 5+ Countries",
    CountryCount >= 3,  "High — 3-4 Countries",
    CountryCount == 2,  "Medium — 2 Countries",
    "Normal — Single Country"
)
| order by CountryCount desc, UniqueIPs desc
| take 100
"""

sp_geo_results = run_kql(sp_signin_query, "Analysing Service Principal sign-in geolocations")

# ── 10.2.3  High-Risk OAuth App Permissions ───────────────────────────────────

sp_highrisk_query = f"""
AADServicePrincipalSignInLogs
| where {TIME_RANGE}
| where ResultType == 0
| summarize
    SignInCount     = count(),
    RecentCountries = make_set(tostring(geo_info_from_ip_address(IPAddress).country), 10),
    UniqueIPs       = dcount(IPAddress),
    LastSeen        = max(TimeGenerated)
  by ServicePrincipalName, AppId, ResourceDisplayName
| join kind=leftouter (
    AuditLogs
    | where {TIME_RANGE}
    | where OperationName in ("Add app role assignment to service principal",
                              "Add delegated permission grant")
    | extend Perms      = tostring(AdditionalDetails)
    | extend IsHighRisk = Perms has_any (
        "Mail.ReadWrite", "Mail.Read", "Mail.Send",
        "Files.ReadWrite.All", "Files.Read.All",
        "Directory.ReadWrite.All", "Directory.Read.All",
        "User.ReadWrite.All", "RoleManagement.ReadWrite.Directory"
      )
    | where IsHighRisk
    | extend AppId2 = tostring(parse_json(tostring(TargetResources))[0].id)
    | summarize HighRiskPerms = make_set(Perms, 5) by AppId2
) on $left.AppId == $right.AppId2
| extend PermRisk = iff(isnotempty(HighRiskPerms), "High-Risk Permissions Granted", "No High-Risk Grant Found")
| project ServicePrincipalName, AppId, ResourceDisplayName, SignInCount,
          UniqueIPs, RecentCountries, PermRisk, HighRiskPerms, LastSeen
| order by SignInCount desc
| take 100
"""

sp_highrisk_results = run_kql(sp_highrisk_query, "Identifying high-risk OAuth app permissions")

# ── 10.2.4  Service Principal Credential Abuse ────────────────────────────────
# SP authenticating from many IPs in a short window → possible stolen client secret / certificate.

sp_abuse_query = f"""
AADServicePrincipalSignInLogs
| where {TIME_RANGE}
| where ResultType == 0
| summarize
    HourlyIPs = dcount(IPAddress),
    Countries = make_set(tostring(geo_info_from_ip_address(IPAddress).country), 20),
    IPs       = make_set(IPAddress, 10)
  by ServicePrincipalName, AppId, bin(TimeGenerated, 1h)
| where HourlyIPs >= 3
| extend CountryCount = array_length(Countries)
| extend AbuseSignal  = case(
    HourlyIPs >= 10 or CountryCount >= 3, "Critical — Likely Credential Abuse",
    HourlyIPs >= 5  or CountryCount == 2, "High — Suspicious Multi-IP Auth",
    "Medium — Review Required"
)
| order by HourlyIPs desc
| take 100
"""

sp_abuse_results = run_kql(sp_abuse_query, "Detecting Service Principal credential abuse patterns")

# ════════════════════════════════════════════════════════════════════════════════
# OUTPUT & VISUALISATION
# ════════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*72}")
print("OAUTH APP & SERVICE PRINCIPAL THREAT HUNTING — SUMMARY")
print(f"{'='*72}")

# ── Consent grants ─────────────────────────────────────────────────────────────
if not oauth_consent_results.empty:
    total    = len(oauth_consent_results)
    highrisk = len(oauth_consent_results[oauth_consent_results.get("RiskLevel", pd.Series(dtype=str)).str.startswith("High", na=False)]) if "RiskLevel" in oauth_consent_results.columns else 0
    print(f"\n  Consent Grants:  {total} events  ({highrisk} high-risk sensitive-permission grants)")
    display(oauth_consent_results.head(20))

    if "Country" in oauth_consent_results.columns and "RiskLevel" in oauth_consent_results.columns:
        hr = oauth_consent_results[oauth_consent_results["RiskLevel"].str.startswith("High", na=False)]
        if not hr.empty:
            ctry = hr["Country"].replace("", "Unknown").fillna("Unknown").value_counts().head(15).reset_index()
            ctry.columns = ["Country", "Count"]
            fig_consent = px.bar(ctry, x="Country", y="Count",
                                 title="High-Risk OAuth Consent Grants by Country",
                                 color="Count", color_continuous_scale="Reds")
            fig_consent.update_layout(xaxis_tickangle=-40)
            fig_consent.show()
else:
    print("\n  No OAuth consent grant events found")

# ── SP geo anomalies ────────────────────────────────────────────────────────────
if not sp_geo_results.empty:
    critical = len(sp_geo_results[sp_geo_results.get("GeoRisk", pd.Series(dtype=str)).str.startswith("Critical", na=False)]) if "GeoRisk" in sp_geo_results.columns else 0
    print(f"\n  Service Principal Geo Anomalies:  {len(sp_geo_results)} SPs  ({critical} critical — 5+ countries)")
    display(sp_geo_results.head(20))

    from collections import Counter
    all_countries: list = []
    if "Countries" in sp_geo_results.columns:
        for val in sp_geo_results["Countries"]:
            if isinstance(val, list):
                all_countries.extend([c for c in val if c and c != ""])
    if all_countries:
        ctry_counts = Counter(all_countries)
        ctry_df = pd.DataFrame(ctry_counts.most_common(30), columns=["Country", "SPSignIns"])
        fig_geo = px.choropleth(
            ctry_df, locations="Country", locationmode="country names",
            color="SPSignIns", color_continuous_scale="Reds",
            title="Service Principal Sign-ins by Country",
        )
        fig_geo.update_layout(height=450)
        fig_geo.show()
        fig_bar = px.bar(ctry_df.head(20), x="Country", y="SPSignIns",
                         title="Top 20 Countries — Service Principal Sign-ins",
                         color="SPSignIns", color_continuous_scale="Oranges")
        fig_bar.update_layout(xaxis_tickangle=-40)
        fig_bar.show()
else:
    print("\n  No Service Principal sign-in geo anomalies found")

# ── High-risk permissions ───────────────────────────────────────────────────────
if not sp_highrisk_results.empty:
    risky_apps = len(sp_highrisk_results[sp_highrisk_results.get("PermRisk", pd.Series(dtype=str)).str.startswith("High", na=False)]) if "PermRisk" in sp_highrisk_results.columns else 0
    print(f"\n  OAuth Apps with High-Risk Permissions:  {risky_apps} / {len(sp_highrisk_results)} apps analysed")
    display(sp_highrisk_results[sp_highrisk_results.get("PermRisk", pd.Series(dtype=str)).str.startswith("High", na=False)].head(20) if "PermRisk" in sp_highrisk_results.columns else sp_highrisk_results.head(20))
else:
    print("\n  No high-risk OAuth app permission grants detected")

# ── Credential abuse ────────────────────────────────────────────────────────────
if not sp_abuse_results.empty:
    critical_abuse = len(sp_abuse_results[sp_abuse_results.get("AbuseSignal", pd.Series(dtype=str)).str.startswith("Critical", na=False)]) if "AbuseSignal" in sp_abuse_results.columns else 0
    print(f"\n  SP Credential Abuse Signals:  {len(sp_abuse_results)} patterns  ({critical_abuse} critical)")
    display(sp_abuse_results.head(20))

    if "ServicePrincipalName" in sp_abuse_results.columns and "HourlyIPs" in sp_abuse_results.columns:
        top_abusers = sp_abuse_results.groupby("ServicePrincipalName", observed=True)["HourlyIPs"].max().reset_index().sort_values("HourlyIPs", ascending=False).head(15)
        fig_abuse = px.bar(top_abusers, x="ServicePrincipalName", y="HourlyIPs",
                           title="Top Service Principals — Max Unique IPs per Hour",
                           color="HourlyIPs", color_continuous_scale="Reds")
        fig_abuse.update_layout(xaxis_tickangle=-40)
        fig_abuse.show()
else:
    print("\n  No Service Principal credential abuse patterns detected")

# ── Store for downstream sections ──────────────────────────────────────────────
oauth_results = {
    "consent":     oauth_consent_results,
    "sp_geo":      sp_geo_results,
    "sp_highrisk": sp_highrisk_results,
    "sp_abuse":    sp_abuse_results,
}
print(f"\n{'='*72}")
print("  Tip: Revoke risky consent grants via Entra ID > Enterprise Applications")
print("  Tip: Block new OAuth app consents via Admin Consent Policy in Entra ID")
print(f"{'='*72}")


🔍 Hunting suspicious OAuth consent grants...
   ✅ Returned 200 rows
🔍 Analysing Service Principal sign-in geolocations...
   ✅ Returned 37 rows
🔍 Identifying high-risk OAuth app permissions...
   ✅ Returned 61 rows
🔍 Detecting Service Principal credential abuse patterns...
   ✅ Returned 100 rows

OAUTH APP & SERVICE PRINCIPAL THREAT HUNTING — SUMMARY

  Consent Grants:  200 events  (0 high-risk sensitive-permission grants)


,TimeGenerated,OperationName,RiskLevel,Actor,ActorIP,Country,City,AppName,Permissions
0,2026-02-20 20:01:46.271516+00:00,Consent to application,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Fabric data-risk assessment scanning,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
1,2026-02-20 20:01:46.261515+00:00,Add delegated permission grant,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
2,2026-02-20 18:06:41.317829+00:00,Consent to application,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Fabric data-risk assessment scanning,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
3,2026-02-20 18:06:41.265833+00:00,Add delegated permission grant,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
4,2026-02-20 18:06:41.225832+00:00,Add app role assignment to service principal,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
5,2026-02-20 18:06:41.161824+00:00,Add app role assignment to service principal,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
6,2026-02-20 18:06:41.102829+00:00,Add app role assignment to service principal,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
7,2026-02-20 18:06:41.041828+00:00,Add app role assignment to service principal,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
8,2026-02-20 18:06:40.984825+00:00,Add app role assignment to service principal,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."
9,2026-02-20 18:06:40.926824+00:00,Add app role assignment to service principal,Standard,u411@a.alpineskihouse.co,20.97.10.99,United States,San Antonio,Microsoft Graph,"[{""key"":""User-Agent"",""value"":""Mozilla/5.0 (Win..."



  Service Principal Geo Anomalies:  37 SPs  (0 critical — 5+ countries)


,ServicePrincipalName,ServicePrincipalId,AppId,TotalSignIns,UniqueIPs,Countries,Cities,SampleIPs,FirstSeen,LastSeen,CountryCount,GeoRisk
0,CDOT Infra,27bf9e71-106d-49bd-b417-818d8ae59794,3bb8dbcb-ee30-4a08-bc75-e85d832e926b,9787,14,"[""Netherlands"",""United States"",""United Kingdom...","[""Amsterdam"",""Washington"",""Boydton"",""London"",""...","[""40.68.200.63"",""74.235.241.47"",""172.176.74.14...",2025-11-26 15:42:52.625788+00:00,2026-02-20 09:12:43.157062+00:00,4,High — 3-4 Countries
1,SecurityCopilotAgentIdentity-dfb94675-3650-4b1...,ae4c883f-febd-4abb-aad1-48c0872770cc,ae4c883f-febd-4abb-aad1-48c0872770cc,163,8,"[""France"",""Sweden"",""Ireland"",""Germany""]","[""Paris"","""",""Dublin"",""Frankfurt am Main""]","[""4.251.29.11"",""135.225.147.168"",""72.145.24.32...",2026-01-23 22:41:29.390031+00:00,2026-02-20 19:45:43.909937+00:00,4,High — 3-4 Countries
2,SecurityCopilotAgentIdentity-a8c2716a-fb00-468...,7c3f06d3-321b-41d3-a306-85851079bf53,7c3f06d3-321b-41d3-a306-85851079bf53,1455,8,"[""Germany"",""Ireland"",""Sweden"",""France""]","[""Frankfurt am Main"",""Dublin"","""",""Paris""]","[""4.182.156.104"",""72.145.24.35"",""72.145.24.32""...",2026-01-05 22:01:07.712356+00:00,2026-02-22 23:01:55.105400+00:00,4,High — 3-4 Countries
3,SecurityCopilotAgentIdentity-2b44e0be-aa16-4df...,897d38e2-23d0-4df9-9d12-2c89b9e542c7,897d38e2-23d0-4df9-9d12-2c89b9e542c7,1667,8,"[""Germany"",""Ireland"",""France"",""Sweden""]","[""Frankfurt am Main"",""Dublin"",""Paris"",""""]","[""4.182.156.104"",""72.145.24.32"",""4.251.29.11"",...",2026-01-13 18:45:44.492903+00:00,2026-02-22 23:23:57.481404+00:00,4,High — 3-4 Countries
4,SecurityCopilotAgentIdentity-300f6453-208f-4eb...,ad490577-b1e3-4690-81b5-fe42381ef9fd,ad490577-b1e3-4690-81b5-fe42381ef9fd,862,8,"[""Germany"",""Ireland"",""France"",""Sweden""]","[""Frankfurt am Main"",""Dublin"",""Paris"",""""]","[""4.182.156.104"",""72.145.24.32"",""4.251.29.11"",...",2026-01-26 16:29:42.507779+00:00,2026-02-22 23:49:22.224708+00:00,4,High — 3-4 Countries
5,SecurityCopilotAgentIdentity-22c8adb7-3d74-41a...,85f59bde-6844-4270-9f84-8a551d589efc,85f59bde-6844-4270-9f84-8a551d589efc,146,8,"[""Germany"",""Sweden"",""France"",""Ireland""]","[""Frankfurt am Main"","""",""Paris"",""Dublin""]","[""4.182.156.104"",""135.225.147.171"",""4.251.29.1...",2026-01-08 16:59:32.163592+00:00,2026-02-20 20:26:34.210029+00:00,4,High — 3-4 Countries
6,SecurityCopilotAgentIdentity-8f1a8f06-e84d-4ab...,0e342b5b-6795-4cce-b117-a42b4d54e544,0e342b5b-6795-4cce-b117-a42b4d54e544,514,7,"[""Sweden"",""Germany"",""Ireland"",""France""]","["""",""Frankfurt am Main"",""Dublin"",""Paris""]","[""135.225.147.171"",""4.182.156.104"",""135.225.14...",2026-01-05 21:52:47.230772+00:00,2026-01-22 16:03:08.820880+00:00,4,High — 3-4 Countries
7,SecurityCopilotAgentIdentity-233ff2b7-781b-494...,dcda3839-0218-40d3-a28e-a23adb851d90,dcda3839-0218-40d3-a28e-a23adb851d90,41,6,"[""Sweden"",""Ireland"",""France"",""Germany""]","["""",""Dublin"",""Paris"",""Frankfurt am Main""]","[""135.225.147.168"",""72.145.24.32"",""4.251.29.11...",2026-02-11 09:22:03.630992+00:00,2026-02-22 09:23:26.381821+00:00,4,High — 3-4 Countries
8,Mimik Emails - AlpineSkiHouse,9944cde6-68dc-42e4-b629-71c91d18e3a9,5c571357-f3bf-44e2-af0e-4813f949a8b8,212063,97,"[""United States"",""Netherlands"",""France""]","[""Washington"",""Boydton"",""Amsterdam"","""",""Issy-l...","[""20.241.180.136"",""20.241.180.93"",""20.241.180....",2025-11-25 11:17:39.958865+00:00,2026-02-22 23:58:55.247894+00:00,3,High — 3-4 Countries
9,Micr0s0ft-NinjaApp,d02d85a9-770e-42af-a61b-d2e4619ab882,c3a3d820-cdf9-4e51-880a-c33cba0f7d71,7,3,"[""Sweden"",""Netherlands"",""Germany""]","["""",""Amsterdam"",""Frankfurt am Main""]","[""45.84.107.33"",""45.66.35.21"",""107.189.12.3""]",2025-12-10 12:51:35.823208+00:00,2025-12-10 13:16:03.720596+00:00,3,High — 3-4 Countries



  OAuth Apps with High-Risk Permissions:  0 / 61 apps analysed


,ServicePrincipalName,AppId,ResourceDisplayName,SignInCount,UniqueIPs,RecentCountries,PermRisk,HighRiskPerms,LastSeen



  SP Credential Abuse Signals:  100 patterns  (1 critical)


,ServicePrincipalName,AppId,TimeGenerated,HourlyIPs,Countries,IPs,CountryCount,AbuseSignal
0,Microsoft Cloud App Security (Internal),25a6a87d-1e19-4c71-9cb0-16e88ff608f1,2025-12-21 11:00:00+00:00,13,"[""""]","[""fd00:4ab1:6a02:5149:6f1b:200:a18:f93"",""fd00:...",1,Critical — Likely Credential Abuse
1,Mimik Emails - AlpineSkiHouse,5c571357-f3bf-44e2-af0e-4813f949a8b8,2025-12-09 11:00:00+00:00,8,"[""United States""]","[""20.241.180.93"",""20.241.180.89"",""20.15.17.134...",1,High — Suspicious Multi-IP Auth
2,Mimik Emails - AlpineSkiHouse,5c571357-f3bf-44e2-af0e-4813f949a8b8,2025-12-10 09:00:00+00:00,8,"[""United States""]","[""20.241.180.136"",""20.15.17.134"",""48.211.229.1...",1,High — Suspicious Multi-IP Auth
3,Mimik Emails - AlpineSkiHouse,5c571357-f3bf-44e2-af0e-4813f949a8b8,2026-01-08 10:00:00+00:00,8,"[""United States""]","[""20.241.180.89"",""20.241.180.93"",""52.251.9.85""...",1,High — Suspicious Multi-IP Auth
4,MonitoringAutomation,5ca795cf-55c4-4f58-b39e-bdb33b15ec27,2025-12-29 17:00:00+00:00,7,"[""United States""]","[""20.246.140.26"",""48.194.119.247"",""52.190.34.1...",1,High — Suspicious Multi-IP Auth
5,MonitoringAutomation,5ca795cf-55c4-4f58-b39e-bdb33b15ec27,2026-01-19 17:00:00+00:00,7,"[""United States""]","[""20.246.140.26"",""48.194.45.8"",""135.222.240.20...",1,High — Suspicious Multi-IP Auth
6,MonitoringAutomation,5ca795cf-55c4-4f58-b39e-bdb33b15ec27,2026-02-16 17:00:00+00:00,7,"[""United States""]","[""20.246.140.26"",""172.212.120.34"",""51.8.240.49...",1,High — Suspicious Multi-IP Auth
7,Mimik Emails - AlpineSkiHouse,5c571357-f3bf-44e2-af0e-4813f949a8b8,2025-12-09 20:00:00+00:00,7,"[""United States""]","[""20.241.180.136"",""20.161.124.232"",""172.212.11...",1,High — Suspicious Multi-IP Auth
8,MonitoringAutomation,5ca795cf-55c4-4f58-b39e-bdb33b15ec27,2026-01-05 17:00:00+00:00,7,"[""United States""]","[""20.246.140.26"",""48.216.138.42"",""20.253.101.5...",1,High — Suspicious Multi-IP Auth
9,Mimik Emails - AlpineSkiHouse,5c571357-f3bf-44e2-af0e-4813f949a8b8,2025-12-02 11:00:00+00:00,7,"[""United States""]","[""20.241.180.89"",""4.153.121.20"",""20.161.93.76""...",1,High — Suspicious Multi-IP Auth



  Tip: Revoke risky consent grants via Entra ID > Enterprise Applications
  Tip: Block new OAuth app consents via Admin Consent Policy in Entra ID


---
## 11. Defender for Office 365 — Email & Collaboration Threat Hunting

Hunt for threats across email, Microsoft Teams, SharePoint, and OneDrive using Defender for Office 365 telemetry.

**Use Cases Covered:**
| # | Use Case | Detection Method |
|---|----------|-----------------|
| 1 | **Spam & Phishing** | High-volume delivery, phishing verdicts, impersonation |
| 2 | **Malware in Email** | Attachment detonation verdicts, malicious file types |
| 3 | **Malware in Teams / SharePoint / OneDrive** | CloudAppEvents file threat signals |
| 4 | **Geolocation Anomalies** | Sender/recipient country mismatch, unusual source regions |
| 5 | **Business Email Compromise (BEC)** | Inbox rule creation, forwarding rules, impersonation patterns |
| 6 | **Adversary-in-the-Middle (AiTM)** | MFA-bypass token theft: successful auth without MFA + new IP/ASN |

**Key Tables:**
`EmailEvents`, `EmailAttachmentInfo`, `EmailUrlInfo`, `UrlClickEvents`, `CloudAppEvents`, `AuditLogs`


In [34]:

# ── 11.1  Spam & Phishing Detection ─────────────────────────────────────────
import importlib, sys
for _mod in ["pandas","plotly.express","collections"]:
    if _mod.split(".")[0] not in sys.modules:
        importlib.import_module(_mod.split(".")[0])
import pandas as pd
import plotly.express as px
from collections import Counter

spam_phishing_query = """
EmailEvents
| where TimeGenerated >= ago(365d)
| where DeliveryAction != "Delivered"
    or ThreatTypes has_any ("Phish", "Spam", "Malware")
    or tostring(column_ifexists("PhishFilterVerdict",""))  in ("Phish", "HighConfidencePhish")
    or tostring(column_ifexists("SpamFilterVerdict",""))   in ("Spam",  "HighConfidenceSpam")
| extend SenderCountry = tostring(geo_info_from_ip_address(SenderIPv4).country)
| extend PhishVerdict = tostring(column_ifexists("PhishFilterVerdict",""))
| extend SpamVerdict  = tostring(column_ifexists("SpamFilterVerdict",""))
| extend ThreatCategory = case(
    PhishVerdict in ("Phish","HighConfidencePhish"), "Phishing",
    ThreatTypes has "Phish",  "Phishing",
    SpamVerdict  in ("Spam","HighConfidenceSpam"),   "Spam",
    ThreatTypes has "Spam",   "Spam",
    ThreatTypes has "Malware","Malware",
    "Other Blocked"
)
| summarize TotalMessages=count(), UniqueTargets=dcount(RecipientEmailAddress),
    UniqueSenders=dcount(SenderFromAddress),
    SenderCountries=make_set(SenderCountry,10),
    FirstSeen=min(TimeGenerated), LastSeen=max(TimeGenerated)
    by ThreatCategory, SenderIPv4
| extend ThreatScore = iff(TotalMessages>1000 or UniqueTargets>100,"High",
                       iff(TotalMessages>100  or UniqueTargets>20, "Medium","Low"))
| order by TotalMessages desc
| take 100
"""

spam_phishing_results = run_kql(spam_phishing_query, "Section 11.1 – Spam & Phishing")

if not spam_phishing_results.empty:
    print(f"\n  Found {len(spam_phishing_results)} spam/phishing source IPs")
    display(spam_phishing_results.head(20))

    if "ThreatCategory" in spam_phishing_results.columns and "TotalMessages" in spam_phishing_results.columns:
        cat_df = spam_phishing_results.groupby("ThreatCategory",observed=True)["TotalMessages"].sum().reset_index()
        px.bar(cat_df, x="ThreatCategory", y="TotalMessages", color="ThreatCategory",
               title="Email Threats by Category").show()

    country_counts = Counter()
    if "SenderCountries" in spam_phishing_results.columns:
        for val in spam_phishing_results["SenderCountries"]:
            if isinstance(val, list):
                country_counts.update([c for c in val if c])
            elif isinstance(val, str) and val:
                country_counts[val] += 1
    if country_counts:
        cc_df = pd.DataFrame(country_counts.most_common(15), columns=["Country","Count"])
        px.bar(cc_df, x="Country", y="Count", title="Top Threat Source Countries",
               color="Count", color_continuous_scale="Reds").show()
else:
    print("No spam/phishing data found (EmailEvents table may not be licensed)")


🔍 Section 11.1 – Spam & Phishing...
   ✅ Returned 100 rows

  Found 100 spam/phishing source IPs


,ThreatCategory,SenderIPv4,TotalMessages,UniqueTargets,UniqueSenders,SenderCountries,FirstSeen,LastSeen,ThreatScore
0,Other Blocked,,107190,7130,1349,"[""""]",2025-08-27 12:35:17+00:00,2026-02-23 11:09:18+00:00,High
1,Other Blocked,40.126.23.162,7482,2847,4256,"[""United States""]",2025-08-27 11:27:06+00:00,2026-02-23 07:21:07+00:00,High
2,Other Blocked,20.190.151.37,7092,2849,4104,"[""United States""]",2025-08-27 12:06:27+00:00,2026-02-21 11:42:09+00:00,High
3,Other Blocked,40.126.23.163,6997,2798,4079,"[""United States""]",2025-08-27 12:51:07+00:00,2026-02-22 00:25:03+00:00,High
4,Other Blocked,40.126.23.96,6997,2812,4119,"[""United States""]",2025-08-27 14:54:05+00:00,2026-02-21 21:54:09+00:00,High
5,Other Blocked,40.126.23.38,6955,2782,4073,"[""United States""]",2025-08-28 00:51:06+00:00,2026-02-23 04:27:07+00:00,High
6,Other Blocked,20.190.151.38,6838,2777,3997,"[""United States""]",2025-08-27 12:18:04+00:00,2026-02-23 08:46:05+00:00,High
7,Other Blocked,40.126.23.97,6796,2742,4096,"[""United States""]",2025-08-27 13:12:02+00:00,2026-02-23 07:39:10+00:00,High
8,Other Blocked,40.126.23.26,6782,2717,4005,"[""United States""]",2025-08-27 13:33:06+00:00,2026-02-22 06:58:02+00:00,High
9,Other Blocked,20.190.151.101,6738,2736,4104,"[""United States""]",2025-08-27 16:42:06+00:00,2026-02-23 05:04:03+00:00,High


In [35]:

# ── 11.2  Malware in Email Attachments ───────────────────────────────────────
import pandas as pd
import plotly.express as px

email_malware_query = """
EmailAttachmentInfo
| where TimeGenerated >= ago(365d)
| where isnotempty(ThreatNames)
    or ThreatTypes has "Malware"
    or tostring(DetectionMethods) has "detonation"
| join kind=leftouter (
    EmailEvents
    | where TimeGenerated >= ago(365d)
    | project NetworkMessageId, RecipientEmailAddress, SenderFromAddress,
              SenderIPv4, Subject, DeliveryAction, DeliveryLocation
) on NetworkMessageId
| extend SenderCountry = tostring(geo_info_from_ip_address(SenderIPv4).country)
| extend SeverityLevel = case(
    DeliveryAction == "Delivered", "CRITICAL - Delivered",
    DeliveryAction == "Junked",    "High - Junked",
    "Blocked/Quarantined"
)
| project TimeGenerated, SeverityLevel, Subject, FileName, FileType,
    MalwareFamily   = tostring(ThreatNames),
    ThreatTypes     = tostring(ThreatTypes),
    SenderFromAddress, SenderIPv4, SenderCountry,
    RecipientEmailAddress, DeliveryAction, SHA256
| order by TimeGenerated desc
| take 100
"""

email_malware_results = run_kql(email_malware_query, "Section 11.2 – Email Malware")

if not email_malware_results.empty:
    print(f"\n  Found {len(email_malware_results)} email malware detections")
    if "DeliveryAction" in email_malware_results.columns:
        delivered = email_malware_results[email_malware_results["DeliveryAction"] == "Delivered"]
        if not delivered.empty:
            print(f"  !! {len(delivered)} malware samples DELIVERED to inboxes!")
    display(email_malware_results.head(20))

    if "MalwareFamily" in email_malware_results.columns:
        mf = (email_malware_results["MalwareFamily"]
              .replace("", pd.NA).dropna()
              .value_counts().head(10).reset_index())
        mf.columns = ["MalwareFamily", "Count"]
        if not mf.empty:
            px.bar(mf, x="MalwareFamily", y="Count",
                   title="Top Malware Families in Email",
                   color="Count", color_continuous_scale="Reds").show()
else:
    print("No email malware data (EmailAttachmentInfo table may not be licensed)")


🔍 Section 11.2 – Email Malware...
   ✅ Returned 100 rows

  Found 100 email malware detections
  !! 99 malware samples DELIVERED to inboxes!


,TimeGenerated,SeverityLevel,Subject,FileName,FileType,MalwareFamily,ThreatTypes,SenderFromAddress,SenderIPv4,SenderCountry,RecipientEmailAddress,DeliveryAction,SHA256
0,2025-10-21 23:38:13+00:00,CRITICAL - Delivered,Immediate Attention Required - Overdue Annual ...,Eicar.txt,txt;text,Malicious Payload,Malware,u6446@int.zava-corp.com,20.190.151.37,United States,u3944@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
1,2025-10-21 23:21:18+00:00,CRITICAL - Delivered,Immediate Attention Required - Overdue Annual ...,zippedeicar.txt,txt;text,Malicious Payload,Malware,u11493@int.zava-corp.com,20.190.151.38,United States,u2517@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
2,2025-10-21 23:19:10+00:00,CRITICAL - Delivered,Launching Our Eco-Friendly Campaign - 2025,zippedeicar.txt,txt;text,Malicious Payload,Malware,u9624@int.zava-corp.com,40.126.23.97,United States,u10971@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
3,2025-10-21 22:44:09+00:00,CRITICAL - Delivered,New design spec Process | SpecDis-2352,zippedeicar.txt,txt;text,Malicious Payload,Malware,u4526@int.zava-corp.com,20.190.151.38,United States,u5519@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
4,2025-10-21 22:33:14+00:00,CRITICAL - Delivered,New design spec Process | SpecDis-2167,zippedeicar.txt,txt;text,Malicious Payload,Malware,u1664@int.zava-corp.com,40.126.23.38,United States,u5567@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
5,2025-10-21 22:21:11+00:00,CRITICAL - Delivered,Experience the Future with Contoso Personal De...,zippedeicar.txt,txt;text,Malicious Payload,Malware,u11248@int.zava-corp.com,20.190.151.37,United States,u12236@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
6,2025-10-21 22:13:29+00:00,CRITICAL - Delivered,New design spec Process | SpecDis-1685,zippedeicar.txt,txt;text,Malicious Payload,Malware,u2700@int.zava-corp.com,20.190.151.38,United States,u2324@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
7,2025-10-21 22:13:10+00:00,CRITICAL - Delivered,Document Review Request,zippedeicar.txt,txt;text,Malicious Payload,Malware,u2904@int.zava-corp.com,20.190.151.100,United States,u4205@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
8,2025-10-21 22:11:07+00:00,CRITICAL - Delivered,Proposal for Collaboration 156,zippedeicar.txt,txt;text,Malicious Payload,Malware,u5552@int.zava-corp.com,40.126.23.162,United States,u9641@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...
9,2025-10-21 22:03:20+00:00,CRITICAL - Delivered,Immediate Attention Required - Overdue Annual ...,Eicar.txt,txt;text,Malicious Payload,Malware,u1206@int.zava-corp.com,20.190.152.153,United States,u873@int.zava-corp.com,Delivered,2e45cc8103b477050bcbd483f0a672107653dcd3dc3e65...


In [36]:

# ── 11.3  Malware in Teams / SharePoint / OneDrive ───────────────────────────
# Sources: FileMaliciousContentInfo (primary — MDO file detections)
#          CloudAppEvents (secondary — activity-level signals)
import pandas as pd
import plotly.express as px

# ── Primary: FileMaliciousContentInfo ────────────────────────────────────────
fmci_query = """
FileMaliciousContentInfo
| where TimeGenerated >= ago(365d)
| extend _Workload    = tostring(column_ifexists("Workload", ""))
| extend _ThreatName  = tostring(column_ifexists("ThreatName", ""))
| extend _FileName    = tostring(column_ifexists("FileName", ""))
| extend _FileType    = tostring(column_ifexists("FileType", ""))
| extend _SHA256      = tostring(column_ifexists("SHA256", ""))
| extend _SiteUrl     = tostring(column_ifexists("SiteUrl", column_ifexists("ContentUrl", "")))
| extend Application = case(
    _Workload == "Teams",      "Microsoft Teams",
    _Workload == "SharePoint", "Microsoft SharePoint Online",
    _Workload == "OneDrive",   "Microsoft OneDrive for Business",
    isnotempty(_Workload),     _Workload,
    "Other M365")
| extend RiskLevel = case(
    isnotempty(_ThreatName), strcat("Malware: ", _ThreatName),
    "Malicious File Detected")
| project TimeGenerated, Source="FileMaliciousContentInfo",
    RiskLevel, Application,
    ActionType = "MaliciousFileDetected",
    FileName   = _FileName,
    FileType   = _FileType,
    ThreatName = _ThreatName,
    SHA256     = _SHA256,
    SiteUrl    = _SiteUrl,
    FileUrl    = _SiteUrl,
    AccountDisplayName = "", IPAddress = "", UserCountry = ""
| order by TimeGenerated desc
"""

# ── Secondary: CloudAppEvents ─────────────────────────────────────────────────
cae_query = """
CloudAppEvents
| where TimeGenerated >= ago(365d)
| where Application in ("Microsoft Teams","Microsoft SharePoint Online",
                        "Microsoft OneDrive for Business","Microsoft OneDrive")
| where ActionType in ("FileMalwareDetected","AntiVirusScanCompleted",
                       "AntiVirusScanFailed","AntiVirusFileDeleted")
    or tostring(RawEventData) has_any ("malware","virus","threat","infected")
| extend FileName    = coalesce(tostring(todynamic(RawEventData).SourceFileName),
                                tostring(todynamic(RawEventData).ObjectId), "Unknown")
| extend UserCountry = tostring(geo_info_from_ip_address(IPAddress).country)
| extend RiskLevel   = case(
    ActionType == "FileMalwareDetected",  "Malware Detected",
    ActionType == "AntiVirusFileDeleted", "File Quarantined",
    Application == "Microsoft Teams",     "Teams Activity",
    "SharePoint/OneDrive Activity")
| project TimeGenerated, Source="CloudAppEvents",
    RiskLevel, Application, ActionType,
    FileName, FileType="",
    ThreatName="", SHA256="",
    SiteUrl="", FileUrl="",
    AccountDisplayName, IPAddress, UserCountry
| order by TimeGenerated desc
| take 100
"""

print("Running collaboration malware queries...")
fmci_results = run_kql(fmci_query, "Section 11.3 – FileMaliciousContentInfo")
cae_results  = run_kql(cae_query,  "Section 11.3 – CloudAppEvents")

frames = [df for df in [fmci_results, cae_results] if not df.empty]
collab_malware_results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if not collab_malware_results.empty:
    print(f"\n  Found {len(collab_malware_results)} collaboration platform threat events")
    print(f"    FileMaliciousContentInfo : {len(fmci_results) if not fmci_results.empty else 0}")
    print(f"    CloudAppEvents           : {len(cae_results)  if not cae_results.empty  else 0}")

    confirmed = collab_malware_results[
        collab_malware_results["ActionType"].isin(["FileMalwareDetected", "AntiVirusFileDeleted", "MaliciousFileDetected"])
    ]
    if not confirmed.empty:
        print(f"  !! {len(confirmed)} confirmed malware/quarantine events!")

    display(collab_malware_results.head(30))

    # Breakdown by platform
    if "Application" in collab_malware_results.columns:
        app_df = collab_malware_results.groupby("Application", observed=True).size().reset_index(name="Count")
        px.pie(app_df, names="Application", values="Count",
               title="11.3 — Collaboration Threat Events by Platform").show()

    # Top threat names from FileMaliciousContentInfo
    if not fmci_results.empty and "ThreatName" in fmci_results.columns:
        tn = (fmci_results["ThreatName"].replace("", pd.NA).dropna()
              .value_counts().head(10).reset_index())
        tn.columns = ["ThreatName", "Count"]
        if not tn.empty:
            px.bar(tn, x="ThreatName", y="Count",
                   title="Top Threat Names in Collaboration Files (FileMaliciousContentInfo)",
                   color="Count", color_continuous_scale="Reds").show()
else:
    print("No collaboration malware data (FileMaliciousContentInfo / CloudAppEvents may not be available)")


Running collaboration malware queries...
🔍 Section 11.3 – FileMaliciousContentInfo...
   ✅ Returned 0 rows
🔍 Section 11.3 – CloudAppEvents...
   ✅ Returned 100 rows

  Found 100 collaboration platform threat events
    FileMaliciousContentInfo : 0
    CloudAppEvents           : 100


,TimeGenerated,Source,RiskLevel,Application,ActionType,FileName,FileType,ThreatName,SHA256,SiteUrl,FileUrl,AccountDisplayName,IPAddress,UserCountry
0,2026-02-11 22:15:53+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,FileAccessed,T-a05a069f-e10d-47a9-a998-81f648d9a102-Vulnera...,,,,,,Caleb Nzimande,4.208.178.91,Ireland
1,2026-02-11 22:15:38+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,FileAccessed,T-a05a069f-e10d-47a9-a998-81f648d9a102-Vulnera...,,,,,,Caleb Nzimande,4.208.178.91,Ireland
2,2026-02-11 22:15:38+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,FileAccessed,T-a05a069f-e10d-47a9-a998-81f648d9a102-Vulnera...,,,,,,Caleb Nzimande,4.208.178.91,Ireland
3,2026-02-11 22:14:13+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft OneDrive for Business,FileAccessed,T-a05a069f-e10d-47a9-a998-81f648d9a102-Vulnera...,,,,,,Caleb Nzimande,4.208.178.91,Ireland
4,2026-02-05 21:14:25+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,DLPRuleMatch,d723d806-39eb-40d0-90e5-97ab4dfb1cf8,,,,,,Gabriel Valentova,,
5,2026-02-05 21:14:25+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,DLPRuleMatch,d723d806-39eb-40d0-90e5-97ab4dfb1cf8,,,,,,Gabriel Valentova,,
6,2026-02-05 21:14:23+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,DLPRuleMatch,e9e2fa24-3210-4db6-ac2b-4dbdd5f99143,,,,,,Gabriel Valentova,,
7,2026-02-05 21:14:23+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,DLPRuleMatch,e9e2fa24-3210-4db6-ac2b-4dbdd5f99143,,,,,,Gabriel Valentova,,
8,2026-02-05 18:05:52+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,UnifiedSimulationRuleMatch,SPO_ZjM5YWNlZjctMTljMC00ZDYwLWJlZTYtYzFlOTE1Nj...,,,,,,,,
9,2026-02-05 18:05:51+00:00,CloudAppEvents,SharePoint/OneDrive Activity,Microsoft SharePoint Online,UnifiedSimulationRuleMatch,SPO_ZjM5YWNlZjctMTljMC00ZDYwLWJlZTYtYzFlOTE1Nj...,,,,,,,,


In [37]:

# ── 11.4  Business Email Compromise (BEC) Detection ──────────────────────────
import pandas as pd
import plotly.express as px

# Query 1: Suspicious inbox/transport rules via OfficeActivity
bec_rules_query = """
OfficeActivity
| where TimeGenerated >= ago(365d)
| where RecordType in ("ExchangeAdmin","MicrosoftTeams")
| where Operation in ("New-InboxRule","Set-InboxRule","UpdateInboxRules",
                      "New-TransportRule","Set-TransportRule")
| where tostring(Parameters) has_any ("ForwardTo","RedirectTo","DeleteMessage",
                                       "ForwardAsAttachmentTo","MoveToFolder")
    or tostring(AffectedItems) has_any ("ForwardTo","RedirectTo","DeleteMessage")
| extend UserCountry = tostring(geo_info_from_ip_address(ClientIP).country)
| project TimeGenerated, BECSignal="Suspicious Inbox Rule",
    User=UserId, Operation,
    Details=tostring(Parameters),
    IPAddress=ClientIP, UserCountry
| order by TimeGenerated desc
| take 50
"""

# Query 2: High-volume external email forwarding via EmailEvents
bec_forward_query = """
EmailEvents
| where TimeGenerated >= ago(365d)
| where EmailDirection == "Outbound"
| extend SenderDomain = extract("@(.+)$", 1, SenderFromAddress)
| where isnotempty(SenderDomain)
| where RecipientEmailAddress !endswith SenderDomain
| summarize ForwardCount=count(),
    ExternalRecipients=make_set(RecipientEmailAddress,10),
    FirstSeen=min(TimeGenerated)
    by SenderFromAddress
| where ForwardCount > 20
| order by ForwardCount desc
| project TimeGenerated=FirstSeen, BECSignal="High External Forwarding",
    User=SenderFromAddress, Operation="EmailForwarding",
    Details=tostring(ExternalRecipients),
    IPAddress="", UserCountry=""
| take 25
"""

print("Running BEC detection queries...")
bec_rules   = run_kql(bec_rules_query,   "BEC – inbox/transport rules")
bec_forward = run_kql(bec_forward_query, "BEC – external forwarding")

bec_frames = [df for df in [bec_rules, bec_forward] if not df.empty]
bec_results = pd.concat(bec_frames, ignore_index=True) if bec_frames else pd.DataFrame()

if not bec_results.empty:
    print(f"\n  Found {len(bec_results)} BEC indicators")
    print(f"  Suspicious rules:   {len(bec_rules) if not bec_rules.empty else 0}")
    print(f"  External forwarding:{len(bec_forward) if not bec_forward.empty else 0}")
    display(bec_results)
    sig_df = bec_results.groupby("BECSignal",observed=True).size().reset_index(name="Count")
    px.bar(sig_df, x="BECSignal", y="Count", color="BECSignal",
           title="BEC Indicators by Type").show()
else:
    print("No BEC indicators detected (OfficeActivity/EmailEvents may not be available)")


Running BEC detection queries...
🔍 BEC – inbox/transport rules...
   ✅ Returned 9 rows
🔍 BEC – external forwarding...
   ✅ Returned 25 rows

  Found 34 BEC indicators
  Suspicious rules:   9
  External forwarding:25


,TimeGenerated,BECSignal,User,Operation,Details,IPAddress,UserCountry
0,2026-01-26 06:54:26+00:00,Suspicious Inbox Rule,u9245@int.zava-corp.com,New-InboxRule,"[{""Name"":""AlwaysDeleteOutlookRulesBlob"",""Value...",57.158.176.248:29791,
1,2026-01-26 06:53:52+00:00,Suspicious Inbox Rule,u9245@int.zava-corp.com,New-InboxRule,"[{""Name"":""AlwaysDeleteOutlookRulesBlob"",""Value...",57.158.176.248:14634,
2,2025-11-10 18:09:23+00:00,Suspicious Inbox Rule,u101@a.alpineskihouse.co,Set-TransportRule,"[{""Name"":""ExceptIfRecipientDomainIs"",""Value"":""...",20.250.113.9:18583,
3,2025-11-10 18:04:58+00:00,Suspicious Inbox Rule,u101@a.alpineskihouse.co,Set-TransportRule,"[{""Name"":""ExceptIfRecipientDomainIs"",""Value"":""...",20.250.113.9:13081,
4,2025-10-23 14:44:48+00:00,Suspicious Inbox Rule,u101@a.alpineskihouse.co,Set-TransportRule,"[{""Name"":""Name"",""Value"":""Users can't receive i...",20.250.113.9:11599,
5,2025-10-22 08:56:24+00:00,Suspicious Inbox Rule,leeg@vnevado.alpineskihouse.co,New-InboxRule,"[{""Name"":""AlwaysDeleteOutlookRulesBlob"",""Value...",185.220.101.23:20896,
6,2025-09-11 08:47:48+00:00,Suspicious Inbox Rule,leeg@vnevado.alpineskihouse.co,New-InboxRule,"[{""Name"":""AlwaysDeleteOutlookRulesBlob"",""Value...",192.42.116.18:15391,
7,2025-09-10 13:56:31+00:00,Suspicious Inbox Rule,leeg@vnevado.alpineskihouse.co,New-InboxRule,"[{""Name"":""AlwaysDeleteOutlookRulesBlob"",""Value...",45.84.107.222:29340,
8,2025-09-02 20:59:36+00:00,Suspicious Inbox Rule,u411@a.alpineskihouse.co,Set-TransportRule,"[{""Name"":""FromScope"",""Value"":""NotInOrganizatio...",20.81.125.41:25196,
9,2025-08-28 15:29:08+00:00,High External Forwarding,postmaster@zava-corp.com,EmailForwarding,"[""mcc365@microsoft.com"",""payroll@fabrikam.com""...",,


In [38]:

# ── 11.5  Adversary-in-the-Middle (AiTM) Detection ───────────────────────────
import pandas as pd
import plotly.express as px

# Signal A: SigninLogs – successful single-factor auth from multiple countries
aitm_signin_query = """
union isfuzzy=true SigninLogs, AADNonInteractiveUserSignInLogs
| where TimeGenerated >= ago(365d)
| where ResultType == 0
| where AuthenticationRequirement == "singleFactorAuthentication"
    or ConditionalAccessStatus     == "notApplied"
    or tostring(AuthenticationDetails) !has "MFA"
| extend Country = tostring(geo_info_from_ip_address(IPAddress).country)
| summarize SignInCount=count(), UniqueIPs=dcount(IPAddress),
    Countries=make_set(Country,10),
    Apps=make_set(AppDisplayName,5),
    FirstSeen=min(TimeGenerated), LastSeen=max(TimeGenerated),
    SampleIP=any(IPAddress)
    by UserPrincipalName
| extend CountryCount = array_length(Countries)
| extend AiTMRisk = case(
    CountryCount >= 3, "High – 3+ Countries",
    CountryCount == 2, "Medium – 2 Countries",
    "Low – Single Country"
)
| order by SignInCount desc
| take 100
"""

# Signal B: phishing link clicks (optional – not all workspaces have this table)
aitm_clicks_query = """
UrlClickEvents
| where TimeGenerated >= ago(365d)
| where ActionType == "ClickAllowed"
| where ThreatTypes has_any ("Phish","HighConfidencePhish")
| extend ClickCountry = tostring(geo_info_from_ip_address(IPAddress).country)
| project TimeGenerated,
    ClickUser    = iif(isnotempty(tostring(AccountUpn)), tostring(AccountUpn), "Unknown"),
    ClickIP      = IPAddress, ClickCountry, ClickURL = Url, NetworkMessageId
| order by TimeGenerated desc
| take 200
"""

print("Running AiTM detection queries...")
aitm_sign_results  = run_kql(aitm_signin_query,  "AiTM – MFA-bypass sign-ins")
aitm_click_results = run_kql(aitm_clicks_query,  "AiTM – phishing link clicks")

print("\n" + "="*60)
print("AiTM DETECTION SUMMARY")
print("="*60)

if not aitm_sign_results.empty:
    print(f"\n  MFA-bypass sign-ins: {len(aitm_sign_results)} users")
    if "AiTMRisk" in aitm_sign_results.columns:
        high = aitm_sign_results[aitm_sign_results["AiTMRisk"].str.startswith("High", na=False)]
        if not high.empty:
            print(f"  HIGH RISK: {len(high)} users sign in from 3+ countries without MFA!")
    display(aitm_sign_results.head(20))
    aitm_results = aitm_sign_results
else:
    print("  No MFA-bypass sign-in patterns detected")
    aitm_results = pd.DataFrame()

if not aitm_click_results.empty:
    print(f"\n  Phishing link clicks (UrlClickEvents): {len(aitm_click_results)}")
    display(aitm_click_results.head(10))
    if not aitm_sign_results.empty and "ClickUser" in aitm_click_results.columns:
        clickers = set(aitm_click_results["ClickUser"].dropna().str.lower())
        signers  = set(aitm_sign_results["UserPrincipalName"].dropna().str.lower())
        overlap  = clickers & signers
        if overlap:
            print(f"\n  {len(overlap)} user(s) clicked phishing link AND bypassed MFA:")
            for u in sorted(overlap):
                print(f"    * {u}")
else:
    print("  UrlClickEvents: no phishing clicks found (or table not available in this workspace)")

if not aitm_sign_results.empty and "AiTMRisk" in aitm_sign_results.columns:
    risk_df = aitm_sign_results.groupby("AiTMRisk",observed=True).size().reset_index(name="Count")
    px.bar(risk_df, x="AiTMRisk", y="Count", color="AiTMRisk",
           title="AiTM Risk – MFA-bypass Sign-ins by Risk Level",
           color_discrete_map={
               "High – 3+ Countries":  "#d63031",
               "Medium – 2 Countries": "#e17055",
               "Low – Single Country": "#fdcb6e"
           }).show()

    # ── Geolocation map — countries seen per AiTM user ────────────────────────
    import plotly.graph_objects as go

    # Explode the Countries array so each country gets its own row
    if "Countries" in aitm_sign_results.columns:
        geo_rows = []
        for _, row in aitm_sign_results.iterrows():
            countries = row.get("Countries", [])
            if isinstance(countries, str):
                import json as _json
                try:
                    countries = _json.loads(countries)
                except Exception:
                    countries = [countries]
            if not isinstance(countries, list):
                countries = [str(countries)]
            for c in countries:
                if c and c.strip().lower() not in ("", "none", "unknown", "null"):
                    geo_rows.append({
                        "Country": c.strip(),
                        "User": row.get("UserPrincipalName", ""),
                        "AiTMRisk": row.get("AiTMRisk", ""),
                        "SignInCount": int(row.get("SignInCount", 1)),
                    })
        if geo_rows:
            geo_df = pd.DataFrame(geo_rows)
            country_summary = (
                geo_df.groupby("Country", observed=True)
                .agg(UserCount=("User", "nunique"), TotalSignIns=("SignInCount", "sum"))
                .reset_index()
            )

            fig_map = px.choropleth(
                country_summary,
                locations="Country",
                locationmode="country names",
                color="UserCount",
                hover_name="Country",
                hover_data={"TotalSignIns": True, "UserCount": True},
                color_continuous_scale=[
                    [0.0, "#2d3436"],
                    [0.1, "#636e72"],
                    [0.4, "#fdcb6e"],
                    [1.0, "#d63031"],
                ],
                title="AiTM Risk – Geographic Distribution of MFA-bypass Sign-in Origins",
                labels={"UserCount": "Affected Users"},
            )
            fig_map.update_layout(
                geo=dict(
                    showframe=False,
                    showcoastlines=True,
                    coastlinecolor="#636e72",
                    showland=True, landcolor="#1e272e",
                    showocean=True, oceancolor="#0a0f14",
                    showcountries=True, countrycolor="#4a5568",
                    projection_type="natural earth",
                ),
                paper_bgcolor="#2d3436",
                font=dict(color="white", size=11),
                coloraxis_colorbar=dict(title="Users"),
                height=520,
            )
            fig_map.show()

            # Also show a scatter-geo for high-risk users with bubble sizing
            high_geo = geo_df[geo_df["AiTMRisk"].str.startswith("High", na=False)]
            if not high_geo.empty:
                high_summary = (
                    high_geo.groupby("Country", observed=True)
                    .agg(UserCount=("User", "nunique"), TotalSignIns=("SignInCount", "sum"))
                    .reset_index()
                )
                fig_bubble = px.scatter_geo(
                    high_summary,
                    locations="Country",
                    locationmode="country names",
                    size="TotalSignIns",
                    color="UserCount",
                    hover_name="Country",
                    hover_data={"TotalSignIns": True, "UserCount": True},
                    color_continuous_scale="Reds",
                    title="AiTM HIGH-RISK Sign-in Origins (3+ Countries, No MFA) — Bubble = Sign-in Volume",
                    labels={"UserCount": "Affected Users", "TotalSignIns": "Sign-ins"},
                    size_max=45,
                )
                fig_bubble.update_layout(
                    geo=dict(
                        showframe=False, showcoastlines=True, coastlinecolor="#636e72",
                        showland=True, landcolor="#1e272e",
                        showocean=True, oceancolor="#0a0f14",
                        showcountries=True, countrycolor="#4a5568",
                        projection_type="natural earth",
                    ),
                    paper_bgcolor="#2d3436",
                    font=dict(color="white", size=11),
                    height=520,
                )
                fig_bubble.show()
        else:
            print("  No country data available for geolocation map")

# Phishing click origins map
if not aitm_click_results.empty and "ClickCountry" in aitm_click_results.columns:
    click_geo = (
        aitm_click_results[aitm_click_results["ClickCountry"].notna() &
                           (aitm_click_results["ClickCountry"] != "")]
        .groupby("ClickCountry", observed=True)
        .size().reset_index(name="Clicks")
    )
    if not click_geo.empty:
        fig_clicks = px.choropleth(
            click_geo,
            locations="ClickCountry",
            locationmode="country names",
            color="Clicks",
            hover_name="ClickCountry",
            color_continuous_scale="Oranges",
            title="AiTM Phishing Link Click Origins by Country (UrlClickEvents)",
            labels={"Clicks": "Click Count"},
        )
        fig_clicks.update_layout(
            geo=dict(
                showframe=False, showcoastlines=True, coastlinecolor="#636e72",
                showland=True, landcolor="#1e272e",
                showocean=True, oceancolor="#0a0f14",
                showcountries=True, countrycolor="#4a5568",
                projection_type="natural earth",
            ),
            paper_bgcolor="#2d3436",
            font=dict(color="white", size=11),
            height=500,
        )
        fig_clicks.show()

# Ensure aitm_count is set for executive summary cell
aitm_count = len(aitm_results)
print(f"\n  aitm_count = {aitm_count}")


Running AiTM detection queries...
🔍 AiTM – MFA-bypass sign-ins...
   ✅ Returned 100 rows
🔍 AiTM – phishing link clicks...
   ✅ Returned 0 rows

AiTM DETECTION SUMMARY

  MFA-bypass sign-ins: 100 users
  HIGH RISK: 29 users sign in from 3+ countries without MFA!


,UserPrincipalName,SignInCount,UniqueIPs,Countries,Apps,FirstSeen,LastSeen,SampleIP,CountryCount,AiTMRisk
0,securitycopilotagentuser-2b44e0be-aa16-4df1-90...,639847,8,"[""Ireland"",""France"",""Germany"",""Sweden""]","[""WindowsDefenderATP"",""SecurityCopilotAgentIde...",2026-01-13 18:40:43.740610+00:00,2026-02-23 11:15:12.493420+00:00,72.145.24.35,4,High – 3+ Countries
1,securitycopilotagentuser-a8c2716a-fb00-4686-a5...,156181,8,"[""Sweden"",""Germany"",""France"",""Ireland""]","[""Office 365 Information Protection"",""Security...",2026-01-05 22:00:36.699956+00:00,2026-02-23 11:06:19.891504+00:00,135.225.147.168,4,High – 3+ Countries
2,securitycopilotagentuser-300f6453-208f-4ebb-aa...,123569,8,"[""France"",""Germany"",""Sweden"",""Ireland""]","[""Security Copilot API"",""Azure Purview"",""Secur...",2026-01-26 16:29:28.324182+00:00,2026-02-23 11:01:05.976977+00:00,4.251.29.11,4,High – 3+ Countries
3,securitycopilotagentuser-8f1a8f06-e84d-4ab5-85...,44569,7,"[""Sweden"",""France"",""Ireland"",""Germany""]","[""SecurityCopilotAgentIdentity-8f1a8f06-e84d-4...",2026-01-05 21:51:41.568378+00:00,2026-01-22 16:31:37.155099+00:00,135.225.147.168,4,High – 3+ Countries
4,sync_vnevado-dc_5513f8b98a4f@valleenevado.onmi...,20680,1,"[""United States""]","["""",""Microsoft Azure Active Directory Connect""]",2025-11-25 11:26:38.510534+00:00,2026-02-23 11:10:18.362751+00:00,137.117.84.85,1,Low – Single Country
5,lydiab@zava-corp.com,15117,22,"[""United States"",""France""]","[""Windows Sign In"",""ZavaBot"",""Microsoft Authen...",2025-12-02 11:02:42.745913+00:00,2026-01-19 07:19:30.056255+00:00,4.152.23.243,2,Medium – 2 Countries
6,rayt@zava-corp.com,12198,17,"[""United States""]","[""Windows Sign In"",""M365ChatClient"",""OneDrive ...",2025-11-25 11:19:10.319564+00:00,2025-12-21 05:09:14.477395+00:00,4.152.23.243,1,Low – Single Country
7,securitycopilotagentuser-22c8adb7-3d74-41ac-b9...,11050,8,"[""Ireland"",""France"",""Sweden"",""Germany""]","[""SecurityCopilotAgentIdentity-22c8adb7-3d74-4...",2026-01-08 16:58:57.153254+00:00,2026-02-20 20:41:22.911334+00:00,72.145.24.35,4,High – 3+ Countries
8,babaks@zava-corp.com,6739,18,"[""United States""]","[""Windows Sign In"",""Graph Files Manager"",""OneD...",2025-12-02 11:01:41.443132+00:00,2025-12-22 04:01:04.122880+00:00,4.152.23.243,1,Low – Single Country
9,laurenceg@zava-corp.com,6359,20,"[""United States""]","[""Windows Sign In"",""Global Secure Access Clien...",2025-12-02 11:01:34.944748+00:00,2025-12-21 03:57:48.226851+00:00,4.152.23.243,1,Low – Single Country


  UrlClickEvents: no phishing clicks found (or table not available in this workspace)



  aitm_count = 100


## 🏛️ Section 16 — Threat Hunting: On-Prem Identity & Domain Controllers

Advanced threat hunting across Active Directory and on-premises infrastructure using **SecurityEvent**, **Defender for Identity** tables, and NTLM/Kerberos telemetry.

| Hunt | Technique | MITRE | Primary Tables |
|------|-----------|-------|----------------|
| **16.1** | Kerberoasting & AS-REP Roasting | T1558.003 / T1558.004 | SecurityEvent (4768/4769) |
| **16.2** | DCSync / Directory Replication Abuse | T1003.006 | IdentityDirectoryEvents, SecurityEvent (4662) |
| **16.3** | LDAP Recon / BloodHound-style queries | T1087.002 | IdentityQueryEvents |
| **16.4** | NTLM Relay & Pass-the-Hash patterns | T1550.002 | SecurityEvent (4776/4624), IdentityLogonEvents |
| **16.5** | Privileged Group Membership Changes | T1098 | SecurityEvent (4728/4732/4756) |
| **16.6** | Suspicious Processes on Domain Controllers | T1059 / T1003 | IdentityDirectoryEvents |
| **16.7** | Account Creation & Manipulation Burst | T1136.001 / T1531 | SecurityEvent (4720/4722/4725/4726) |
| **16.8** | Golden / Silver Ticket Indicators | T1558.001 / T1558.002 | SecurityEvent (4769/4770), IdentityLogonEvents |


In [39]:

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# ── 16.1  Kerberoasting & AS-REP Roasting ─────────────────────────────────────
# EventID 4769: TGS-REQ with RC4 encryption type (0x17) for service accounts  → Kerberoasting
# EventID 4768: TGT-REQ with DONT_REQUIRE_PREAUTH flag or RC4 from non-DC     → AS-REP Roasting

kerberoast_query = f"""
SecurityEvent
| where {TIME_RANGE}
| where EventID in (4768, 4769, 4771)
| extend TicketEncryptionType = tostring(EventData) has "0x17" or
                                tostring(EventData) has "0x18"
| extend ServiceName    = tostring(parse_xml(tostring(EventData)).EventData.Data[6])
| extend ClientAddress  = tostring(parse_xml(tostring(EventData)).EventData.Data[9])
| extend FailureCode    = tostring(parse_xml(tostring(EventData)).EventData.Data[4])
| extend TicketOptions  = tostring(parse_xml(tostring(EventData)).EventData.Data[3])
| extend IsRC4Request   = tostring(EventData) has "0x17"
| extend AttackType = case(
    EventID == 4769 and IsRC4Request, "Kerberoasting - RC4 TGS Request",
    EventID == 4768 and TicketOptions has "0x40810010", "AS-REP Roasting - Pre-auth Disabled",
    EventID == 4771 and FailureCode == "0x18", "Brute Force - Bad Kerberos Password",
    EventID == 4768 and IsRC4Request, "Possible AS-REP - Weak Encryption",
    "Kerberos Event - Review"
)
| extend GeoInfo = geo_info_from_ip_address(ClientAddress)
| extend Country = tostring(GeoInfo.country)
| summarize
    EventCount   = count(),
    TargetAccounts = make_set(TargetAccount, 20),
    SourceIPs    = make_set(ClientAddress, 10),
    Countries    = make_set(Country, 10),
    FirstSeen    = min(TimeGenerated),
    LastSeen     = max(TimeGenerated)
  by AttackType, Computer, EventID
| order by EventCount desc
| take 100
"""

# ── 16.2  DCSync / Directory Replication Rights Abuse ─────────────────────────
# EventID 4662: GUID {1131f6aa} = DS-Replication-Get-Changes / {9923a32a} = Get-Changes-All
# Defender for Identity: IdentityDirectoryEvents ActionType = "Suspected DCSync attack"

dcsync_query = f"""
let DCReplicationGuids = dynamic([
    "{{1131f6aa-9c07-11d1-f79f-00c04fc2dcd2}}",  // DS-Replication-Get-Changes
    "{{1131f6ab-9c07-11d1-f79f-00c04fc2dcd2}}",  // DS-Replication-Get-Changes-All
    "{{9923a32a-3607-11d2-b9be-0000f87a36b2}}"   // DS-Install-Replica-into-Site
]);
SecurityEvent
| where {TIME_RANGE}
| where EventID == 4662
| where tostring(Properties) has_any ("1131f6aa","1131f6ab","9923a32a","DS-Replication")
| extend Actor    = tostring(SubjectUserName)
| extend TargetObj = tostring(ObjectName)
| extend GeoInfo  = geo_info_from_ip_address(IpAddress)
| extend Country  = tostring(GeoInfo.country)
| summarize
    ReplicationRequests = count(),
    TargetObjects = make_set(TargetObj, 10),
    FirstSeen     = min(TimeGenerated),
    LastSeen      = max(TimeGenerated)
  by Actor, Computer, IpAddress, Country
| extend RiskLevel = case(
    ReplicationRequests >= 10, "CRITICAL - Possible DCSync running",
    ReplicationRequests >= 3,  "HIGH - Multiple replication requests",
    "MEDIUM - Review required"
)
| order by ReplicationRequests desc
| take 50
"""

# ── 16.3  LDAP Reconnaissance / BloodHound-style Queries ──────────────────────
# IdentityQueryEvents captures LDAP/SAMR/LDAPS queries from Defender for Identity sensors

ldap_recon_query = f"""
IdentityQueryEvents
| where {TIME_RANGE}
| extend QueryCount = case(
    QueryType == "Ldap",  1,
    QueryType == "Dns",   1,
    QueryType == "Samr",  1,
    1
)
| summarize
    TotalQueries    = count(),
    QueryTypes      = make_set(QueryType, 10),
    QueriedObjects  = make_set(QueryTarget, 20),
    DCsQueried      = make_set(DestinationDeviceName, 5),
    FirstSeen       = min(TimeGenerated),
    LastSeen        = max(TimeGenerated)
  by AccountName, AccountDomain, DeviceName, IPAddress
| extend RiskScore = case(
    TotalQueries >= 1000, "CRITICAL - Mass LDAP Enumeration",
    TotalQueries >= 200,  "HIGH - Possible BloodHound/ADExplorer",
    TotalQueries >= 50,   "MEDIUM - Elevated LDAP Activity",
    "LOW - Normal"
)
| where TotalQueries >= 20
| order by TotalQueries desc
| take 100
"""

# ── 16.4  NTLM Relay & Pass-the-Hash Patterns ─────────────────────────────────
# EventID 4776: NTLM credential validation (workstation auth against DC)
# EventID 4624 LogonType 3 + NTLM from non-domain machine IP = suspicious
# IdentityLogonEvents: Protocol == "Ntlm" with unusual source

ntlm_pth_query = f"""
IdentityLogonEvents
| where {TIME_RANGE}
| where Protocol == "Ntlm"
| extend GeoInfo  = geo_info_from_ip_address(IPAddress)
| extend Country  = tostring(GeoInfo.country)
| summarize
    NTLMAuths     = count(),
    TargetDevices = dcount(DestinationDeviceName),
    SourceDevices = dcount(DeviceName),
    Countries     = make_set(Country, 5),
    SampleIPs     = make_set(IPAddress, 5),
    FirstSeen     = min(TimeGenerated),
    LastSeen      = max(TimeGenerated)
  by AccountName, AccountDomain
| extend CountryCount = array_length(Countries)
| extend RiskLevel = case(
    TargetDevices >= 20 and CountryCount >= 2, "CRITICAL - PTH/Relay Across Multiple Countries",
    TargetDevices >= 10, "HIGH - Lateral NTLM to Many Devices",
    TargetDevices >= 5,  "MEDIUM - Elevated NTLM Lateral",
    "LOW"
)
| where TargetDevices >= 3
| order by TargetDevices desc
| take 100
"""

# ── 16.5  Privileged Group Membership Changes ─────────────────────────────────
# EventID 4728/4732/4756 = member added to global/local/universal security group
# Target: Domain Admins, Enterprise Admins, Schema Admins, Administrators, Domain Controllers

privgroup_query = f"""
SecurityEvent
| where {TIME_RANGE}
| where EventID in (4728, 4732, 4756, 4761, 4746, 4751)
| extend TargetGroup  = tostring(parse_xml(tostring(EventData)).EventData.Data[2])
| extend AddedMember  = tostring(parse_xml(tostring(EventData)).EventData.Data[0])
| extend Actor        = tostring(SubjectUserName)
| extend IsSensitive  = TargetGroup has_any (
    "Domain Admins","Enterprise Admins","Schema Admins",
    "Administrators","Domain Controllers","Protected Users",
    "Group Policy Creator Owners","Account Operators","Backup Operators"
)
| where IsSensitive
| project TimeGenerated, Computer, Actor, AddedMember, TargetGroup,
          EventID, IpAddress, Activity
| order by TimeGenerated desc
| take 200
"""

# ── 16.6  Suspicious Processes on Domain Controllers ──────────────────────────
# Defender for Identity flags execution of recon/exfil tools on DC
# IdentityDirectoryEvents ActionType includes process exec on DC context

dc_process_query = f"""
IdentityDirectoryEvents
| where {TIME_RANGE}
| where ActionType in (
    "Suspected DCSync attack (replication of directory services)",
    "Suspected identity theft (pass-the-ticket)",
    "Suspected Brute Force attack (LDAP)",
    "Domain dominance",
    "Suspected skeleton key attack",
    "Suspected Golden Ticket usage",
    "Suspected overpass-the-hash attack",
    "Suspected DCShadow attack",
    "Suspicious service creation",
    "Suspicious communication over DNS",
    "Remote code execution attempt")
| extend Actor   = tostring(AccountName)
| extend GeoInfo = geo_info_from_ip_address(IPAddress)
| extend Country = tostring(GeoInfo.country)
| project TimeGenerated, ActionType, Actor, AccountDomain,
          TargetDeviceName, IPAddress, Country, AdditionalFields
| order by TimeGenerated desc
| take 200
"""

# ── 16.7  Account Creation & Manipulation Burst ───────────────────────────────
# Attackers create accounts for backdoor access, or bulk-disable accounts for disruption.
# EventID 4720=Created, 4722=Enabled, 4725=Disabled, 4726=Deleted, 4767=Unlocked
# T1136.001: Create Account / T1531: Account Access Removal

account_manip_query = f"""
SecurityEvent
| where {TIME_RANGE}
| where EventID in (4720, 4722, 4723, 4724, 4725, 4726, 4738, 4767)
| extend Actor       = tostring(SubjectUserName)
| extend TargetAcct  = tostring(TargetAccount)
| extend EventMeaning = case(
    EventID == 4720, "Account Created",
    EventID == 4722, "Account Enabled",
    EventID == 4723, "Password Change Attempt",
    EventID == 4724, "Password Reset by Admin",
    EventID == 4725, "Account Disabled",
    EventID == 4726, "Account Deleted",
    EventID == 4738, "Account Modified",
    EventID == 4767, "Account Unlocked",
    "Unknown"
)
| summarize
    ActionCount   = count(),
    EventTypes    = make_set(EventMeaning, 10),
    TargetAccts   = make_set(TargetAcct, 20),
    DomainCount   = dcount(TargetDomainName),
    Computers     = make_set(Computer, 10),
    FirstSeen     = min(TimeGenerated),
    LastSeen      = max(TimeGenerated),
    SpanMinutes   = datetime_diff("minute", max(TimeGenerated), min(TimeGenerated))
  by Actor, IpAddress
| extend ActionRate = iff(SpanMinutes > 0, todouble(ActionCount) / SpanMinutes, todouble(ActionCount))
| extend AccountsTargeted = array_length(TargetAccts)
| extend RiskLevel = case(
    ActionCount >= 50 or ActionRate >= 5,
        "CRITICAL - Bulk Account Manipulation / Possible Wiper",
    AccountsTargeted >= 20 or ActionCount >= 20,
        "HIGH - Mass Account Operations",
    array_length(EventTypes) >= 3 and ActionCount >= 5,
        "HIGH - Mixed Create/Enable/Delete Pattern",
    ActionCount >= 5,
        "MEDIUM - Multiple Account Changes",
    "LOW"
)
| where ActionCount >= 3
| order by ActionCount desc
| take 150
"""

# ── 16.8  Golden / Silver Ticket Indicators ───────────────────────────────────
# Golden Ticket: forged TGT — anomalous ticket lifetime, non-DC TGS issuance, RC4 on new DCs
# Silver Ticket: forged service ticket — no TGT-REQ for the service, only TGS seen
# EventID 4769: TGS request with abnormal ticket encryption / lifetime
# IdentityLogonEvents: Kerberos logon without preceding TGT visible = Silver Ticket indicator
# T1558.001 / T1558.002

golden_silver_query = f"""
// Golden Ticket: TGS with encryption type 0x17 (RC4) from a non-standard account on a new DC,
// OR ticket options indicating forged TGT (lifetime > 10 hours default, or forwardable + pkinit combo)
let GoldenTicketPattern = SecurityEvent
| where {TIME_RANGE}
| where EventID in (4769, 4770)
| extend EncType   = tostring(parse_xml(tostring(EventData)).EventData.Data[5])
| extend TicketOpt = tostring(parse_xml(tostring(EventData)).EventData.Data[3])
| extend Service   = tostring(parse_xml(tostring(EventData)).EventData.Data[6])
| extend ClientIP  = tostring(parse_xml(tostring(EventData)).EventData.Data[9])
// RC4 encryption on a modern DC for new requests is suspicious
| where EncType == "0x17" or EncType == "0x18"
| extend IndicatorType = case(
    Service == "krbtgt", "CRITICAL - Golden Ticket: krbtgt service with RC4",
    EncType == "0x17",   "HIGH - RC4 TGS Request (Possible Golden/Silver Ticket)",
    "MEDIUM - Weak Kerberos Encryption"
)
| project TimeGenerated, IndicatorType, Computer, Account,
          Service, EncType, TicketOpt, ClientIP, EventID;
// Silver Ticket: Kerberos logon (LogonType 3) with no matching TGT request from same source
let SilverTicketPattern = IdentityLogonEvents
| where {TIME_RANGE}
| where Protocol == "Kerberos" and ActionType == "LogonSuccess"
| extend GeoInfo  = geo_info_from_ip_address(IPAddress)
| extend Country  = tostring(GeoInfo.country)
| summarize
    KerberosLogons  = count(),
    TargetServices  = make_set(DestinationDeviceName, 10),
    SourceIPs       = make_set(IPAddress, 5),
    Countries       = make_set(Country, 5),
    FirstSeen       = min(TimeGenerated),
    LastSeen        = max(TimeGenerated)
  by AccountName, AccountDomain
| extend ServicesAccessed = array_length(TargetServices)
| extend CountryCount     = array_length(Countries)
| extend IndicatorType = case(
    ServicesAccessed >= 20 and CountryCount >= 2,
        "CRITICAL - Kerberos to 20+ Services from Multiple Countries",
    ServicesAccessed >= 10,
        "HIGH - Kerberos to Many Services (Silver Ticket pattern)",
    ServicesAccessed >= 5 and CountryCount >= 2,
        "HIGH - Cross-Country Kerberos Spread",
    "LOW"
)
| where ServicesAccessed >= 5
| project TimeGenerated = FirstSeen, IndicatorType, Computer=AccountDomain,
          Account=AccountName, Service=tostring(TargetServices),
          EncType="Kerberos", TicketOpt="Silver Ticket Pattern", ClientIP=tostring(SourceIPs),
          EventID=0;
union GoldenTicketPattern, SilverTicketPattern
| order by TimeGenerated desc
| take 200
"""

# ── Run all hunts ──────────────────────────────────────────────────────────────
print("Running Section 16 — On-Prem Identity & Domain Controller Threat Hunts...")

kerberoast_results  = run_kql(kerberoast_query,   "16.1 Kerberoasting / AS-REP Roasting")
dcsync_results      = run_kql(dcsync_query,       "16.2 DCSync / Directory Replication Abuse")
ldap_recon_results  = run_kql(ldap_recon_query,   "16.3 LDAP Recon / BloodHound-style queries")
ntlm_pth_results    = run_kql(ntlm_pth_query,     "16.4 NTLM Relay / Pass-the-Hash patterns")
privgroup_results   = run_kql(privgroup_query,     "16.5 Privileged Group Membership Changes")
dc_process_results  = run_kql(dc_process_query,   "16.6 Suspicious processes on Domain Controllers")

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print("SECTION 16 — ON-PREM IDENTITY & DC THREAT HUNT SUMMARY")
print(f"{'='*72}")

findings = {
    "16.1 Kerberoasting / AS-REP":       kerberoast_results,
    "16.2 DCSync Indicators":            dcsync_results,
    "16.3 LDAP Recon (BloodHound)":      ldap_recon_results,
    "16.4 NTLM Relay / Pass-the-Hash":   ntlm_pth_results,
    "16.5 Priv Group Changes":           privgroup_results,
    "16.6 Suspicious DC Processes":      dc_process_results,
}

for name, df in findings.items():
    count = len(df) if not df.empty else 0
    status = f"{count} events found" if count > 0 else "Clean / No data"
    print(f"  {name:<40}  {status}")

# ── Display each result ────────────────────────────────────────────────────────
if not kerberoast_results.empty:
    print("\n--- 16.1 Kerberoasting / AS-REP Roasting ---")
    display(kerberoast_results)

if not dcsync_results.empty:
    print("\n--- 16.2 DCSync Indicators ---")
    display(dcsync_results)

if not ldap_recon_results.empty:
    print("\n--- 16.3 LDAP Reconnaissance ---")
    display(ldap_recon_results)
    if "RiskScore" in ldap_recon_results.columns and "TotalQueries" in ldap_recon_results.columns:
        fig = px.bar(
            ldap_recon_results.head(20).sort_values("TotalQueries", ascending=False),
            x="AccountName" if "AccountName" in ldap_recon_results.columns else ldap_recon_results.index,
            y="TotalQueries",
            color="RiskScore",
            title="16.3 — LDAP Recon: Top Accounts by Query Volume",
            color_discrete_map={
                "CRITICAL - Mass LDAP Enumeration": "#d63031",
                "HIGH - Possible BloodHound/ADExplorer": "#e17055",
                "MEDIUM - Elevated LDAP Activity": "#fdcb6e",
                "LOW - Normal": "#00b894"
            }
        )
        fig.update_layout(xaxis_tickangle=-40)
        fig.show()

if not ntlm_pth_results.empty:
    print("\n--- 16.4 NTLM Relay / Pass-the-Hash ---")
    display(ntlm_pth_results)
    if "AccountName" in ntlm_pth_results.columns and "TargetDevices" in ntlm_pth_results.columns:
        fig2 = px.scatter(
            ntlm_pth_results,
            x="NTLMAuths", y="TargetDevices",
            color="RiskLevel" if "RiskLevel" in ntlm_pth_results.columns else None,
            hover_data=["AccountName"],
            title="16.4 — NTLM Authentications vs Targeted Devices (Pass-the-Hash scatter)",
            color_discrete_map={
                "CRITICAL - PTH/Relay Across Multiple Countries": "#d63031",
                "HIGH - Lateral NTLM to Many Devices": "#e17055",
                "MEDIUM - Elevated NTLM Lateral": "#fdcb6e",
                "LOW": "#00b894"
            }
        )
        fig2.show()

if not privgroup_results.empty:
    print("\n--- 16.5 Privileged Group Membership Changes ---")
    display(privgroup_results)
    if "TargetGroup" in privgroup_results.columns:
        grp = privgroup_results["TargetGroup"].fillna("Unknown").value_counts().head(10).reset_index()
        grp.columns = ["Group", "Count"]
        px.bar(grp, x="Group", y="Count",
               title="16.5 — Privileged Group Changes by Group",
               color="Count", color_continuous_scale="Reds").show()

if not dc_process_results.empty:
    print("\n--- 16.6 Suspicious Processes on Domain Controllers ---")
    display(dc_process_results)
    if "ActionType" in dc_process_results.columns:
        at = dc_process_results["ActionType"].value_counts().reset_index()
        at.columns = ["ActionType", "Count"]
        px.bar(at, x="ActionType", y="Count",
               title="16.6 — Defender for Identity Attack Category Distribution",
               color="Count", color_continuous_scale="Reds"
        ).update_layout(xaxis_tickangle=-40).show()

# Store for cross-reference in MITRE / executive summary
identity_onprem_results = {
    "kerberoast": kerberoast_results,
    "dcsync":     dcsync_results,
    "ldap_recon": ldap_recon_results,
    "ntlm_pth":   ntlm_pth_results,
    "privgroup":  privgroup_results,
    "dc_process": dc_process_results,
}
print(f"\n{'='*72}")


Running Section 16 — On-Prem Identity & Domain Controller Threat Hunts...
🔍 16.1 Kerberoasting / AS-REP Roasting...
   ✅ Returned 3 rows
🔍 16.2 DCSync / Directory Replication Abuse...
   ✅ Returned 6 rows
🔍 16.3 LDAP Recon / BloodHound-style queries...
   ✅ Returned 36 rows
🔍 16.4 NTLM Relay / Pass-the-Hash patterns...
   ✅ Returned 0 rows
🔍 16.5 Privileged Group Membership Changes...
   ✅ Returned 1 rows
🔍 16.6 Suspicious processes on Domain Controllers...
   ✅ Returned 0 rows

SECTION 16 — ON-PREM IDENTITY & DC THREAT HUNT SUMMARY
  16.1 Kerberoasting / AS-REP               3 events found
  16.2 DCSync Indicators                    6 events found
  16.3 LDAP Recon (BloodHound)              36 events found
  16.4 NTLM Relay / Pass-the-Hash           Clean / No data
  16.5 Priv Group Changes                   1 events found
  16.6 Suspicious DC Processes              Clean / No data

--- 16.1 Kerberoasting / AS-REP Roasting ---


,AttackType,Computer,EventID,EventCount,TargetAccounts,SourceIPs,Countries,FirstSeen,LastSeen
0,Kerberos Event - Review,main-DC.zava-corp.com,4768,221726,"[""ZAVA-CORP.COM\\MAIN-ENTRA$"",""ZAVA-CORP.COM\\...","[""{\""@Name\"":\""IpAddress\"",\""#text\"":\""::ffff:...","[""""]",2025-12-17 20:52:19.759621+00:00,2026-02-22 23:58:58.764982+00:00
1,Kerberos Event - Review,ashtravel-dc.ashtravel.alpineskihouse.co,4768,199134,"[""ASHTRAVEL.ALPINESKIHOUSE.CO\\MSOL_b418ebc4de...","[""{\""@Name\"":\""IpAddress\"",\""#text\"":\""::1\""}""...","[""""]",2025-08-27 11:21:09.021812+00:00,2026-02-22 23:57:41.483165+00:00
2,Kerberos Event - Review,VNEVADO-DC.vnevado.alpineskihouse.co,4768,42453,"[""VNEVADO.ALPINESKIHOUSE.CO\\AVORIAZ-SQL1$"",""V...","[""{\""@Name\"":\""IpAddress\"",\""#text\"":\""::ffff:...","[""""]",2025-08-27 11:32:35.707331+00:00,2026-02-22 23:02:55.586309+00:00



--- 16.2 DCSync Indicators ---


,Actor,Computer,IpAddress,Country,ReplicationRequests,TargetObjects,FirstSeen,LastSeen,RiskLevel
0,MSOL_b418ebc4de62,ashtravel-dc.ashtravel.alpineskihouse.co,,,129058,"[""%{232f5ce9-f0a4-4054-8756-ce2d3fb7c604}""]",2025-08-27 11:19:08.448739+00:00,2026-02-22 23:59:41.539328+00:00,CRITICAL - Possible DCSync running
1,MSOL_4852d82e1806,main-DC.zava-corp.com,,,48255,"[""%{3145a524-7a62-45a3-b0f9-8f9f9b9821f2}""]",2025-12-17 21:06:43.908220+00:00,2026-02-22 23:58:58.769681+00:00,CRITICAL - Possible DCSync running
2,VNEVADO-DC$,VNEVADO-DC.vnevado.alpineskihouse.co,,,4309,"[""%{e5a7a378-ea79-4e5b-a5f8-c18b9418fe13}""]",2025-08-27 11:44:41.529803+00:00,2026-02-22 23:17:05.788069+00:00,CRITICAL - Possible DCSync running
3,ASHTRAVEL-DC$,ashtravel-dc.ashtravel.alpineskihouse.co,,,4309,"[""%{232f5ce9-f0a4-4054-8756-ce2d3fb7c604}""]",2025-08-27 11:45:12.114020+00:00,2026-02-22 23:11:37.698172+00:00,CRITICAL - Possible DCSync running
4,main-DC$,main-DC.zava-corp.com,,,1611,"[""%{3145a524-7a62-45a3-b0f9-8f9f9b9821f2}""]",2025-12-17 21:51:11.313422+00:00,2026-02-22 23:47:21.958640+00:00,CRITICAL - Possible DCSync running
5,MB-DC1$,MB-DC1.internal.niseko.alpineskihouse.co,,,1052,"[""%{a35e537a-7c5e-4f30-a0e9-926ff6153f70}""]",2026-01-20 13:21:42.777813+00:00,2026-02-22 23:32:01.730027+00:00,CRITICAL - Possible DCSync running



--- 16.3 LDAP Reconnaissance ---


,AccountName,AccountDomain,DeviceName,IPAddress,TotalQueries,QueryTypes,QueriedObjects,DCsQueried,FirstSeen,LastSeen,RiskScore
0,,,vnevado-dc.vnevado.alpineskihouse.co,10.60.0.10,24466,"[""Srv"",""Ns""]","[""_http._tcp.azure.archive.ubuntu.com"",""_https...","["""",""ns1-201.azure-dns.com"",""ns1-06.azure-dns....",2025-08-27 11:28:25.366169+00:00,2026-02-22 23:33:15.814114+00:00,CRITICAL - Mass LDAP Enumeration
1,,,vnevado-win10s.vnevado.alpineskihouse.co,10.60.0.46,16886,"[""None"",""QueryUser"",""QueryGroup"",""AllObjects""]","[""Joni Sherman"",""HelpDesk""]","[""vnevado-dc.vnevado.alpineskihouse.co""]",2025-09-01 08:10:46.964020+00:00,2026-02-22 23:13:15.218828+00:00,CRITICAL - Mass LDAP Enumeration
2,,,kenvins-pc.zava-corp.com,10.2.0.115,11641,"[""None"",""QueryGroup"",""Srv""]","[""Domain Admins"",""HelpDesk users"",""_gc._tcp.De...","[""main-dc.zava-corp.com""]",2025-08-27 12:13:42.305021+00:00,2026-02-03 23:22:21.269653+00:00,CRITICAL - Mass LDAP Enumeration
3,,,lydiab-pc.zava-corp.com,10.2.0.110,11400,"[""None"",""QueryGroup"",""Srv""]","[""Domain Admins"",""HelpDesk users"",""_gc._tcp.De...","[""main-dc.zava-corp.com""]",2025-08-27 11:21:28.013148+00:00,2026-02-04 00:00:53.797455+00:00,CRITICAL - Mass LDAP Enumeration
4,,,vnevado-win11u.vnevado.alpineskihouse.co,10.60.0.41,11121,"[""QueryUser"",""QueryGroup"",""None""]","[""Mario Rogers"",""HelpDesk""]","[""vnevado-dc.vnevado.alpineskihouse.co""]",2025-08-27 12:08:05.559469+00:00,2026-02-22 23:11:16.186013+00:00,CRITICAL - Mass LDAP Enumeration
5,,,celesteb-pc.zava-corp.com,10.2.0.137,10886,"[""QueryUser"",""QueryGroup"",""None"",""AllObjects""]","[""Celeste Burton"",""Domain Admins"",""HelpDesk us...","[""main-dc.zava-corp.com""]",2025-09-15 11:55:31.224299+00:00,2026-02-03 23:20:34.182223+00:00,CRITICAL - Mass LDAP Enumeration
6,,,rayt-pc.zava-corp.com,10.2.0.132,10404,"[""None"",""QueryGroup"",""Ns""]","[""Domain Admins"",""HelpDesk users"",""download.wi...","[""main-dc.zava-corp.com""]",2025-08-27 12:14:53.569114+00:00,2026-01-19 13:06:42.507000+00:00,CRITICAL - Mass LDAP Enumeration
7,,,vnevado-win10r.vnevado.alpineskihouse.co,10.60.0.35,9060,"[""QueryGroup"",""QueryUser"",""None"",""AllObjects""]","[""HelpDesk"",""Pradeep Gupta""]","[""vnevado-dc.vnevado.alpineskihouse.co""]",2025-09-01 08:12:31.159829+00:00,2025-12-03 16:08:06.629513+00:00,CRITICAL - Mass LDAP Enumeration
8,,,main-entra.zava-corp.com,10.1.0.6,8734,"[""None"",""QueryGroup"",""AllObjects""]","[""Domain Admins"",""Aadi Kapoor"",""Miguel Garcia""...","[""main-dc.zava-corp.com""]",2025-08-27 11:45:11.298209+00:00,2026-02-22 23:23:10.907882+00:00,CRITICAL - Mass LDAP Enumeration
9,,,10.60.0.12,10.60.0.12,8662,"[""Srv""]","[""_https._tcp.esm.ubuntu.com"",""_http._tcp.azur...","[""vnevado-dc.vnevado.alpineskihouse.co""]",2025-08-27 11:28:25.366600+00:00,2026-02-22 23:31:34.663145+00:00,CRITICAL - Mass LDAP Enumeration



--- 16.5 Privileged Group Membership Changes ---


,TimeGenerated,Computer,Actor,AddedMember,TargetGroup,EventID,IpAddress,Activity
0,2025-09-15 12:09:11.738245+00:00,ashtravel-dc.ashtravel.alpineskihouse.co,ASHTRAVEL-DC$,"{""@Name"":""MemberName"",""#text"":""-""}","{""@Name"":""TargetUserName"",""#text"":""Administrat...",4732,,4732 - A member was added to a security-enable...


## ☁️ Section 17 — Threat Hunting: Azure Environment

Proactive threat hunting across Azure control-plane and data-plane activity using **AzureActivity**, **AzureDiagnostics**, **SigninLogs**, **AuditLogs**, and **Defender for Cloud** signals.

| Hunt | Scenario | MITRE | Primary Tables |
|------|----------|-------|----------------|
| **17.1** | Cryptojacking — Suspicious VM / compute deployments | T1496 | AzureActivity |
| **17.2** | Azure Privilege Escalation — Role assignment abuse | T1098.003 | AzureActivity, AuditLogs |
| **17.3** | Mass Resource Deletion — Ransomware / sabotage | T1485 | AzureActivity |
| **17.4** | Key Vault Enumeration & Secret Exfiltration | T1552.001 | AzureDiagnostics |
| **17.5** | Storage Exfiltration — Blob/SAS abuse | T1530 | AzureDiagnostics |
| **17.6** | Impossible Travel in Azure Management | T1078.004 | AzureActivity |
| **17.7** | Suspicious Automation / Logic App Deployments | T1059.009 | AzureActivity |
| **17.8** | Resource Group Mass Deletion | T1485 | AzureActivity |
| **17.9** | NSG Rule Modification — Port Exposure | T1562.007 | AzureActivity |
| **17.10** | Brute Force on Azure Portal / Management Plane | T1110.001 | SigninLogs, AADNonInteractiveUserSignInLogs |
| **17.11** | Azure Resource Enumeration / Reconnaissance | T1526 | AzureActivity, AuditLogs |
| **17.12** | Service Principal Credential & Secret Additions | T1098.001 | AuditLogs |


In [40]:

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# ── 17.1  Cryptojacking Detection ─────────────────────────────────────────────
cryptojacking_query = f"""
AzureActivity
| where {TIME_RANGE}
| where OperationNameValue in (
    "MICROSOFT.COMPUTE/VIRTUALMACHINES/WRITE",
    "MICROSOFT.COMPUTE/VIRTUALMACHINESCALESETS/WRITE",
    "MICROSOFT.COMPUTE/DISKENCRYPTIONSETS/WRITE"
  )
| where ActivityStatusValue == "Success"
| extend Properties_ = todynamic(Properties)
| extend VMSize    = tostring(Properties_.responseBody.properties.hardwareProfile.vmSize)
| extend VMRegion  = tostring(Properties_.location)
| extend GeoInfo   = geo_info_from_ip_address(CallerIpAddress)
| extend Country   = tostring(GeoInfo.country)
| extend IsCryptoRisk = VMSize has_any (
    "Standard_NC","Standard_ND","Standard_NV",
    "Standard_H","Standard_HB","Standard_HC",
    "Standard_F","Standard_FX"
  )
| summarize
    VMDeployments  = count(),
    VMSizes        = make_set(VMSize, 10),
    Regions        = make_set(VMRegion, 10),
    SourceCountries = make_set(Country, 5),
    GPUDeployments = countif(IsCryptoRisk),
    FirstSeen      = min(TimeGenerated),
    LastSeen       = max(TimeGenerated)
  by Caller, CallerIpAddress
| extend RiskLevel = case(
    GPUDeployments >= 5 or VMDeployments >= 20, "CRITICAL - Possible Cryptojacking",
    GPUDeployments >= 2 or VMDeployments >= 10, "HIGH - Suspicious Compute Deployment",
    GPUDeployments >= 1,                        "MEDIUM - GPU VM Deployed",
    "LOW"
)
| order by GPUDeployments desc, VMDeployments desc
| take 100
"""

# ── 17.2  Azure Privilege Escalation ──────────────────────────────────────────
azure_privesc_query = f"""
AzureActivity
| where {TIME_RANGE}
| where OperationNameValue in (
    "MICROSOFT.AUTHORIZATION/ROLEASSIGNMENTS/WRITE",
    "MICROSOFT.AUTHORIZATION/ROLEDEFINITIONS/WRITE",
    "MICROSOFT.AUTHORIZATION/POLICYASSIGNMENTS/WRITE"
  )
| where ActivityStatusValue == "Success"
| extend Properties_ = todynamic(Properties)
| extend RoleId      = tostring(Properties_.requestbody.properties.roleDefinitionId)
| extend Scope       = tostring(Properties_.requestbody.properties.scope)
| extend PrincipalId = tostring(Properties_.requestbody.properties.principalId)
| extend GeoInfo     = geo_info_from_ip_address(CallerIpAddress)
| extend Country     = tostring(GeoInfo.country)
| extend IsHighScope = Scope has_any ("/subscriptions/","/providers/Microsoft.Management/")
| extend IsOwnerOrUA = RoleId has_any (
    "8e3af657-a8ff-443c-a75c-2fe8c4bcb635",
    "18d7d88d-d35e-4fb5-a5c3-7773c20a72d9",
    "b24988ac-6180-42a0-ab88-20f7382dd24c"
  )
| extend PrivEscRisk = case(
    IsOwnerOrUA and IsHighScope, "CRITICAL - Owner/UAA at Subscription Scope",
    IsOwnerOrUA,                 "HIGH - Privileged Role Assigned",
    IsHighScope,                 "MEDIUM - Subscription-level Role Change",
    "LOW - Resource-level Role Change"
)
| project TimeGenerated, PrivEscRisk, Caller, CallerIpAddress, Country,
          OperationNameValue, Scope, RoleId, PrincipalId, ResourceGroup, SubscriptionId
| order by TimeGenerated desc
| take 200
"""

# ── 17.3  Mass Resource Deletion ──────────────────────────────────────────────
mass_delete_query = f"""
AzureActivity
| where {TIME_RANGE}
| where OperationNameValue endswith "/DELETE"
| where ActivityStatusValue == "Success"
| extend GeoInfo  = geo_info_from_ip_address(CallerIpAddress)
| extend Country  = tostring(GeoInfo.country)
| extend ResourceType = tostring(split(OperationNameValue, "/")[1])
| summarize
    DeleteCount    = count(),
    ResourceTypes  = make_set(ResourceType, 15),
    DeletedObjects = make_set(Resource, 20),
    Countries      = make_set(Country, 5),
    FirstDelete    = min(TimeGenerated),
    LastDelete     = max(TimeGenerated),
    SpanMinutes    = datetime_diff("minute", max(TimeGenerated), min(TimeGenerated))
  by Caller, CallerIpAddress, ResourceGroup, SubscriptionId
| extend DeletionRate = iff(SpanMinutes > 0, todouble(DeleteCount) / SpanMinutes, todouble(DeleteCount))
| extend RiskLevel = case(
    DeleteCount >= 50 or DeletionRate >= 5, "CRITICAL - Mass Deletion / Possible Ransomware",
    DeleteCount >= 20 or DeletionRate >= 2, "HIGH - Large-scale Deletion Activity",
    DeleteCount >= 10, "MEDIUM - Multiple Resource Deletes",
    "LOW"
)
| order by DeleteCount desc
| take 100
"""

# ── 17.4  Key Vault Enumeration & Secret Exfiltration ─────────────────────────
keyvault_query = f"""
AzureDiagnostics
| where {TIME_RANGE}
| where ResourceProvider == "MICROSOFT.KEYVAULT"
| where OperationName in (
    "SecretList","SecretGet","KeyList","KeyGet",
    "CertificateList","CertificateGet","VaultGet","VaultList"
  )
| extend GeoInfo  = geo_info_from_ip_address(CallerIPAddress)
| extend Country  = tostring(GeoInfo.country)
| summarize
    TotalOps      = count(),
    SecretReads   = countif(OperationName in ("SecretGet","SecretList")),
    KeyReads      = countif(OperationName in ("KeyGet","KeyList")),
    VaultNames    = make_set(Resource, 10),
    Countries     = make_set(Country, 10),
    UniqueIPs     = dcount(CallerIPAddress),
    FirstSeen     = min(TimeGenerated),
    LastSeen      = max(TimeGenerated)
  by identity_claim_oid_g, CallerIPAddress
| extend RiskLevel = case(
    SecretReads >= 50, "CRITICAL - Mass Secret Exfiltration",
    SecretReads >= 20, "HIGH - Many Secrets Read",
    SecretReads >= 5 and array_length(Countries) >= 2, "HIGH - Secrets from Multiple Countries",
    SecretReads >= 5,  "MEDIUM - Elevated Secret Access",
    "LOW"
)
| order by SecretReads desc, TotalOps desc
| take 100
"""

# ── 17.5  Storage Exfiltration — Blob / SAS Abuse ─────────────────────────────
storage_exfil_query = f"""
AzureDiagnostics
| where {TIME_RANGE}
| where ResourceProvider == "MICROSOFT.STORAGE"
| where OperationName in (
    "BlobRead","BlobList","GetBlob","GetBlobProperties",
    "GetBlobServiceProperties","ListBlobs","ListContainers"
  )
| extend GeoInfo    = geo_info_from_ip_address(CallerIPAddress)
| extend Country    = tostring(GeoInfo.country)
| extend AuthType   = tostring(column_ifexists("AuthenticationType_s", ""))
| extend BlobUri    = tostring(column_ifexists("Uri_s", ""))
| extend BytesSent  = tolong(column_ifexists("ResponseBodySize_d", long(0)))
| extend IsAnon     = AuthType == "Anonymous"
| extend IsSASToken = AuthType == "SAS"
| summarize
    ReadOps        = count(),
    AnonymousReads = countif(IsAnon),
    SASTokenReads  = countif(IsSASToken),
    UniqueBlobs    = dcount(BlobUri),
    Countries      = make_set(Country, 10),
    UniqueIPs      = dcount(CallerIPAddress),
    TotalBytes     = sum(BytesSent),
    FirstSeen      = min(TimeGenerated),
    LastSeen       = max(TimeGenerated)
  by Resource, CallerIPAddress
| extend TotalMB = round(TotalBytes / 1048576.0, 2)
| extend CountryCount = array_length(Countries)
| extend RiskLevel = case(
    AnonymousReads >= 100 or TotalMB >= 1000, "CRITICAL - Mass Unauthenticated / Large Exfil",
    SASTokenReads >= 50 or TotalMB >= 500,    "HIGH - Bulk SAS Read / Possible Exfil",
    TotalMB >= 100 or CountryCount >= 3,       "MEDIUM - Cross-Border or Large Data Read",
    "LOW"
)
| order by TotalMB desc, ReadOps desc
| take 100
"""

# ── 17.6  Impossible Travel in Azure Management ────────────────────────────────
azure_travel_query = f"""
AzureActivity
| where {TIME_RANGE}
| where ActivityStatusValue == "Success"
| extend GeoInfo  = geo_info_from_ip_address(CallerIpAddress)
| extend Country  = tostring(GeoInfo.country)
| where isnotempty(Country)
| summarize
    Countries     = make_set(Country, 20),
    IPs           = make_set(CallerIpAddress, 10),
    Operations    = dcount(OperationNameValue),
    FirstSeen     = min(TimeGenerated),
    LastSeen      = max(TimeGenerated)
  by Caller
| extend CountryCount = array_length(Countries)
| extend SpanHours    = datetime_diff("hour", LastSeen, FirstSeen)
| extend TravelRisk = case(
    CountryCount >= 5, "CRITICAL - 5+ Countries",
    CountryCount >= 3, "HIGH - 3-4 Countries in Period",
    CountryCount == 2 and SpanHours <= 4, "HIGH - 2 Countries within 4 Hours",
    CountryCount == 2, "MEDIUM - 2 Countries",
    "NORMAL"
)
| where CountryCount >= 2
| order by CountryCount desc, SpanHours asc
| take 100
"""

# ── 17.7  Suspicious Automation / Logic App Deployments ───────────────────────
automation_query = f"""
AzureActivity
| where {TIME_RANGE}
| where OperationNameValue in (
    "MICROSOFT.AUTOMATION/AUTOMATIONACCOUNTS/WRITE",
    "MICROSOFT.AUTOMATION/AUTOMATIONACCOUNTS/RUNBOOKS/WRITE",
    "MICROSOFT.AUTOMATION/AUTOMATIONACCOUNTS/WEBHOOKS/WRITE",
    "MICROSOFT.LOGIC/WORKFLOWS/WRITE",
    "MICROSOFT.LOGIC/WORKFLOWS/TRIGGERS/WRITE",
    "MICROSOFT.WEB/SITES/WRITE",
    "MICROSOFT.WEB/SITES/FUNCTIONS/WRITE",
    "MICROSOFT.CONTAINERSERVICE/MANAGEDCLUSTERS/WRITE"
  )
| where ActivityStatusValue == "Success"
| extend GeoInfo  = geo_info_from_ip_address(CallerIpAddress)
| extend Country  = tostring(GeoInfo.country)
| extend ResourceKind = tostring(split(OperationNameValue, "/")[1])
| summarize
    Deployments    = count(),
    ResourceTypes  = make_set(ResourceKind, 10),
    DeployedItems  = make_set(Resource, 15),
    Countries      = make_set(Country, 5),
    FirstSeen      = min(TimeGenerated),
    LastSeen       = max(TimeGenerated)
  by Caller, CallerIpAddress
| extend CountryCount = array_length(Countries)
| extend RiskLevel = case(
    Deployments >= 10 or CountryCount >= 2, "HIGH - Many Automation Resources or Cross-Country",
    Deployments >= 5,  "MEDIUM - Multiple Automation Deployments",
    "LOW - Review"
)
| order by Deployments desc
| take 100
"""

# ── 17.8  Resource Group Mass Deletion ────────────────────────────────────────
# Adversaries targeting RG deletion wipe all child resources in one operation.
# T1485: Data Destruction at infrastructure level.

rg_deletion_query = f"""
AzureActivity
| where {TIME_RANGE}
| where OperationNameValue == "MICROSOFT.RESOURCES/SUBSCRIPTIONS/RESOURCEGROUPS/DELETE"
| where ActivityStatusValue in ("Success", "Accepted")
| extend GeoInfo  = geo_info_from_ip_address(CallerIpAddress)
| extend Country  = tostring(GeoInfo.country)
| summarize
    RGDeletions    = count(),
    DeletedRGs     = make_set(ResourceGroup, 30),
    Subscriptions  = make_set(SubscriptionId, 5),
    Countries      = make_set(Country, 5),
    FirstDelete    = min(TimeGenerated),
    LastDelete     = max(TimeGenerated),
    SpanMinutes    = datetime_diff("minute", max(TimeGenerated), min(TimeGenerated))
  by Caller, CallerIpAddress
| extend DeletionRate = iff(SpanMinutes > 0, todouble(RGDeletions) / SpanMinutes, todouble(RGDeletions))
| extend RiskLevel = case(
    RGDeletions >= 10 or DeletionRate >= 2, "CRITICAL - Mass RG Deletion / Infrastructure Wipe",
    RGDeletions >= 5,                       "HIGH - Multiple Resource Groups Deleted",
    RGDeletions >= 2,                       "MEDIUM - More than 1 RG Deleted",
    "LOW - Single RG Deletion"
)
| order by RGDeletions desc
| take 100
"""

# ── 17.9  NSG Rule Modification — Port Exposure ───────────────────────────────
# Attackers modify NSG inbound rules to open sensitive ports (RDP/SSH/SQL) to 0.0.0.0/0.
# T1562.007: Disable or Modify Cloud Firewall

nsg_rule_query = f"""
AzureActivity
| where {TIME_RANGE}
| where OperationNameValue in (
    "MICROSOFT.NETWORK/NETWORKSECURITYGROUPS/SECURITYRULES/WRITE",
    "MICROSOFT.NETWORK/NETWORKSECURITYGROUPS/WRITE"
  )
| where ActivityStatusValue == "Success"
| extend Properties_  = todynamic(Properties)
| extend NSGName       = tostring(split(ResourceId, "/")[-1])
| extend RuleProps     = todynamic(Properties_.requestbody.properties)
| extend Direction     = tostring(RuleProps.direction)
| extend Access        = tostring(RuleProps.access)
| extend DestPort      = tostring(RuleProps.destinationPortRange)
| extend SourcePrefix  = tostring(RuleProps.sourceAddressPrefix)
| extend GeoInfo       = geo_info_from_ip_address(CallerIpAddress)
| extend Country       = tostring(GeoInfo.country)
| extend IsInternet    = SourcePrefix in ("*", "0.0.0.0/0", "Internet", "Any")
| extend IsDangerousPort = DestPort has_any (
    "22","3389","1433","3306","5432","5986","5985",
    "445","135","137","138","139","23","21","8080","8443"
  )
| extend Severity = case(
    Direction == "Inbound" and Access == "Allow" and IsInternet and IsDangerousPort,
        "CRITICAL - Internet-facing dangerous port opened",
    Direction == "Inbound" and Access == "Allow" and IsInternet,
        "HIGH - Inbound allow rule to Internet",
    Direction == "Inbound" and Access == "Allow" and IsDangerousPort,
        "MEDIUM - Dangerous port allow rule",
    "LOW - NSG change (review)"
)
| project TimeGenerated, Severity, Caller, CallerIpAddress, Country,
          ResourceGroup, NSGName, Direction, Access, DestPort, SourcePrefix
| order by TimeGenerated desc
| take 300
"""

# ── 17.10  Brute Force on Azure Portal / Management Plane ─────────────────────
# Multiple sign-in failures (ResultType != 0) to Azure portal / ARM / Graph apps
# followed by eventual success = credential stuffing or brute force.
# T1110.001: Brute Force - Password Guessing

azure_brute_query = f"""
SigninLogs
| where {TIME_RANGE}
| where AppDisplayName has_any (
    "Azure Portal","Microsoft Azure","Windows Azure Service Management API",
    "Azure Active Directory","Microsoft Graph","Azure Resource Manager"
  )
| extend IsFailure = ResultType != 0
| extend GeoInfo  = geo_info_from_ip_address(IPAddress)
| extend Country  = tostring(GeoInfo.country)
| summarize
    TotalAttempts  = count(),
    FailedAttempts = countif(IsFailure),
    SuccessAttempts = countif(not(IsFailure)),
    FailureCodes   = make_set(ResultDescription, 5),
    SourceIPs      = make_set(IPAddress, 10),
    Countries      = make_set(Country, 10),
    FirstSeen      = min(TimeGenerated),
    LastSeen       = max(TimeGenerated)
  by UserPrincipalName, AppDisplayName
| extend FailureRate  = round(100.0 * FailedAttempts / TotalAttempts, 1)
| extend BruteForceRisk = case(
    FailedAttempts >= 100 and SuccessAttempts >= 1,
        "CRITICAL - Mass Failures then Success (Brute Force / Spray)",
    FailedAttempts >= 50 and SuccessAttempts >= 1,
        "HIGH - Many Failures then Success",
    FailedAttempts >= 50,
        "HIGH - Ongoing Brute Force (No Success Yet)",
    FailedAttempts >= 20,
        "MEDIUM - Elevated Failures",
    "LOW"
)
| where FailedAttempts >= 10
| order by FailedAttempts desc
| take 200
"""

# ── 17.11  Azure Resource Enumeration / Reconnaissance ────────────────────────
# Adversaries list subscriptions, resource groups, resources, and role assignments
# to understand the target environment before moving laterally or exfiltrating.
# T1526: Cloud Service Discovery

azure_enum_query = f"""
AzureActivity
| where {TIME_RANGE}
| where ActivityStatusValue == "Success"
| where OperationNameValue in (
    "MICROSOFT.RESOURCES/SUBSCRIPTIONS/RESOURCEGROUPS/READ",
    "MICROSOFT.RESOURCES/SUBSCRIPTIONS/RESOURCES/READ",
    "MICROSOFT.RESOURCES/SUBSCRIPTIONS/READ",
    "MICROSOFT.AUTHORIZATION/ROLEASSIGNMENTS/READ",
    "MICROSOFT.AUTHORIZATION/ROLEDEFINITIONS/READ",
    "MICROSOFT.AUTHORIZATION/PERMISSIONS/READ",
    "MICROSOFT.COMPUTE/VIRTUALMACHINES/READ",
    "MICROSOFT.NETWORK/VIRTUALNETWORKS/READ",
    "MICROSOFT.NETWORK/NETWORKSECURITYGROUPS/READ",
    "MICROSOFT.KEYVAULT/VAULTS/READ",
    "MICROSOFT.STORAGE/STORAGEACCOUNTS/READ"
  )
| extend GeoInfo  = geo_info_from_ip_address(CallerIpAddress)
| extend Country  = tostring(GeoInfo.country)
| extend ResourceCategory = tostring(split(OperationNameValue, "/")[1])
| summarize
    TotalReadOps     = count(),
    ResourceCategories = make_set(ResourceCategory, 15),
    SubscriptionsRead = dcount(SubscriptionId),
    RGsEnumerated    = dcount(ResourceGroup),
    SourceIPs        = make_set(CallerIpAddress, 5),
    Countries        = make_set(Country, 5),
    FirstSeen        = min(TimeGenerated),
    LastSeen         = max(TimeGenerated),
    SpanMinutes      = datetime_diff("minute", max(TimeGenerated), min(TimeGenerated))
  by Caller
| extend EnumRate   = iff(SpanMinutes > 0, todouble(TotalReadOps) / SpanMinutes, todouble(TotalReadOps))
| extend ReconRisk  = case(
    TotalReadOps >= 500 or EnumRate >= 10,
        "CRITICAL - Mass Enumeration / Active Recon",
    TotalReadOps >= 200 or RGsEnumerated >= 20,
        "HIGH - Broad Resource Enumeration",
    TotalReadOps >= 50  or RGsEnumerated >= 5,
        "MEDIUM - Elevated Read Activity",
    "LOW"
)
| where TotalReadOps >= 30
| order by TotalReadOps desc
| take 150
"""

# ── 17.12  Service Principal Credential & Secret Additions ────────────────────
# Adding new credentials/secrets to existing SPNs is a persistence technique.
# T1098.001: Account Manipulation - Additional Cloud Credentials

sp_credential_query = f"""
AuditLogs
| where {TIME_RANGE}
| where OperationName in (
    "Add service principal credentials",
    "Update application – Certificates and secrets management",
    "Add app role assignment to service principal",
    "Add owner to application",
    "Add member to role",
    "Add application",
    "Update service principal"
  )
| extend InitiatedBy_UPN = tostring(InitiatedBy.user.userPrincipalName)
| extend InitiatedBy_SP  = tostring(InitiatedBy.app.servicePrincipalName)
| extend Actor           = coalesce(InitiatedBy_UPN, InitiatedBy_SP, "Unknown")
| extend ActorIP         = tostring(InitiatedBy.user.ipAddress)
| extend TargetApp = tostring(TargetResources[0].displayName)
| extend GeoInfo   = geo_info_from_ip_address(ActorIP)
| extend Country   = tostring(GeoInfo.country)
| extend CredentialRisk = case(
    OperationName has "credentials" and Actor != "Unknown",
        "HIGH - Credential Added to Existing SPN",
    OperationName has "role assignment" or OperationName has "Add owner",
        "HIGH - Privilege Granted to SPN",
    OperationName has "Add application",
        "MEDIUM - New Application Registered",
    "LOW - SP/App Modification"
)
| project TimeGenerated, CredentialRisk, Actor, ActorIP, Country,
          OperationName, TargetApp, Result, CorrelationId
| order by TimeGenerated desc
| take 300
"""

# ── Run all hunts ──────────────────────────────────────────────────────────────
print("Running Section 17 — Azure Environment Threat Hunts...")

cryptojacking_results  = run_kql(cryptojacking_query,  "17.1 Cryptojacking / Suspicious VM Deploy")
azure_privesc_results  = run_kql(azure_privesc_query,  "17.2 Azure Privilege Escalation")
mass_delete_results    = run_kql(mass_delete_query,    "17.3 Mass Resource Deletion")
keyvault_results       = run_kql(keyvault_query,       "17.4 Key Vault Enumeration / Secret Exfil")
storage_exfil_results  = run_kql(storage_exfil_query,  "17.5 Storage Exfiltration / SAS Abuse")
azure_travel_results   = run_kql(azure_travel_query,   "17.6 Impossible Travel in Azure Management")
automation_results     = run_kql(automation_query,     "17.7 Suspicious Automation Deployments")
rg_deletion_results    = run_kql(rg_deletion_query,    "17.8 Resource Group Mass Deletion")
nsg_rule_results       = run_kql(nsg_rule_query,       "17.9 NSG Rule Modification / Port Exposure")
azure_brute_results    = run_kql(azure_brute_query,    "17.10 Brute Force on Azure Portal")
azure_enum_results     = run_kql(azure_enum_query,     "17.11 Azure Resource Enumeration / Recon")
sp_credential_results  = run_kql(sp_credential_query,  "17.12 Service Principal Credential Additions")

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print("SECTION 17 — AZURE ENVIRONMENT THREAT HUNT SUMMARY")
print(f"{'='*72}")

azure_findings = {
    "17.1  Cryptojacking":                   cryptojacking_results,
    "17.2  Azure Privilege Escalation":       azure_privesc_results,
    "17.3  Mass Resource Deletion":           mass_delete_results,
    "17.4  Key Vault Exfil":                  keyvault_results,
    "17.5  Storage Exfiltration":             storage_exfil_results,
    "17.6  Impossible Travel":                azure_travel_results,
    "17.7  Automation Deployments":           automation_results,
    "17.8  RG Mass Deletion":                 rg_deletion_results,
    "17.9  NSG Port Exposure":                nsg_rule_results,
    "17.10 Brute Force (Azure Portal)":       azure_brute_results,
    "17.11 Resource Enumeration / Recon":     azure_enum_results,
    "17.12 SPN Credential Additions":         sp_credential_results,
}

for name, df in azure_findings.items():
    count = len(df) if not df.empty else 0
    flag  = "  *** FINDINGS ***" if count > 0 else ""
    print(f"  {name:<44}  {count:>4} events{flag}")

# ── Visualisations ─────────────────────────────────────────────────────────────
# 17.1 Cryptojacking
if not cryptojacking_results.empty and "Caller" in cryptojacking_results.columns:
    top_cx = cryptojacking_results.nlargest(15, "VMDeployments")
    fig_cx = px.bar(top_cx, x="Caller", y=["VMDeployments", "GPUDeployments"],
                    title="17.1 — VM Deployments: Total vs GPU/HPC (Cryptojacking Risk)",
                    barmode="overlay", color_discrete_sequence=["#74b9ff","#d63031"])
    fig_cx.update_layout(xaxis_tickangle=-40)
    fig_cx.show()

# 17.2 Privilege escalation
if not azure_privesc_results.empty and "PrivEscRisk" in azure_privesc_results.columns:
    pe_df = azure_privesc_results["PrivEscRisk"].value_counts().reset_index()
    pe_df.columns = ["RiskLevel", "Count"]
    px.pie(pe_df, names="RiskLevel", values="Count",
           title="17.2 — Azure Role Assignment Risk Distribution",
           color_discrete_sequence=px.colors.sequential.Reds_r).show()

# 17.3 Mass deletions
if not mass_delete_results.empty and "Caller" in mass_delete_results.columns:
    top_del = mass_delete_results.nlargest(15, "DeleteCount")
    px.bar(top_del, x="Caller", y="DeleteCount",
           color="RiskLevel" if "RiskLevel" in top_del.columns else None,
           title="17.3 — Top Callers by Resource Delete Count",
           color_discrete_map={
               "CRITICAL - Mass Deletion / Possible Ransomware": "#d63031",
               "HIGH - Large-scale Deletion Activity": "#e17055",
               "MEDIUM - Multiple Resource Deletes": "#fdcb6e",
               "LOW": "#00b894"
           }).update_layout(xaxis_tickangle=-40).show()

# 17.8 Resource Group Deletion
if not rg_deletion_results.empty:
    print("\n--- 17.8 Resource Group Mass Deletion ---")
    display(rg_deletion_results)
    if "RiskLevel" in rg_deletion_results.columns and len(rg_deletion_results) > 1:
        px.bar(rg_deletion_results.nlargest(15, "RGDeletions"),
               x="Caller", y="RGDeletions", color="RiskLevel",
               title="17.8 — Resource Group Deletions per Caller",
               color_discrete_map={
                   "CRITICAL - Mass RG Deletion / Infrastructure Wipe": "#d63031",
                   "HIGH - Multiple Resource Groups Deleted": "#e17055",
                   "MEDIUM - More than 1 RG Deleted": "#fdcb6e",
                   "LOW - Single RG Deletion": "#00b894"
               }).update_layout(xaxis_tickangle=-40).show()

# 17.9 NSG Port Exposure
if not nsg_rule_results.empty:
    print("\n--- 17.9 NSG Rule Modifications (Port Exposure) ---")
    display(nsg_rule_results.head(40))
    if "Severity" in nsg_rule_results.columns:
        sev_df = nsg_rule_results["Severity"].value_counts().reset_index()
        sev_df.columns = ["Severity", "Count"]
        px.bar(sev_df, x="Severity", y="Count",
               title="17.9 — NSG Rule Changes by Severity",
               color="Count", color_continuous_scale="Reds"
        ).update_layout(xaxis_tickangle=-30).show()

# 17.10 Brute Force
if not azure_brute_results.empty:
    print("\n--- 17.10 Brute Force on Azure Management ---")
    display(azure_brute_results.head(30))
    if "BruteForceRisk" in azure_brute_results.columns:
        top_bf = azure_brute_results.nlargest(15, "FailedAttempts")
        fig_bf = px.bar(top_bf, x="UserPrincipalName", y=["FailedAttempts","SuccessAttempts"],
                        title="17.10 — Brute Force: Failed vs Successful Logins per User",
                        barmode="group", color_discrete_sequence=["#d63031","#00b894"])
        fig_bf.update_layout(xaxis_tickangle=-40)
        fig_bf.show()

# 17.11 Resource Enumeration
if not azure_enum_results.empty:
    print("\n--- 17.11 Azure Resource Enumeration / Recon ---")
    display(azure_enum_results.head(30))
    if "TotalReadOps" in azure_enum_results.columns:
        px.bar(azure_enum_results.nlargest(15, "TotalReadOps"),
               x="Caller", y="TotalReadOps", color="ReconRisk" if "ReconRisk" in azure_enum_results.columns else None,
               title="17.11 — Top Resource Enumerators",
               color_discrete_map={
                   "CRITICAL - Mass Enumeration / Active Recon": "#d63031",
                   "HIGH - Broad Resource Enumeration": "#e17055",
                   "MEDIUM - Elevated Read Activity": "#fdcb6e",
                   "LOW": "#00b894"
               }).update_layout(xaxis_tickangle=-40).show()

# 17.12 SPN Credentials
if not sp_credential_results.empty:
    print("\n--- 17.12 Service Principal Credential Additions ---")
    risky_spn = sp_credential_results[
        sp_credential_results.get("CredentialRisk", pd.Series(dtype=str)).str.startswith("HIGH", na=False)
    ] if "CredentialRisk" in sp_credential_results.columns else sp_credential_results
    display((risky_spn if not risky_spn.empty else sp_credential_results).head(30))
    if "OperationName" in sp_credential_results.columns:
        op_df = sp_credential_results["OperationName"].value_counts().reset_index()
        op_df.columns = ["OperationName", "Count"]
        px.bar(op_df, x="OperationName", y="Count",
               title="17.12 — SPN/App Audit Operations",
               color="Count", color_continuous_scale="Reds"
        ).update_layout(xaxis_tickangle=-35).show()

# 17.6 Impossible Travel geo map
if not azure_travel_results.empty and "Countries" in azure_travel_results.columns:
    from collections import Counter
    ct = Counter()
    for row in azure_travel_results["Countries"]:
        if isinstance(row, list):
            ct.update([c for c in row if c])
    if ct:
        ct_df = pd.DataFrame(ct.most_common(30), columns=["Country", "Count"])
        px.choropleth(ct_df, locations="Country", locationmode="country names",
                      color="Count", color_continuous_scale="OrRd",
                      title="17.6 — Azure Management Activity by Country (Impossible Travel)"
        ).update_layout(height=420).show()

# 17.4 Key Vault / 17.5 Storage
if not keyvault_results.empty:
    print("\n--- 17.4 Key Vault Enumeration ---")
    display(keyvault_results.head(20))

if not storage_exfil_results.empty:
    print("\n--- 17.5 Storage Exfiltration ---")
    display(storage_exfil_results.head(20))

if not automation_results.empty and "RiskLevel" in automation_results.columns:
    suspicious_auto = automation_results[~automation_results["RiskLevel"].str.startswith("LOW", na=False)]
    if not suspicious_auto.empty:
        print("\n--- 17.7 Suspicious Automation Deployments ---")
        display(suspicious_auto.head(20))

# ── Store for downstream sections ──────────────────────────────────────────────
azure_threat_hunt_results = {
    "cryptojacking":  cryptojacking_results,
    "privesc":        azure_privesc_results,
    "mass_delete":    mass_delete_results,
    "keyvault":       keyvault_results,
    "storage_exfil":  storage_exfil_results,
    "azure_travel":   azure_travel_results,
    "automation":     automation_results,
    "rg_deletion":    rg_deletion_results,
    "nsg_rules":      nsg_rule_results,
    "brute_force":    azure_brute_results,
    "enumeration":    azure_enum_results,
    "sp_credentials": sp_credential_results,
}
print(f"\n{'='*72}")


Running Section 17 — Azure Environment Threat Hunts...
🔍 17.1 Cryptojacking / Suspicious VM Deploy...
   ✅ Returned 12 rows
🔍 17.2 Azure Privilege Escalation...
   ✅ Returned 147 rows
🔍 17.3 Mass Resource Deletion...
   ✅ Returned 53 rows
🔍 17.4 Key Vault Enumeration / Secret Exfil...
   ✅ Returned 100 rows
🔍 17.5 Storage Exfiltration / SAS Abuse...
   ✅ Returned 0 rows
🔍 17.6 Impossible Travel in Azure Management...
   ✅ Returned 96 rows
🔍 17.7 Suspicious Automation Deployments...
   ✅ Returned 21 rows
🔍 17.8 Resource Group Mass Deletion...
   ✅ Returned 2 rows
🔍 17.9 NSG Rule Modification / Port Exposure...
   ✅ Returned 213 rows
🔍 17.10 Brute Force on Azure Portal...
   ✅ Returned 200 rows
🔍 17.11 Azure Resource Enumeration / Recon...
   ✅ Returned 0 rows
🔍 17.12 Service Principal Credential Additions...
   ✅ Returned 300 rows

SECTION 17 — AZURE ENVIRONMENT THREAT HUNT SUMMARY
  17.1  Cryptojacking                             12 events  *** FINDINGS ***
  17.2  Azure Privilege Esca


--- 17.8 Resource Group Mass Deletion ---


,Caller,CallerIpAddress,RGDeletions,DeletedRGs,Subscriptions,Countries,FirstDelete,LastDelete,SpanMinutes,DeletionRate,RiskLevel
0,02ce8729-158b-4a3d-aad5-6a1cee6c7e46,172.191.219.25,6,"[""MC_ALPINE-MDC-RG_AKSHELM_EASTUS"",""MC_ALPINE-...","[""4fc2c46b-4b55-49c7-9621-730e0a08c4aa""]","[""United States""]",2026-01-28 10:19:23.733682+00:00,2026-02-09 09:33:40.004002+00:00,17234,0.000348,HIGH - Multiple Resource Groups Deleted
1,02ce8729-158b-4a3d-aad5-6a1cee6c7e46,172.191.219.27,2,"[""MC_ALPINE-MDC-RG_HELMCLUSTER_EASTUS"",""MC_ALP...","[""4fc2c46b-4b55-49c7-9621-730e0a08c4aa""]","[""United States""]",2026-02-09 10:12:54.014869+00:00,2026-02-09 13:25:56.811925+00:00,193,0.010363,MEDIUM - More than 1 RG Deleted



--- 17.9 NSG Rule Modifications (Port Exposure) ---


,TimeGenerated,Severity,Caller,CallerIpAddress,Country,ResourceGroup,NSGName,Direction,Access,DestPort,SourcePrefix
0,2026-02-21 19:36:54.539565+00:00,LOW - NSG change (review),8c96267f-aa20-4cf8-8690-73660c9ae778,57.152.107.100,United States,MC_ALPINE-MDC-RG_AKS-AUTO_EASTUS,,,,,
1,2026-02-21 18:23:28.820199+00:00,LOW - NSG change (review),f57719fe-de9b-4ae0-96ea-150c7638ce9e,172.191.220.76,United States,MC_ALPINE-MDC-RG_ALPINE-K8-MDC_EASTUS,,,,,
2,2026-02-21 18:23:28.024923+00:00,LOW - NSG change (review),f57719fe-de9b-4ae0-96ea-150c7638ce9e,172.191.220.76,United States,MC_ALPINE-MDC-RG_ALPINE-K8-MDC_EASTUS,,,,,
3,2026-02-21 18:23:26.901967+00:00,LOW - NSG change (review),f57719fe-de9b-4ae0-96ea-150c7638ce9e,172.191.220.76,United States,MC_ALPINE-MDC-RG_ALPINE-K8-MDC_EASTUS,,,,,
4,2026-02-21 18:23:26.082380+00:00,LOW - NSG change (review),f57719fe-de9b-4ae0-96ea-150c7638ce9e,172.191.220.76,United States,MC_ALPINE-MDC-RG_ALPINE-K8-MDC_EASTUS,,,,,
5,2026-02-21 18:23:24.835768+00:00,LOW - NSG change (review),f57719fe-de9b-4ae0-96ea-150c7638ce9e,172.191.220.76,United States,MC_ALPINE-MDC-RG_ALPINE-K8-MDC_EASTUS,,,,,
6,2026-02-21 18:23:23.738075+00:00,LOW - NSG change (review),f57719fe-de9b-4ae0-96ea-150c7638ce9e,172.191.220.76,United States,MC_ALPINE-MDC-RG_ALPINE-K8-MDC_EASTUS,,,,,
7,2026-02-20 19:04:52.419677+00:00,LOW - NSG change (review),8c96267f-aa20-4cf8-8690-73660c9ae778,57.152.107.100,United States,MC_ALPINE-MDC-RG_AKS-AUTO_EASTUS,,,,,
8,2026-02-20 19:04:51.764547+00:00,LOW - NSG change (review),8c96267f-aa20-4cf8-8690-73660c9ae778,57.152.107.100,United States,MC_ALPINE-MDC-RG_AKS-AUTO_EASTUS,,,,,
9,2026-02-20 18:31:52.047678+00:00,LOW - NSG change (review),f57719fe-de9b-4ae0-96ea-150c7638ce9e,172.191.220.76,United States,MC_ALPINE-MDC-RG_ALPINE-K8-MDC_EASTUS,,,,,


...

,TimeGenerated,CredentialRisk,Actor,ActorIP,Country,OperationName,TargetApp,Result,CorrelationId
173,2026-02-20 18:06:41.225832+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,b7c6c9f6-4a26-496c-92f5-c6885bfa1e5f
174,2026-02-20 18:06:41.161824+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,b7c6c9f6-4a26-496c-92f5-c6885bfa1e5f
175,2026-02-20 18:06:41.102829+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,b7c6c9f6-4a26-496c-92f5-c6885bfa1e5f
176,2026-02-20 18:06:41.041828+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,b7c6c9f6-4a26-496c-92f5-c6885bfa1e5f
177,2026-02-20 18:06:40.984825+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,b7c6c9f6-4a26-496c-92f5-c6885bfa1e5f
178,2026-02-20 18:06:40.926824+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,b7c6c9f6-4a26-496c-92f5-c6885bfa1e5f
191,2026-02-20 16:23:32.975971+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add owner to application,,success,d0c79dce-36a3-44b9-acf4-3ff40b2f2f05
194,2026-02-20 16:20:29.954304+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add owner to application,,success,c3619bb5-f914-4412-a52a-7eac3b89af68
196,2026-02-20 16:04:35.219497+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,8da1f49e-ca5a-44a4-a76e-5299371acf6d
197,2026-02-20 16:04:35.166500+00:00,HIGH - Privilege Granted to SPN,u411@a.alpineskihouse.co,20.97.10.99,United States,Add app role assignment to service principal,Microsoft Graph,success,8da1f49e-ca5a-44a4-a76e-5299371acf6d



--- 17.4 Key Vault Enumeration ---


,identity_claim_oid_g,CallerIPAddress,TotalOps,SecretReads,KeyReads,VaultNames,Countries,UniqueIPs,FirstSeen,LastSeen,RiskLevel
0,27bf9e71-106d-49bd-b417-818d8ae59794,74.235.241.47,692,692,0,"[""MODERNWORK-KV-BSO2"",""COREID-KV-94I7"",""MDTIAC...","[""United States""]",1,2026-01-09 23:21:54.270087+00:00,2026-01-27 09:13:57.783063+00:00,CRITICAL - Mass Secret Exfiltration
1,27bf9e71-106d-49bd-b417-818d8ae59794,91.165.169.168,401,401,0,"[""COREID-KV-94I7"",""MODERNWORK-KV-BSO2"",""SENTIN...","[""France""]",1,2025-11-28 07:39:26.616888+00:00,2026-01-10 21:26:24.227630+00:00,CRITICAL - Mass Secret Exfiltration
2,27bf9e71-106d-49bd-b417-818d8ae59794,108.142.230.59,374,374,0,"[""NETSEC-KV-H09G"",""COREID-KV-94I7"",""MODERNWORK...","[""Netherlands""]",1,2025-12-10 14:56:29.960852+00:00,2026-01-19 10:55:16.143825+00:00,CRITICAL - Mass Secret Exfiltration
3,27bf9e71-106d-49bd-b417-818d8ae59794,167.220.197.217,331,331,0,"[""MODERNWORK-KV-BSO2""]","[""United Kingdom""]",1,2025-11-27 09:18:41.542158+00:00,2025-11-27 15:08:22.655657+00:00,CRITICAL - Mass Secret Exfiltration
4,27bf9e71-106d-49bd-b417-818d8ae59794,167.220.196.89,229,229,0,"[""MODERNWORK-KV-BSO2""]","[""United Kingdom""]",1,2025-11-27 09:19:31.525030+00:00,2025-11-27 14:39:43.573279+00:00,CRITICAL - Mass Secret Exfiltration
5,27bf9e71-106d-49bd-b417-818d8ae59794,20.107.5.167,211,211,0,"[""COREID-KV-94I7"",""NETSEC-KV-H09G"",""MODERNWORK...","[""Netherlands""]",1,2025-12-10 14:56:54.560956+00:00,2026-01-19 10:55:16.020329+00:00,CRITICAL - Mass Secret Exfiltration
6,27bf9e71-106d-49bd-b417-818d8ae59794,20.107.46.209,203,203,0,"[""NETSEC-KV-H09G"",""COREID-KV-94I7"",""MODERNWORK...","[""Netherlands""]",1,2025-12-10 14:56:24.591721+00:00,2026-01-19 10:55:16.089781+00:00,CRITICAL - Mass Secret Exfiltration
7,27bf9e71-106d-49bd-b417-818d8ae59794,52.174.182.198,129,129,0,"[""MODERNWORK-KV-BSO2"",""COREID-KV-94I7""]","[""Netherlands""]",1,2026-01-22 15:24:54.958585+00:00,2026-01-22 17:14:49.617833+00:00,CRITICAL - Mass Secret Exfiltration
8,27bf9e71-106d-49bd-b417-818d8ae59794,40.68.199.203,118,118,0,"[""MODERNWORK-KV-BSO2"",""COREID-KV-94I7""]","[""Netherlands""]",1,2026-01-22 15:48:13.411564+00:00,2026-01-22 17:08:37.773385+00:00,CRITICAL - Mass Secret Exfiltration
9,5089008a-13ac-47de-b01f-dfedc9d9937a,10.1.0.4,87,87,0,"[""MODERNWORK-KV-BSO2""]","[""""]",1,2025-11-27 14:32:41.294900+00:00,2026-01-23 07:41:41.076324+00:00,CRITICAL - Mass Secret Exfiltration



--- 17.7 Suspicious Automation Deployments ---


,Caller,CallerIpAddress,Deployments,ResourceTypes,DeployedItems,Countries,FirstSeen,LastSeen,CountryCount,RiskLevel
0,ecfa59d3-9cb3-4245-b3c6-08bd76710fad,52.162.111.194,4948,"[""WORKFLOWS""]","[""""]","[""United States""]",2025-11-25 13:53:55.042487+00:00,2026-02-22 13:58:58.083665+00:00,1,HIGH - Many Automation Resources or Cross-Country
1,238b9693-7595-4d31-b268-a2e5cedbbda2,20.246.140.26,451,"[""WORKFLOWS"",""MANAGEDCLUSTERS""]","[""""]","[""United States""]",2025-11-26 00:10:44.458170+00:00,2026-02-19 19:08:16.502089+00:00,1,HIGH - Many Automation Resources or Cross-Country
2,u5470@int.zava-corp.com,2a04:7f80:1076:ca00:f8b5:d529:6267:2c9,37,"[""WORKFLOWS""]","[""""]","[""Qatar""]",2025-12-03 10:01:30.183568+00:00,2025-12-03 14:45:05.025773+00:00,1,HIGH - Many Automation Resources or Cross-Country
3,u17189@ops.zava-corp.com,70.37.27.17,8,"[""WORKFLOWS""]","[""""]","[""United States""]",2026-01-22 20:39:18.508050+00:00,2026-01-23 19:59:06.605923+00:00,1,MEDIUM - Multiple Automation Deployments
4,u17189@ops.zava-corp.com,70.37.27.13,8,"[""WORKFLOWS""]","[""""]","[""United States""]",2026-01-23 16:53:15.162197+00:00,2026-01-27 05:55:55.837766+00:00,1,MEDIUM - Multiple Automation Deployments
5,u13155@int.zava-corp.com,167.220.197.129,7,"[""WORKFLOWS""]","[""""]","[""United Kingdom""]",2026-01-09 15:13:02.569899+00:00,2026-01-09 16:03:37.164662+00:00,1,MEDIUM - Multiple Automation Deployments
6,u441@a.alpineskihouse.co,2409:4091:9005:760d:4594:1735:6d80:ec31,6,"[""MANAGEDCLUSTERS""]","[""""]","[""India""]",2026-02-09 10:33:00.690566+00:00,2026-02-09 15:17:57.003377+00:00,1,MEDIUM - Multiple Automation Deployments
7,u13155@int.zava-corp.com,2a01:110:8012:1010:826e:39d0:4824:3269,6,"[""WORKFLOWS""]","[""""]","[""United Kingdom""]",2026-01-20 15:17:48.045582+00:00,2026-01-20 15:40:19.185100+00:00,1,MEDIUM - Multiple Automation Deployments



---
## 🔬 Section 12 — Forensic Investigation Hub

Deep-dive investigation toolkit for SOC analysts. Set the target value in each sub-section and run to pull a full cross-table timeline.

| Sub-section | Target | Tables Queried |
|-------------|--------|----------------|
| **12.1 IP Forensics** | `INVESTIGATE_IP` | SigninLogs, DeviceNetworkEvents, EmailEvents, ThreatIntelligenceIndicator |
| **12.2 Device Forensics** | `INVESTIGATE_DEVICE` | DeviceLogonEvents, DeviceNetworkEvents, DeviceProcessEvents, DeviceFileEvents, SecurityAlert |
| **12.3 Identity Forensics** | `INVESTIGATE_UPN` | SigninLogs, AuditLogs, AADRiskyUsers, DeviceLogonEvents, EmailEvents |

> **Usage:** Change the `INVESTIGATE_*` variables in each cell, then run the cell to generate a full forensic timeline.


In [41]:

# ── 12.1  IP Forensics ────────────────────────────────────────────────────────
# Full pivot on a single IP address across all relevant tables.
# Change INVESTIGATE_IP to the IP you want to investigate.

INVESTIGATE_IP = "185.220.101.60"          # <── SET YOUR TARGET IP HERE
FORENSIC_DAYS  = 90                   # look-back window for this investigation

print(f"{'='*72}")
print(f"🔍 IP FORENSIC INVESTIGATION: {INVESTIGATE_IP}")
print(f"   Window: last {FORENSIC_DAYS} days")
print(f"{'='*72}")

ip_signin_query = f"""
SigninLogs
| where TimeGenerated > ago({FORENSIC_DAYS}d)
| where IPAddress == "{INVESTIGATE_IP}"
| project TimeGenerated, UserPrincipalName, AppDisplayName,
          ResultType, ResultDescription, Location, DeviceDetail,
          RiskLevelDuringSignIn, ConditionalAccessStatus
| order by TimeGenerated desc
| take 100
"""

ip_device_query = f"""
DeviceNetworkEvents
| where TimeGenerated > ago({FORENSIC_DAYS}d)
| where RemoteIP == "{INVESTIGATE_IP}" or LocalIP == "{INVESTIGATE_IP}"
| project TimeGenerated, DeviceName, ActionType,
          LocalIP, LocalPort, RemoteIP, RemotePort, RemoteUrl,
          InitiatingProcessAccountName, InitiatingProcessFileName
| order by TimeGenerated desc
| take 100
"""

ip_email_query = f"""
EmailEvents
| where TimeGenerated > ago({FORENSIC_DAYS}d)
| where SenderIPv4 == "{INVESTIGATE_IP}"
| project TimeGenerated, SenderFromAddress, SenderDisplayName,
          RecipientEmailAddress, Subject, DeliveryAction, ThreatTypes
| order by TimeGenerated desc
| take 50
"""

ip_ti_query = f"""
ThreatIntelligenceIndicator
| where TimeGenerated > ago(90d)
| where NetworkIP == "{INVESTIGATE_IP}"
    or NetworkDestinationIP == "{INVESTIGATE_IP}"
    or NetworkSourceIP == "{INVESTIGATE_IP}"
| project TimeGenerated, ThreatType, ConfidenceScore, Description,
          IndicatorId, ExpirationDateTime, Tags
| order by ConfidenceScore desc
"""

print("\n📡 [1/4] Sign-in activity from this IP...")
ip_signins = run_kql(ip_signin_query, "")
print(f"\n🖥️  [2/4] Device network events involving this IP...")
ip_devices = run_kql(ip_device_query, "")
print(f"\n📧 [3/4] Email traffic from this IP...")
ip_emails  = run_kql(ip_email_query, "")
print(f"\n🎯 [4/4] Threat Intelligence indicators for this IP...")
ip_ti      = run_kql(ip_ti_query, "")

# ─ Summary ─
print(f"\n{'='*72}")
print(f"📋 IP FORENSIC SUMMARY: {INVESTIGATE_IP}")
print(f"{'='*72}")
print(f"   Sign-ins:                {len(ip_signins)} events")
print(f"   Device network events:   {len(ip_devices)} events")
print(f"   Email messages sent:     {len(ip_emails)} events")
ti_hits = len(ip_ti)
if ti_hits > 0:
    max_conf = ip_ti["ConfidenceScore"].max() if "ConfidenceScore" in ip_ti.columns else "N/A"
    print(f"   🔴 TI HITS:              {ti_hits} indicators (max confidence: {max_conf})")
    display(ip_ti)
else:
    print(f"   TI indicators:           0 (not known-malicious)")

if not ip_signins.empty:
    users = ip_signins["UserPrincipalName"].nunique() if "UserPrincipalName" in ip_signins.columns else 0
    if users > 1:
        print(f"\n   ⚠️  Multiple users ({users}) signed in from this IP — possible shared/VPN/proxy")
    display(ip_signins.head(10))

if not ip_devices.empty:
    devices = ip_devices["DeviceName"].nunique() if "DeviceName" in ip_devices.columns else 0
    print(f"\n   Contacted by {devices} unique device(s):")
    display(ip_devices.head(10))

if not ip_emails.empty:
    print(f"\n   Emails sent from this IP:")
    display(ip_emails.head(10))

# ─ Timeline chart combining all events ─
all_timeline = []
for df, source in [(ip_signins, "SignIn"), (ip_devices, "Device"), (ip_emails, "Email")]:
    if not df.empty and "TimeGenerated" in df.columns:
        tmp = df[["TimeGenerated"]].copy()
        tmp["Source"] = source
        tmp["TimeGenerated"] = pd.to_datetime(tmp["TimeGenerated"], errors="coerce")
        tmp = tmp.dropna()
        all_timeline.append(tmp)

if all_timeline:
    timeline_df = pd.concat(all_timeline)
    timeline_df["Hour"] = timeline_df["TimeGenerated"].dt.floor("h")
    chart_df = timeline_df.groupby(["Hour","Source"]).size().reset_index(name="Count")
    fig = px.bar(chart_df, x="Hour", y="Count", color="Source", barmode="stack",
                 title=f"📡 IP Activity Timeline: {INVESTIGATE_IP}",
                 labels={"Hour": "Time", "Count": "Events"})
    fig.show()


🔍 IP FORENSIC INVESTIGATION: 185.220.101.60
   Window: last 90 days

📡 [1/4] Sign-in activity from this IP...
   ✅ Returned 89 rows

🖥️  [2/4] Device network events involving this IP...
   ✅ Returned 0 rows

📧 [3/4] Email traffic from this IP...
   ✅ Returned 0 rows

🎯 [4/4] Threat Intelligence indicators for this IP...
   ✅ Returned 0 rows

📋 IP FORENSIC SUMMARY: 185.220.101.60
   Sign-ins:                89 events
   Device network events:   0 events
   Email messages sent:     0 events
   TI indicators:           0 (not known-malicious)


,TimeGenerated,UserPrincipalName,AppDisplayName,ResultType,ResultDescription,Location,DeviceDetail,RiskLevelDuringSignIn,ConditionalAccessStatus
0,2026-02-06 09:28:04.117274+00:00,elviaa@zava-corp.com,ADIbizaUX,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
1,2026-02-06 09:28:03.917307+00:00,elviaa@zava-corp.com,ADIbizaUX,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
2,2026-02-06 09:28:01.108203+00:00,elviaa@zava-corp.com,Azure Portal,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
3,2026-02-06 09:27:56.543980+00:00,elviaa@zava-corp.com,Azure Portal,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
4,2026-02-06 09:27:24.536184+00:00,elviaa@zava-corp.com,ADIbizaUX,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
5,2026-02-06 09:26:26.374548+00:00,elviaa@zava-corp.com,Azure Portal,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
6,2026-02-06 09:26:16.180060+00:00,elviaa@zava-corp.com,Azure Portal,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
7,2026-02-06 09:26:12.574475+00:00,elviaa@zava-corp.com,Azure Portal,50140,This occurred due to 'Keep me signed in' inter...,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
8,2026-02-06 09:25:57.569488+00:00,elviaa@zava-corp.com,Microsoft_AAD_RegisteredApps,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success
9,2026-02-06 09:25:56.918090+00:00,elviaa@zava-corp.com,Microsoft_AAD_RegisteredApps,0,,DE,"{""deviceId"":"""",""operatingSystem"":""Windows10"",""...",medium,success


In [42]:

# ── 12.2  Device Forensics ────────────────────────────────────────────────────
# Full activity timeline on a single device name.
# Set INVESTIGATE_DEVICE to the DeviceName (partial match supported).

INVESTIGATE_DEVICE = "ash-irvins"       # <── SET YOUR TARGET DEVICE NAME (prefix OK)
FORENSIC_DAYS_DEV  = 30

print(f"{'='*72}")
print(f"🖥️  DEVICE FORENSIC INVESTIGATION: {INVESTIGATE_DEVICE}")
print(f"   Window: last {FORENSIC_DAYS_DEV} days")
print(f"{'='*72}")

dev_logon_query = f"""
DeviceLogonEvents
| where TimeGenerated > ago({FORENSIC_DAYS_DEV}d)
| where DeviceName startswith "{INVESTIGATE_DEVICE}"
| project TimeGenerated, DeviceName, AccountName, AccountDomain,
          LogonType, ActionType, RemoteIP, RemoteDeviceName,
          InitiatingProcessFileName
| order by TimeGenerated desc | take 200
"""

dev_network_query = f"""
DeviceNetworkEvents
| where TimeGenerated > ago({FORENSIC_DAYS_DEV}d)
| where DeviceName startswith "{INVESTIGATE_DEVICE}"
| summarize Connections = count(), UniqueRemoteIPs = dcount(RemoteIP),
            FirstSeen = min(TimeGenerated), LastSeen = max(TimeGenerated)
  by RemoteIP, RemotePort, InitiatingProcessFileName
| order by Connections desc | take 50
"""

dev_process_query = f"""
DeviceProcessEvents
| where TimeGenerated > ago({FORENSIC_DAYS_DEV}d)
| where DeviceName startswith "{INVESTIGATE_DEVICE}"
| where InitiatingProcessFileName !in ("svchost.exe","explorer.exe","SearchHost.exe","RuntimeBroker.exe")
| project TimeGenerated, DeviceName, AccountName, FileName,
          ProcessCommandLine, InitiatingProcessFileName,
          InitiatingProcessCommandLine, SHA256
| order by TimeGenerated desc | take 100
"""

dev_file_query = f"""
DeviceFileEvents
| where TimeGenerated > ago({FORENSIC_DAYS_DEV}d)
| where DeviceName startswith "{INVESTIGATE_DEVICE}"
| where ActionType in ("FileCreated","FileModified","FileRenamed")
| where FolderPath !startswith "C:\\\\Windows\\\\"
    and FolderPath !startswith "C:\\\\Program Files"
| project TimeGenerated, DeviceName, ActionType, FileName,
          FolderPath, InitiatingProcessAccountName,
          InitiatingProcessFileName, SHA256
| order by TimeGenerated desc | take 100
"""

dev_alert_query = f"""
SecurityAlert
| where TimeGenerated > ago({FORENSIC_DAYS_DEV}d)
| where Entities has "{INVESTIGATE_DEVICE}"
| project TimeGenerated, AlertName, AlertSeverity, Description,
          ProviderName, Status, Tactics
| order by AlertSeverity asc, TimeGenerated desc | take 50
"""

print("\n🔑 [1/5] Logon events...")
dev_logons   = run_kql(dev_logon_query, "")
print("\n🌐 [2/5] Network connections...")
dev_network  = run_kql(dev_network_query, "")
print("\n⚙️  [3/5] Process executions (non-system)...")
dev_procs    = run_kql(dev_process_query, "")
print("\n📁 [4/5] File changes (user/temp paths)...")
dev_files    = run_kql(dev_file_query, "")
print("\n🚨 [5/5] Security alerts involving device...")
dev_alerts   = run_kql(dev_alert_query, "")

# ─ Summary ─
print(f"\n{'='*72}")
print(f"📋 DEVICE FORENSIC SUMMARY: {INVESTIGATE_DEVICE}")
print(f"{'='*72}")
print(f"   Logon events:            {len(dev_logons)}")
print(f"   Unique remote IPs:       {dev_network['RemoteIP'].nunique() if not dev_network.empty and 'RemoteIP' in dev_network.columns else 0}")
print(f"   Process executions:      {len(dev_procs)}")
print(f"   File changes:            {len(dev_files)}")
alerts_cnt = len(dev_alerts)
sev_label = ("🔴" if any(s in ["High","Critical"] for s in (dev_alerts["AlertSeverity"].tolist() if not dev_alerts.empty and "AlertSeverity" in dev_alerts.columns else [])) else "🟠") if alerts_cnt > 0 else "🟢"
print(f"   {sev_label} Security alerts:         {alerts_cnt}")

if not dev_alerts.empty:
    print("\n🚨 ALERTS ON THIS DEVICE:")
    display(dev_alerts)
if not dev_logons.empty:
    users_on_device = dev_logons["AccountName"].nunique() if "AccountName" in dev_logons.columns else 0
    print(f"\n👤 {users_on_device} unique account(s) logged on:")
    display(dev_logons.head(10))
if not dev_procs.empty:
    print("\n⚙️ Notable process executions:")
    display(dev_procs.head(15))
if not dev_network.empty:
    print("\n🌐 Top remote connections:")
    display(dev_network.head(15))
if not dev_files.empty:
    print("\n📁 File changes in user paths:")
    display(dev_files.head(15))

# ─ Process timeline chart ─
if not dev_procs.empty and "TimeGenerated" in dev_procs.columns:
    dev_procs["TimeGenerated"] = pd.to_datetime(dev_procs["TimeGenerated"], errors="coerce")
    hourly = dev_procs.dropna(subset=["TimeGenerated"]).groupby(
        dev_procs["TimeGenerated"].dt.floor("h")).size().reset_index(name="ProcessCount")
    fig = px.line(hourly, x="TimeGenerated", y="ProcessCount",
                  title=f"⚙️ Process Execution Frequency: {INVESTIGATE_DEVICE}",
                  markers=True)
    fig.show()


🖥️  DEVICE FORENSIC INVESTIGATION: ash-irvins
   Window: last 30 days

🔑 [1/5] Logon events...
   ✅ Returned 91 rows

🌐 [2/5] Network connections...
   ✅ Returned 50 rows

⚙️  [3/5] Process executions (non-system)...
   ✅ Returned 100 rows

📁 [4/5] File changes (user/temp paths)...
   ✅ Returned 100 rows

🚨 [5/5] Security alerts involving device...
   ✅ Returned 5 rows

📋 DEVICE FORENSIC SUMMARY: ash-irvins
   Logon events:            91
   Unique remote IPs:       42
   Process executions:      100
   File changes:            100
   🟠 Security alerts:         5

🚨 ALERTS ON THIS DEVICE:


,TimeGenerated,AlertName,AlertSeverity,Description,ProviderName,Status,Tactics
0,2026-02-03 05:38:12.232909+00:00,DLP policy (DLP Policy - Classified Project Fi...,Medium,,MicrosoftEndPointDlp,Resolved,Exfiltration
1,2026-01-27 19:35:04.076345+00:00,DLP policy (DLP Policy - Classified Project Fi...,Medium,,MicrosoftEndPointDlp,New,Exfiltration
2,2026-01-27 04:23:08.359589+00:00,DLP policy (DLP Policy - Classified Project Fi...,Medium,,MicrosoftEndPointDlp,New,Exfiltration
3,2026-01-26 22:56:05.211977+00:00,DLP policy (DLP Policy - Classified Project Fi...,Medium,,MicrosoftEndPointDlp,New,Exfiltration
4,2026-01-26 22:29:09.582735+00:00,DLP policy (DLP Policy - Classified Project Fi...,Medium,,MicrosoftEndPointDlp,New,Exfiltration



👤 8 unique account(s) logged on:


,TimeGenerated,DeviceName,AccountName,AccountDomain,LogonType,ActionType,RemoteIP,RemoteDeviceName,InitiatingProcessFileName
0,2026-02-22 03:05:42.601909+00:00,ash-irvins,-,,Batch,LogonFailed,,,svchost.exe
1,2026-02-22 00:38:40.051685+00:00,ash-irvins,-,,Batch,LogonFailed,,,svchost.exe
2,2026-02-22 00:35:15.848482+00:00,ash-irvins,dwm-1,window manager,Interactive,LogonSuccess,,,winlogon.exe
3,2026-02-22 00:35:15.848461+00:00,ash-irvins,dwm-1,window manager,Interactive,LogonSuccess,,,winlogon.exe
4,2026-02-22 00:35:15.848432+00:00,ash-irvins,dwm-1,window manager,Unknown,LogonAttempted,-,,winlogon.exe
5,2026-02-22 00:35:15.373611+00:00,ash-irvins,umfd-0,font driver host,Interactive,LogonSuccess,,,wininit.exe
6,2026-02-22 00:35:15.373589+00:00,ash-irvins,umfd-1,font driver host,Interactive,LogonSuccess,,,winlogon.exe
7,2026-02-22 00:35:15.373544+00:00,ash-irvins,umfd-0,font driver host,Unknown,LogonAttempted,-,,wininit.exe
8,2026-02-22 00:35:15.373544+00:00,ash-irvins,umfd-1,font driver host,Unknown,LogonAttempted,-,,winlogon.exe
9,2026-02-22 00:32:59.436427+00:00,ash-irvins,dwm-1,window manager,Interactive,LogonSuccess,,,winlogon.exe



⚙️ Notable process executions:


,TimeGenerated,DeviceName,AccountName,FileName,ProcessCommandLine,InitiatingProcessFileName,InitiatingProcessCommandLine,SHA256
0,2026-02-23 10:49:41.561590+00:00,ash-irvins,system,conhost.exe,conhost.exe 0xffffffff -ForceV1,mpcmdrun.exe,"""MpCmdRun.exe"" Scan -ScheduleJob -ScanTrigger 55",f3dbb469e96320311cd8d6e6ca00dd538d5bbead3712b4...
1,2026-02-23 10:34:46.007218+00:00,ash-irvins,,TrustedInstaller.exe,TrustedInstaller.exe,services.exe,services.exe,
2,2026-02-23 10:34:45.991498+00:00,ash-irvins,,conhost.exe,conhost.exe 0xffffffff -ForceV1,powershell.exe,powershell.exe -ExecutionPolicy AllSigned -NoP...,
3,2026-02-23 10:34:45.975547+00:00,ash-irvins,,SenseImdsCollector.exe,"""SenseImdsCollector.exe"" 1",MsSense.exe,"""MsSense.exe""",
4,2026-02-23 10:34:45.959863+00:00,ash-irvins,,svchost.exe,svchost.exe -k netsvcs -p -s wlidsvc,services.exe,services.exe,
5,2026-02-23 10:34:45.943952+00:00,ash-irvins,,MpCmdRun.exe,"""MpCmdRun.exe"" GetDeviceTicket -AccessKey C7E1...",MsMpEng.exe,"""MsMpEng.exe""",
6,2026-02-23 10:34:45.912164+00:00,ash-irvins,,updater.exe,"""updater.exe"" --crash-handler --system ""--data...",updater.exe,"""updater.exe"" --system --windows-service --ser...",
7,2026-02-23 10:34:45.896287+00:00,ash-irvins,,svchost.exe,svchost.exe -k netsvcs -p -s wuauserv,services.exe,services.exe,
8,2026-02-23 10:34:45.880361+00:00,ash-irvins,,MpCmdRun.exe,"""MpCmdRun.exe"" GetDeviceTicket -AccessKey 2271...",MsMpEng.exe,"""MsMpEng.exe""",
9,2026-02-23 10:34:45.848980+00:00,ash-irvins,,svchost.exe,svchost.exe -k wsappx -p -s AppXSvc,services.exe,services.exe,



🌐 Top remote connections:


,RemoteIP,RemotePort,InitiatingProcessFileName,Connections,UniqueRemoteIPs,FirstSeen,LastSeen
0,92.223.96.6,80,svchost.exe,7690,1,2026-01-24 11:36:49.805781+00:00,2026-02-23 10:52:42.810106+00:00
1,192.168.50.1,53,,2807,1,2026-01-24 13:58:41.405920+00:00,2026-02-23 11:04:47.374680+00:00
2,199.232.210.172,80,svchost.exe,1291,1,2026-01-24 11:38:50.996416+00:00,2026-02-23 10:52:48.852231+00:00
3,199.232.214.172,80,svchost.exe,1201,1,2026-01-24 14:29:14.975758+00:00,2026-02-23 10:52:48.852479+00:00
4,169.254.169.254,80,SenseImdsCollector.exe,682,1,2026-01-24 11:47:01.310157+00:00,2026-02-23 10:24:08.635320+00:00
5,23.61.94.21,443,svchost.exe,681,1,2026-01-24 11:38:51.267029+00:00,2026-02-23 10:15:58.060680+00:00
6,::1,40342,SenseImdsCollector.exe,680,1,2026-01-24 11:47:01.294411+00:00,2026-02-23 10:24:08.667379+00:00
7,127.0.0.1,40342,SenseImdsCollector.exe,651,1,2026-01-24 11:47:01.278730+00:00,2026-02-23 10:24:08.651489+00:00
8,40.74.25.3,443,Microsoft.Management.Services.IntuneWindowsAge...,619,1,2026-01-24 11:38:51.345989+00:00,2026-02-23 10:15:58.076507+00:00
9,23.193.194.14,80,svchost.exe,571,1,2026-01-24 14:08:32.889932+00:00,2026-02-23 10:53:49.214873+00:00



📁 File changes in user paths:


,TimeGenerated,DeviceName,ActionType,FileName,FolderPath,InitiatingProcessAccountName,InitiatingProcessFileName,SHA256
0,2026-02-23 07:03:11.401479+00:00,ash-irvins,FileRenamed,WINDOWS.SIUF.xml,C:\ProgramData\Microsoft\Diagnosis\DownloadedS...,system,svchost.exe,bc3c203fb09e54b9eb967b4407064381c77784ad9950e9...
1,2026-02-23 06:45:24.132716+00:00,ash-irvins,FileModified,{6C2D27F1-5DF2-4B36-A544-9B83F36B0BC7}_1514095...,C:\ProgramData\Microsoft\Windows Defender Adva...,system,svchost.exe,fff6c7f244d198edf03ae6c77299e6f328cb7fbb35f096...
2,2026-02-23 02:45:16.884062+00:00,ash-irvins,FileRenamed,8AFD76F1-8CC4-4925-81E9-0FD8DEA009E4_Device_65...,C:\ProgramData\Microsoft\DC\HostOS\8AFD76F1-8C...,system,svchost.exe,f75579ef04d4cb90ed5995de6788e3d5eabda9960b643e...
3,2026-02-23 02:45:12.214613+00:00,ash-irvins,FileRenamed,8AFD76F1-8CC4-4925-81E9-0FD8DEA009E4_Device_23...,C:\ProgramData\Microsoft\DC\HostOS\8AFD76F1-8C...,system,svchost.exe,1fdb1457424c657fc762c01dc4c849af99da92235673f3...
4,2026-02-23 02:03:12.206677+00:00,ash-irvins,FileRenamed,WINDOWS.DIAGNOSTICS.xml,C:\ProgramData\Microsoft\Diagnosis\DownloadedS...,system,svchost.exe,575203b135f08ae2eff77c0a3dabf2fec5bf2eed4627db...
5,2026-02-23 00:37:24.169387+00:00,ash-irvins,FileCreated,energy-report.html,C:\ProgramData\Microsoft\Windows\Power Efficie...,system,taskhostw.exe,07a397141e75c2abe9a69a03e78816e0def9f59738547e...
6,2026-02-23 00:37:24.038629+00:00,ash-irvins,FileCreated,energy-report-2026-02-22.xml,C:\ProgramData\Microsoft\Windows\Power Efficie...,system,taskhostw.exe,f5521f47bf4e7be283bcdf07f85d7fd2cc46884c6d3476...
7,2026-02-23 00:37:23.516264+00:00,ash-irvins,FileCreated,energy-report-latest.xml,C:\ProgramData\Microsoft\Windows\Power Efficie...,system,taskhostw.exe,f5521f47bf4e7be283bcdf07f85d7fd2cc46884c6d3476...
8,2026-02-23 00:33:11.948939+00:00,ash-irvins,FileRenamed,WINDOWS.PERFTRACKPOINTDATA.xml,C:\ProgramData\Microsoft\Diagnosis\DownloadedS...,system,svchost.exe,575203b135f08ae2eff77c0a3dabf2fec5bf2eed4627db...
9,2026-02-23 00:33:11.858585+00:00,ash-irvins,FileRenamed,WINDOWS.PERFTRACKESCALATIONS.xml,C:\ProgramData\Microsoft\Diagnosis\DownloadedS...,system,svchost.exe,575203b135f08ae2eff77c0a3dabf2fec5bf2eed4627db...


In [43]:

# ── 12.3  Identity Forensics ──────────────────────────────────────────────────
# Full account timeline — sign-ins, admin actions, risk events, device logons,
# email sent/received. Set INVESTIGATE_UPN to the target user.

INVESTIGATE_UPN   = "elviaa"   # <── SET TARGET USER UPN
FORENSIC_DAYS_ID  = 30

print(f"{'='*72}")
print(f"👤 IDENTITY FORENSIC INVESTIGATION: {INVESTIGATE_UPN}")
print(f"   Window: last {FORENSIC_DAYS_ID} days")
print(f"{'='*72}")

id_signin_query = f"""
SigninLogs
| where TimeGenerated > ago({FORENSIC_DAYS_ID}d)
| where UserPrincipalName =~ "{INVESTIGATE_UPN}"
| extend GeoInfo = geo_info_from_ip_address(IPAddress)
| project TimeGenerated, AppDisplayName, IPAddress,
          Country = tostring(GeoInfo.country), City = tostring(GeoInfo.city),
          ResultType, RiskLevelDuringSignIn, AuthenticationRequirement,
          ConditionalAccessStatus, DeviceDetail
| order by TimeGenerated desc | take 200
"""

id_audit_query = f"""
AuditLogs
| where TimeGenerated > ago({FORENSIC_DAYS_ID}d)
| where InitiatedBy has "{INVESTIGATE_UPN}"
    or TargetResources has "{INVESTIGATE_UPN}"
| project TimeGenerated, OperationName, Category, Result,
          InitiatedByUser = tostring(InitiatedBy.user.userPrincipalName),
          TargetObjects   = tostring(TargetResources),
          IPAddress = tostring(InitiatedBy.user.ipAddress)
| order by TimeGenerated desc | take 100
"""

id_risk_query = f"""
AADUserRiskEvents
| where TimeGenerated > ago({FORENSIC_DAYS_ID}d)
| where UserPrincipalName =~ "{INVESTIGATE_UPN}"
| project TimeGenerated, RiskEventType, RiskLevel, RiskState,
          IpAddress, Location, AdditionalInfo
| order by RiskLevel asc, TimeGenerated desc
"""

# DeviceLogonEvents has no AccountUpn column — match on AccountName (prefix of UPN)
_upn_username = INVESTIGATE_UPN.split("@")[0] if "@" in INVESTIGATE_UPN else INVESTIGATE_UPN
id_device_query = f"""
DeviceLogonEvents
| where TimeGenerated > ago({FORENSIC_DAYS_ID}d)
| where AccountName =~ "{_upn_username}"
| summarize LogonCount = count(), FirstSeen = min(TimeGenerated),
            LastSeen = max(TimeGenerated)
  by DeviceName, AccountDomain, LogonType, ActionType
| order by LogonCount desc | take 30
"""

id_email_query = f"""
EmailEvents
| where TimeGenerated > ago({FORENSIC_DAYS_ID}d)
| where SenderFromAddress =~ "{INVESTIGATE_UPN}"
    or RecipientEmailAddress =~ "{INVESTIGATE_UPN}"
| project TimeGenerated, Direction = EmailDirection,
          From = SenderFromAddress, To = RecipientEmailAddress,
          Subject, DeliveryAction, ThreatTypes
| order by TimeGenerated desc | take 50
"""

print(f"\n🔑 [1/5] Sign-in history...")
id_signins  = run_kql(id_signin_query, "")
print(f"\n📋 [2/5] Audit / admin actions...")
id_audits   = run_kql(id_audit_query, "")
print(f"\n⚠️  [3/5] Risk events...")
id_risks    = run_kql(id_risk_query, "")
print(f"\n🖥️  [4/5] Device logons by this account...")
id_devices  = run_kql(id_device_query, "")
print(f"\n📧 [5/5] Email activity...")
id_emails   = run_kql(id_email_query, "")

# ─ Summary ─
print(f"\n{'='*72}")
print(f"📋 IDENTITY FORENSIC SUMMARY: {INVESTIGATE_UPN}")
print(f"{'='*72}")
print(f"   Sign-in events:          {len(id_signins)}")
if not id_signins.empty and "Country" in id_signins.columns:
    countries = id_signins["Country"].dropna().unique()
    print(f"   Countries signed in from: {', '.join(countries[:5])}")
    if len(countries) > 3:
        print(f"   ⚠️  RISK: Multiple countries — review for impossible travel")
print(f"   Audit operations:        {len(id_audits)}")
risk_cnt = len(id_risks)
print(f"   {'🔴' if risk_cnt > 0 else '🟢'} Risk events:             {risk_cnt}")
print(f"   Unique devices logged on: {id_devices['DeviceName'].nunique() if not id_devices.empty and 'DeviceName' in id_devices.columns else 0}")
print(f"   Email events:            {len(id_emails)}")

if not id_risks.empty:
    print(f"\n⚠️ IDENTITY RISK EVENTS:")
    display(id_risks)

if not id_signins.empty:
    print(f"\n🔑 SIGN-IN HISTORY (latest 10):")
    display(id_signins.head(10))
    # Sign-in country chart
    if "Country" in id_signins.columns:
        country_counts = id_signins.groupby("Country").size().reset_index(name="Count")
        fig = px.bar(country_counts.sort_values("Count", ascending=False).head(15),
                     x="Country", y="Count",
                     title=f"👤 Sign-in Countries: {INVESTIGATE_UPN}",
                     color="Count", color_continuous_scale="Reds")
        fig.show()

if not id_audits.empty:
    print(f"\n📋 ADMIN OPERATIONS (latest 10):")
    display(id_audits.head(10))

if not id_devices.empty:
    print(f"\n🖥️  DEVICES ACCESSED:")
    display(id_devices)

if not id_emails.empty:
    print(f"\n📧 EMAIL ACTIVITY (latest 10):")
    display(id_emails.head(10))


👤 IDENTITY FORENSIC INVESTIGATION: elviaa
   Window: last 30 days

🔑 [1/5] Sign-in history...
   ✅ Returned 0 rows

📋 [2/5] Audit / admin actions...
   ✅ Returned 100 rows

⚠️  [3/5] Risk events...
   ✅ Returned 0 rows

🖥️  [4/5] Device logons by this account...
   ✅ Returned 0 rows

📧 [5/5] Email activity...
   ✅ Returned 0 rows

📋 IDENTITY FORENSIC SUMMARY: elviaa
   Sign-in events:          0
   Audit operations:        100
   🟢 Risk events:             0
   Unique devices logged on: 0
   Email events:            0

📋 ADMIN OPERATIONS (latest 10):


,TimeGenerated,OperationName,Category,Result,InitiatedByUser,TargetObjects,IPAddress
0,2026-02-19 07:23:12.074624+00:00,Synchronization rule action,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",
1,2026-02-19 07:23:12.032617+00:00,Synchronization rule action,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",
2,2026-02-19 07:23:12.023621+00:00,Synchronization rule action,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",
3,2026-02-19 07:23:11.979617+00:00,Import,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",
4,2026-02-19 06:58:08.460278+00:00,Disable account,UserManagement,success,,"[{""id"":""befd56cb-ba43-4f04-a630-252ee3870c7f"",...",
5,2026-02-19 06:58:08.459278+00:00,Update user,UserManagement,success,,"[{""id"":""befd56cb-ba43-4f04-a630-252ee3870c7f"",...",
6,2026-02-19 06:42:13.356350+00:00,Synchronization rule action,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",
7,2026-02-19 06:42:13.315350+00:00,Synchronization rule action,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",
8,2026-02-19 06:42:13.307350+00:00,Synchronization rule action,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",
9,2026-02-19 06:42:13.255350+00:00,Import,ProvisioningManagement,success,,"[{""id"":""3af32bf9-cd49-4b8b-aa72-d0fa47b0f40f"",...",



---
## 📊 Section 13 — SOC KPI Dashboard

Key Performance Indicators for the Security Operations Center, rendered as a KPI scorecard and trend charts.

| KPI | Target | Description |
|-----|--------|-------------|
| **MTTD** | < 24 h | Mean Time to Detect — first alert to analyst awareness |
| **Alert Volume Trend** | Stable ↓ | Daily alert count over analysis period |
| **Alert Severity Mix** | > 80% H/M actioned | High/Medium/Low/Info breakdown |
| **True-Positive Rate** | > 70% | Closed-as-TP vs total closed alerts |
| **Top Alert Sources** | Balanced | Alerts by provider — identify noisy sources |
| **Credential Attack Volume** | 0 trend ↓ | Brute force + spray over time |


In [44]:

# ── 13.  SOC KPI Dashboard ────────────────────────────────────────────────────
from plotly.subplots import make_subplots

# ─ KPI Query 1: Alert volume + severity ──────────────────────────────────────
kpi_alerts_query = f"""
SecurityAlert
| where {TIME_RANGE}
| extend Day = bin(TimeGenerated, 1d)
| summarize
    TotalAlerts  = count(),
    HighAlerts   = countif(AlertSeverity == "High"),
    MedAlerts    = countif(AlertSeverity == "Medium"),
    LowAlerts    = countif(AlertSeverity == "Low"),
    InfoAlerts   = countif(AlertSeverity == "Informational"),
    ClosedTotal  = countif(Status == "Resolved")
  by Day
| order by Day asc
"""

kpi_provider_query = f"""
SecurityAlert
| where {TIME_RANGE}
| summarize AlertCount = count(), HighAlerts = countif(AlertSeverity in ("High","Medium"))
  by ProviderName
| order by AlertCount desc
| take 15
"""

kpi_mttd_query = f"""
SecurityAlert
| where {TIME_RANGE}
| where isnotempty(StartTime) and isnotempty(TimeGenerated)
| extend MTTD_Hours = datetime_diff("hour", TimeGenerated, StartTime)
| where MTTD_Hours >= 0 and MTTD_Hours <= 720
| summarize
    Avg_MTTD_Hours    = round(avg(MTTD_Hours), 1),
    Median_MTTD_Hours = percentile(MTTD_Hours, 50),
    P95_MTTD_Hours    = percentile(MTTD_Hours, 95),
    AlertCount        = count()
"""

print("📊 Loading SOC KPI data...")
kpi_daily    = run_kql(kpi_alerts_query,   "Daily alert KPIs")
kpi_provider = run_kql(kpi_provider_query, "Alerts by provider")
kpi_mttd_df  = run_kql(kpi_mttd_query,    "MTTD calculation")

# ─ Derived KPIs from notebook findings ───────────────────────────────────────
_bf  = len(brute_force_results)    if 'brute_force_results'    in dir() and not brute_force_results.empty    else 0
_ps  = len(password_spray_results) if 'password_spray_results' in dir() and not password_spray_results.empty else 0
_ru  = len(risky_users)            if 'risky_users'            in dir() and not risky_users.empty            else 0
_it  = len(impossible_travel)      if 'impossible_travel'      in dir() and not impossible_travel.empty      else 0
_da  = len(defender_alerts)        if 'defender_alerts'        in dir() and not defender_alerts.empty        else 0
_total_ioc = sum([
    len(network_ioc_matches)  if 'network_ioc_matches'  in dir() and not network_ioc_matches.empty  else 0,
    len(file_ioc_matches)     if 'file_ioc_matches'     in dir() and not file_ioc_matches.empty     else 0,
    len(domain_ioc_matches)   if 'domain_ioc_matches'   in dir() and not domain_ioc_matches.empty   else 0,
    len(url_ioc_matches)      if 'url_ioc_matches'      in dir() and not url_ioc_matches.empty      else 0,
    len(process_ioc_matches)  if 'process_ioc_matches'  in dir() and not process_ioc_matches.empty  else 0,
    len(email_ioc_matches)    if 'email_ioc_matches'    in dir() and not email_ioc_matches.empty    else 0,
])

# ─ MTTD ───────────────────────────────────────────────────────────────────────
if not kpi_mttd_df.empty and "Avg_MTTD_Hours" in kpi_mttd_df.columns:
    avg_mttd    = float(kpi_mttd_df["Avg_MTTD_Hours"].iloc[0])
    median_mttd = float(kpi_mttd_df["Median_MTTD_Hours"].iloc[0]) if "Median_MTTD_Hours" in kpi_mttd_df.columns else avg_mttd
else:
    avg_mttd = median_mttd = None

# ─ Alert totals ───────────────────────────────────────────────────────────────
if not kpi_daily.empty:
    total_alerts = int(kpi_daily["TotalAlerts"].sum())
    total_high   = int(kpi_daily["HighAlerts"].sum())
    total_med    = int(kpi_daily["MedAlerts"].sum())
    total_low    = int(kpi_daily["LowAlerts"].sum())
    total_info   = int(kpi_daily["InfoAlerts"].sum())
    total_closed = int(kpi_daily["ClosedTotal"].sum()) if "ClosedTotal" in kpi_daily.columns else 0
    tp_rate      = round(100 * total_closed / total_alerts, 1) if total_alerts > 0 else 0
else:
    total_alerts = total_high = total_med = total_low = total_info = total_closed = 0
    tp_rate = 0

# ─ KPI Scorecard ──────────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print("📊 SOC KPI SCORECARD — 12 MONTH PERIOD")
print(f"{'='*72}")
if avg_mttd is not None:
    mttd_str    = f"{avg_mttd:.1f} h (median {median_mttd:.1f} h)"
    mttd_status = "🟢 GOOD" if avg_mttd < 4 else ("🟡 ACCEPTABLE" if avg_mttd < 24 else "🔴 NEEDS IMPROVEMENT")
else:
    mttd_str    = "N/A (SecurityAlert.StartTime unavailable)"
    mttd_status = "⚪ UNKNOWN"
print(f"   ⏱️  Mean Time to Detect (MTTD):   {mttd_str}")
print(f"                                   → Target < 4 h  {mttd_status}")
print(f"\n   🚨 Total Alerts:                  {total_alerts:,}")
print(f"      🔴 High:                       {total_high:,}")
print(f"      🟠 Medium:                     {total_med:,}")
print(f"      🟡 Low:                        {total_low:,}")
print(f"      ℹ️  Informational:              {total_info:,}")
sig_rate   = round(100 * (total_high + total_med) / total_alerts, 1) if total_alerts > 0 else 0
sig_status = "🟢 GOOD" if sig_rate > 30 else "🟡 REVIEW — possible alert fatigue"
print(f"\n   📈 High+Medium Signal Rate:       {sig_rate}%  → Target >30%  {sig_status}")
print(f"\n   🦠 Threat Intel IOC Hits:         {_total_ioc:,}")
print(f"   🔐 Credential Attack Events:       {_bf + _ps:,}  (Brute Force: {_bf}, Spray: {_ps})")
print(f"   👤 Risky Users Identified:         {_ru:,}")
print(f"   🌍 Impossible Travel Events:       {_it:,}")
print(f"{'='*72}")

# ─ Dashboard: 2×2 chart grid ──────────────────────────────────────────────────
# Position (2,2) is a pie/domain chart — must declare type='domain' in specs
fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{"type": "xy"},     {"type": "xy"}],
        [{"type": "xy"},     {"type": "domain"}],
    ],
    subplot_titles=(
        "Daily Alert Volume by Severity",
        "Alerts by Provider / Source",
        "Key Findings Overview",
        "Alert Severity Distribution",
    ),
    vertical_spacing=0.18,
)

# Plot 1 — Daily alert volume stacked bar
if not kpi_daily.empty and "Day" in kpi_daily.columns:
    kpi_daily["Day"] = pd.to_datetime(kpi_daily["Day"], errors="coerce")
    for col, color, name in [
        ("HighAlerts", "#d63031", "High"),
        ("MedAlerts",  "#e17055", "Medium"),
        ("LowAlerts",  "#fdcb6e", "Low"),
        ("InfoAlerts", "#74b9ff", "Info"),
    ]:
        if col in kpi_daily.columns:
            fig.add_trace(go.Bar(
                x=kpi_daily["Day"], y=kpi_daily[col],
                name=name, marker_color=color,
                legendgroup="sev", showlegend=True,
            ), row=1, col=1)
    fig.update_layout(barmode="stack")

# Plot 2 — Alerts by provider (horizontal bar)
if not kpi_provider.empty and "ProviderName" in kpi_provider.columns:
    top15 = kpi_provider.head(15)
    fig.add_trace(go.Bar(
        x=top15["AlertCount"], y=top15["ProviderName"],
        orientation="h", marker_color="#6c5ce7", showlegend=False,
        text=top15["AlertCount"], textposition="outside",
    ), row=1, col=2)

# Plot 3 — Key findings bar
kpi_findings = {
    "Brute Force":      _bf,
    "Password Spray":   _ps,
    "Risky Users":      _ru,
    "Impossible Travel":_it,
    "IOC Hits":         _total_ioc,
    "Defender Alerts":  _da,
}
fig.add_trace(go.Bar(
    x=list(kpi_findings.keys()),
    y=list(kpi_findings.values()),
    marker_color=["#d63031","#e17055","#fdcb6e","#0984e3","#6c5ce7","#00b894"],
    showlegend=False,
    text=list(kpi_findings.values()),
    textposition="outside",
), row=2, col=1)

# Plot 4 — Severity pie (domain subplot)
if total_alerts > 0:
    fig.add_trace(go.Pie(
        labels=["High", "Medium", "Low", "Info"],
        values=[total_high, total_med, total_low, total_info],
        marker_colors=["#d63031", "#e17055", "#fdcb6e", "#74b9ff"],
        hole=0.4,
        showlegend=True,
    ), row=2, col=2)
else:
    # placeholder so subplot isn't empty
    fig.add_trace(go.Pie(
        labels=["No Data"], values=[1],
        marker_colors=["#b2bec3"], showlegend=False,
    ), row=2, col=2)

fig.update_layout(
    title_text="📊 SOC KPI Dashboard — 12 Month Analysis",
    height=750,
    plot_bgcolor="#f8f9fa",
    paper_bgcolor="white",
    font=dict(size=11),
)
fig.show()


📊 Loading SOC KPI data...
🔍 Daily alert KPIs...
   ✅ Returned 90 rows
🔍 Alerts by provider...
   ✅ Returned 12 rows
🔍 MTTD calculation...
   ✅ Returned 1 rows

📊 SOC KPI SCORECARD — 12 MONTH PERIOD
   ⏱️  Mean Time to Detect (MTTD):   95.1 h (median 1.0 h)
                                   → Target < 4 h  🔴 NEEDS IMPROVEMENT

   🚨 Total Alerts:                  44,522
      🔴 High:                       11,119
      🟠 Medium:                     12,830
      🟡 Low:                        17,537
      ℹ️  Informational:              3,036

   📈 High+Medium Signal Rate:       53.8%  → Target >30%  🟢 GOOD

   🦠 Threat Intel IOC Hits:         300
   🔐 Credential Attack Events:       200  (Brute Force: 100, Spray: 100)
   👤 Risky Users Identified:         518
   🌍 Impossible Travel Events:       50



---
## 🏢 Section 14 — C-Level / Board Executive Dashboard

Designed for non-technical stakeholders. Shows:
- **Overall security posture** as a risk gauge (RAG)
- **Business risk translation** — findings mapped to business impact
- **Month-over-month trend** – are we improving?
- **Regulatory / compliance exposure** (GDPR, NIST, ISO 27001 control areas)
- **Top 5 recommendations** with estimated effort and impact


In [45]:

# ── 14.  C-Level Executive Dashboard ─────────────────────────────────────────
from plotly.subplots import make_subplots

# ─ Collect all findings ───────────────────────────────────────────────────────
_bf        = len(brute_force_results)    if 'brute_force_results'    in dir() and not brute_force_results.empty    else 0
_ps        = len(password_spray_results) if 'password_spray_results' in dir() and not password_spray_results.empty else 0
_ru        = len(risky_users)            if 'risky_users'            in dir() and not risky_users.empty            else 0
_it        = len(impossible_travel)      if 'impossible_travel'      in dir() and not impossible_travel.empty      else 0
_la        = len(lateral_accounts)       if 'lateral_accounts'       in dir() and not lateral_accounts.empty       else 0
_sp        = len(spam_phishing_results)  if 'spam_phishing_results'  in dir() and not spam_phishing_results.empty  else 0
_bec       = len(bec_results)            if 'bec_results'            in dir() and not bec_results.empty            else 0
_aitm      = len(aitm_results)           if 'aitm_results'           in dir() and not aitm_results.empty           else 0
_nioc      = len(network_ioc_matches)    if 'network_ioc_matches'    in dir() and not network_ioc_matches.empty    else 0
_fioc      = len(file_ioc_matches)       if 'file_ioc_matches'       in dir() and not file_ioc_matches.empty       else 0
_dioc      = len(domain_ioc_matches)     if 'domain_ioc_matches'     in dir() and not domain_ioc_matches.empty     else 0

# ─ Risk Score (0–100) — weighted by business criticality ─────────────────────
risk_score = min(100, (
    _bf   * 2   +   # brute force — credential risk
    _ps   * 3   +   # spray — broader blast radius
    _ru   * 1   +   # risky users
    _it   * 5   +   # impossible travel — likely compromise
    _la   * 8   +   # lateral movement — active attacker in network
    _bec  * 10  +   # BEC — direct financial risk
    _aitm * 12  +   # AiTM — session bypass, hardest to detect
    (_nioc + _fioc + _dioc) * 4  # IOC hits — confirmed TI match
))

# RAG classification
if risk_score >= 60:
    rag = "CRITICAL"; rag_color = "#d63031"; rag_emoji = "🔴"
elif risk_score >= 30:
    rag = "HIGH";     rag_color = "#e17055"; rag_emoji = "🟠"
elif risk_score >= 10:
    rag = "MEDIUM";   rag_color = "#fdcb6e"; rag_emoji = "🟡"
else:
    rag = "LOW";      rag_color = "#00b894"; rag_emoji = "🟢"

# ─ Business Risk Areas ────────────────────────────────────────────────────────
business_risks = [
    ("Data Breach Risk",           min(100, (_nioc+_fioc+_dioc)*6 + _la*10),   "Unauthorized data access via IOC/lateral movement"),
    ("Account Takeover Risk",      min(100, (_bf+_ps)*2 + _ru*2 + _it*6),      "Compromised credentials or impossible travel"),
    ("Email & Fraud Risk",         min(100, (_bec*15 + _aitm*12 + _sp*2)),     "BEC, AiTM, phishing reaching users"),
    ("Regulatory / GDPR Exposure", min(100, (_ru*3 + _la*8 + _nioc*5)),        "Risk events triggering mandatory reporting"),
    ("Business Continuity Risk",   min(100, (_la*6 + _aitm*8 + _fioc*6)),      "Ransomware, lateral movement, persistence"),
]

# ─ Compliance Control Mapping (NIST CSF / ISO 27001 / CIS) ───────────────────
compliance_gaps = [
    ("Identity (AC-2, A.9.2)",   "MFA / conditional access",     min(100, (_bf+_ps)*3 + _ru*2)),
    ("Detection (DE-3, A.16)",   "Alert tuning / SIEM coverage",  min(100, _la*5 + _nioc*3)),
    ("Response (RS-2, A.16.1)",  "Incident response playbooks",    min(100, (_bec+_aitm)*8)),
    ("Data Protection (PR-DS)",  "DLP / email filtering",          min(100, _bec*8 + _sp*3)),
    ("Supply Chain (ID-SC)",     "Threat intel / IOC feeds",       min(100, (_nioc+_fioc+_dioc)*4)),
]

# ─ Print executive brief ──────────────────────────────────────────────────────
print("━"*72)
print("  🏢 EXECUTIVE SECURITY BRIEFING  ·  Microsoft Sentinel  ·  12-Month")
print("━"*72)
print(f"\n  {rag_emoji} OVERALL SECURITY POSTURE:  {rag}  (Score: {risk_score}/100)")
print(f"\n  BUSINESS RISK AREAS:")
for area, score, note in business_risks:
    bar = "█" * (score // 10) + "░" * (10 - score // 10)
    level = "CRITICAL" if score>=60 else "HIGH" if score>=30 else "MEDIUM" if score>=10 else "LOW"
    print(f"   {area:<35}  [{bar}] {score:3d}  {level}")
    print(f"   {'':35}  {note}")
print(f"\n  COMPLIANCE CONTROL GAPS (estimated):")
for ctrl, desc, score in compliance_gaps:
    bar = "█" * (score // 10) + "░" * (10 - score // 10)
    print(f"   {ctrl:<30}  [{bar}] {score:3d}  — {desc}")
print(f"\n  TOP 5 RECOMMENDED INVESTMENTS:")
recs = [
    ("🔐", "Enforce Phishing-Resistant MFA (FIDO2)",       "High",   "Low",    f"Eliminates credential spray ({_ps} campaigns) and AiTM ({_aitm} events)"),
    ("🛡️", "Deploy Safe Links + Safe Attachments (MDO)",   "High",   "Low",    f"Blocks phishing delivery ({_sp} source IPs) and BEC ({_bec} indicators)"),
    ("🔍", "Privileged Identity Management (PIM/JIT)",      "Medium", "Medium", f"Limits lateral movement blast radius ({_la} accounts at risk)"),
    ("🌐", "Implement Zero Trust Network Segmentation",     "Medium", "High",   f"Contains active IOC traffic ({_nioc+_fioc+_dioc} TI hits)"),
    ("📋", "Incident Response Tabletop Exercise",           "Low",    "Low",    "Validate AiTM + BEC playbooks with SOC team"),
]
for i, (icon, rec, impact, effort, rationale) in enumerate(recs, 1):
    print(f"   {i}. {icon} {rec}")
    print(f"      Impact: {impact}  |  Effort: {effort}  |  Rationale: {rationale}")
print("━"*72)

# ─ Dashboard: Gauge + Risk Bars + Compliance Radar ───────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "indicator"}, {"type": "bar"}],
           [{"type": "bar"},       {"type": "bar"}]],
    subplot_titles=(
        "Overall Security Risk Score",
        "Business Risk Areas (0–100)",
        "Compliance Control Gaps (0=clean)",
        "Finding Heatmap by Category"
    ),
    vertical_spacing=0.18, horizontal_spacing=0.12
)

# Gauge
fig.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=risk_score,
    delta={'reference': 40, 'increasing': {'color': '#d63031'}, 'decreasing': {'color': '#00b894'}},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': rag_color},
        'steps': [
            {'range': [0,  10], 'color': '#dfe6e9'},
            {'range': [10, 30], 'color': '#b2d3c2'},
            {'range': [30, 60], 'color': '#ffeaa7'},
            {'range': [60,100], 'color': '#fab1a0'},
        ],
        'threshold': {'line': {'color': 'black', 'width': 3}, 'thickness': 0.8, 'value': 60}
    },
    title={'text': f"Risk Score  ·  {rag}", 'font': {'size': 14}},
    number={'font': {'size': 40, 'color': rag_color}}
), row=1, col=1)

# Business risk bars
biz_labels  = [x[0] for x in business_risks]
biz_scores  = [x[1] for x in business_risks]
biz_colors  = ["#d63031" if s>=60 else "#e17055" if s>=30 else "#fdcb6e" if s>=10 else "#00b894" for s in biz_scores]
fig.add_trace(go.Bar(
    y=biz_labels, x=biz_scores, orientation='h',
    marker_color=biz_colors, showlegend=False,
    text=[f"{s}" for s in biz_scores], textposition='outside'
), row=1, col=2)

# Compliance gaps
comp_labels = [x[0] for x in compliance_gaps]
comp_scores = [x[2] for x in compliance_gaps]
comp_colors = ["#d63031" if s>=60 else "#e17055" if s>=30 else "#fdcb6e" if s>=10 else "#00b894" for s in comp_scores]
fig.add_trace(go.Bar(
    y=comp_labels, x=comp_scores, orientation='h',
    marker_color=comp_colors, showlegend=False,
    text=[f"{s}" for s in comp_scores], textposition='outside'
), row=2, col=1)

# Finding heatmap per category
categories = ["Credential Attacks", "Risky Identities", "Impossible Travel",
              "Lateral Movement", "BEC/AiTM", "IOC Hits", "Phishing/Malware"]
cat_vals   = [_bf+_ps, _ru, _it, _la, _bec+_aitm, _nioc+_fioc+_dioc, _sp]
cat_colors = ["#d63031" if v>10 else "#e17055" if v>0 else "#636e72" for v in cat_vals]
fig.add_trace(go.Bar(
    x=categories, y=cat_vals,
    marker_color=cat_colors, showlegend=False,
    text=cat_vals, textposition='outside'
), row=2, col=2)

fig.update_layout(
    title_text=f"🏢 C-Level Security Dashboard  ·  Risk: {rag}  ({risk_score}/100)",
    height=820,
    paper_bgcolor='white',
    font=dict(size=11)
)
# Cap x-axis at 100 for the risk bars
fig.update_xaxes(range=[0, 115], row=1, col=2)
fig.update_xaxes(range=[0, 115], row=2, col=1)
fig.show()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🏢 EXECUTIVE SECURITY BRIEFING  ·  Microsoft Sentinel  ·  12-Month
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  🔴 OVERALL SECURITY POSTURE:  CRITICAL  (Score: 100/100)

  BUSINESS RISK AREAS:
   Data Breach Risk                     [░░░░░░░░░░]   0  LOW
                                        Unauthorized data access via IOC/lateral movement
   Account Takeover Risk                [██████████] 100  CRITICAL
                                        Compromised credentials or impossible travel
   Email & Fraud Risk                   [██████████] 100  CRITICAL
                                        BEC, AiTM, phishing reaching users
   Regulatory / GDPR Exposure           [██████████] 100  CRITICAL
                                        Risk events triggering mandatory reporting
   Business Continuity Risk             [██████████] 100  CRITICAL
                                        


---
## 🧩 Section 15 — MITRE ATT&CK Framework Coverage

Maps every detection from this notebook to MITRE ATT&CK tactics and techniques.  
A heatmap shows **detection depth** — dark cells are coverage gaps needing additional rules or data sources.

| Tactic | Key Techniques Covered |
|--------|------------------------|
| **TA0001 Initial Access** | T1566 Phishing, T1078 Valid Accounts |
| **TA0006 Credential Access** | T1110 Brute Force, T1110.003 Password Spray, T1539 Session Cookie, T1557 AiTM |
| **TA0007 Discovery** | T1087 Account Discovery, T1083 File Discovery |
| **TA0008 Lateral Movement** | T1021 Remote Services, T1550 Alternate Auth |
| **TA0009 Collection** | T1114 Email Collection, T1213 Data Repositories |
| **TA0010 Exfiltration** | T1048 Alt Protocol, T1567 Cloud Storage |
| **TA0011 C2** | T1071 App Layer, T1102 Web Service |
| **TA0040 Impact** | T1486 Data Encrypted, T1565 Data Manipulation |


In [46]:

import pandas as pd
import plotly.graph_objects as go
from datetime import timedelta

print("MITRE ATT&CK — querying SecurityIncident & SecurityAlert for technique coverage...")

# ── Helper to safely count rows from a prior-cell variable ───────────────────
def _count(var_name):
    try:
        val = globals()[var_name]
        if val is None:
            return 0
        if hasattr(val, 'empty') and val.empty:
            return 0
        return len(val)
    except Exception:
        return 0

# ── 1. Query SecurityIncident for MITRE techniques ───────────────────────────
kql_incident = """
SecurityIncident
| where TimeGenerated > ago(365d)
| where isnotempty(AdditionalData)
| mv-expand Tactic = todynamic(AdditionalData).tactics
| mv-expand Tech   = todynamic(AdditionalData).techniques
| where isnotempty(Tech)
| summarize IncidentCount = dcount(IncidentNumber) by
    Tactic   = tostring(Tactic),
    Technique = tostring(Tech)
| order by IncidentCount desc
"""

kql_alert = """
SecurityAlert
| where TimeGenerated > ago(365d)
| where isnotempty(Tactics)
| mv-expand Tactic = todynamic(Tactics)
| extend TechArray = todynamic(ExtendedProperties)["Mitre techniques"]
| mv-expand Tech = TechArray
| where isnotempty(Tech)
| summarize AlertCount = dcount(SystemAlertId) by
    Tactic   = tostring(Tactic),
    Technique = tostring(Tech)
| order by AlertCount desc
"""

incident_df = pd.DataFrame()
alert_df    = pd.DataFrame()

if "run_kql" in dir() or "run_kql" in globals():
    incident_df = run_kql(kql_incident, "SecurityIncident — MITRE technique counts")
    alert_df    = run_kql(kql_alert,    "SecurityAlert   — MITRE technique counts")
else:
    print("   ⚠️  run_kql not available — run the Setup / Auth cells first.")

# ── 2. Build a lookup: technique_id → event count (from live data) ────────────
live_counts: dict[str, int] = {}

for df, col in [(incident_df, "IncidentCount"), (alert_df, "AlertCount")]:
    if df.empty or "Technique" not in df.columns:
        continue
    for _, row in df.iterrows():
        tech = str(row["Technique"]).strip().upper()
        cnt  = int(row.get(col, 0))
        live_counts[tech] = live_counts.get(tech, 0) + cnt

print(f"\n   Live techniques found in SecurityIncident/Alert: {len(live_counts)}")
if live_counts:
    for t, c in sorted(live_counts.items(), key=lambda x: -x[1])[:10]:
        print(f"      {t:15}  {c:,} events")

# ── 3. Fallback variable-based counts (from earlier notebook cells) ───────────
_bf   = _count('brute_force_results')
_ps   = _count('password_spray_results')
_ru   = _count('risky_users')
_it   = _count('impossible_travel')
_nioc = _count('network_ioc_matches')
_fioc = _count('file_ioc_matches')
_la   = _count('lateral_accounts')
_ld   = _count('lateral_devices')
_sp   = _count('spam_phishing_results')
_em   = _count('email_malware_results')
_cm   = _count('collab_malware_results')
_bec  = _count('bec_results')
_aitm = _count('aitm_results')

# ── 4. Technique definitions — live count wins, fallback to variable count ────
def _c(tech_id: str, fallback: int = 0) -> int:
    """Return live count for a technique if available, else the fallback."""
    return live_counts.get(tech_id.upper(), fallback) or fallback

mitre_findings = [
    ("TA0001 Initial Access",    "T1566",     "Phishing",                       "MDO",       _c("T1566",     _sp + _em)),
    ("TA0001 Initial Access",    "T1566.001", "Spearphishing Attachment",        "MDO",       _c("T1566.001", _em)),
    ("TA0001 Initial Access",    "T1566.002", "Spearphishing Link",              "MDO",       _c("T1566.002", _sp)),
    ("TA0001 Initial Access",    "T1078",     "Valid Accounts",                  "AAD",       _c("T1078",     _bf + _it)),
    ("TA0001 Initial Access",    "T1078.004", "Cloud Accounts",                  "AAD",       _c("T1078.004", _ru)),
    ("TA0006 Credential Access", "T1110",     "Brute Force",                     "AAD",       _c("T1110",     _bf)),
    ("TA0006 Credential Access", "T1110.003", "Password Spraying",               "AAD",       _c("T1110.003", _ps)),
    ("TA0006 Credential Access", "T1539",     "Steal Web Session Cookie",        "MDO",       _c("T1539",     _aitm)),
    ("TA0006 Credential Access", "T1557",     "Adversary-in-the-Middle",         "MDO",       _c("T1557",     _aitm)),
    ("TA0007 Discovery",         "T1087",     "Account Discovery",               "MDE",       _c("T1087",     _la)),
    ("TA0007 Discovery",         "T1083",     "File and Directory Discovery",    "MDE",       _c("T1083",     _fioc)),
    ("TA0007 Discovery",         "T1046",     "Network Service Discovery",       "MDE",       _c("T1046",     _ld)),
    ("TA0008 Lateral Movement",  "T1021",     "Remote Services",                 "MDE",       _c("T1021",     _la + _ld)),
    ("TA0008 Lateral Movement",  "T1550",     "Use Alternate Auth Material",     "Identity",  _c("T1550",     _la)),
    ("TA0009 Collection",        "T1114",     "Email Collection",                "MDO",       _c("T1114",     _bec)),
    ("TA0009 Collection",        "T1213",     "Data from Repositories",          "MDO",       _c("T1213",     _cm)),
    ("TA0010 Exfiltration",      "T1048",     "Exfil Over Alternative Protocol", "MDE",       _c("T1048",     _nioc)),
    ("TA0010 Exfiltration",      "T1567",     "Exfil to Cloud Storage",          "MDO",       _c("T1567",     _bec)),
    ("TA0011 C2",                "T1071",     "Application Layer Protocol",      "MDE",       _c("T1071",     _nioc)),
    ("TA0011 C2",                "T1102",     "Web Service",                     "MDO",       _c("T1102",     _aitm)),
    ("TA0040 Impact",            "T1486",     "Data Encrypted for Impact",       "MDE",       _c("T1486",     _fioc)),
    ("TA0040 Impact",            "T1565",     "Data Manipulation",               "MDO",       _c("T1565",     _bec)),
]

# Add any EXTRA techniques found in live data that aren't already mapped above
mapped_ids = {t[1].upper() for t in mitre_findings}

tactic_map = {
    "InitialAccess":        "TA0001 Initial Access",
    "CredentialAccess":     "TA0006 Credential Access",
    "Discovery":            "TA0007 Discovery",
    "LateralMovement":      "TA0008 Lateral Movement",
    "Collection":           "TA0009 Collection",
    "Exfiltration":         "TA0010 Exfiltration",
    "CommandAndControl":    "TA0011 C2",
    "Impact":               "TA0040 Impact",
    "Persistence":          "TA0003 Persistence",
    "PrivilegeEscalation":  "TA0004 Privilege Escalation",
    "DefenseEvasion":       "TA0005 Defense Evasion",
    "Execution":            "TA0002 Execution",
    "Reconnaissance":       "TA0043 Reconnaissance",
    "ResourceDevelopment":  "TA0042 Resource Development",
    "PreAttack":            "TA0043 Reconnaissance",
}

for df, col in [(incident_df, "IncidentCount"), (alert_df, "AlertCount")]:
    if df.empty or "Technique" not in df.columns:
        continue
    for _, row in df.iterrows():
        tid = str(row["Technique"]).strip().upper()
        if tid in mapped_ids:
            continue
        tactic_raw = str(row.get("Tactic", "")).strip()
        tactic_label = tactic_map.get(tactic_raw, tactic_raw or "Unknown")
        cnt = int(row.get(col, 0))
        mitre_findings.append((tactic_label, tid, tid, "Sentinel", cnt))
        mapped_ids.add(tid)

print(f"\n   Total mapped techniques (including live extras): {len(mitre_findings)}")

# ── 5. Build DataFrame & signal labels ───────────────────────────────────────
mitre_df = pd.DataFrame(
    mitre_findings,
    columns=["Tactic", "TechniqueID", "TechniqueName", "Sources", "Count"]
)
mitre_df["Signal"] = mitre_df["Count"].apply(
    lambda x: "High" if x > 10 else ("Low" if x > 0 else "Not Detected")
)

# ── 6. Heatmap ────────────────────────────────────────────────────────────────
tactics_order = [
    "TA0001 Initial Access",
    "TA0006 Credential Access",
    "TA0007 Discovery",
    "TA0008 Lateral Movement",
    "TA0009 Collection",
    "TA0010 Exfiltration",
    "TA0011 C2",
    "TA0040 Impact",
    "TA0003 Persistence",
    "TA0004 Privilege Escalation",
    "TA0005 Defense Evasion",
    "TA0002 Execution",
    "TA0043 Reconnaissance",
    "TA0042 Resource Development",
]
present_tactics = [t for t in tactics_order if t in mitre_df["Tactic"].values]
# Add any unmapped tactics
for t in mitre_df["Tactic"].unique():
    if t not in present_tactics:
        present_tactics.append(t)

pivot = mitre_df.pivot_table(
    index="TechniqueID", columns="Tactic", values="Count", aggfunc="sum"
).fillna(0)
present_tactics = [t for t in present_tactics if t in pivot.columns]
pivot = pivot[present_tactics]
short_labels = [t.split(" ", 1)[1] if " " in t else t for t in pivot.columns]

data_source_note = ""
if not incident_df.empty or not alert_df.empty:
    sources = []
    if not incident_df.empty: sources.append(f"SecurityIncident ({len(incident_df)} rows)")
    if not alert_df.empty:    sources.append(f"SecurityAlert ({len(alert_df)} rows)")
    data_source_note = "  |  Source: " + ", ".join(sources)

fig = go.Figure(data=go.Heatmap(
    z=pivot.values.tolist(),
    x=short_labels,
    y=list(pivot.index),
    colorscale=[[0, "#2d3436"], [0.01, "#636e72"], [0.3, "#fdcb6e"], [1.0, "#d63031"]],
    showscale=True,
    colorbar=dict(title="Event Count"),
    hovertemplate="<b>%{y}</b><br>%{x}<br>Count: %{z}<extra></extra>",
))
fig.update_layout(
    title=f"MITRE ATT&CK Detection Coverage — Last 12 Months{data_source_note}",
    height=650,
    xaxis=dict(tickangle=-30, side="top"),
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="#1e272e",
    paper_bgcolor="#2d3436",
    font=dict(color="white", size=11),
)
fig.show()

# ── 7. Coverage summary ───────────────────────────────────────────────────────
sep = "=" * 72
print(f"\n{sep}")
print("MITRE ATT&CK COVERAGE SUMMARY")
print(sep)
detected    = int((mitre_df["Count"] > 0).sum())
high_signal = int((mitre_df["Count"] > 10).sum())
gaps        = int((mitre_df["Count"] == 0).sum())
src_note = "(SecurityIncident + SecurityAlert)" if (not incident_df.empty or not alert_df.empty) else "(fallback: notebook variables only)"
print(f"   Data source:                 {src_note}")
print(f"   Techniques mapped:           {len(mitre_df)}")
print(f"   Techniques with detections:  {detected} / {len(mitre_df)}")
print(f"   High-signal (>10 events):    {high_signal}")
print(f"   Coverage gaps (0 events):    {gaps}")

if detected > 0:
    print("\n   ACTIVE TECHNIQUES:")
    active = mitre_df[mitre_df["Count"] > 0].sort_values("Count", ascending=False)
    for _, r in active.iterrows():
        print(f"   {r['Signal']:<18} {r['TechniqueID']:12}  {r['TechniqueName']:<45}  ({r['Count']:,})")
else:
    print("\n   ⚠️  No events detected in SecurityIncident or SecurityAlert for these techniques.")
    print("      Check that the workspace has data and the auth cell has run.")

print("\n   COVERAGE GAPS:")
for _, r in mitre_df[mitre_df["Count"] == 0].iterrows():
    print(f"   No Data          {r['TechniqueID']:12}  {r['TechniqueName']}")


MITRE ATT&CK — querying SecurityIncident & SecurityAlert for technique coverage...
🔍 SecurityIncident — MITRE technique counts...
   ✅ Returned 612 rows
🔍 SecurityAlert   — MITRE technique counts...
   ✅ Returned 0 rows

   Live techniques found in SecurityIncident/Alert: 75
      T1566            481 events
      T0890            445 events
      T1071            326 events
      T1204            203 events
      T1078            131 events
      T1534            106 events
      T1110            49 events
      T1539            45 events
      T1569            44 events
      T1021            43 events

   Total mapped techniques (including live extras): 85



MITRE ATT&CK COVERAGE SUMMARY
   Data source:                 (SecurityIncident + SecurityAlert)
   Techniques mapped:           85
   Techniques with detections:  82 / 85
   High-signal (>10 events):    24
   Coverage gaps (0 events):    3

   ACTIVE TECHNIQUES:
   High               T1078.004     Cloud Accounts                                 (518)
   High               T1566         Phishing                                       (481)
   High               T0890         T0890                                          (434)
   High               T1071         Application Layer Protocol                     (326)
   High               T1204         T1204                                          (158)
   High               T1078         Valid Accounts                                 (131)
   High               T1102         Web Service                                    (100)
   High               T1110.003     Password Spraying                              (100)
   High               T

---
## 11. Visualization Dashboard

Comprehensive visualizations summarizing the security analysis findings.

In [47]:

# Create Summary Dashboard
from plotly.subplots import make_subplots

# Collect summary statistics
summary_stats = {
    'Brute Force Incidents':    len(brute_force_results)    if 'brute_force_results'    in dir() and not brute_force_results.empty    else 0,
    'Password Spray Campaigns': len(password_spray_results) if 'password_spray_results' in dir() and not password_spray_results.empty else 0,
    'Risky Users':              len(risky_users)            if 'risky_users'            in dir() and not risky_users.empty            else 0,
    'Anomaly Hours':            len(anomalies)              if 'anomalies'              in dir()                                       else 0,
    'Impossible Travel':        len(impossible_travel)      if 'impossible_travel'      in dir() and not impossible_travel.empty      else 0,
    # IOC — broken out by entity type
    'IOC: IP Address':          len(network_ioc_matches)    if 'network_ioc_matches'    in dir() and not network_ioc_matches.empty    else 0,
    'IOC: File Hash':           len(file_ioc_matches)       if 'file_ioc_matches'       in dir() and not file_ioc_matches.empty       else 0,
    'IOC: Domain':              len(domain_ioc_matches)     if 'domain_ioc_matches'     in dir() and not domain_ioc_matches.empty     else 0,
    'IOC: URL':                 len(url_ioc_matches)        if 'url_ioc_matches'        in dir() and not url_ioc_matches.empty        else 0,
    'IOC: Process/LOLBin':      len(process_ioc_matches)    if 'process_ioc_matches'    in dir() and not process_ioc_matches.empty    else 0,
    'IOC: Email Sender':        len(email_ioc_matches)      if 'email_ioc_matches'      in dir() and not email_ioc_matches.empty      else 0,
    # Lateral + Alerts
    'Lateral Movement Accounts':len(lateral_accounts)       if 'lateral_accounts'       in dir() and not lateral_accounts.empty       else 0,
    'Defender Alerts':          len(defender_alerts)        if 'defender_alerts'        in dir() and not defender_alerts.empty        else 0,
    # MDO
    'Spam/Phishing IPs':        len(spam_phishing_results)  if 'spam_phishing_results'  in dir() and not spam_phishing_results.empty  else 0,
    'Email Malware':            len(email_malware_results)  if 'email_malware_results'  in dir() and not email_malware_results.empty  else 0,
    'Collab Malware Events':    len(collab_malware_results) if 'collab_malware_results' in dir() and not collab_malware_results.empty else 0,
    'BEC Indicators':           len(bec_results)            if 'bec_results'            in dir() and not bec_results.empty            else 0,
    'AiTM Patterns':            len(aitm_results)           if 'aitm_results'           in dir() and not aitm_results.empty           else 0,
}

# Create summary bar chart
fig = go.Figure()

colors = [
    '#ff6b6b', '#feca57', '#48dbfb', '#ff9ff3', '#54a0ff',
    '#d63031', '#c0392b', '#e17055', '#b71c1c', '#c62828', '#922b21',   # IOC reds
    '#5f27cd', '#00d2d3', '#1dd1a1', '#fd9644', '#a29bfe',
    '#00b894', '#e17055', '#74b9ff'
]

fig.add_trace(go.Bar(
    x=list(summary_stats.keys()),
    y=list(summary_stats.values()),
    marker_color=colors[:len(summary_stats)],
    text=list(summary_stats.values()),
    textposition='auto'
))

fig.update_layout(
    title='🛡️ Security Analysis Summary - 12 Month Overview',
    xaxis_title='Detection Category',
    yaxis_title='Count',
    xaxis_tickangle=-45,
    height=500
)

fig.show()

# Print summary
print("\n" + "="*60)
print("📊 ANALYSIS SUMMARY")
print("="*60)
for category, count in summary_stats.items():
    status = "🔴" if count > 10 else "🟠" if count > 0 else "🟢"
    print(f"{status} {category}: {count}")



📊 ANALYSIS SUMMARY
🔴 Brute Force Incidents: 100
🔴 Password Spray Campaigns: 100
🔴 Risky Users: 518
🔴 Anomaly Hours: 12
🔴 Impossible Travel: 50
🟢 IOC: IP Address: 0
🟢 IOC: File Hash: 0
🟢 IOC: Domain: 0
🟢 IOC: URL: 0
🔴 IOC: Process/LOLBin: 300
🟢 IOC: Email Sender: 0
🟢 Lateral Movement Accounts: 0
🔴 Defender Alerts: 4677
🔴 Spam/Phishing IPs: 100
🔴 Email Malware: 100
🔴 Collab Malware Events: 100
🔴 BEC Indicators: 34
🔴 AiTM Patterns: 100


---
## 12. Executive Summary

### 📋 Key Findings and Recommendations

In [48]:

# ── Data Volume Scanned During Investigation ──────────────────────────────────
# Queries the Usage table to get GB ingested/available per table
# over the analysis period — used in the executive summary below.
data_scan_query = """
Usage
| where TimeGenerated > ago(365d)
| summarize TotalGB = round(sum(Quantity) / 1024.0, 2) by DataType
| where TotalGB > 0
| order by TotalGB desc
"""

data_scan_df = run_kql(data_scan_query, "Calculating data volume scanned per table")

# Totals used by the executive summary
investigation_tables = [
    "SigninLogs", "AADNonInteractiveUserSignInLogs", "AADServicePrincipalSignInLogs",
    "AuditLogs", "AADRiskyUsers", "AADUserRiskEvents", "RiskySignIns",
    "CloudAppEvents", "DeviceLogonEvents", "DeviceNetworkEvents",
    "DeviceProcessEvents", "DeviceFileEvents", "IdentityLogonEvents",
    "IdentityQueryEvents", "SecurityAlert", "ThreatIntelligenceIndicator",
]

if not data_scan_df.empty:
    total_gb_all    = round(float(data_scan_df["TotalGB"].sum()), 2)
    inv_mask        = data_scan_df["DataType"].isin(investigation_tables)
    total_gb_inv    = round(float(data_scan_df.loc[inv_mask, "TotalGB"].sum()), 2)
    tables_scanned  = int(inv_mask.sum())

    print(f"\n📦 DATA SCAN SUMMARY")
    print(f"   Total GB ingested (all tables):        {total_gb_all:,.2f} GB")
    print(f"   Total GB across investigation tables:  {total_gb_inv:,.2f} GB")
    print(f"   Investigation tables with data:        {tables_scanned}")
    print()
    display(data_scan_df.head(20))

    # Bar chart – top 15 tables by volume
    top15 = data_scan_df.head(15).copy()
    fig = px.bar(
        top15, x="DataType", y="TotalGB",
        title="Data Volume by Table (GB) — Analysis Period",
        labels={"TotalGB": "GB Ingested", "DataType": "Table"},
        color="TotalGB", color_continuous_scale="Blues",
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    total_gb_all   = 0.0
    total_gb_inv   = 0.0
    tables_scanned = 0
    print("⚠️  Usage table returned no data — data scan stats unavailable.")


🔍 Calculating data volume scanned per table...
   ✅ Returned 79 rows

📦 DATA SCAN SUMMARY
   Total GB ingested (all tables):        888.05 GB
   Total GB across investigation tables:  486.37 GB
   Investigation tables with data:        12



,DataType,TotalGB
0,AADNonInteractiveUserSignInLogs,422.90
1,Testvakirosh_KQL_CL,62.96
2,MicrosoftGraphActivityLogs,61.08
3,ASimDnsActivityLogs,44.79
4,DeviceCustomImageLoadEvents,39.89
5,DeviceCustomScriptEvents,35.30
6,SecurityEvent,34.80
7,AzureDiagnostics,29.72
8,DeviceCustomFileEvents,22.79
9,CloudAppEvents,20.49


In [49]:

# ══════════════════════════════════════════════════════════════════════════════
#  EXECUTIVE SUMMARY  —  Microsoft Sentinel Security Analysis
#  Pulls counts from every variable set throughout the notebook, with
#  fallbacks to SecurityIncident / SecurityAlert data (incident_df / alert_df
#  from Section 15) when earlier analysis cells were not executed this session.
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime
import pandas as pd

# ── Helpers ───────────────────────────────────────────────────────────────────
def _df_count(var_name: str) -> int:
    """Return len() of a DataFrame variable if it exists and is non-empty."""
    try:
        val = globals()[var_name]
        if val is None or (hasattr(val, "empty") and val.empty):
            return 0
        return len(val)
    except (KeyError, NameError):
        return 0

def _int_var(var_name: str, default: int = 0) -> int:
    try:
        return int(globals()[var_name])
    except (KeyError, NameError, TypeError, ValueError):
        return default

# ── Pull SecurityIncident / SecurityAlert counts (MITRE section fallback) ─────
# incident_df and alert_df are produced by Section 15 (MITRE ATT&CK cell)
_inc  = globals().get("incident_df", pd.DataFrame())
_alrt = globals().get("alert_df",    pd.DataFrame())

def _live_total_by_tactic(tactic_substring: str) -> int:
    """Sum IncidentCount/AlertCount rows whose Tactic contains the given string."""
    total = 0
    for df, col in [(_inc, "IncidentCount"), (_alrt, "AlertCount")]:
        if df.empty or "Tactic" not in df.columns or col not in df.columns:
            continue
        mask = df["Tactic"].astype(str).str.contains(tactic_substring, case=False, na=False)
        total += int(df.loc[mask, col].sum())
    return total

def _live_total_by_technique(tech_ids: list) -> int:
    """Sum counts for specific technique IDs from live_counts dict."""
    lc = globals().get("live_counts", {})
    return sum(lc.get(t.upper(), 0) for t in tech_ids)

# ── Data scan stats ───────────────────────────────────────────────────────────
_total_gb_all   = _int_var("total_gb_all",   0)
_total_gb_inv   = _int_var("total_gb_inv",   0)
_tables_scanned = _int_var("tables_scanned", 0)

# ── CREDENTIAL ATTACKS ────────────────────────────────────────────────────────
# Priority: direct analysis variable → SecurityIncident/Alert tactic → live_counts
_bf_direct = _df_count("brute_force_results")
_ps_direct = _df_count("password_spray_results")

# SecurityIncident / SecurityAlert CredentialAccess
_cred_live = _live_total_by_tactic("CredentialAccess")

# Technique-specific from live_counts
_bf_live    = _live_total_by_technique(["T1110", "T1110.001", "T1110.002", "T1110.004"])
_ps_live    = _live_total_by_technique(["T1110.003"])

# Azure Resource brute force (Section 17)
_azure_bf   = _df_count("azure_brute_results")

bf_count = _bf_direct or _bf_live or _azure_bf
ps_count = _ps_direct or _ps_live

# ── RISKY USERS ───────────────────────────────────────────────────────────────
_ru_direct    = _df_count("risky_users")
# sp_highrisk_results is populated by the Service Principal section
_ru_sp        = _df_count("sp_highrisk_results")
# AADRiskyUsers events from SecurityIncident/Alert
_ru_live      = _live_total_by_tactic("InitialAccess") // 2  # rough proxy when direct not available
ru_count      = _ru_direct or _ru_sp
# Annotate data source
_ru_src = ("AAD risky users" if _ru_direct else
           "Service principal high-risk" if _ru_sp else
           "Not computed this session")

# ── ANOMALY DETECTION ─────────────────────────────────────────────────────────
_anomaly_direct  = _df_count("anomalies") if "anomalies" in globals() and hasattr(globals().get("anomalies"), "__len__") else 0
_it_direct       = _df_count("impossible_travel")
_it_count_var    = _int_var("it_count", 0)       # set by anomaly/travel cell
_anomaly_count_v = _int_var("anomaly_count", 0)   # set by anomaly/spike cell
# Live fallback
_it_live         = _live_total_by_technique(["T1534", "T1078"])

anomaly_count = _anomaly_count_v or _anomaly_direct
it_count      = _it_count_var or _it_direct or _it_live

# ── IOC MATCHES ───────────────────────────────────────────────────────────────
net_ioc      = _df_count("network_ioc_matches")
file_ioc     = _df_count("file_ioc_matches")
domain_ioc   = _df_count("domain_ioc_matches")
url_ioc      = _df_count("url_ioc_matches")
process_ioc  = _df_count("process_ioc_matches")
emailsnd_ioc = _df_count("email_ioc_matches")
total_ioc    = net_ioc + file_ioc + domain_ioc + url_ioc + process_ioc + emailsnd_ioc

# ── LATERAL MOVEMENT ──────────────────────────────────────────────────────────
lm_accounts = _df_count("lateral_accounts")
lm_devices  = _df_count("lateral_devices")
_lm_live    = _live_total_by_tactic("LateralMovement")
if lm_accounts == 0 and lm_devices == 0 and _lm_live > 0:
    lm_accounts = _lm_live  # use live count as proxy

# ── DEFENDER FOR OFFICE 365 ───────────────────────────────────────────────────
sp_count   = _df_count("spam_phishing_results")
em_count   = _df_count("email_malware_results")
cm_count   = _df_count("collab_malware_results")
bec_count  = _df_count("bec_results")
aitm_count = _int_var("aitm_count", _df_count("aitm_results"))

# ── IDENTIY PROTECTION + ON-PREM HUNTING ─────────────────────────────────────
_onprem = globals().get("identity_onprem_results", {})
_dcsync = _df_count("dcsync_results")
_kerberoast = _df_count("kerberoast_results")
_ntlm_pth   = _df_count("ntlm_pth_results")
onprem_total = _dcsync + _kerberoast + _ntlm_pth

# ── AZURE CLOUD THREAT HUNTING (Section 17) ───────────────────────────────────
crypto_count   = _df_count("cryptojacking_results")
privesc_count  = _df_count("azure_privesc_results")
mass_del_count = _df_count("mass_delete_results")
kv_count       = _df_count("keyvault_results")
storage_count  = _df_count("storage_exfil_results")
rg_del_count   = _df_count("rg_deletion_results")
auto_count     = _df_count("automation_results")
azure_travel   = _df_count("azure_travel_results")
azure_cloud_total = (crypto_count + privesc_count + mass_del_count +
                     kv_count + storage_count + rg_del_count + auto_count + azure_travel)

# ── MITRE ATT&CK COVERAGE ─────────────────────────────────────────────────────
_mitre_df       = globals().get("mitre_df", pd.DataFrame())
_mitre_detected = int((_mitre_df["Count"] > 0).sum()) if not _mitre_df.empty and "Count" in _mitre_df.columns else 0
_mitre_total    = len(_mitre_df) if not _mitre_df.empty else 22
_mitre_gaps     = _mitre_total - _mitre_detected

# ── Sentinel incident/alert totals (Section 15 live data) ────────────────────
_inc_total  = int(_inc["IncidentCount"].sum())  if not _inc.empty  and "IncidentCount" in _inc.columns  else 0
_alrt_total = int(_alrt["AlertCount"].sum())    if not _alrt.empty and "AlertCount"    in _alrt.columns else 0

# ═══════════════════════════════════════════════════════════════════════════════
#  PRINT REPORT
# ═══════════════════════════════════════════════════════════════════════════════
SEP  = "═" * 79
SEP2 = "─" * 79
print(SEP)
print("                  🛡️  MICROSOFT SENTINEL SECURITY ANALYSIS")
print("                          EXECUTIVE SUMMARY REPORT")
print(SEP)
print(f"\n📅 Analysis Period : {START_DATE.strftime('%Y-%m-%d')}  →  {END_DATE.strftime('%Y-%m-%d')}  (12 months)")
print(f"📊 Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
if _inc_total > 0 or _alrt_total > 0:
    print(f"🔗 SecurityIncident events analysed : {_inc_total:,}")
    print(f"🔗 SecurityAlert   events analysed   : {_alrt_total:,}")
print(f"💾 Total workspace data ingested     : {_total_gb_all:,.2f} GB  ({_tables_scanned} tables)")

# ── 1. CREDENTIAL ATTACKS ─────────────────────────────────────────────────────
print(f"\n{SEP2}")
print("🔐  CREDENTIAL ATTACK DETECTION")
print(SEP2)
_bf_src = ("SigninLogs analysis" if _bf_direct
           else "SecurityIncident/Alert — T1110" if _bf_live
           else "Azure Resource brute-force" if _azure_bf else "cells not executed this session")
_ps_src = ("SigninLogs analysis" if _ps_direct
           else "SecurityIncident/Alert — T1110.003" if _ps_live else "cells not executed this session")
print(f"   • Brute Force events   : {bf_count:>6,}   [{_bf_src}]")
print(f"   • Password Spray events: {ps_count:>6,}   [{_ps_src}]")
if bf_count > 0 or ps_count > 0:
    print(f"   ⚠️  RISK : Active credential attacks detected")
    print(f"   📝 ACTION: Enable MFA, implement sign-in risk Conditional Access policies")
else:
    print(f"   ✅  No credential attack signals detected")

# ── 2. RISKY USERS / IDENTITY PROTECTION ─────────────────────────────────────
print(f"\n{SEP2}")
print("👤  RISKY USERS & IDENTITY PROTECTION")
print(SEP2)
print(f"   • Users at Risk        : {ru_count:>6,}   [{_ru_src}]")
_aitm_src = "AiTM / MFA-bypass analysis (Sections 11.5)"
print(f"   • AiTM Attack Patterns : {aitm_count:>6,}   [{_aitm_src}]")
_oauth = _df_count("oauth_consent_results")
if _oauth:
    print(f"   • Suspicious OAuth Consents: {_oauth:>4,}")
if ru_count > 0 or aitm_count > 0:
    print(f"   ⚠️  RISK : Compromised or at-risk identities present")
    print(f"   📝 ACTION: Remediate risky users; revoke suspicious OAuth consents; review MFA gaps")

# ── 3. ANOMALY DETECTION ──────────────────────────────────────────────────────
print(f"\n{SEP2}")
print("📈  ANOMALY & TRAVEL DETECTION")
print(SEP2)
_at_src  = ("Anomaly detection cell" if _anomaly_count_v or _anomaly_direct else "data source not available")
_it_src  = ("Impossible-travel analysis" if _it_direct or _it_count_var
            else "SecurityIncident/Alert proxy" if _it_live else "cells not executed this session")
print(f"   • Anomalous time-period spikes : {anomaly_count:>5,}   [{_at_src}]")
print(f"   • Impossible Travel events     : {it_count:>5,}   [{_it_src}]")
_az_tr_src = "Azure Management (Section 17.6)"
print(f"   • Azure Mgmt cross-country     : {azure_travel:>5,}   [{_az_tr_src}]")
if anomaly_count > 5 or it_count > 0 or azure_travel > 0:
    print(f"   ⚠️  RISK : Unusual sign-in patterns — possible account compromise or travel anomaly")
    print(f"   📝 ACTION: Investigate flagged accounts; enable named-location CA policies")
else:
    print(f"   ✅  No significant anomalies detected")

# ── 4. THREAT INTELLIGENCE MATCHES ───────────────────────────────────────────
print(f"\n{SEP2}")
print("🧠  THREAT INTELLIGENCE IOC MATCHES")
print(SEP2)
print(f"   • IP Address IOC matches   : {net_ioc:>6,}")
print(f"   • File Hash IOC matches    : {file_ioc:>6,}")
print(f"   • Domain IOC matches       : {domain_ioc:>6,}")
print(f"   • URL IOC matches          : {url_ioc:>6,}")
print(f"   • Process / LOLBin matches : {process_ioc:>6,}")
print(f"   • Email Sender IOC matches : {emailsnd_ioc:>6,}")
print(f"   {'─'*38}")
print(f"   • TOTAL IOC hits           : {total_ioc:>6,}")
if total_ioc > 0:
    print(f"   🔴 CRITICAL: Known malicious indicators active in environment!")
    print(f"   📝 ACTION  : Immediate IR — isolate affected assets; block IOCs at perimeter")
else:
    print(f"   ✅  No TI IOC matches detected")

# ── 5. LATERAL MOVEMENT ───────────────────────────────────────────────────────
print(f"\n{SEP2}")
print("🔄  LATERAL MOVEMENT DETECTION")
print(SEP2)
print(f"   • Accounts with unusual device access   : {lm_accounts:>5,}")
print(f"   • Devices with unusual network activity : {lm_devices:>5,}")
print(f"   • DCSync / Credential dump events       : {_dcsync:>5,}")
print(f"   • Kerberoasting detections              : {_kerberoast:>5,}")
print(f"   • NTLM / Pass-the-Hash patterns         : {_ntlm_pth:>5,}")
_onprem_src = "[Section 16 — On-Prem Identity Hunting]"
if lm_accounts + lm_devices + onprem_total > 0:
    print(f"   ⚠️  RISK : Lateral movement / credential theft patterns {_onprem_src}")
    print(f"   📝 ACTION: Segment network; enforce LAPS; investigate flagged accounts")
else:
    print(f"   ✅  No lateral movement signals detected")

# ── 6. DEFENDER FOR OFFICE 365 ───────────────────────────────────────────────
print(f"\n{SEP2}")
print("🛡️  MICROSOFT DEFENDER FOR OFFICE 365 THREATS")
print(SEP2)
print(f"   • Spam / Phishing source IPs          : {sp_count:>5,}")
print(f"   • Email Malware detections            : {em_count:>5,}")
print(f"   • Teams / SharePoint / OneDrive files : {cm_count:>5,}")
print(f"   • BEC indicators                      : {bec_count:>5,}")
print(f"   • AiTM / MFA-bypass patterns          : {aitm_count:>5,}")
_mdo_total = sp_count + em_count + cm_count + bec_count + aitm_count
if _mdo_total > 0:
    print(f"   ⚠️  RISK : Email & collaboration threats detected")
    print(f"   📝 ACTION: Review MDO policies; enable Safe Links/Attachments; revoke session tokens")
else:
    print(f"   ✅  No MDO threats detected in period")

# ── 7. AZURE CLOUD THREAT HUNTING ────────────────────────────────────────────
print(f"\n{SEP2}")
print("☁️  AZURE CLOUD THREAT HUNTING  [Section 17]")
print(SEP2)
print(f"   • Cryptojacking / GPU abuse            : {crypto_count:>5,}")
print(f"   • Privilege escalation attempts        : {privesc_count:>5,}")
print(f"   • Azure resource mass deletion         : {mass_del_count:>5,}")
print(f"   • Resource Group deletions             : {rg_del_count:>5,}")
print(f"   • Key Vault enumeration/exfiltration   : {kv_count:>5,}")
print(f"   • Storage / SAS abuse                  : {storage_count:>5,}")
print(f"   • Suspicious automation deployments    : {auto_count:>5,}")
print(f"   • Multi-country Azure management       : {azure_travel:>5,}")
if azure_cloud_total > 0:
    print(f"   ⚠️  RISK : Cloud infrastructure threats detected")
    print(f"   📝 ACTION: Review Azure role assignments; enable Defender for Cloud; restrict SAS tokens")
else:
    print(f"   ✅  No Azure cloud threats detected")

# ── 8. MITRE ATT&CK COVERAGE ─────────────────────────────────────────────────
print(f"\n{SEP2}")
print("🗺️  MITRE ATT&CK FRAMEWORK COVERAGE  [Section 15]")
print(SEP2)
_src_label = "(SecurityIncident + SecurityAlert)" if (_inc_total + _alrt_total) > 0 else "(variable fallback — run Section 15)"
print(f"   • Source                  : {_src_label}")
print(f"   • Techniques mapped       : {_mitre_total:>5,}")
print(f"   • Techniques with events  : {_mitre_detected:>5,}  /  {_mitre_total}")
print(f"   • Coverage gaps (0 events): {_mitre_gaps:>5,}")
if not _mitre_df.empty and "Count" in _mitre_df.columns:
    _high_sig = int((_mitre_df["Count"] > 10).sum())
    print(f"   • High-signal (>10 events): {_high_sig:>5,}")
    if _mitre_gaps > 0:
        print(f"   ⚠️  {_mitre_gaps} MITRE techniques lack coverage — consider new detection rules")
    else:
        print(f"   ✅  Full MITRE coverage achieved for mapped techniques")

# ── 9. DATA SCAN ──────────────────────────────────────────────────────────────
print(f"\n{SEP2}")
print("💾  DATA SCAN COVERAGE  [Section 2]")
print(SEP2)
print(f"   • Total workspace data ingested        : {_total_gb_all:>8,.2f} GB")
print(f"   • Investigation table coverage         : {_total_gb_inv:>8,.2f} GB")
print(f"   • Tables with data                     : {_tables_scanned:>8,}")
if _total_gb_inv > 500:
    print(f"   📝 NOTE: Large volume — consider narrowing query time windows to reduce cost")
elif _total_gb_inv > 0:
    print(f"   ✅  Data coverage adequate for analysis scope")
else:
    print(f"   ⚠️  Usage data unavailable — re-run the Data Scan cell (Section 2)")

# ── 10. OVERALL RISK SCORE ────────────────────────────────────────────────────
total_findings = (bf_count + ps_count + ru_count + anomaly_count + it_count +
                  total_ioc + lm_accounts + lm_devices + onprem_total +
                  sp_count + em_count + cm_count + bec_count + aitm_count +
                  azure_cloud_total)

print(f"\n{SEP}")
print("🎯  OVERALL RISK ASSESSMENT")
print(SEP)

if total_findings > 100 or total_ioc > 20:
    risk_level = "🔴 CRITICAL"
    risk_desc  = "Multiple high-severity threats active — escalate to IR team immediately"
elif total_findings > 50 or total_ioc > 5:
    risk_level = "🟠 HIGH"
    risk_desc  = "Significant security findings requiring prompt investigation"
elif total_findings > 10:
    risk_level = "🟡 MEDIUM"
    risk_desc  = "Security findings present — schedule review and monitoring"
else:
    risk_level = "🟢 LOW"
    risk_desc  = "Security posture healthy — continue routine monitoring"

print(f"\n   Overall Risk Level : {risk_level}")
print(f"   Assessment         : {risk_desc}")
print(f"   Total Findings     : {total_findings:,}")
print(f"\n   Breakdown by Domain:")
print(f"     Credential Attacks     : {bf_count + ps_count:>6,}  (Brute Force + Spray)")
print(f"     Identity / Risky Users : {ru_count + aitm_count:>6,}  (Risky Users + AiTM)")
print(f"     Anomalies / Travel     : {anomaly_count + it_count + azure_travel:>6,}")
print(f"     Threat Intel IOCs      : {total_ioc:>6,}  (IP + File + Domain + URL + Process + Email)")
print(f"     Lateral Movement       : {lm_accounts + lm_devices + onprem_total:>6,}")
print(f"     Defender for MDO       : {_mdo_total:>6,}  (Phish + Malware + BEC + AiTM)")
print(f"     Azure Cloud Threats    : {azure_cloud_total:>6,}  (Crypto + PrivEsc + Deletion + KV + Storage)")
print(f"     MITRE Coverage Gaps    : {_mitre_gaps:>6,}  techniques with zero detections")

# ── TOP RECOMMENDATIONS ───────────────────────────────────────────────────────
print(f"\n{SEP}")
print("📋  PRIORITISED RECOMMENDATIONS")
print(SEP)

recs: list[tuple[str, str, str]] = []   # (priority_tag, finding, action)

if total_ioc > 0:
    recs.append(("🔴 P1 – CRITICAL", f"{total_ioc} TI IOC matches active",
                 "Isolate affected assets; block IOCs at firewall/proxy/EDR immediately"))
if aitm_count > 0:
    recs.append(("🔴 P1 – CRITICAL", f"{aitm_count} AiTM / MFA-bypass patterns",
                 "Revoke session tokens; enforce FIDO2/phish-resistant MFA; review CA policies"))
if bec_count > 0:
    recs.append(("🔴 P1 – CRITICAL", f"{bec_count} BEC indicators",
                 "Remove suspicious forwarding rules; audit OAuth consents; alert impacted users"))
if onprem_total > 0:
    recs.append(("🔴 P1 – CRITICAL", f"{onprem_total} on-prem identity threats (DCSync/Kerberoast/PtH)",
                 "Reset krbtgt; reset affected accounts; audit Tier-0 access immediately"))
if ru_count > 0:
    recs.append(("🟠 P2 – HIGH", f"{ru_count} users at risk (Identity Protection)",
                 "Remediate risky users; force password reset; require MFA re-registration"))
if bf_count + ps_count > 0:
    recs.append(("🟠 P2 – HIGH", f"{bf_count + ps_count} credential attack events",
                 "Enable sign-in risk CA policy; lockout after N failures; reduce attack surface"))
if privesc_count + kv_count > 0:
    recs.append(("🟠 P2 – HIGH", f"{privesc_count} PrivEsc + {kv_count} Key Vault access events",
                 "Audit role assignments; enforce PIM JIT; enable KV diagnostic logs + alerts"))
if em_count + cm_count > 0:
    recs.append(("🟡 P3 – MEDIUM", f"{em_count} email + {cm_count} collaboration malware detections",
                 "Enable Safe Attachments with Dynamic Delivery; review MDO anti-malware policies"))
if it_count + azure_travel > 0:
    recs.append(("🟡 P3 – MEDIUM", f"{it_count + azure_travel} impossible/cross-country travel events",
                 "Enable named-location CA; review sign-ins; consider travel-block policies"))
if _mitre_gaps > 10:
    recs.append(("🟡 P3 – MEDIUM", f"{_mitre_gaps} MITRE ATT&CK coverage gaps",
                 "Deploy Sentinel analytic rule templates to close gaps (see Section 15 heatmap)"))
if crypto_count > 0:
    recs.append(("🟡 P3 – MEDIUM", f"{crypto_count} cryptojacking / GPU VM anomalies",
                 "Review compute deployments; set Azure Policy to restrict GPU SKUs; alert on cost spikes"))
if mass_del_count + rg_del_count > 0:
    recs.append(("🟡 P3 – MEDIUM", f"{mass_del_count + rg_del_count} mass deletion events",
                 "Enable Azure Resource Locks on critical resources; review delete RBAC scope"))
if not recs:
    recs.append(("🟢 INFO", "No high-priority findings",
                 "Continue routine monitoring; re-run notebook monthly"))

# Always-on best practices
recs += [
    ("🔵 BASELINE", "MFA coverage",          "Enforce MFA for all users — especially admins and cloud-only accounts"),
    ("🔵 BASELINE", "MDO Safe Links/Attachments", "Confirm Safe Links and Safe Attachments are enabled for all users"),
    ("🔵 BASELINE", "Privileged access",      "Implement PIM / JIT for all Azure AD and Azure RBAC privileged roles"),
    ("🔵 BASELINE", "SOC KPIs",               "Review Section 13 MTTD trend — target sub-4h detection on High alerts"),
    ("🔵 BASELINE", "C-Level Dashboard",      "Review Section 14 risk score and compliance gap recommendations"),
]

for i, (tag, finding, action) in enumerate(recs, 1):
    print(f"\n   {i:>2}. {tag}")
    print(f"       Finding : {finding}")
    print(f"       Action  : {action}")

# ── DRILL-DOWN SECTIONS ───────────────────────────────────────────────────────
print(f"\n{SEP}")
print("🔍  AVAILABLE DRILL-DOWN SECTIONS")
print(SEP)
sections = [
    ("Section 7",  "Threat Intelligence IOC sweep  — IP / Domain / URL / File / Process / Email"),
    ("Section 8",  "Lateral movement & unusual device access"),
    ("Section 11", "Defender for Office 365  — Phishing, Malware, BEC, AiTM"),
    ("Section 12", "Forensic Investigation Hub  — INVESTIGATE_IP / _DEVICE / _UPN deep-dives"),
    ("Section 13", "SOC KPI Dashboard  — MTTD, alert volume, severity mix, true-positive rate"),
    ("Section 14", "C-Level / Board Dashboard  — Risk gauge, compliance gaps, investment recs"),
    ("Section 15", "MITRE ATT&CK Heatmap  — Coverage gaps across 22 techniques"),
    ("Section 16", "On-Prem Identity Hunting  — DCSync, Kerberoast, NTLM, Pass-the-Hash"),
    ("Section 17", "Azure Cloud Threat Hunting  — Cryptojacking, PrivEsc, KV, Storage, NSG"),
]
for sec, desc in sections:
    print(f"   📌 {sec:<12} {desc}")

print(f"\n{SEP}")
print("                         END OF EXECUTIVE SUMMARY")
print(SEP)


═══════════════════════════════════════════════════════════════════════════════
                  🛡️  MICROSOFT SENTINEL SECURITY ANALYSIS
                          EXECUTIVE SUMMARY REPORT
═══════════════════════════════════════════════════════════════════════════════

📅 Analysis Period : 2025-02-23  →  2026-02-23  (12 months)
📊 Report Generated: 2026-02-23 12:19:48
🔗 SecurityIncident events analysed : 2,827
🔗 SecurityAlert   events analysed   : 0
💾 Total workspace data ingested     : 888.00 GB  (12 tables)

───────────────────────────────────────────────────────────────────────────────
🔐  CREDENTIAL ATTACK DETECTION
───────────────────────────────────────────────────────────────────────────────
   • Brute Force events   :    100   [SigninLogs analysis]
   • Password Spray events:    100   [SigninLogs analysis]
   ⚠️  RISK : Active credential attacks detected
   📝 ACTION: Enable MFA, implement sign-in risk Conditional Access policies

──────────────────────────────────────────────────

---
## 📚 Appendix: Data Tables Reference

| Table | Description | Key Fields |
|-------|-------------|------------|
| **SigninLogs** | Interactive user sign-ins | UserPrincipalName, ResultType, IPAddress, Location |
| **AADNonInteractiveUserSignInLogs** | Non-interactive sign-ins | UserPrincipalName, ResultType, AppDisplayName |
| **AADServicePrincipalSignInLogs** | Service principal sign-ins | ServicePrincipalId, ResultType |
| **AuditLogs** | Azure AD audit events | OperationName, InitiatedBy, TargetResources |
| **AADRiskyUsers** | Users flagged as risky | UserPrincipalName, RiskLevel, RiskState |
| **AADUserRiskEvents** | Individual risk events | UserPrincipalName, RiskEventType |
| **RiskySignIns** | Sign-ins with risk | UserPrincipalName, RiskLevel, IpAddress |
| **CloudAppEvents** | Cloud app activity | AccountDisplayName, ActionType, Application |
| **DeviceLogonEvents** | Device logon events | DeviceName, AccountName, LogonType |
| **DeviceNetworkEvents** | Network connections | DeviceName, RemoteIP, RemotePort |
| **DeviceProcessEvents** | Process execution | DeviceName, FileName, ProcessCommandLine |
| **DeviceFileEvents** | File system events | DeviceName, FileName, SHA256 |
| **IdentityLogonEvents** | MDI logon events | AccountName, DeviceName, Protocol |
| **IdentityQueryEvents** | MDI query events | AccountName, QueryType |
| **SecurityAlert** | Security alerts | AlertName, AlertSeverity, ProviderName |
| **EmailEvents** | Email delivery & filtering events | SenderFromAddress, RecipientEmailAddress, ThreatTypes, EmailDirection |
| **EmailAttachmentInfo** | Email attachment metadata | FileName, SHA256, ThreatTypes, MalwareFamily |
| **EmailUrlInfo** | URLs embedded in emails | Url, UrlLocation, ThreatTypes |
| **UrlClickEvents** | Safe Links click telemetry | Url, AccountUpn, ActionType, IPAddress, IsClickedThrough |
